# Nutrelle — treino em GPU (seções 8, 10 e 11: uma única foto de cima)

Notebook gerado por `eval/treino/kaggle/monta_pacote.py`; **não edite à mão** (edite `treino_gpu.py` e gere de novo). Roda sem interação: baixa as fotos zenitais do Nutrition5k do bucket público, treina as configurações da rodada (padrão: execução 2, as 11 configurações do pipeline congelado da seção 11) até o orçamento de tempo e grava previsões (treino fora-do-fold, 450 pratos de avaliação e teste2, sem ler gabarito), registros, os arquivos para servir cada modelo e `servico.json` (latências) na pasta de saída.

Kaggle: *Settings → Accelerator → GPU T4 x2* e *Internet → On*. Colab: *Ambiente de execução → GPU*. Instruções completas em `eval/treino/kaggle/LEIA-ME.md`.

In [ ]:
# Parâmetros (os padrões servem para o Kaggle)
import json, os
ORCAMENTO_H = 11.0   # horas de relógio; o que não couber fica marcado como 'sem_orcamento' e pode ser retomado
SAIDA = '/kaggle/working' if os.path.isdir('/kaggle/working') else '/content/nutrelle_saida'   # no Colab, aponte para o Drive para retomar
EXTRA = []           # ex.: ['--rodada', '1'] (as 15 da seção 10); ['--so', 'ridge'] para rodar só parte; ['--retoma', '/content/drive/MyDrive/nutrelle_saida']
# as variáveis de ambiente NUTRELLE_SAIDA e NUTRELLE_EXTRA (lista em JSON) substituem os valores acima sem editar o notebook
SAIDA = os.environ.get('NUTRELLE_SAIDA') or SAIDA
EXTRA = json.loads(os.environ.get('NUTRELLE_EXTRA') or 'null') or EXTRA
os.makedirs('/tmp/nutrelle_codigo/listas', exist_ok=True)

In [ ]:
%%writefile /tmp/nutrelle_codigo/treino_gpu.py
#!/usr/bin/env python3
"""Treino em GPU (Kaggle/Colab) da seção 8 e nova execução para o teste2 (seção 11), de ponta a ponta e sem interação.

Premissa do produto: a inferência usa UMA foto RGB tirada de cima. Critério: proteína dentro de ±20% ou ±5 g.

Uso (Kaggle/Colab, pelo notebook nutrelle_treino_gpu.ipynb, ou direto):
  python3 treino_gpu.py                       execução 2 (padrão): as 11 configurações de RODADAS["2"], até o orçamento
  python3 treino_gpu.py --rodada 1            as 15 configurações da execução 1 (seção 10)
  python3 treino_gpu.py --lista               mostra as configurações da rodada e sai
  python3 treino_gpu.py --so "ft siglip2"     só as configurações cujo nome contém o texto
  python3 treino_gpu.py --smoke               teste rápido em CPU (backbone minúsculo, poucas dezenas de pratos)
  python3 treino_gpu.py --checa-modelos       monta cada backbone sem pesos e passa uma imagem (sem rede)
  python3 treino_gpu.py --autoteste           testes da máquina de estados (sem torch)

O que faz:
  1. Baixa do bucket público do Nutrition5k as fotos zenitais (e a profundidade, só se a destilação for rodar)
     dos pratos de listas/treino.csv, listas/aval_ids.csv e, se existir, listas/aval2_ids.csv (teste2), em paralelo,
     com novas tentativas e retomada.
  2. Para cada configuração (ordem de CONFIGS = prioridade): previsões fora-do-fold no treino (os MESMOS 5 folds por
     cadeia da CPU, coluna `fold` de listas/treino.csv) + previsões nos 450 pratos de avaliação e nos pratos do teste2
     por um modelo treinado em todo o treino, com média de 8 orientações (grupo diedral).
  3. Grava o que é preciso para servir: pesos__<config>.pt (ajuste fino, meia precisão), cabeca__<config>.npz (ridge
     sobre backbone congelado; o backbone público não é regravado: ficam a URL, o tamanho e o sha256) e
     servir__<config>.json (pré-processamento, orientações, saídas).
  4. Fase de serviço, depois de todo o treino e com a máquina livre: remonta cada modelo só com esses arquivos, confere
     que ele reproduz as previsões gravadas e mede latência (uma T4 em fp16, lotes de 1 e de 8, 1 e 8 orientações;
     CPU com o número de threads registrado). Resultado em servico.json, com os totais dos pipelines de PIPELINES.
  5. Tudo na pasta de saída, sem subpastas: pred__<config>.npz/.csv (esquema do banco zen_banco + teste2),
     log__<config>.json, parte__*.npz (retomada), manifesto.json, eventos.jsonl, tamanhos.json e treino_gpu.log.

Higiene (igual à da CPU, ver docs/dados/precisao/resultado_modelo_treinado.md, 8.1 e 11.1):
  - treino = os 1.684 pratos de listas/treino.csv (regra estrita de cadeia); os 450 pratos de avaliação e os do
    teste2 nunca entram no treino, nem como imagem sem rótulo (o script confere as interseções);
  - este script NÃO lê nenhum gabarito de avaliação (calib/teste/n5k_v1/teste2): só grava previsões;
  - sem parada antecipada: épocas fixas por receita; a validação por época só é registrada;
  - a execução 2 não muda nada no treino: mesmas receitas, sementes, folds, ordem dos lotes e lotes de extração
    das features do treino + 450 (os pratos do teste2 são extraídos em lotes à parte, depois).

Retomada: cada parte (fold0..fold4, final; tudo no ridge) vira um arquivo parte__*.npz ao terminar. Ao iniciar, o
script copia partes de --retoma (ou de /kaggle/input/**, se houver um manifesto.json deste formato) e pula o que já
existe e está completo para esta execução (uma parte final/tudo sem teste2 ou sem os arquivos de servir é refeita).
"""
import argparse, csv, glob, hashlib, json, math, os, shutil, subprocess, sys, tempfile, threading, time, traceback, zlib
from concurrent.futures import ThreadPoolExecutor
from enum import Enum
import numpy as np

FORMATO = "nutrelle-treino-gpu/2"          # /2 = /1 + teste2, arquivos de servir e servico.json
FORMATOS_ACEITOS = ("nutrelle-treino-gpu/1", FORMATO)
AQUI = os.path.dirname(os.path.abspath(__file__))
BUCKET = "https://storage.googleapis.com/nutrition5k_dataset/nutrition5k_dataset/imagery/realsense_overhead"
SPLITS = ["calib", "teste", "n5k_v1"]
EPS_D = 0.01
FAIXAS = np.array([5, 15, 25, 40])   # faixas de proteína do sorteio do v2
I_M, I_P, I_K, I_D, I_F, I_C = 0, 1, 2, 3, 4, 5
SAI_OK, SAI_SEM_ORCAMENTO, SAI_MEMORIA = 0, 3, 4

# ------------------------------------------------------------------ backbones e configurações (pré-declaradas)
_BV, _AUG, _FB = ("https://storage.googleapis.com/big_vision/siglip2/", "https://storage.googleapis.com/vit_models/augreg/",
                  "https://dl.fbaipublicfiles.com/dinov2/")
BACKBONES = {
    # chave: timm = arquitetura.etiqueta dos pesos no Hugging Face; hw = (H, W) da entrada; url = o mesmo checkpoint no
    # servidor de quem publicou; fontes = ordem em que as origens são tentadas. SigLIP 2: primeiro o .npz do big_vision,
    # que é o arquivo usado na CPU (zen_extrai.py); DINOv2: primeiro o Hugging Face.
    "siglip2_b16_224": dict(timm="vit_base_patch16_siglip_224.v2_webli", hw=(224, 224), url=_BV + "siglip2_b16_224.npz", fontes=("url", "hf")),  # foto inteira achatada, como na CPU
    "siglip2_b16_384x512": dict(timm="vit_base_patch16_siglip_512.v2_webli", hw=(384, 512), url=_BV + "siglip2_b16_512.npz", fontes=("url", "hf")),  # proporção original, pesos de 512 px
    "siglip2_l16_384x512": dict(timm="vit_large_patch16_siglip_512.v2_webli", hw=(384, 512), url=_BV + "siglip2_l16_512.npz", fontes=("url", "hf")),
    "siglip2_so400m16_384x512": dict(timm="vit_so400m_patch16_siglip_512.v2_webli", hw=(384, 512), url=_BV + "siglip2_so400m16_512.npz", fontes=("url", "hf")),
    "dinov2_b14_336x448": dict(timm="vit_base_patch14_reg4_dinov2.lvd142m", hw=(336, 448), url=_FB + "dinov2_vitb14/dinov2_vitb14_reg4_pretrain.pth", fontes=("hf", "url")),
    "dinov2_l14_336x448": dict(timm="vit_large_patch14_reg4_dinov2.lvd142m", hw=(336, 448), url=_FB + "dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth", fontes=("hf", "url")),
    "dinov2_l14_224x308": dict(timm="vit_large_patch14_reg4_dinov2.lvd142m", hw=(224, 308), url=_FB + "dinov2_vitl14/dinov2_vitl14_reg4_pretrain.pth", fontes=("hf", "url")),
    # só para --smoke
    "vit_tiny_224": dict(timm="vit_tiny_patch16_224.augreg_in21k", hw=(224, 224), url=_AUG + "Ti_16-i21k-300ep-lr_0.001-aug_none-wd_0.03-do_0.0-sd_0.0.npz", fontes=("url", "hf")),
    "vit_tiny_96x128": dict(timm="vit_tiny_patch16_224.augreg_in21k", hw=(96, 128), url=_AUG + "Ti_16-i21k-300ep-lr_0.001-aug_none-wd_0.03-do_0.0-sd_0.0.npz", fontes=("url", "hf")),
}
PASTA_PESOS = os.path.join(tempfile.gettempdir(), "nutrelle_cache", "pesos")   # redefinida a partir de --cache
# receitas de ajuste fino (lr do topo, decaimento de lr por camada, épocas); fixadas antes de qualquer resultado
RECEITAS = {
    "r1": dict(lr=3e-5, lld=0.75, epocas=15),   # receita principal
    "r2": dict(lr=6e-5, lld=0.65, epocas=25),   # passo maior no topo, base quase parada, mais épocas
    "r3": dict(lr=2e-5, lld=0.90, epocas=8),    # a receita da CPU (lr 2e-5, 8 épocas) estendida ao backbone inteiro
}
PARTES_FT = ["fold0", "fold1", "fold2", "fold3", "fold4", "final"]


def _ridge(bb, est_min):
    return dict(nome=f"gpu ridge {bb} o8", tipo="ridge", bb=bb, bs=32, est_min=est_min)


def _ft(bb, receita, est_min, tipo="ft", sufixo="", **kw):
    c = dict(nome=f"gpu {tipo} {bb} {receita}{sufixo}", tipo=tipo, bb=bb, receita=receita, bs=32, acum=1, escala=(0.7, 1.3),
             ckpt_grad=False, wd=0.05, drop_path=0.1, ema=0.99, aquec=0.1, est_min=est_min, **RECEITAS[receita])
    c.update(kw)
    return c


_ALTA = dict(bs=16, acum=2, ckpt_grad=True)     # 768 tokens: lote menor + checkpoint de gradiente (16 GB)
# Ordem = prioridade. est_min = minutos de GPU MEDIDOS na execução 1 (10/11/2026, 2 × Tesla T4) para a configuração
# inteira (2.134 pratos; com o teste2 a extração das features cresce ~39%). Só informa; não entra no treino.
CONFIGS = [
    _ridge("siglip2_b16_224", 2),                                   # paridade com a CPU: "ridge siglip2_b16 o8" (RMSE 0,2551)
    _ridge("siglip2_b16_384x512", 5),                               # 8 orientações na resolução maior (não rodado na CPU)
    _ridge("dinov2_b14_336x448", 4),                                # backbone alternativo, congelado
    _ft("siglip2_b16_224", "r1", 26),                               # ajuste fino completo a 224
    _ft("siglip2_b16_384x512", "r1", 101, epocas=12, **_ALTA),      # ajuste fino completo na resolução maior
    _ft("dinov2_b14_336x448", "r1", 99, epocas=12, **_ALTA),        # backbone alternativo, ajuste fino completo
    _ft("siglip2_b16_224", "r2", 41),
    _ft("siglip2_b16_224", "r3", 15),
    _ridge("dinov2_l14_336x448", 11),
    _ridge("siglip2_l16_384x512", 13),
    _ridge("siglip2_so400m16_384x512", 18),
    _ft("siglip2_b16_384x512", "r2", 133, epocas=16, **_ALTA),
    _ft("siglip2_b16_224", "r1", 25, sufixo=" sem-escala", escala=None),   # ablação: só orientações, sem escala aleatória
    _ft("dinov2_l14_224x308", "r1", 122, epocas=10, lld=0.85, bs=16, acum=2, ckpt_grad=True),
    _ft("siglip2_b16_224", "r1", 59, tipo="destila-rgbd"),          # professor RGB-D -> aluno RGB (opcional, por último)
]
_MASSA5 = ["gpu ridge siglip2_b16_384x512 o8", "gpu ridge dinov2_b14_336x448 o8", "gpu ridge dinov2_l14_336x448 o8",
           "gpu ridge siglip2_l16_384x512 o8", "gpu ridge siglip2_so400m16_384x512 o8"]
_FT4 = ["gpu ft siglip2_b16_224 r1", "gpu ft siglip2_b16_384x512 r1", "gpu ft dinov2_b14_336x448 r1", "gpu ft dinov2_l14_224x308 r1"]
# Execução 2 (seção 11): só o que o pipeline congelado usa + o candidato a rodar no aparelho + a configuração de paridade.
RODADAS = {"1": [c["nome"] for c in CONFIGS],
           "2": ["gpu ridge siglip2_b16_224 o8"] + _MASSA5 + _FT4 + ["gpu ft siglip2_b16_224 r2"]}
# Pipelines congelados da seção 11.2, para os totais de servico.json (o custo de cada modelo aparece uma vez).
PIPELINES = {"massa_5_backbones": list(_MASSA5), "imagem_9_modelos": _MASSA5 + _FT4}
NOTA_PIPELINES = ("imagem_9_modelos: a densidade do pipeline congelado usa ainda três cabeças de frações treinadas na CPU "
                  "(soft-mlp) sobre features congeladas do SigLIP 2 B/16 a 384×512 (2 orientações) e, numa delas ('ambos'), "
                  "também a 224 px. O custo do backbone a 384×512 já está contado em 'gpu ridge siglip2_b16_384x512 o8' se as "
                  "features forem compartilhadas; o do backbone a 224 px é o de 'gpu ridge siglip2_b16_224 o8' (fora dos totais). "
                  "As cabeças em si são MLPs pequenas: custo desprezível, não medido aqui.")
CONFIGS_SMOKE = [
    dict(nome="smoke ridge vit_tiny_224 o8", tipo="ridge", bb="vit_tiny_224", bs=8, est_min=1),
    _ft("vit_tiny_224", "r1", 1, epocas=1, bs=8),
    _ft("vit_tiny_96x128", "r1", 1, epocas=1, bs=4, acum=2, ckpt_grad=True),
    _ft("vit_tiny_224", "r1", 1, tipo="destila-rgbd", epocas=1, bs=8),
]
for _c in CONFIGS_SMOKE:
    _c["nome"] = _c["nome"].replace("gpu ", "smoke ")


def slug(nome):
    return "".join(ch if ch.isalnum() else "_" for ch in nome)


def partes_de(cfg):
    return ["tudo"] if cfg["tipo"] == "ridge" else list(PARTES_FT)


def configs_da_rodada(rodada, smoke=False):
    if smoke:
        return list(CONFIGS_SMOKE)
    return [c for c in CONFIGS if c["nome"] in RODADAS[rodada]]     # a ordem de CONFIGS (prioridade) é mantida


# ------------------------------------------------------------------ máquina de estados de cada parte (config × fold)
class Estado(str, Enum):
    PENDENTE = "pendente"            # na fila (tentativa e divisor de lote guardados para a próxima execução)
    RODANDO = "rodando"              # um processo de treino está com ela
    FEITA = "feita"                  # parte__*.npz gravado (nesta sessão ou retomado)
    FALHOU = "falhou"                # erro repetido: a configuração inteira é abandonada
    SEM_ORCAMENTO = "sem_orcamento"  # não cabia no tempo que restava
    CANCELADA = "cancelada"          # outra parte da mesma configuração falhou ou ficou sem orçamento


TERMINAIS = {Estado.FEITA, Estado.FALHOU, Estado.SEM_ORCAMENTO, Estado.CANCELADA}
MAX_TENTATIVAS, MAX_DIV_LOTE = 2, 4


def parte_inicial():
    return dict(estado=Estado.PENDENTE.value, tentativa=0, div_lote=1, seg=None, erro=None, dispositivo=None, retomada=False)


def transicao(p, ev):
    """Função pura (estado da parte, evento) -> novo estado. Eventos: retomou | iniciou | concluiu | estourou_memoria |
    reduziu_lote | falhou | sem_orcamento | cancelou. Transição não prevista levanta ValueError (estado impossível não é gravado)."""
    e, t = Estado(p["estado"]), ev["tipo"]
    n = dict(p)
    if e is Estado.PENDENTE and t == "retomou":
        n.update(estado=Estado.FEITA.value, retomada=True, seg=ev.get("seg"))
    elif e is Estado.PENDENTE and t == "iniciou":
        n.update(estado=Estado.RODANDO.value, tentativa=p["tentativa"] + 1, dispositivo=ev["dispositivo"])
    elif e is Estado.PENDENTE and t == "sem_orcamento":
        n.update(estado=Estado.SEM_ORCAMENTO.value)
    elif e is Estado.PENDENTE and t == "reduziu_lote":       # outra parte da configuração estourou a memória
        n.update(div_lote=max(p["div_lote"], ev["div_lote"]))
    elif e is Estado.PENDENTE and t == "cancelou":
        n.update(estado=Estado.CANCELADA.value, erro=ev.get("motivo"))
    elif e is Estado.RODANDO and t == "concluiu":
        n.update(estado=Estado.FEITA.value, seg=ev["seg"], erro=None)
    elif e is Estado.RODANDO and t == "sem_orcamento":
        n.update(estado=Estado.SEM_ORCAMENTO.value)
    elif e is Estado.RODANDO and t == "estourou_memoria":
        if p["div_lote"] < MAX_DIV_LOTE:      # tenta de novo com metade do lote (e o dobro de acumulação)
            n.update(estado=Estado.PENDENTE.value, div_lote=p["div_lote"] * 2, tentativa=0, erro="memória")
        else:
            n.update(estado=Estado.FALHOU.value, erro="memória insuficiente mesmo com lote ÷ %d" % MAX_DIV_LOTE)
    elif e is Estado.RODANDO and t == "falhou":
        n.update(estado=(Estado.PENDENTE if p["tentativa"] < MAX_TENTATIVAS else Estado.FALHOU).value, erro=ev.get("erro"))
    else:
        raise ValueError(f"transição inválida: {e.value} + {t}")
    return n


def estado_config(partes):
    """Estado da configuração, derivado das partes: completa | falhou | sem_orcamento | em_andamento | pendente."""
    es = {Estado(p["estado"]) for p in partes.values()}
    if es == {Estado.FEITA}:
        return "completa"
    if Estado.FALHOU in es:
        return "falhou"
    if Estado.SEM_ORCAMENTO in es:
        return "sem_orcamento"
    if es == {Estado.PENDENTE}:
        return "pendente"
    return "em_andamento"


def autoteste():
    p = parte_inicial()
    assert Estado(transicao(p, dict(tipo="retomou"))["estado"]) is Estado.FEITA
    r = transicao(p, dict(tipo="iniciou", dispositivo="cuda:0"))
    assert r["estado"] == "rodando" and r["tentativa"] == 1 and p["tentativa"] == 0           # pura: não altera a entrada
    assert transicao(r, dict(tipo="concluiu", seg=10.0)) == dict(r, estado="feita", seg=10.0)
    m = transicao(r, dict(tipo="estourou_memoria"))
    assert m["estado"] == "pendente" and m["div_lote"] == 2 and m["tentativa"] == 0
    m4 = dict(r, div_lote=MAX_DIV_LOTE)
    assert transicao(m4, dict(tipo="estourou_memoria"))["estado"] == "falhou"
    f1 = transicao(r, dict(tipo="falhou", erro="x"))
    assert f1["estado"] == "pendente" and f1["erro"] == "x"                                   # 1ª falha: tenta de novo
    f2 = transicao(transicao(f1, dict(tipo="iniciou", dispositivo="cpu")), dict(tipo="falhou", erro="y"))
    assert f2["estado"] == "falhou"                                                           # 2ª falha: desiste
    assert transicao(r, dict(tipo="sem_orcamento"))["estado"] == "sem_orcamento"
    assert transicao(p, dict(tipo="cancelou", motivo="z"))["estado"] == "cancelada"
    assert transicao(p, dict(tipo="reduziu_lote", div_lote=2)) == dict(p, div_lote=2)
    for terminal in TERMINAIS:                                                                # terminais não saem do lugar
        for t in ("retomou", "iniciou", "concluiu", "estourou_memoria", "reduziu_lote", "falhou", "sem_orcamento", "cancelou"):
            try:
                transicao(dict(p, estado=terminal.value), dict(tipo=t, dispositivo="x", seg=1, div_lote=2))
                raise AssertionError((terminal, t))
            except ValueError:
                pass
    for t in ("concluiu", "estourou_memoria", "falhou"):                                      # só quem está rodando termina
        try:
            transicao(p, dict(tipo=t, seg=1)); raise AssertionError(t)
        except ValueError:
            pass
    f = lambda *es: estado_config({i: dict(estado=e) for i, e in enumerate(es)})
    assert f("feita", "feita") == "completa" and f("feita", "falhou", "cancelada") == "falhou"
    assert f("feita", "sem_orcamento") == "sem_orcamento" and f("pendente", "pendente") == "pendente"
    assert f("feita", "pendente") == "em_andamento" and f("rodando", "pendente") == "em_andamento"
    assert json.loads(json.dumps(m)) == m                                                     # serializável
    assert len({slug(c["nome"]) for c in CONFIGS}) == len(CONFIGS) and all(c["bb"] in BACKBONES for c in CONFIGS + CONFIGS_SMOKE)
    nomes = {c["nome"] for c in CONFIGS}
    assert all(set(v) <= nomes and len(set(v)) == len(v) for v in list(RODADAS.values()) + list(PIPELINES.values()))
    assert len(RODADAS["2"]) == 11 and all(set(v) <= set(RODADAS["2"]) for v in PIPELINES.values())
    assert [c["nome"] for c in configs_da_rodada("2")][0] == "gpu ridge siglip2_b16_224 o8"      # a paridade roda primeiro
    print("autoteste: ok")


# ------------------------------------------------------------------ dados (listas do repositório; sem gabarito de avaliação)
class Dados:
    """Treino zenital estrito (ids, folds por cadeia, alvos em log) + ids de avaliação (sem rótulo):
    aval = calib/teste/n5k_v1 (aval_ids.csv); conj = aval + os conjuntos de aval2_ids.csv (teste2), se o arquivo existir."""

    def __init__(self, pasta, smoke=0):
        arq_tr, arq_av = os.path.join(pasta, "treino.csv"), os.path.join(pasta, "aval_ids.csv")
        with open(arq_tr) as f:
            rows = list(csv.DictReader(f))
        self.grupos = [c[2:] for c in rows[0] if c.startswith("m_")]
        G = len(self.grupos)
        self.N = 6 + 2 * G
        self.I_MG, self.I_PG = slice(6, 6 + G), slice(6 + G, 6 + 2 * G)
        self.ids = [r["dish_id"] for r in rows]
        self.fold = np.array([int(r["fold"]) for r in rows])
        col = lambda c: np.array([float(r[c]) for r in rows])
        m, p = col("massa"), col("proteina")
        mg = np.array([[float(r[f"m_{g}"]) for g in self.grupos] for r in rows])
        pg = np.array([[float(r[f"p_{g}"]) for g in self.grupos] for r in rows])
        self.massa, self.proteina = m, p
        # mesmos alvos de zen_comum.alvos: log m | log1p p | log1p kcal | log(p/m + eps) | log1p gordura | log1p carbo | grupos
        self.T = np.c_[np.log(m), np.log1p(p), np.log1p(col("kcal")), np.log(p / m + EPS_D), np.log1p(col("gordura")),
                       np.log1p(col("carbo")), np.log1p(mg), np.log1p(pg)]
        fx = np.digitize(p, FAIXAS)
        self.w_cv = (1 / np.bincount(fx, minlength=5))[fx]; self.w_cv /= self.w_cv.sum()   # 5 faixas com o mesmo peso
        with open(arq_av) as f:
            av = list(csv.DictReader(f))
        self.aval = {s: [r["dish_id"] for r in av if r["split"] == s] for s in SPLITS}
        ev = [d for s in SPLITS for d in self.aval[s]]
        assert len(set(self.ids)) == len(self.ids) and len(set(ev)) == len(ev), "ids repetidos nas listas"
        assert not set(self.ids) & set(ev), "prato de avaliação na lista de treino"
        self.hash = hashlib.sha256(open(arq_tr, "rb").read() + open(arq_av, "rb").read()).hexdigest()[:16]
        # conjuntos novos, só para previsão (teste2): não mudam o hash das listas de treino/avaliação da execução 1
        self.conj, self.hash2 = dict(self.aval), None
        arq_av2 = os.path.join(pasta, "aval2_ids.csv")
        if os.path.exists(arq_av2):
            with open(arq_av2) as f:
                av2 = list(csv.DictReader(f))
            for s in dict.fromkeys(r["split"] for r in av2):
                assert s not in self.conj and s != "oof", f"conjunto repetido: {s}"
                self.conj[s] = [r["dish_id"] for r in av2 if r["split"] == s]
            ev2 = [r["dish_id"] for r in av2]
            assert len(set(ev2)) == len(ev2), "ids repetidos em aval2_ids.csv"
            assert not set(ev2) & set(self.ids), "prato do teste2 na lista de treino"
            assert not set(ev2) & set(ev), "prato do teste2 já está na avaliação antiga"
            # hash do CONTEÚDO (o arquivo do repositório tem fim de linha CRLF e o notebook o regrava com LF)
            self.hash2 = hashlib.sha256("\n".join(f"{r['prato_id']},{r['split']},{r['dish_id']}" for r in av2).encode()).hexdigest()[:16]
        self.extras = [s for s in self.conj if s not in SPLITS]
        # --smoke: poucos pratos por fold e por conjunto; o resto fica NaN nas saídas
        self.usa_tr = np.ones(len(self.ids), bool)
        self.usa_av = {s: np.ones(len(v), bool) for s, v in self.conj.items()}
        if smoke:
            self.usa_tr[:] = False
            for k in range(5):
                self.usa_tr[np.where(self.fold == k)[0][:smoke]] = True
            for s in self.conj:
                self.usa_av[s][:] = False; self.usa_av[s][:4] = True
        usados = lambda nomes: [d for s in nomes for d, u in zip(self.conj[s], self.usa_av[s]) if u]
        base = [d for d, u in zip(self.ids, self.usa_tr) if u] + usados(SPLITS)
        self.n_base = len(base)                     # treino + 450: mesma ordem (e mesmos lotes de extração) da execução 1
        self.todos = base + usados(self.extras)
        self.chave = hashlib.sha256("\n".join(self.todos).encode()).hexdigest()[:10]


def proteinas(T):
    massa = np.exp(T[:, I_M]); dens = np.clip(np.exp(T[:, I_D]) - EPS_D, 0, None)
    direto = np.expm1(T[:, I_P]).clip(0)
    return dict(direto=direto, mxd=massa * dens, geo=np.sqrt(direto * massa * dens), massa=massa)


def acerto(real, est):
    err = np.abs(est - real)
    return (err <= 5) | (err <= 0.20 * np.maximum(real, 1e-9))


def resumo_cv(D, oof):
    """Métricas de CV no TREINO (as mesmas de zen_avalia): RMSE de log-massa e critério ponderado por faixa."""
    v = ~np.isnan(oof[:, 0])
    if not v.any():
        return dict(n=0)
    P = proteinas(oof[v]); w = D.w_cv[v]
    cr = lambda est: float(100 * (acerto(D.proteina[v], est) * w).sum() / w.sum())
    return dict(n=int(v.sum()), rmse_logm=float(np.sqrt(np.mean((oof[v, I_M] - D.T[v, I_M]) ** 2))),
                mape_massa=float(100 * np.mean(np.abs(P["massa"] - D.massa[v]) / D.massa[v])),
                crit_direto=cr(P["direto"]), crit_mxd=cr(P["mxd"]), crit_geo=cr(P["geo"]))


# ------------------------------------------------------------------ ridge (cópia de zen_comum: mesmo cálculo da CPU)
class Ridge:
    ALFAS = np.logspace(-1, 5, 19)

    def _decomp(self, X, T):
        mu, sd = X.mean(0), X.std(0) + 1e-6
        Z = (X - mu) / sd
        mt = T.mean(0)
        s2, V = np.linalg.eigh((Z.T @ Z).astype(np.float64))
        return mu, sd, mt, s2.clip(0), V, V.T @ (Z.T @ (T - mt))

    def caminho(self, X, T, Xv):
        mu, sd, mt, s2, V, VtZtT = self._decomp(X, T)
        ZvV = ((Xv - mu) / sd) @ V
        return np.stack([ZvV @ (VtZtT / (s2 + a)[:, None]) + mt for a in self.ALFAS])

    def fit(self, X, T, alfa):
        mu, sd, mt, s2, V, VtZtT = self._decomp(X, T)
        self.mu, self.sd, self.mt = mu, sd, mt
        self.W = np.stack([V @ (VtZtT[:, j] / (s2 + a)) for j, a in enumerate(alfa)], 1)
        return self

    def predict(self, X):
        return ((X - self.mu) / self.sd) @ self.W + self.mt


def ridge_cv(X, T, fold):
    """X[n, orientações, D]: cada orientação é uma amostra de treino; previsão = média nas orientações.
    Devolve (previsões fora-do-fold, alfa por alvo, MSE de CV por alvo)."""
    n, o, Dm = X.shape
    r = Ridge()
    P = np.zeros((len(r.ALFAS), n, T.shape[1]))
    for k in np.unique(fold):
        tr, va = fold != k, fold == k
        cam = r.caminho(X[tr].reshape(-1, Dm), np.repeat(T[tr], o, 0), X[va].reshape(-1, Dm))
        P[:, va] = cam.reshape(len(r.ALFAS), va.sum(), o, -1).mean(2)
    mse = ((P - T) ** 2).mean(1)
    best = mse.argmin(0)
    return np.stack([P[best[j], :, j] for j in range(T.shape[1])], 1), r.ALFAS[best], mse.min(0)


def ridge_final(X, T, alfa):
    n, o, Dm = X.shape
    m = Ridge().fit(X.reshape(-1, Dm), np.repeat(T, o, 0), alfa)
    return (lambda Xn: m.predict(Xn.reshape(-1, Dm)).reshape(Xn.shape[0], Xn.shape[1], -1).mean(1)), m


# ------------------------------------------------------------------ download e cache das imagens
def _png_ok(dados):
    return len(dados) > 64 and dados[:8] == b"\x89PNG\r\n\x1a\n" and dados[-8:-4] == b"IEND"


def baixa_um(dish, arquivo, raiz, tentativas=6):
    """Baixa <BUCKET>/<dish>/<arquivo> para raiz/img/<dish>/<arquivo>. Retomada: arquivo PNG íntegro é mantido."""
    import urllib.error, urllib.request
    dest = os.path.join(raiz, "img", dish, arquivo)
    if os.path.exists(dest):
        with open(dest, "rb") as f:
            if _png_ok(f.read()):
                return "tinha"
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    erro = None
    for t in range(tentativas):
        try:
            with urllib.request.urlopen(f"{BUCKET}/{dish}/{arquivo}", timeout=60) as r:
                dados = r.read()
                tam = r.headers.get("Content-Length")
            if (tam and int(tam) != len(dados)) or not _png_ok(dados):
                raise IOError(f"arquivo truncado ({len(dados)} bytes)")
            tmp = f"{dest}.parte{os.getpid()}"
            with open(tmp, "wb") as f:
                f.write(dados)
            os.replace(tmp, dest)
            return "baixou"
        except urllib.error.HTTPError as e:
            erro = f"HTTP {e.code}"
            if e.code in (403, 404):     # não existe no bucket: tentar de novo não adianta
                break
        except Exception as e:           # rede: espera e tenta de novo
            erro = f"{type(e).__name__}: {e}"
        time.sleep(min(30, 2 ** t) * (0.5 + np.random.rand()))
    return f"erro: {erro}"


class Trava:
    """Trava entre processos por arquivo: dois processos não baixam os mesmos pesos nem extraem as mesmas features."""

    def __init__(self, arq):
        self.arq = arq

    def __enter__(self):
        while True:
            try:
                fd = os.open(self.arq, os.O_CREAT | os.O_EXCL | os.O_WRONLY)
                os.write(fd, str(os.getpid()).encode()); os.close(fd)
                return self
            except FileExistsError:
                try:
                    os.kill(int(open(self.arq).read() or 0), 0)     # o dono ainda existe?
                except (ValueError, ProcessLookupError, FileNotFoundError):
                    try:
                        os.remove(self.arq)                          # trava órfã
                    except FileNotFoundError:
                        pass
                    continue
                except PermissionError:
                    pass
                time.sleep(2)

    def __exit__(self, *exc):
        try:
            os.remove(self.arq)
        except FileNotFoundError:
            pass


def baixa_arquivo(url, dest, tentativas=8):
    """Download de arquivo grande (pesos) com novas tentativas e retomada por Range. Só existe em `dest` se completo."""
    import urllib.error, urllib.request
    if os.path.exists(dest):
        return dest
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    with Trava(dest + ".trava"):
        parte, erro = dest + ".parte", None
        for t in range(tentativas):
            if os.path.exists(dest):
                return dest
            try:
                ja = os.path.getsize(parte) if os.path.exists(parte) else 0
                req = urllib.request.Request(url, headers={"Range": f"bytes={ja}-"} if ja else {})
                with urllib.request.urlopen(req, timeout=120) as r:
                    if ja and r.status != 206:      # servidor ignorou o Range: recomeça
                        ja = 0
                    total = ja + int(r.headers.get("Content-Length") or 0)
                    with open(parte, "ab" if ja else "wb") as f:
                        shutil.copyfileobj(r, f, 1 << 22)
                if total and os.path.getsize(parte) != total:
                    raise IOError(f"incompleto: {os.path.getsize(parte)} de {total} bytes")
                os.replace(parte, dest)
                print(f"pesos: {os.path.basename(dest)} ({os.path.getsize(dest) / 1e6:.0f} MB) de {url}", flush=True)
                return dest
            except urllib.error.HTTPError as e:
                erro = f"HTTP {e.code}"
                if e.code == 416 and os.path.exists(parte):    # Range além do fim: o parcial não serve
                    os.remove(parte)
                elif e.code in (401, 403, 404):
                    break
            except Exception as e:
                erro = f"{type(e).__name__}: {e}"
            time.sleep(min(60, 2 ** t) * (0.5 + np.random.rand()))
    raise IOError(f"não foi possível baixar {url}: {erro}")


def baixa(dishes, arquivo, raiz, threads=16):
    t0 = time.time()
    with ThreadPoolExecutor(threads) as ex:
        res = list(ex.map(lambda d: baixa_um(d, arquivo, raiz), dishes))
    falhas = [(d, r) for d, r in zip(dishes, res) if r.startswith("erro")]
    diz(f"download {arquivo}: {res.count('baixou')} baixados, {res.count('tinha')} já no disco, {len(falhas)} falhas ({time.time() - t0:.0f}s)")
    if falhas:
        raise SystemExit(f"Não foi possível baixar {len(falhas)} arquivos (ex.: {falhas[:3]}). Sem todos os pratos a higiene "
                         "do treino/avaliação não é a mesma da CPU; rode de novo (o download é retomado).")


class Cache:
    """Imagens decodificadas em disco (memmap), na ordem de Dados.todos: rgb uint8 [n, 480, 640, 3], cor da mesa
    [n, 3] (mediana da moldura, para o aumento de escala) e, se pedido, altura acima da mesa em cm [n, 240, 320]."""
    H, W = 480, 640

    def __init__(self, raiz, D):
        self.raiz, self.D = raiz, D
        self.pasta = os.path.join(raiz, f"cache_{D.chave}")
        self.pos = {d: i for i, d in enumerate(D.todos)}
        self.rgb = self.cor = self.alt = None

    def _arq(self, nome):
        return os.path.join(self.pasta, nome)

    def prepara(self, profundidade=False):
        os.makedirs(self.pasta, exist_ok=True)
        n = len(self.D.todos)
        if not os.path.exists(self._arq("rgb.npy")):
            baixa(self.D.todos, "rgb.png", self.raiz)
            from PIL import Image
            t0 = time.time()
            mm = np.lib.format.open_memmap(self._arq("rgb.tmp.npy"), mode="w+", dtype=np.uint8, shape=(n, self.H, self.W, 3))
            cor = np.zeros((n, 3), np.float32)

            def um(i):
                im = Image.open(os.path.join(self.raiz, "img", self.D.todos[i], "rgb.png")).convert("RGB")
                fora = im.size != (self.W, self.H)
                if fora:
                    im = im.resize((self.W, self.H), Image.BICUBIC)
                a = np.asarray(im); mm[i] = a
                k = max(2, int(0.03 * min(a.shape[:2])))   # moldura externa de 3%, como zen_extrai._borda
                cor[i] = np.median(np.concatenate([a[:k].reshape(-1, 3), a[-k:].reshape(-1, 3), a[:, :k].reshape(-1, 3), a[:, -k:].reshape(-1, 3)]), 0) / 255
                return fora
            with ThreadPoolExecutor(os.cpu_count() or 2) as ex:
                fora = sum(ex.map(um, range(n)))
            mm.flush(); del mm
            np.save(self._arq("cor.npy"), cor)
            os.replace(self._arq("rgb.tmp.npy"), self._arq("rgb.npy"))
            diz(f"cache RGB: {n} fotos decodificadas ({fora} fora de 640×480, redimensionadas) em {time.time() - t0:.0f}s")
        if profundidade and not os.path.exists(self._arq("alt.npy")):
            baixa(self.D.todos, "depth_raw.png", self.raiz)
            from PIL import Image
            t0 = time.time()
            alt = np.zeros((n, self.H // 2, self.W // 2), np.float16)

            def um(i):
                d = np.asarray(Image.open(os.path.join(self.raiz, "img", self.D.todos[i], "depth_raw.png"))).astype(np.float32)
                if d.shape != (self.H, self.W):
                    return 1
                val = d > 0
                if val.sum() < 1000:
                    return 1
                if np.median(d[val]) < 1000:      # primeiros pratos vêm em mm; o resto em 0,1 mm
                    d = d * 10
                k = int(0.08 * self.H)
                borda = np.concatenate([d[:k].ravel(), d[-k:].ravel(), d[:, :k].ravel(), d[:, -k:].ravel()])
                mesa = np.median(borda[borda > 0]) if (borda > 0).any() else np.median(d[val])
                a = np.where(val, (mesa - d) / 100.0, 0.0).clip(-2, 15)     # cm acima da mesa; sem leitura = 0
                alt[i] = a.reshape(self.H // 2, 2, self.W // 2, 2).mean((1, 3))
                return 0
            with ThreadPoolExecutor(os.cpu_count() or 2) as ex:
                ruins = sum(ex.map(um, range(n)))
            np.save(self._arq("alt.tmp.npy"), alt); os.replace(self._arq("alt.tmp.npy"), self._arq("alt.npy"))
            diz(f"cache de profundidade: {n} mapas ({ruins} sem leitura útil, ficam em zero) em {time.time() - t0:.0f}s")
        return self

    def abre(self, profundidade=False):
        self.rgb = np.load(self._arq("rgb.npy"), mmap_mode="r")
        self.cor = np.load(self._arq("cor.npy"))
        if profundidade:
            self.alt = np.load(self._arq("alt.npy"), mmap_mode="r")
        assert len(self.rgb) == len(self.D.todos)
        return self


# ------------------------------------------------------------------ modelo e pré-processamento (torch só aqui para baixo)
def orienta(x, k):
    """k-ésimo elemento do grupo diedral aplicado a [..., H, W]; mesma ordem de zen_extrai.orienta
    (original, 180°, espelho H, espelho V, 90°, 270°, transposta, antitransposta)."""
    if k == 0: return x
    if k == 1: return x.flip(-2, -1)
    if k == 2: return x.flip(-1)
    if k == 3: return x.flip(-2)
    t = x.transpose(-2, -1)
    return [t.flip(-2), t.flip(-1), t, t.flip(-2, -1)][k - 4]


def orienta_lote(x, ks):
    """Orientação por amostra. Entrada não quadrada: todas as amostras do lote no mesmo formato (ks < 4 ou ks >= 4)."""
    ks = np.asarray(ks)
    if (ks == ks[0]).all():
        return orienta(x, int(ks[0]))
    assert x.shape[-1] == x.shape[-2] or (ks < 4).all() or (ks >= 4).all()
    forma = x.shape if ks[0] < 4 else (*x.shape[:-2], x.shape[-1], x.shape[-2])
    import torch
    out = x.new_empty(forma)
    for k in np.unique(ks):
        m = torch.from_numpy(np.where(ks == k)[0]).to(x.device)
        out[m] = orienta(x[m], int(k))
    return out


def redimensiona(x, hw, zoom=None, cor=None):
    """x [B, C, H, W] em [0, 1] -> [B, C, h, w]. zoom por amostra, com a semântica de zen_extrai.zoom: z > 1 aproxima
    (recorte central de 1/z); z < 1 afasta (mais 'mesa' em volta, na cor mediana da moldura da foto)."""
    import torch
    F = torch.nn.functional
    if zoom is None:
        return F.interpolate(x, size=hw, mode="bicubic", antialias=True, align_corners=False)
    B, Cn, H, W = x.shape
    out = []
    for i in range(B):
        z, xi = float(zoom[i]), x[i:i + 1]
        if z > 1:
            ch, cw = H / z, W / z
            xi = xi[:, :, round((H - ch) / 2):round((H + ch) / 2), round((W - cw) / 2):round((W + cw) / 2)]
        elif z < 1:
            ph, pw = int(round((H / z - H) / 2)), int(round((W / z - W) / 2))
            tela = cor[i].view(1, Cn, 1, 1).expand(1, Cn, H + 2 * ph, W + 2 * pw).clone()
            tela[:, :, ph:ph + H, pw:pw + W] = xi
            xi = tela
        out.append(F.interpolate(xi, size=hw, mode="bicubic", antialias=True, align_corners=False))
    return torch.cat(out)


def monta_lote(C, linhas, dev, hw, canais=3, zoom=None):
    """Lê as fotos do cache e devolve [B, canais, h, w]: RGB em [0, 1] (+ altura em cm / 5 no 4º canal)."""
    import torch
    linhas = np.asarray(linhas)
    x = torch.from_numpy(np.ascontiguousarray(C.rgb[linhas])).to(dev).permute(0, 3, 1, 2).float().div_(255)
    cor = torch.from_numpy(C.cor[linhas]).to(dev)
    if canais == 4:
        a = torch.from_numpy(np.stack([C.alt[i] for i in linhas]).astype(np.float32)).to(dev)[:, None] / 5
        x = torch.cat([x, torch.nn.functional.interpolate(a, size=x.shape[-2:], mode="bilinear", align_corners=False)], 1)
        cor = torch.cat([cor, torch.zeros(len(linhas), 1, device=dev)], 1)
    x = redimensiona(x, hw, zoom, cor)
    x[:, :3].clamp_(0, 1)
    return x


def cria_rede(bb, n_saidas, pretrained=True, drop_path=0.0, ckpt_grad=False, canais=3):
    """Backbone do timm + cabeça linear sobre [agregado ‖ média dos tokens] (os blocos 'agregado' e 'média' da CPU)."""
    import timm, torch

    class Rede(torch.nn.Module):
        def __init__(self):
            super().__init__()
            b = BACKBONES[bb]
            arq_timm = b["timm"].split(".")[0]
            kw = dict(num_classes=0, img_size=b["hw"], dynamic_img_size=b["hw"][0] != b["hw"][1], drop_path_rate=drop_path)
            erros = []
            for fonte in (b["fontes"] if pretrained else ()):
                try:
                    if fonte == "url":   # checkpoint original num arquivo local; o timm converte (.npz) ou filtra (.pth)
                        arq = baixa_arquivo(b["url"], os.path.join(PASTA_PESOS, os.path.basename(b["url"])))
                        self.m = timm.create_model(arq_timm, pretrained=True, pretrained_cfg_overlay=dict(file=arq, custom_load=arq.endswith(".npz")), **kw)
                    else:
                        self.m = timm.create_model(b["timm"], pretrained=True, **kw)
                    _anota_origem(bb, fonte)
                    break
                except Exception as e:
                    erros.append(f"{fonte}: {type(e).__name__}: {str(e)[:300]}")
                    print(f"pesos de {bb} pela origem '{fonte}' falharam ({erros[-1]})", flush=True)
            else:
                if pretrained:
                    raise RuntimeError(f"sem pesos para {bb}: {erros}")
                self.m = timm.create_model(arq_timm, pretrained=False, **kw)
            if ckpt_grad:
                self.m.set_grad_checkpointing(True)
            pc = self.m.pretrained_cfg
            media, desvio = list(pc.get("mean", (0.5,) * 3)), list(pc.get("std", (0.5,) * 3))
            if canais == 4:   # 4º canal (altura) entra com peso zero: o professor parte exatamente do modelo RGB
                p = self.m.patch_embed.proj
                novo = torch.nn.Conv2d(4, p.out_channels, p.kernel_size, p.stride, p.padding, bias=p.bias is not None)
                with torch.no_grad():
                    novo.weight.zero_(); novo.weight[:, :3] = p.weight
                    if p.bias is not None:
                        novo.bias.copy_(p.bias)
                self.m.patch_embed.proj = novo
                media, desvio = media + [0.0], desvio + [1.0]
            self.register_buffer("media", torch.tensor(media).view(1, -1, 1, 1))
            self.register_buffer("desvio", torch.tensor(desvio).view(1, -1, 1, 1))
            self.dim = 2 * self.m.num_features
            self.head = torch.nn.Linear(self.dim, n_saidas)

        def feats(self, x):
            t = self.m.forward_features((x - self.media) / self.desvio)
            return torch.cat([self.m.forward_head(t, pre_logits=True), t[:, self.m.num_prefix_tokens:].mean(1)], 1)

        def forward(self, x):
            f = self.feats(x)
            return self.head(f), f

    return Rede()


def _anota_origem(bb, fonte):
    """Registra de qual origem os pesos públicos do backbone foram de fato carregados (para servir__*.json)."""
    try:
        os.makedirs(PASTA_PESOS, exist_ok=True)
        with open(os.path.join(PASTA_PESOS, f"origem__{bb}.txt"), "w") as f:
            f.write(fonte)
    except OSError:
        pass


def sha256_arquivo(arq):
    h = hashlib.sha256()
    with open(arq, "rb") as f:
        for bloco in iter(lambda: f.read(1 << 24), b""):
            h.update(bloco)
    return h.hexdigest()


def descreve_backbone(bb):
    """Checkpoint público do backbone: origem usada, URL ou repositório, arquivo, tamanho e sha256. É o que permite
    servir sem regravar o backbone. Chamar antes de apagar o arquivo do cache."""
    b = BACKBONES[bb]
    try:
        fonte = open(os.path.join(PASTA_PESOS, f"origem__{bb}.txt")).read().strip()
    except OSError:
        fonte = b["fontes"][0]
    o = dict(chave=bb, timm=b["timm"], fonte=fonte, url=b.get("url"), hf_id=None, arquivo=None, bytes=None, sha256=None)
    arq = None
    try:
        import timm
        o["timm_versao"] = timm.__version__
        o["hf_id"] = timm.models.get_pretrained_cfg(b["timm"]).hf_hub_id
    except Exception:
        pass
    if fonte == "url":
        arq = os.path.join(PASTA_PESOS, os.path.basename(b["url"]))
    elif o["hf_id"]:
        try:
            from huggingface_hub import try_to_load_from_cache
            for nome in ("model.safetensors", "open_clip_model.safetensors", "pytorch_model.bin", "open_clip_pytorch_model.bin"):
                cam = try_to_load_from_cache(o["hf_id"], nome)
                if isinstance(cam, str) and os.path.exists(cam):
                    arq = cam
                    break
        except Exception:
            pass
    if arq and os.path.exists(arq):
        o.update(arquivo=os.path.basename(arq), bytes=os.path.getsize(arq), sha256=sha256_arquivo(arq))
    return o


def grupos_lr(rede, lr, lld, wd):
    """Decaimento de lr por camada: a cabeça e o topo usam lr; cada bloco abaixo multiplica por lld."""
    L = len(rede.m.blocks)
    grupos = {}
    for n, p in rede.named_parameters():
        if not p.requires_grad:
            continue
        if n.startswith("m.blocks."):
            cam = int(n.split(".")[2]) + 1
        elif n.startswith(("m.patch_embed", "m.pos_embed", "m.cls_token", "m.reg_token")):
            cam = 0
        else:                      # cabeça, norma final, agregação por atenção
            cam = L + 1
        sem_wd = p.ndim <= 1 or n.endswith(("pos_embed", "cls_token", "reg_token", "latent"))
        grupos.setdefault((cam, sem_wd), []).append(p)
    return [dict(params=ps, lr=lr * lld ** (L + 1 - cam), weight_decay=0.0 if sem_wd else wd) for (cam, sem_wd), ps in sorted(grupos.items())]


class SemOrcamento(Exception):
    pass


def autocast(dev):
    import torch
    return torch.autocast(device_type=dev.type, dtype=torch.float16, enabled=dev.type == "cuda")


def feats_congeladas(bb, D, C, dev, bs, prazo):
    """Features do backbone congelado para todos os pratos, 8 orientações: [n_todos, 8, 2·dim] float16 (cache em disco).
    É a entrada do ridge e do ponto de partida da cabeça no ajuste fino."""
    arq = os.path.join(C.pasta, f"feat__{bb}.npy")
    with Trava(arq + ".trava"):
        if not os.path.exists(arq):
            _extrai_feats(arq, bb, D, C, dev, bs, prazo)
    return np.load(arq)


def _extrai_feats(arq, bb, D, C, dev, bs, prazo):
    import torch
    t0 = time.time()
    rede = cria_rede(bb, D.N).to(dev).eval()
    hw = BACKBONES[bb]["hw"]
    X = np.zeros((len(D.todos), 8, rede.dim), np.float16)
    with torch.inference_mode():
        for ini, fim in ((0, D.n_base), (D.n_base, len(D.todos))):   # treino + 450 nos mesmos lotes da execução 1; teste2 depois
            for i in range(ini, fim, bs):
                if time.time() > prazo:
                    raise SemOrcamento()
                x = monta_lote(C, list(range(i, min(i + bs, fim))), dev, hw)
                for k in range(8):
                    with autocast(dev):
                        f = rede.feats(orienta(x, k))
                    X[i:i + len(x), k] = f.float().cpu().numpy()
    tmp = f"{arq}.{os.getpid()}.npy"
    np.save(tmp, X); os.replace(tmp, arq)
    print(f"features congeladas {bb}: {X.shape} em {time.time() - t0:.0f}s ({len(D.todos) * 8 / (time.time() - t0):.0f} img/s)", flush=True)


def pesos_perda(D, n_saidas):
    w = np.ones(n_saidas, np.float32)   # mesmos pesos por alvo do ajuste fino da CPU (zen_finetune.py)
    w[I_K] = 0.5; w[I_F] = 0.3; w[I_C] = 0.3; w[D.I_MG] = 0.15; w[D.I_PG] = 0.15
    return w


def treina(cfg, D, C, tr, va, dev, prazo, div_lote=1, canais=3, professor=None, fator_orc=1.0, semente=0, rotulo=""):
    """Ajuste fino completo. tr/va: índices em D.ids (va só para registrar a curva de validação; não decide nada).
    Devolve (rede com os pesos da média móvel, função de previsão, histórico por época, (média, desvio) dos alvos)."""
    import copy, torch
    torch.manual_seed(semente)
    rng = np.random.default_rng(semente)
    hw = BACKBONES[cfg["bb"]]["hw"]
    quad = hw[0] == hw[1]
    bs = max(1, min(cfg["bs"] // div_lote, len(tr))); acum = cfg["acum"] * div_lote
    T = D.T
    tm, ts = T[tr].mean(0), T[tr].std(0) + 1e-6
    # cabeça iniciada na solução do ridge sobre as features congeladas dos mesmos pratos (alfa por CV interna):
    # o ajuste fino parte do melhor modelo congelado, como na CPU
    Xc = feats_congeladas(cfg["bb"], D, C, dev, max(1, cfg["bs"] // div_lote), prazo)
    Xr = Xc[[C.pos[D.ids[i]] for i in tr]].astype(np.float32)
    _, alfa, _ = ridge_cv(Xr, T[tr], D.fold[tr])
    _, m = ridge_final(Xr, T[tr], alfa)
    del Xc, Xr
    rede = cria_rede(cfg["bb"], D.N, drop_path=cfg["drop_path"], ckpt_grad=cfg["ckpt_grad"], canais=canais).to(dev)
    with torch.no_grad():
        rede.head.weight.copy_(torch.from_numpy((((m.W / m.sd[:, None]) / ts).T).astype(np.float32)))
        rede.head.bias.copy_(torch.from_numpy(((m.mt - (m.mu / m.sd) @ m.W - tm) / ts).astype(np.float32)))
    ema = copy.deepcopy(rede).eval()
    ema.m.set_grad_checkpointing(False)
    for p in ema.parameters():
        p.requires_grad_(False)
    opt = torch.optim.AdamW(grupos_lr(rede, cfg["lr"], cfg["lld"], cfg["wd"]))
    passos_ep = max(1, len(tr) // (bs * acum)); total = cfg["epocas"] * passos_ep
    n_aq = max(1, int(cfg["aquec"] * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / n_aq if s < n_aq else 0.5 * (1 + math.cos(math.pi * (s - n_aq) / max(1, total - n_aq))))
    try:
        scaler = torch.amp.GradScaler(dev.type, enabled=dev.type == "cuda")
    except (AttributeError, TypeError):
        scaler = torch.cuda.amp.GradScaler(enabled=dev.type == "cuda")
    w = torch.from_numpy(pesos_perda(D, D.N)).to(dev)
    Yn = torch.from_numpy(((T - tm) / ts).astype(np.float32)).to(dev)
    tm_t, ts_t = torch.from_numpy(tm.astype(np.float32)).to(dev), torch.from_numpy(ts.astype(np.float32)).to(dev)
    huber = torch.nn.functional.smooth_l1_loss
    linha = np.array([C.pos.get(d, -1) for d in D.ids])   # -1: prato fora do subconjunto do --smoke

    def preve(ids, n_or=8):
        """Média das previsões (em log) nas n_or orientações, com os pesos da média móvel."""
        out = np.zeros((len(ids), D.N), np.float32)
        with torch.inference_mode():
            for i in range(0, len(ids), bs):
                x = monta_lote(C, [C.pos[d] for d in ids[i:i + bs]], dev, hw, canais)
                soma = 0
                for k in range(n_or):
                    with autocast(dev):
                        soma = soma + ema(orienta(x, k))[0].float()
                out[i:i + len(x)] = (soma / n_or * ts_t + tm_t).cpu().numpy()
        return out

    def valida():
        pv = preve([D.ids[i] for i in va], 1); P = proteinas(pv); wv = D.w_cv[va]
        cr = lambda est: float(100 * (acerto(D.proteina[va], est) * wv).sum() / wv.sum())
        return dict(val_rmse_logm=float(np.sqrt(np.mean((pv[:, I_M] - T[va, I_M]) ** 2))), val_crit_direto=cr(P["direto"]), val_crit_mxd=cr(P["mxd"]))

    hist, passo, t0 = [], 0, time.time()
    if va is not None and len(va):
        hist.append(dict(epoca=0, **valida()))
        print(f"{rotulo} época 0 (cabeça = ridge): {hist[-1]}", flush=True)
    t_treino = time.time()
    for ep in range(cfg["epocas"]):
        rede.train()
        perm = rng.permutation(len(tr)); perdas = []
        for b in range(passos_ep):
            for a in range(acum):
                j = tr[perm[(b * acum + a) * bs:(b * acum + a + 1) * bs]]
                if quad:
                    ks = rng.integers(0, 8, len(j))
                else:   # lote inteiro em paisagem (0–3) ou em retrato (4–7); a orientação dentro do formato é por amostra
                    ks = rng.integers(0, 4, len(j)) + 4 * int(rng.integers(0, 2))
                zoom = rng.uniform(*cfg["escala"], len(j)) if cfg["escala"] else None
                x = orienta_lote(monta_lote(C, linha[j], dev, hw, 4 if professor is not None else canais, zoom), ks)
                x4, x = (x, x[:, :3]) if professor is not None else (None, x)
                jt = torch.from_numpy(j).to(dev)
                with autocast(dev):
                    out, f = rede(x)
                    perda = (huber(out.float(), Yn[jt], reduction="none") * w).mean()
                    if professor is not None:   # destilação: saídas e features do professor RGB-D na mesma vista
                        with torch.no_grad():
                            op, fp = professor(x4)
                        perda = perda + (huber(out.float(), op.float(), reduction="none") * w).mean() \
                            + 0.5 * (1 - torch.nn.functional.cosine_similarity(f.float(), fp.float(), dim=1)).mean()
                if not torch.isfinite(perda):     # precisão mista pode estourar: melhor falhar do que gravar lixo
                    raise RuntimeError(f"perda não finita no passo {passo} (época {ep + 1})")
                scaler.scale(perda / acum).backward()
                perdas.append(float(perda.detach()))
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(rede.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()
            passo += 1
            d = min(cfg["ema"], (1 + passo) / (10 + passo))
            with torch.no_grad():
                for pe, p in zip(ema.parameters(), rede.parameters()):
                    pe.lerp_(p, 1 - d)
            if passo == min(total, max(3, total // 20)) or passo % 50 == 0:   # projeção de tempo: cabe no orçamento?
                por_passo = (time.time() - t_treino) / passo
                fim = time.time() + (total - passo) * por_passo * 1.1 + (total * por_passo * 1.1 + 120) * (fator_orc - 1) + 90
                if fim > prazo:
                    print(f"{rotulo} sem orçamento: faltariam {(fim - time.time()) / 60:.0f} min e restam {(prazo - time.time()) / 60:.0f} min", flush=True)
                    raise SemOrcamento()
        h = dict(epoca=ep + 1, perda=float(np.mean(perdas)), lr=float(sched.get_last_lr()[-1]), seg=round(time.time() - t0, 1),
                 img_s=round((ep + 1) * passos_ep * acum * bs / (time.time() - t_treino), 1))
        if va is not None and len(va):
            h.update(valida())
        hist.append(h)
        print(f"{rotulo} {h}", flush=True)
    return ema, preve, hist, (tm, ts)


# ------------------------------------------------------------------ uma parte (processo filho)
def arq_parte(saida, cfg, parte):
    return os.path.join(saida, f"parte__{slug(cfg['nome'])}__{parte}.npz")


def arq_de(saida, cfg, tipo):
    """Arquivos por configuração: pesos (.pt), cabeca (.npz), servir (.json), servico (.json)."""
    return os.path.join(saida, f"{tipo}__{slug(cfg['nome'])}." + dict(pesos="pt", cabeca="npz", servir="json", servico="json")[tipo])


def nomes_saidas(D):
    return ["log_massa", "log1p_proteina", "log1p_kcal", "log_densidade", "log1p_gordura", "log1p_carbo"] \
        + [f"log1p_m_{g}" for g in D.grupos] + [f"log1p_p_{g}" for g in D.grupos]


def parte_pronta(a, cfg, parte, D):
    """A parte está em disco COM tudo o que esta execução precisa? Folds: só o arquivo. final/tudo: previsões de todos
    os conjuntos das listas (inclusive teste2) e os arquivos de servir. Parte de execução antiga sem isso é refeita."""
    arq = arq_parte(a.saida, cfg, parte)
    if not os.path.exists(arq):
        return False
    if parte in ("tudo", "final"):
        with np.load(arq, allow_pickle=False) as z:
            if any(sp not in z.files for sp in D.conj):
                return False
        if not os.path.exists(arq_de(a.saida, cfg, "servir")):
            return False
        if parte == "tudo" and not os.path.exists(arq_de(a.saida, cfg, "cabeca")):
            return False
        if parte == "final" and a.guarda_pesos == "todos" and not os.path.exists(arq_de(a.saida, cfg, "pesos")):
            return False
    return True


def grava_servir(a, cfg, D, dim, cabeca):
    """servir__<config>.json: o que é preciso para reproduzir a previsão fora deste script (pré-processamento, backbone
    público com URL e sha256, orientações, cabeça, leitura das 22 saídas)."""
    b = BACKBONES[cfg["bb"]]
    try:        # nada aqui pode derrubar uma parte já treinada: o que falhar fica registrado no próprio arquivo
        import timm
        pc = timm.models.get_pretrained_cfg(b["timm"])
        media, desvio = list(pc.mean), list(pc.std)
        bk = descreve_backbone(cfg["bb"])
    except Exception as e:
        media = desvio = None
        bk = dict(chave=cfg["bb"], timm=b["timm"], url=b.get("url"), hf_id=None, fonte=None, arquivo=None, bytes=None, sha256=None, erro=f"{type(e).__name__}: {e}")
    spec = dict(
        formato=FORMATO, nome=cfg["nome"], tipo=cfg["tipo"], listas_hash=D.hash, n_treino=int(D.usa_tr.sum()),
        backbone=dict(bk, basta_o_checkpoint_publico=cfg["tipo"] == "ridge"),
        entrada=dict(foto="RGB zenital inteira, sem recorte (640×480 no Nutrition5k)", hw=list(b["hw"]),
                     redimensiona="bicúbico com antialias, da foto inteira direto para (H, W) = hw "
                                  "(torch.nn.functional.interpolate, align_corners=False); com hw quadrado a proporção é achatada",
                     escala="uint8 / 255, limitado a [0, 1] depois de redimensionar", media=media, desvio=desvio),
        orientacoes=dict(n=8, ordem=["original", "180°", "espelho horizontal", "espelho vertical", "90°", "270°", "transposta", "antitransposta"],
                         regra="média das 22 saídas (em log) nas 8 orientações; com hw não quadrado, as 4 últimas entram em retrato (W × H)"),
        features="[saída agregada do backbone (timm forward_head com pre_logits=True) ‖ média dos tokens de patch depois da norma final]",
        dim_features=int(dim), precisao="backbone em fp16 (torch.autocast) na GPU, como nas previsões gravadas",
        cabeca=cabeca,
        saidas=dict(nomes=nomes_saidas(D),
                    leitura="massa_g = exp(s[0]); proteína direta_g = expm1(s[1]); kcal = expm1(s[2]); densidade (g de proteína por g) = "
                            "exp(s[3]) − 0.01; gordura_g = expm1(s[4]); carboidrato_g = expm1(s[5]); massa e proteína por grupo: expm1"),
        cfg=cfg)
    grava_json(arq_de(a.saida, cfg, "servir"), spec)


def roda_parte(a, cfg, parte, cfgs=()):
    global PASTA_PESOS
    import torch
    t0 = time.time()
    PASTA_PESOS = os.path.join(a.cache, "pesos")
    os.environ.setdefault("HF_HOME", os.path.join(a.cache, "hf"))     # pesos do Hugging Face fora da pasta de saída
    torch.set_num_threads(max(1, int(os.environ.get("OMP_NUM_THREADS", "2"))))
    dev = torch.device(a.dispositivo)
    D = Dados(a.listas, a.smoke)
    destila = cfg["tipo"] == "destila-rgbd"
    C = Cache(a.cache, D).abre(profundidade=destila)
    prazo = a.prazo or float("inf")
    rot = f"[{cfg['nome']} | {parte}]"
    tr_tudo = np.where(D.usa_tr)[0]
    nan = lambda n: np.full((n, D.N), np.nan, np.float32)
    saida = dict(cfg=json.dumps(cfg), parte=parte, dispositivo=str(torch.cuda.get_device_name(0) if dev.type == "cuda" else "cpu"))
    if cfg["tipo"] == "ridge":
        X = feats_congeladas(cfg["bb"], D, C, dev, cfg["bs"], prazo).astype(np.float32)
        Xtr = X[[C.pos[D.ids[i]] for i in tr_tudo]]
        oof = nan(len(D.ids))
        oof[tr_tudo], alfa, mse = ridge_cv(Xtr, D.T[tr_tudo], D.fold[tr_tudo])
        pred, m = ridge_final(Xtr, D.T[tr_tudo], alfa)
        saida.update(oof=oof, alfa=alfa, cv_mse=mse)
        for s in D.conj:
            p = nan(len(D.conj[s])); u = np.where(D.usa_av[s])[0]
            p[u] = pred(X[[C.pos[D.conj[s][i]] for i in u]])
            saida[s] = p
        hist = []
        # para servir: a cabeça do ridge (o backbone é o checkpoint público, descrito em servir__*.json)
        arq_c = arq_de(a.saida, cfg, "cabeca")
        np.savez(arq_c + ".tmp.npz", mu=m.mu, sd=m.sd, W=m.W, mt=m.mt, alfa=alfa, nome=cfg["nome"], listas_hash=D.hash)
        os.replace(arq_c + ".tmp.npz", arq_c)
        grava_servir(a, cfg, D, X.shape[-1], dict(
            tipo="ridge sobre o backbone congelado", arquivo=os.path.basename(arq_c), bytes=os.path.getsize(arq_c),
            formula="s = ((f − mu) / sd) @ W + mt, com as features f em float32 (cabeça fora do autocast); arrays mu, sd, W, mt no arquivo"))
        url = BACKBONES[cfg["bb"]].get("url")       # checkpoint grande que nenhuma outra configuração usa: libera o disco se ele estiver curto
        arq = os.path.join(PASTA_PESOS, os.path.basename(url or "-"))
        if url and os.path.exists(arq) and sum(BACKBONES[c["bb"]].get("url") == url for c in cfgs) == 1 \
                and shutil.disk_usage(PASTA_PESOS).free < 15 * 2 ** 30:
            os.remove(arq)
    else:
        k = None if parte == "final" else int(parte[4:])
        tr = tr_tudo if k is None else tr_tudo[D.fold[tr_tudo] != k]
        va = None if k is None else tr_tudo[D.fold[tr_tudo] == k]
        sem = zlib.crc32(f"{cfg['nome']}|{parte}".encode()) % 2 ** 31
        prof, hist_prof = None, None
        if destila:   # professor RGB-D treinado nos mesmos pratos (nunca vê o fold de fora); só o aluno RGB faz previsões
            prof, _, hist_prof, _ = treina(cfg, D, C, tr, va, dev, prazo, a.div_lote, canais=4, fator_orc=2.0, semente=sem + 1, rotulo=rot + " professor")
        rede, preve, hist, (tm, ts) = treina(cfg, D, C, tr, va, dev, prazo, a.div_lote, professor=prof, semente=sem, rotulo=rot)
        if hist_prof is not None:
            saida["hist_professor"] = json.dumps(hist_prof)
        if k is not None:
            oof = nan(len(D.ids)); oof[va] = preve([D.ids[i] for i in va])
            saida["oof"] = oof
        else:
            for s in D.conj:
                p = nan(len(D.conj[s])); u = np.where(D.usa_av[s])[0]
                p[u] = preve([D.conj[s][i] for i in u])
                saida[s] = p
            if a.guarda_pesos != "nenhum":   # pesos da média móvel em meia precisão (ficam fora do git); só tensores e tipos simples
                arq = arq_de(a.saida, cfg, "pesos")
                torch.save(dict(formato=FORMATO, nome=cfg["nome"], cfg=cfg, timm=BACKBONES[cfg["bb"]], tm=[float(v) for v in tm], ts=[float(v) for v in ts],
                                grupos=list(D.grupos), listas_hash=D.hash,
                                estado={n: (v.half() if v.is_floating_point() else v).cpu() for n, v in rede.state_dict().items()}), arq + ".tmp")
                os.replace(arq + ".tmp", arq)
                grava_servir(a, cfg, D, rede.dim, dict(
                    tipo="linear, treinada junto com o backbone", arquivo=os.path.basename(arq), bytes=os.path.getsize(arq),
                    formula="s = head(f) * ts + tm; o arquivo traz 'estado' (rede inteira com a média móvel dos pesos, meia precisão), 'tm' e 'ts'",
                    carregar="torch.load(arquivo, weights_only=True); cria_rede(bb, 22, pretrained=False).load_state_dict(estado em float32)"))
    for k in ["oof"] + list(D.conj):      # só previsões finitas viram arquivo (NaN é reservado a pratos fora da parte)
        if k in saida and not np.isfinite(saida[k][~np.isnan(saida[k][:, 0])]).all():
            raise RuntimeError(f"previsões não finitas em '{k}'")
    saida.update(hist=json.dumps(hist), seg=time.time() - t0)
    if dev.type == "cuda":
        saida["mem_gb"] = torch.cuda.max_memory_allocated() / 2 ** 30
    dest = arq_parte(a.saida, cfg, parte)
    np.savez(dest + ".tmp.npz", **saida); os.replace(dest + ".tmp.npz", dest)
    print(f"{rot} pronto em {(time.time() - t0) / 60:.1f} min", flush=True)


# ------------------------------------------------------------------ fase de serviço (processo filho, um modelo por vez)
def mede_servico(a, cfg):
    """Remonta o modelo SÓ com os arquivos de servir, confere que ele reproduz as previsões gravadas e mede o custo:
    parâmetros, tamanho dos pesos e latência por imagem (GPU em fp16, lotes de 1 e de 8, 1 e 8 orientações; CPU em
    float32 com o número de threads registrado). O tempo inclui redimensionar e normalizar, a partir de uint8 480×640
    já no dispositivo; não inclui decodificar o PNG nem a cópia para a GPU. Grava servico__<config>.json."""
    global PASTA_PESOS
    import torch
    F = torch.nn.functional
    t0 = time.time()
    PASTA_PESOS = os.path.join(a.cache, "pesos")
    os.environ.setdefault("HF_HOME", os.path.join(a.cache, "hf"))
    n_thr = max(1, int(os.environ.get("OMP_NUM_THREADS", "2")))
    torch.set_num_threads(n_thr)
    dev = torch.device(a.dispositivo)
    D = Dados(a.listas, a.smoke)
    C = Cache(a.cache, D).abre()
    prazo = a.prazo or float("inf")
    hw = BACKBONES[cfg["bb"]]["hw"]
    quad, ridge = hw[0] == hw[1], cfg["tipo"] == "ridge"
    spec = json.load(open(arq_de(a.saida, cfg, "servir")))
    if ridge:      # backbone público + cabeça do ridge como camada linear
        rede = cria_rede(cfg["bb"], D.N)
        z = np.load(arq_de(a.saida, cfg, "cabeca"), allow_pickle=False)
        with torch.no_grad():
            rede.head.weight.copy_(torch.from_numpy((z["W"] / z["sd"][:, None]).T.astype(np.float32)))
            rede.head.bias.copy_(torch.from_numpy((z["mt"] - (z["mu"] / z["sd"]) @ z["W"]).astype(np.float32)))
        tm, ts = np.zeros(D.N, np.float32), np.ones(D.N, np.float32)
        arquivos = [dict(arquivo=spec["backbone"]["arquivo"], bytes=spec["backbone"]["bytes"], publico=True, url=spec["backbone"]["url"],
                         hf_id=spec["backbone"]["hf_id"], fonte=spec["backbone"]["fonte"], sha256=spec["backbone"]["sha256"]),
                    dict(arquivo=os.path.basename(arq_de(a.saida, cfg, "cabeca")), bytes=os.path.getsize(arq_de(a.saida, cfg, "cabeca")), publico=False)]
    else:          # só o arquivo de pesos gravado (sem baixar nada)
        ck = torch.load(arq_de(a.saida, cfg, "pesos"), map_location="cpu", weights_only=True)
        rede = cria_rede(cfg["bb"], D.N, pretrained=False)
        rede.load_state_dict({n: (v.float() if v.is_floating_point() else v) for n, v in ck["estado"].items()})
        tm, ts = np.array(ck["tm"], np.float32), np.array(ck["ts"], np.float32)
        arquivos = [dict(arquivo=os.path.basename(arq_de(a.saida, cfg, "pesos")), bytes=os.path.getsize(arq_de(a.saida, cfg, "pesos")), publico=False)]
    rede.eval()
    n_par, n_cab = sum(p.numel() for p in rede.parameters()), sum(p.numel() for p in rede.head.parameters())

    def infere(d, xu8, n_or, agrupado=False):
        """uint8 [B, 480, 640, 3] no dispositivo -> 22 saídas em log por imagem (média das orientações).
        agrupado: as 8 orientações em um lote só (entrada quadrada) ou em dois de 4 (paisagem e retrato)."""
        x = F.interpolate(xu8.permute(0, 3, 1, 2).float().div(255), size=hw, mode="bicubic", antialias=True, align_corners=False).clamp_(0, 1)
        grupos = [[k] for k in range(n_or)] if not agrupado else ([list(range(8))] if quad else [[0, 1, 2, 3], [4, 5, 6, 7]])
        soma = 0
        for g in grupos:
            xb = torch.cat([orienta(x, k) for k in g])
            with autocast(d):
                f = rede.feats(xb)
                o = None if ridge else rede.head(f)
            if ridge:      # cabeça do ridge em float32, fora do autocast (como as previsões gravadas)
                o = rede.head(f.float())
            soma = soma + o.float().reshape(len(g), len(x), -1).sum(0)
        return soma / n_or * torch.from_numpy(ts).to(d) + torch.from_numpy(tm).to(d)

    def cronometra(d, xu8, n_or, agrupado, reps, aquece):
        sinc = torch.cuda.synchronize if d.type == "cuda" else (lambda: None)
        with torch.inference_mode():
            for _ in range(aquece):
                infere(d, xu8, n_or, agrupado)
            if d.type == "cuda":
                torch.cuda.reset_peak_memory_stats()
            tempos = []
            for _ in range(reps):
                sinc(); t = time.perf_counter(); infere(d, xu8, n_or, agrupado); sinc()
                tempos.append(time.perf_counter() - t)
        r = dict(lote=len(xu8), orientacoes=n_or, oito_em_lote=bool(agrupado), repeticoes=reps,
                 ms_por_imagem=round(1000 * float(np.median(tempos)) / len(xu8), 2), ms_por_imagem_min=round(1000 * min(tempos) / len(xu8), 2),
                 ms_por_imagem_max=round(1000 * max(tempos) / len(xu8), 2))
        if d.type == "cuda":
            r["mem_pico_mb"] = round(torch.cuda.max_memory_allocated() / 2 ** 20)
        return r

    u8 = lambda ids, d: torch.from_numpy(np.ascontiguousarray(C.rgb[[C.pos[i] for i in ids]])).to(d)
    rede.to(dev)
    # 1. o modelo remontado reproduz as previsões gravadas? (até 8 pratos do calib, 8 orientações; não usa gabarito)
    parte = np.load(arq_parte(a.saida, cfg, "tudo" if ridge else "final"), allow_pickle=False)
    u = np.where(D.usa_av["calib"])[0][:8]
    with torch.inference_mode():
        p = infere(dev, u8([D.conj["calib"][i] for i in u], dev), 8).cpu().numpy()
    dif = np.abs(p - parte["calib"][u])
    res = dict(formato=FORMATO, nome=cfg["nome"], tipo=cfg["tipo"], backbone=cfg["bb"], timm=BACKBONES[cfg["bb"]]["timm"], hw=list(hw),
               parametros=dict(total=int(n_par), backbone=int(n_par - n_cab), cabeca=int(n_cab)), mb_fp16_em_memoria=round(n_par * 2 / 1e6, 1),
               arquivos=arquivos, bytes_pesos=int(sum(x["bytes"] or 0 for x in arquivos)), bytes_gravados_na_saida=int(sum(x["bytes"] or 0 for x in arquivos if not x["publico"])),
               conferencia=dict(pratos=int(len(u)), max_dif_log=float(dif.max()), max_dif_log_massa=float(dif[:, I_M].max()), max_dif_log1p_proteina=float(dif[:, I_P].max()),
                                dispositivo=dev.type, nota="diferença máxima, em log, entre o modelo remontado com os arquivos de servir e as previsões gravadas"),
               inclui="redimensionar (bicúbico) + normalizar + rede + cabeça, a partir de uint8 480×640 já no dispositivo; não inclui decodificar o PNG nem a cópia para a GPU",
               gpu=None, cpu=None)
    print(f"[{cfg['nome']} | servico] {n_par / 1e6:.1f} M parâmetros; remontado × gravado: diferença máxima {dif.max():.4f} em log", flush=True)
    ids8 = D.todos[:8]
    # 2. GPU (uma só, fp16)
    if dev.type == "cuda":
        x8 = u8(ids8, dev)
        med = {}
        for nome, (lote, n_or, agr, reps, aq) in dict(lote1_o1=(1, 1, False, 30, 5), lote1_o8=(1, 8, False, 10, 2), lote1_o8_em_lote=(1, 8, True, 10, 2),
                                                       lote8_o1=(8, 1, False, 10, 2), lote8_o8=(8, 8, False, 5, 1)).items():
            if time.time() + 60 > prazo:
                raise SemOrcamento()
            med[nome] = cronometra(dev, x8[:lote], n_or, agr, reps, aq)
        res["gpu"] = dict(nome=torch.cuda.get_device_name(0), precisao="fp16 (torch.autocast)", medidas=med)
        print(f"[{cfg['nome']} | servico] GPU ms por imagem: " + ", ".join(f"{k} {v['ms_por_imagem']}" for k, v in med.items()), flush=True)
    # 3. CPU (float32), poucas imagens
    rede.float().cpu()
    cpu = torch.device("cpu")
    x8 = u8(ids8, cpu)
    try:
        modelo_cpu = next(l.split(":", 1)[1].strip() for l in open("/proc/cpuinfo") if l.startswith("model name"))
    except Exception:
        modelo_cpu = None
    med = dict(lote1_o1=cronometra(cpu, x8[:1], 1, False, 2 if a.smoke else 3, 1))
    rapido = med["lote1_o1"]["ms_por_imagem"] < 500
    if time.time() + 10 * med["lote1_o1"]["ms_por_imagem"] / 1000 + 30 < prazo:
        med["lote1_o8"] = cronometra(cpu, x8[1:2], 8, False, 3 if rapido and not a.smoke else 1, 0)
    res["cpu"] = dict(threads=n_thr, nucleos=os.cpu_count(), modelo=modelo_cpu, precisao="float32", medidas=med)
    print(f"[{cfg['nome']} | servico] CPU ({n_thr} threads) ms por imagem: " + ", ".join(f"{k} {v['ms_por_imagem']}" for k, v in med.items()), flush=True)
    res["seg"] = round(time.time() - t0, 1)
    grava_json(arq_de(a.saida, cfg, "servico"), res)


def principal_parte(a, cfgs):
    nome, parte = a.job.rsplit(":", 1)
    cfg = next(c for c in cfgs if slug(c["nome"]) == nome)
    simula = os.environ.get("NUTRELLE_SIMULA", "")     # só para testar o orquestrador: "memoria" ou "erro"
    if simula == "memoria" and a.div_lote == 1 and parte in ("fold0", "tudo"):
        sys.exit(SAI_MEMORIA)
    if simula == "erro" and parte == "fold1":
        sys.exit(1)
    try:
        if parte == "servico":
            mede_servico(a, cfg)
        else:
            roda_parte(a, cfg, parte, cfgs)
    except SemOrcamento:
        sys.exit(SAI_SEM_ORCAMENTO)
    except Exception as e:
        traceback.print_exc()
        sys.exit(SAI_MEMORIA if "out of memory" in str(e).lower() and parte != "servico" else 1)


# ------------------------------------------------------------------ orquestrador
_trava_log = threading.Lock()
_arq_log = None


def diz(*partes):
    s = time.strftime("%H:%M:%S ") + " ".join(str(p) for p in partes)
    with _trava_log:
        print(s, flush=True)
        if _arq_log:
            _arq_log.write(s + "\n"); _arq_log.flush()


def grava_json(arq, obj):
    with open(arq + ".tmp", "w") as f:
        json.dump(obj, f, indent=1, ensure_ascii=False, default=float)
    os.replace(arq + ".tmp", arq)


def garante_timm(cfgs):
    """Kaggle/Colab trazem timm; as etiquetas '.v2_webli' (SigLIP 2) pedem timm >= 1.0.15. Atualiza se faltar."""
    def faltam():
        import timm
        tem = set(timm.list_pretrained())
        return sorted({BACKBONES[c["bb"]]["timm"] for c in cfgs} - tem), timm.__version__
    try:
        f, v = faltam()
    except ImportError:
        f, v = ["timm"], None
    if f:
        diz(f"timm {v} não conhece {f}: atualizando com pip")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "timm"], check=False)
        r = subprocess.run([sys.executable, "-c", "import timm,sys; t=set(timm.list_pretrained()); sys.exit(any(x not in t for x in sys.argv[1:]))"] + f)
        if r.returncode:
            raise SystemExit(f"Mesmo depois de atualizar, o timm não tem os pesos {f}.")


def retoma(a):
    """Copia partes prontas de execuções anteriores (mesmas listas) para a pasta de saída. Partes de fold da execução 1
    são aceitas como estão; partes final/tudo sem teste2 ou sem os arquivos de servir são refeitas (ver parte_pronta)."""
    fontes = list(a.retoma)
    if os.path.isdir("/kaggle/input"):
        fontes += [os.path.dirname(m) for m in glob.glob("/kaggle/input/**/manifesto.json", recursive=True)]
    n = 0
    for pasta in fontes:
        try:
            man = json.load(open(os.path.join(pasta, "manifesto.json")))
        except Exception:
            continue
        if man.get("formato") not in FORMATOS_ACEITOS or man.get("listas_hash") != a._hash or bool(man.get("smoke")) != bool(a.smoke):
            diz(f"retomada: {pasta} ignorada (outro formato, outras listas ou smoke)")
            continue
        for padrao in ("parte__*.npz", "pesos__*.pt", "cabeca__*.npz", "servir__*.json", "servico__*.json"):
            for arq in glob.glob(os.path.join(pasta, padrao)):
                dest = os.path.join(a.saida, os.path.basename(arq))
                if not os.path.exists(dest):
                    shutil.copy(arq, dest); n += 1
    if n:
        diz(f"retomada: {n} arquivos copiados de {fontes}")


def junta(a, cfg, D):
    """Reúne as partes de uma configuração completa: pred__*.npz (esquema do banco + teste2), pred__*.csv e log__*.json."""
    s = slug(cfg["nome"])
    partes = {p: dict(np.load(arq_parte(a.saida, cfg, p), allow_pickle=False)) for p in partes_de(cfg)}
    if cfg["tipo"] == "ridge":
        oof, av = partes["tudo"]["oof"], partes["tudo"]
    else:
        oof = np.full((len(D.ids), D.N), np.nan, np.float32)
        for k in range(5):
            m = (D.fold == k) & D.usa_tr
            oof[m] = partes[f"fold{k}"]["oof"][m]
        av = partes["final"]
    assert not np.isnan(oof[D.usa_tr]).any(), "previsão fora-do-fold faltando"
    for sp in D.conj:
        assert av[sp].shape == (len(D.conj[sp]), D.N) and not np.isnan(av[sp][D.usa_av[sp]]).any(), f"previsão de {sp} faltando"
    pred = dict(oof=oof, **{sp: av[sp] for sp in D.conj})
    np.savez(os.path.join(a.saida, f"pred__{s}.tmp.npz"), nome=cfg["nome"], listas_hash=D.hash, listas2_hash=D.hash2 or "", fold=D.fold,
             ids_treino=np.array(D.ids), **{f"ids_{sp}": np.array(D.conj[sp]) for sp in D.conj}, **pred)
    os.replace(os.path.join(a.saida, f"pred__{s}.tmp.npz"), os.path.join(a.saida, f"pred__{s}.npz"))
    with open(os.path.join(a.saida, f"pred__{s}.csv"), "w", newline="") as f:
        wr = csv.writer(f); wr.writerow(["conjunto", "dish_id", "fold"] + nomes_saidas(D))
        for conj, ids, M, folds in [("treino_oof", D.ids, oof, D.fold)] + [(sp, D.conj[sp], av[sp], None) for sp in D.conj]:
            for i, d in enumerate(ids):
                if not np.isnan(M[i, 0]):
                    wr.writerow([conj, d, "" if folds is None else int(folds[i])] + [f"{v:.5f}" for v in M[i]])
    cv = resumo_cv(D, oof)
    extra = {"hist_professor": {p: json.loads(str(v["hist_professor"])) for p, v in partes.items() if "hist_professor" in v}}
    grava_json(os.path.join(a.saida, f"log__{s}.json"),
               dict(nome=cfg["nome"], cfg=cfg, cv_treino=cv, seg_total=float(sum(float(v["seg"]) for v in partes.values())),
                    partes={p: dict(seg=float(v["seg"]), dispositivo=str(v["dispositivo"]), mem_gb=float(v["mem_gb"]) if "mem_gb" in v else None,
                                    hist=json.loads(str(v["hist"]))) for p, v in partes.items()}, **(extra if extra["hist_professor"] else {})))
    return cv


def monta_servico(a, cfgs, pipelines):
    """servico.json: uma entrada por modelo medido + totais de cada pipeline (soma das latências por imagem, um modelo
    depois do outro numa só GPU; soma de parâmetros e de bytes de pesos; maior pico de memória de GPU)."""
    modelos = {}
    for c in cfgs:
        if os.path.exists(arq_de(a.saida, c, "servico")):
            modelos[c["nome"]] = json.load(open(arq_de(a.saida, c, "servico")))

    def total(nomes):
        tem, falta = [n for n in nomes if n in modelos], [n for n in nomes if n not in modelos]
        soma = lambda disp: {k: round(sum(modelos[n][disp]["medidas"][k]["ms_por_imagem"] for n in tem), 1)
                             for k in (modelos[tem[0]][disp] or {}).get("medidas", {})
                             if all(modelos[n][disp] and k in modelos[n][disp]["medidas"] for n in tem)} if tem else {}
        picos = [v["mem_pico_mb"] for n in tem if modelos[n]["gpu"] for v in modelos[n]["gpu"]["medidas"].values()]
        return dict(modelos=list(nomes), completo=not falta, faltam=falta,
                    parametros=int(sum(modelos[n]["parametros"]["total"] for n in tem)),
                    bytes_pesos=int(sum(modelos[n]["bytes_pesos"] for n in tem)),
                    bytes_gravados_na_saida=int(sum(modelos[n]["bytes_gravados_na_saida"] for n in tem)),
                    mb_fp16_em_memoria=round(sum(modelos[n]["mb_fp16_em_memoria"] for n in tem), 1),
                    gpu_ms_por_imagem=soma("gpu"), gpu_mem_pico_mb_maior_modelo=max(picos) if picos else None, cpu_ms_por_imagem=soma("cpu"))

    um = next(iter(modelos.values()), {})
    obj = dict(formato=FORMATO, gerado=time.strftime("%Y-%m-%d %H:%M:%S"),
               condicoes=dict(gpu=(um.get("gpu") or {}).get("nome"), gpu_precisao="fp16 (torch.autocast)", cpu=um.get("cpu"),
                              inclui=um.get("inclui"), medido="um modelo por vez, com a máquina sem outro treino em andamento",
                              chaves="loteB_oK = lote de B imagens, K orientações em sequência; lote1_o8_em_lote = as 8 orientações de uma imagem "
                                     "num lote só (entrada quadrada) ou em dois de 4 (paisagem e retrato); ms_por_imagem = mediana do tempo do lote ÷ B"),
               pipelines={k: total(v) for k, v in pipelines.items()}, nota_pipelines=NOTA_PIPELINES, modelos=modelos)
    for v in obj["condicoes"]["cpu"] and [obj["condicoes"]["cpu"]] or []:
        v.pop("medidas", None)
    grava_json(os.path.join(a.saida, "servico.json"), obj)
    return obj


def grava_tamanhos(a):
    """tamanhos.json: bytes de cada arquivo da pasta de saída, por tipo e no total (limite do Kaggle: ~20 GB)."""
    arqs = {n: os.path.getsize(os.path.join(a.saida, n)) for n in sorted(os.listdir(a.saida))
            if os.path.isfile(os.path.join(a.saida, n)) and n != "tamanhos.json" and ".tmp" not in n}
    tipo = lambda n: n.split("__")[0] if "__" in n else "outros"
    por_tipo = {}
    for n, b in arqs.items():
        por_tipo[tipo(n)] = por_tipo.get(tipo(n), 0) + b
    obj = dict(total_bytes=int(sum(arqs.values())), total_gb=round(sum(arqs.values()) / 2 ** 30, 3), n_arquivos=len(arqs), limite_kaggle_gb=20,
               maior_arquivo=max(arqs.items(), key=lambda x: x[1]) if arqs else None,
               por_tipo_mb={k: round(v / 2 ** 20, 1) for k, v in sorted(por_tipo.items())}, arquivos=arqs)
    grava_json(os.path.join(a.saida, "tamanhos.json"), obj)
    return obj


def orquestra(a, cfgs):
    global _arq_log
    t_ini = time.time()
    os.makedirs(a.saida, exist_ok=True)
    try:
        os.makedirs(a.cache, exist_ok=True)
    except OSError:      # /kaggle/temp pode não existir ou não aceitar escrita
        a.cache = os.path.join(tempfile.gettempdir(), "nutrelle_cache"); os.makedirs(a.cache, exist_ok=True)
    for t in glob.glob(os.path.join(a.cache, "**", "*.trava"), recursive=True):   # nenhum processo filho vivo aqui
        os.remove(t)
    _arq_log = open(os.path.join(a.saida, "treino_gpu.log"), "a")
    prazo = t_ini + a.orcamento_h * 3600 - 300          # 5 min de reserva para fechar os arquivos
    D = Dados(a.listas, a.smoke)
    a._hash = D.hash
    livre = lambda pasta: round(shutil.disk_usage(pasta).free / 2 ** 30, 1)
    diz(f"{FORMATO} | rodada {'smoke' if a.smoke else a.rodada}: {len(cfgs)} configurações | treino {int(D.usa_tr.sum())} pratos, avaliação "
        + ", ".join(f"{s} {int(D.usa_av[s].sum())}" for s in D.conj) + f" | listas {D.hash}, teste2 {D.hash2} | orçamento {a.orcamento_h} h"
        f" | saída {a.saida} ({livre(a.saida)} GB livres) | cache {a.cache} ({livre(a.cache)} GB livres) | pesos: {a.guarda_pesos}")
    garante_timm(cfgs)
    import torch
    n_gpu = torch.cuda.device_count()
    disp = [f"cuda:{i}" for i in range(n_gpu)] if n_gpu and not a.cpu else ["cpu"] * a.workers
    try:      # nomes pelo nvidia-smi: perguntar ao torch abriria um contexto CUDA (memória de GPU) neste processo
        gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], capture_output=True, text=True, timeout=30).stdout.strip().splitlines()
    except Exception:
        gpus = []
    gpus = gpus if len(gpus) == n_gpu else [f"cuda:{i}" for i in range(n_gpu)]
    try:
        import timm
        v_timm = timm.__version__
    except Exception:
        v_timm = None
    diz(f"torch {torch.__version__}, timm {v_timm} | dispositivos: {gpus or 'CPU'} -> {len(disp)} processo(s) em paralelo")
    if not n_gpu and not (a.smoke or a.cpu):
        raise SystemExit("Sem GPU. No Kaggle: Settings > Accelerator > GPU T4 x2. (Para testar em CPU: --smoke.)")
    retoma(a)
    estado = {slug(c["nome"]): {p: parte_inicial() for p in partes_de(c)} for c in cfgs}
    est_serv = {slug(c["nome"]): {"servico": parte_inicial()} for c in cfgs}      # fase de serviço: mesma máquina de estados
    trava = threading.Lock()
    info = dict(formato=FORMATO, rodada="smoke" if a.smoke else a.rodada, smoke=bool(a.smoke), listas_hash=D.hash, listas2_hash=D.hash2,
                n_treino=int(D.usa_tr.sum()), conjuntos={s: int(D.usa_av[s].sum()) for s in D.conj}, inicio=time.strftime("%Y-%m-%d %H:%M:%S"),
                orcamento_h=a.orcamento_h, gpus=gpus, torch=torch.__version__, timm=v_timm, python=sys.version.split()[0], guarda_pesos=a.guarda_pesos,
                disco_livre_gb_inicio=dict(saida=livre(a.saida), cache=livre(a.cache)), argv=sys.argv[1:], configs={}, melhor_modelo=None)

    def evento(cfg, parte, **ev):     # único lugar em que o estado muda (sempre com a trava)
        s = slug(cfg["nome"])
        mapa = est_serv if parte == "servico" else estado
        mapa[s][parte] = transicao(mapa[s][parte], ev)
        with open(os.path.join(a.saida, "eventos.jsonl"), "a") as f:
            f.write(json.dumps(dict(t=round(time.time(), 1), config=cfg["nome"], parte=parte, **ev, estado=mapa[s][parte]["estado"]), ensure_ascii=False) + "\n")

    def manifesto():
        for c in cfgs:
            s = slug(c["nome"])
            info["configs"].setdefault(c["nome"], dict(slug=s, tipo=c["tipo"], bb=c["bb"], cv_treino=None, min=None))
            info["configs"][c["nome"]].update(estado=estado_config(estado[s]), partes=estado[s], servico=est_serv[s]["servico"])
        info["duracao_min"] = round((time.time() - t_ini) / 60, 1)
        grava_json(os.path.join(a.saida, "manifesto.json"), info)

    def fecha_config(c):
        """Configuração completa: junta as partes, calcula a CV no treino e, com --guarda-pesos melhor, mantém só os pesos do melhor modelo."""
        s = slug(c["nome"])
        cv = junta(a, c, D)
        ic = info["configs"][c["nome"]]
        ic["cv_treino"] = cv; ic["min"] = round(sum(p["seg"] or 0 for p in estado[s].values()) / 60, 1)
        diz(f"== {c['nome']}: CV no treino (n={cv['n']}) RMSE log-massa {cv['rmse_logm']:.4f} | MAPE massa {cv['mape_massa']:.1f}% | "
            f"critério ponderado direto {cv['crit_direto']:.1f}% m×d {cv['crit_mxd']:.1f}% geo {cv['crit_geo']:.1f}% | {ic['min']} min de GPU")
        nota = lambda n: max(info["configs"][n]["cv_treino"][r] for r in ("crit_direto", "crit_mxd", "crit_geo"))
        com_pesos = [x["nome"] for x in cfgs if info["configs"][x["nome"]]["cv_treino"] and os.path.exists(arq_de(a.saida, x, "pesos"))]
        if com_pesos:
            melhor = max(com_pesos, key=nota)
            info["melhor_modelo"] = dict(nome=melhor, arquivo=f"pesos__{slug(melhor)}.pt", criterio_cv=nota(melhor),
                                         regra="maior critério ponderado na CV do treino entre as rotas direto, m×d e geo")
            if a.guarda_pesos == "melhor":
                for n in com_pesos:
                    if n != melhor:
                        os.remove(os.path.join(a.saida, f"pesos__{slug(n)}.pt"))

    def lanca(c, p, dev, div_lote, threads):
        """Roda uma parte num processo filho e devolve (código de saída, últimas linhas, segundos)."""
        env = dict(os.environ, PYTHONUNBUFFERED="1", OMP_NUM_THREADS=str(threads))
        cmd = [sys.executable, "-u", os.path.abspath(__file__), "--job", f"{slug(c['nome'])}:{p}", "--prazo", str(prazo), "--div-lote", str(div_lote),
               "--saida", a.saida, "--cache", a.cache, "--listas", a.listas, "--guarda-pesos", a.guarda_pesos]
        if a.smoke:
            cmd += ["--smoke", str(a.smoke)]
        if dev.startswith("cuda"):
            env["CUDA_VISIBLE_DEVICES"] = dev.split(":")[1]; cmd += ["--dispositivo", "cuda:0"]
        else:
            cmd += ["--dispositivo", "cpu"]
        t0 = time.time()
        pr = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env)
        cauda = []
        for linha in pr.stdout:
            diz(f"[{dev}] {linha.rstrip()}"); cauda = (cauda + [linha.rstrip()])[-8:]
        return pr.wait(), cauda, time.time() - t0

    with trava:
        for c in cfgs:                      # partes já em disco e completas para esta execução (retomada) -> FEITA
            for p in partes_de(c):
                if parte_pronta(a, c, p, D):
                    evento(c, p, tipo="retomou", seg=float(np.load(arq_parte(a.saida, c, p))["seg"]))
                elif os.path.exists(arq_parte(a.saida, c, p)):
                    diz(f"retomada: {c['nome']} | {p} existe, mas sem teste2 ou sem os arquivos de servir: será refeita")
        manifesto()
        for c in cfgs:
            if estado_config(estado[slug(c["nome"])]) == "completa":
                fecha_config(c)
        manifesto()
    pendentes = [c for c in cfgs if estado_config(estado[slug(c["nome"])]) != "completa"]
    if pendentes or not a.sem_servico:
        Cache(a.cache, D).prepara(profundidade=any(c["tipo"] == "destila-rgbd" for c in pendentes))

    def proxima(dev):
        """Próxima parte pendente, na ordem de prioridade. Parte que não cabe no tempo restante vira sem_orcamento."""
        for c in cfgs:
            s = slug(c["nome"])
            for p in partes_de(c):
                e = estado[s][p]
                if e["estado"] != Estado.PENDENTE.value:
                    continue
                feitas = [x["seg"] for x in estado[s].values() if x["estado"] == Estado.FEITA.value and x["seg"] and not x["retomada"]]
                est = float(np.median(feitas)) if feitas else 120.0
                if time.time() + est > prazo:
                    evento(c, p, tipo="sem_orcamento")
                    cancela(c, "sem orçamento para a configuração inteira")
                    continue
                evento(c, p, tipo="iniciou", dispositivo=dev)
                return c, p
        return None

    def cancela(c, motivo):
        for p, e in estado[slug(c["nome"])].items():
            if e["estado"] == Estado.PENDENTE.value:
                evento(c, p, tipo="cancelou", motivo=motivo)

    def trabalhador(i, dev):
        while True:
            with trava:
                job = proxima(dev); manifesto()
            if job is None:
                return
            c, p = job
            e = estado[slug(c["nome"])][p]
            diz(f"[{dev}] começa {c['nome']} | {p} (tentativa {e['tentativa']}, lote ÷ {e['div_lote']}); restam {(prazo - time.time()) / 60:.0f} min")
            rc, cauda, seg = lanca(c, p, dev, e["div_lote"], max(1, (os.cpu_count() or 2) // len(disp)) if not a.smoke else 1)
            with trava:
                if rc == SAI_OK and parte_pronta(a, c, p, D):
                    evento(c, p, tipo="concluiu", seg=seg)
                    if estado_config(estado[slug(c["nome"])]) == "completa":
                        try:
                            fecha_config(c)
                        except Exception as ex:
                            diz(f"ERRO ao juntar {c['nome']}: {ex}")
                elif rc == SAI_SEM_ORCAMENTO:
                    evento(c, p, tipo="sem_orcamento"); cancela(c, "sem orçamento para a configuração inteira")
                elif rc == SAI_MEMORIA:
                    evento(c, p, tipo="estourou_memoria")
                    for q, eq in estado[slug(c["nome"])].items():   # as outras partes da configuração usam o mesmo lote reduzido
                        if eq["estado"] == Estado.PENDENTE.value and q != p:
                            evento(c, q, tipo="reduziu_lote", div_lote=estado[slug(c["nome"])][p]["div_lote"])
                else:
                    evento(c, p, tipo="falhou", erro=f"código {rc}: " + " | ".join(cauda[-3:]))
                if estado[slug(c["nome"])][p]["estado"] == Estado.FALHOU.value:
                    diz(f"[{dev}] {c['nome']} | {p} FALHOU: {estado[slug(c['nome'])][p]['erro']}")
                    cancela(c, f"a parte {p} falhou")
                manifesto()

    if pendentes:
        ts = [threading.Thread(target=trabalhador, args=(i, d)) for i, d in enumerate(disp)]
        [t.start() for t in ts]; [t.join() for t in ts]

    # ---- fase de serviço: depois de todo o treino, um modelo por vez numa só GPU, com a CPU livre (latências limpas)
    def serve(c):
        s = slug(c["nome"])
        if os.path.exists(arq_de(a.saida, c, "servico")):
            return evento(c, "servico", tipo="retomou")
        if estado_config(estado[s]) != "completa" or not os.path.exists(arq_de(a.saida, c, "servir")) \
                or not os.path.exists(arq_de(a.saida, c, "cabeca" if c["tipo"] == "ridge" else "pesos")):
            return evento(c, "servico", tipo="cancelou", motivo="configuração incompleta ou sem arquivos de servir")
        while est_serv[s]["servico"]["estado"] == Estado.PENDENTE.value:
            if time.time() + 300 > prazo:
                return evento(c, "servico", tipo="sem_orcamento")
            evento(c, "servico", tipo="iniciou", dispositivo=disp[0])
            diz(f"[{disp[0]}] serviço: {c['nome']}; restam {(prazo - time.time()) / 60:.0f} min")
            rc, cauda, seg = lanca(c, "servico", disp[0], 1, (os.cpu_count() or 2) if not a.smoke else 1)
            if rc == SAI_OK and os.path.exists(arq_de(a.saida, c, "servico")):
                evento(c, "servico", tipo="concluiu", seg=seg)
            elif rc == SAI_SEM_ORCAMENTO:
                evento(c, "servico", tipo="sem_orcamento")
            else:
                evento(c, "servico", tipo="falhou", erro=f"código {rc}: " + " | ".join(cauda[-3:]))

    with trava:
        if not a.sem_servico:
            for c in cfgs:
                serve(c); manifesto()
            sv = monta_servico(a, cfgs, {"smoke_todos": [c["nome"] for c in cfgs]} if a.smoke else PIPELINES)
            for k, v in sv["pipelines"].items():
                diz(f"serviço, {k}: {'completo' if v['completo'] else 'FALTAM ' + str(v['faltam'])} | {v['parametros'] / 1e6:.0f} M parâmetros | "
                    f"pesos {v['bytes_pesos'] / 2 ** 30:.2f} GB | GPU ms/imagem {v['gpu_ms_por_imagem']} | CPU ms/imagem {v['cpu_ms_por_imagem']}")
        info["disco_livre_gb_fim"] = dict(saida=livre(a.saida), cache=livre(a.cache))
        manifesto()
        tam = grava_tamanhos(a)
        info["tamanho_saida"] = {k: tam[k] for k in ("total_gb", "n_arquivos", "por_tipo_mb", "maior_arquivo")}
        manifesto()
    diz("\n" + "\n".join(f"  {ic['estado']:14s} {n:44s} " + (f"RMSE {ic['cv_treino']['rmse_logm']:.4f} crit {ic['cv_treino']['crit_direto']:.1f}/{ic['cv_treino']['crit_mxd']:.1f}/{ic['cv_treino']['crit_geo']:.1f}% {ic['min']} min"
                                                                  if ic["cv_treino"] else "") + f" | serviço: {ic['servico']['estado']}" for n, ic in info["configs"].items()))
    diz(f"melhor modelo único pela CV: {info['melhor_modelo']}")
    diz(f"saída: {tam['total_gb']} GB em {tam['n_arquivos']} arquivos {tam['por_tipo_mb']} (MB por tipo)")
    diz(f"fim: {info['duracao_min']} min. Baixe a pasta de saída e rode eval/treino/kaggle/ingere_kaggle.py <pasta>.")


def checa_modelos():
    """Monta cada backbone SEM pesos e passa uma imagem em paisagem e em retrato (confere nomes, formatos e orientações)."""
    import torch
    for bb, b in BACKBONES.items():
        t0 = time.time()
        rede = cria_rede(bb, 22, pretrained=False).eval()
        x = torch.rand(1, 3, *b["hw"])
        with torch.inference_mode():
            formas = [tuple(rede(orienta(x, k))[0].shape) for k in (0, 5)]
        print(f"{bb:28s} {b['timm']:44s} {sum(p.numel() for p in rede.parameters()) / 1e6:6.1f} M parâmetros | dim {rede.dim} | "
              f"blocos {len(rede.m.blocks)} | grupos de lr {len(grupos_lr(rede, 1e-4, 0.75, 0.05))} | saídas {formas} | {time.time() - t0:.1f}s", flush=True)


def main():
    kaggle = os.path.isdir("/kaggle/working")
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--saida", default="/kaggle/working" if kaggle else os.path.abspath("nutrelle_saida"))
    ap.add_argument("--cache", default="/kaggle/temp/nutrelle_cache" if kaggle else os.path.join(tempfile.gettempdir(), "nutrelle_cache"))
    ap.add_argument("--listas", default=os.path.join(AQUI, "listas"))
    ap.add_argument("--orcamento-h", type=float, default=11.0, help="horas de relógio desde o início; o que não couber fica 'sem_orcamento'")
    ap.add_argument("--so", action="append", default=[], help="só configurações cujo nome contém o texto (pode repetir)")
    ap.add_argument("--retoma", action="append", default=[], help="pasta com a saída de uma execução anterior (pode repetir)")
    ap.add_argument("--rodada", choices=sorted(RODADAS), default="2", help="quais configurações: 1 = as 15 da seção 10; 2 = as 11 da nova execução (seção 11)")
    ap.add_argument("--guarda-pesos", choices=["melhor", "todos", "nenhum"], default="todos",
                    help="pesos finais dos ajustes finos: todos (padrão; é o que serve), só o melhor pela CV, ou nenhum")
    ap.add_argument("--sem-servico", action="store_true", help="não mede latência nem grava servico.json")
    ap.add_argument("--smoke", type=int, nargs="?", const=8, default=0, help="teste rápido: N pratos por fold (padrão 8), backbone minúsculo, CPU")
    ap.add_argument("--workers", type=int, default=1, help="processos em paralelo quando não há GPU (--smoke/--cpu)")
    ap.add_argument("--cpu", action="store_true", help="ignora a GPU")
    ap.add_argument("--lista", action="store_true"); ap.add_argument("--checa-modelos", action="store_true"); ap.add_argument("--autoteste", action="store_true")
    ap.add_argument("--job"); ap.add_argument("--dispositivo", default="cpu"); ap.add_argument("--prazo", type=float); ap.add_argument("--div-lote", type=int, default=1)
    a = ap.parse_args()
    a.saida, a.cache, a.listas = (os.path.abspath(p) for p in (a.saida, a.cache, a.listas))
    if a.autoteste:
        return autoteste()
    if a.checa_modelos:
        return checa_modelos()
    if a.job:       # processo filho: procura a configuração entre todas as declaradas
        return principal_parte(a, CONFIGS_SMOKE if a.smoke else CONFIGS)
    cfgs = configs_da_rodada(a.rodada, a.smoke)
    if a.so:
        cfgs = [c for c in cfgs if any(t in c["nome"] for t in a.so)]
    if a.lista:
        for c in cfgs:
            print(f"{c['nome']:46s} {c['tipo']:13s} {BACKBONES[c['bb']]['timm']:44s} {BACKBONES[c['bb']]['hw']} {c['est_min']:4d} min de GPU na execução 1 "
                  + (f"| lr {c['lr']:g} lld {c['lld']} épocas {c['epocas']} lote {c['bs']}×{c['acum']} escala {c['escala']}" if c["tipo"] != "ridge" else ""))
        return print(f"rodada {a.rodada}: {len(cfgs)} configurações, {sum(c['est_min'] for c in cfgs) / 60:.1f} h de GPU medidas na execução 1 "
                     "(2 × T4: perto de metade disso em relógio); a execução 2 soma o teste2 (+39% na extração de features) e a fase de serviço")
    orquestra(a, cfgs)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile /tmp/nutrelle_codigo/listas/treino.csv
dish_id,cadeia,fold,split_oficial,massa,proteina,kcal,gordura,carbo,m_proteina_animal,m_leguminosa,m_laticinio,m_amido,m_vegetal,m_gordura_molho,m_misto,m_outros,p_proteina_animal,p_leguminosa,p_laticinio,p_amido,p_vegetal,p_gordura_molho,p_misto,p_outros
dish_1556572657,167,3,train,36.0,0.288,41.4,3.852,2.268,0.0,0.0,0.0,0.0,36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.288,0.0,0.0,0.0
dish_1556573514,168,2,train,23.0,0.506,6.44,0.092,1.219,0.0,0.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.506,0.0,0.0,0.0
dish_1556575014,169,1,train,62.0,0.496,71.3,6.634,3.906,0.0,0.0,0.0,0.0,62.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.496,0.0,0.0,0.0
dish_1556575083,169,1,train,64.0,2.176,27.52,0.192,5.76,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.176,0.0,0.0,0.0
dish_1556575124,169,1,train,28.0,0.196,4.48,0.056,0.952,0.0,0.0,0.0,0.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.196,0.0,0.0,0.0
dish_1556575273,170,0,train,152.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1556575327,170,0,test,152.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1556575386,170,0,train,152.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1556575446,170,0,train,152.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1556575499,170,0,train,152.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1556575558,170,0,train,151.0,2.868,103.3,6.882,10.618,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.868,0.0,0.0,0.0
dish_1557863012,177,4,train,31.0,0.31,3.1,0.0,0.62,0.0,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.31,0.0,0.0,0.0
dish_1557863056,177,4,train,47.0,0.94,75.2,7.05,4.23,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.94,0.0,0.0,0.0
dish_1557863104,177,4,train,40.0,0.32,46.0,4.28,2.52,0.0,0.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.32,0.0,0.0,0.0
dish_1557863170,177,4,train,142.0,2.266,129.82,11.426,8.234,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.266,0.0,0.0,0.0
dish_1557937037,179,0,test,49.0,0.588,25.97,0.343,5.88,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.588,0.0,0.0,0.0
dish_1557937079,179,0,test,27.0,9.99,146.07,11.34,0.378,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1557937157,179,0,test,271.0,24.342,355.25,22.123,14.7,123.0,0.0,0.0,0.0,148.0,0.0,0.0,0.0,21.99,0.0,0.0,0.0,2.352,0.0,0.0,0.0
dish_1557937335,180,1,test,38.0,0.608,47.158,0.076,11.172,0.0,0.0,0.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.608,0.0,0.0,0.0,0.0
dish_1557937615,181,4,test,21.0,0.147,6.93,0.063,1.68,0.0,0.0,0.0,0.0,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.147,0.0,0.0,0.0
dish_1557937715,181,4,test,39.0,0.468,20.67,0.273,4.68,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.468,0.0,0.0,0.0
dish_1557937758,181,4,test,128.0,3.53,106.783,2.262,18.932,17.0,0.0,0.0,38.0,73.0,0.0,0.0,0.0,2.125,0.0,0.0,0.608,0.797,0.0,0.0,0.0
dish_1558026623,182,3,train,83.0,0.415,29.88,0.083,7.47,0.0,0.0,0.0,0.0,83.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.415,0.0,0.0,0.0
dish_1558026684,182,3,train,71.0,0.568,24.14,0.142,5.68,0.0,0.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.568,0.0,0.0,0.0
dish_1558026714,182,3,train,24.0,8.88,129.84,10.08,0.336,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.88,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558026756,182,3,train,178.0,9.863,183.86,10.305,13.486,24.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,8.88,0.0,0.0,0.0,0.983,0.0,0.0,0.0
dish_1558027057,183,1,train,65.0,1.04,80.665,0.13,19.11,0.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.04,0.0,0.0,0.0,0.0
dish_1558027164,183,1,train,30.0,11.1,162.3,12.6,0.42,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.1,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558028007,184,0,train,71.0,1.42,54.67,0.071,12.07,0.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.42,0.0,0.0,0.0,0.0
dish_1558028057,184,0,train,28.0,10.36,151.48,11.76,0.392,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558028492,185,4,train,61.0,0.488,20.74,0.122,4.88,0.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.488,0.0,0.0,0.0
dish_1558028690,185,4,train,66.0,0.33,23.76,0.066,5.94,0.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.33,0.0,0.0,0.0
dish_1558029148,186,2,train,108.0,0.54,38.88,0.108,9.72,0.0,0.0,0.0,0.0,108.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.54,0.0,0.0,0.0
dish_1558029230,186,2,train,115.0,2.3,88.55,0.115,19.55,0.0,0.0,0.0,115.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.3,0.0,0.0,0.0,0.0
dish_1558029278,186,2,train,27.0,9.99,146.07,11.34,0.378,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558029324,186,2,train,250.0,12.83,273.5,11.563,29.648,27.0,0.0,0.0,115.0,108.0,0.0,0.0,0.0,9.99,0.0,0.0,2.3,0.54,0.0,0.0,0.0
dish_1558029518,187,0,train,111.0,1.776,137.751,0.222,32.634,0.0,0.0,0.0,111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.776,0.0,0.0,0.0,0.0
dish_1558029686,188,4,train,22.0,8.14,119.02,9.24,0.308,22.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.14,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558029842,189,3,train,67.0,0.536,22.78,0.134,5.36,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.536,0.0,0.0,0.0
dish_1558029923,189,3,train,200.0,10.452,279.551,9.596,38.302,22.0,0.0,0.0,111.0,67.0,0.0,0.0,0.0,8.14,0.0,0.0,1.776,0.536,0.0,0.0,0.0
dish_1558030182,190,1,test,88.0,1.76,67.76,0.088,14.96,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.76,0.0,0.0,0.0,0.0
dish_1558030289,190,1,test,107.0,0.535,38.52,0.107,9.63,0.0,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.535,0.0,0.0,0.0
dish_1558030383,190,1,test,26.0,9.62,140.66,10.92,0.364,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558030435,190,1,test,221.0,11.915,246.94,11.115,24.954,26.0,0.0,0.0,88.0,107.0,0.0,0.0,0.0,9.62,0.0,0.0,1.76,0.535,0.0,0.0,0.0
dish_1558030832,191,4,train,72.0,1.152,89.352,0.144,21.168,0.0,0.0,0.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.152,0.0,0.0,0.0,0.0
dish_1558030971,191,4,train,49.0,0.245,17.64,0.049,4.41,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.245,0.0,0.0,0.0
dish_1558031133,191,4,test,110.0,1.76,136.51,0.22,32.34,0.0,0.0,0.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.76,0.0,0.0,0.0,0.0
dish_1558031237,191,4,test,93.0,0.744,31.62,0.186,7.44,0.0,0.0,0.0,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.744,0.0,0.0,0.0
dish_1558031302,191,4,test,74.0,1.48,56.98,0.074,12.58,0.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.48,0.0,0.0,0.0,0.0
dish_1558032156,193,3,train,26.0,9.62,140.66,10.92,0.364,26.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.62,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558109864,196,3,train,37.0,8.288,213.897,18.315,8.399,0.0,0.0,0.0,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.288
dish_1558109907,196,3,train,63.0,0.441,43.47,0.126,11.34,0.0,0.0,0.0,0.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.441,0.0,0.0,0.0
dish_1558109986,196,3,train,75.0,14.475,258.45,20.475,3.15,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.475,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558110245,197,3,train,51.0,0.357,29.07,0.153,7.14,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.357,0.0,0.0,0.0
dish_1558110297,197,3,test,51.0,0.357,29.07,0.153,7.14,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.357,0.0,0.0,0.0
dish_1558110385,197,3,test,34.0,4.25,52.87,3.57,0.408,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.25,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558110436,197,3,test,69.0,1.104,85.629,0.138,20.286,0.0,0.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.104,0.0,0.0,0.0,0.0
dish_1558110478,197,3,test,161.0,8.301,205.439,6.801,27.932,41.0,0.0,0.0,69.0,51.0,0.0,0.0,0.0,6.84,0.0,0.0,1.104,0.357,0.0,0.0,0.0
dish_1558113154,198,0,train,54.0,0.378,37.26,0.108,9.72,0.0,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.378,0.0,0.0,0.0
dish_1558113204,198,0,train,31.0,6.944,179.211,15.345,7.037,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.944
dish_1558113255,198,0,train,90.0,0.72,30.6,0.18,7.2,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.72,0.0,0.0,0.0
dish_1558113304,198,0,train,175.0,8.042,247.071,15.633,23.957,0.0,0.0,0.0,0.0,144.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,1.098,0.0,0.0,6.944
dish_1558113430,199,2,test,28.0,0.616,7.84,0.112,1.484,0.0,0.0,0.0,0.0,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.616,0.0,0.0,0.0
dish_1558113477,199,2,test,50.0,11.2,289.05,24.75,11.35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.2
dish_1558113941,200,2,test,78.0,11.816,296.89,24.862,12.834,0.0,0.0,0.0,0.0,28.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.616,0.0,0.0,11.2
dish_1558114145,201,1,test,83.0,0.581,57.27,0.166,14.94,0.0,0.0,0.0,0.0,83.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.581,0.0,0.0,0.0
dish_1558114209,201,1,test,90.0,0.72,30.6,0.18,7.2,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.72,0.0,0.0,0.0
dish_1558114937,203,0,train,49.0,1.078,13.72,0.196,2.597,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.078,0.0,0.0,0.0
dish_1558115011,203,0,train,20.0,4.48,115.62,9.9,4.54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.48
dish_1558115764,205,4,train,28.0,10.36,151.48,11.76,0.392,28.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558115825,205,4,train,62.0,0.434,42.78,0.124,11.16,0.0,0.0,0.0,0.0,62.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.434,0.0,0.0,0.0
dish_1558115867,205,4,train,23.0,5.152,132.963,11.385,5.221,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.152
dish_1558115915,205,4,train,59.0,11.387,203.314,16.107,2.478,59.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.387,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558115960,205,4,train,51.0,1.02,39.27,0.051,8.67,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.02,0.0,0.0,0.0,0.0
dish_1558116001,205,4,train,223.0,28.353,569.807,39.427,27.921,87.0,0.0,0.0,51.0,62.0,0.0,0.0,23.0,21.747,0.0,0.0,1.02,0.434,0.0,0.0,5.152
dish_1558116090,205,4,train,91.0,0.728,30.94,0.182,7.28,0.0,0.0,0.0,0.0,91.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.728,0.0,0.0,0.0
dish_1558116146,205,4,train,111.0,0.777,76.59,0.222,19.98,0.0,0.0,0.0,0.0,111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.777,0.0,0.0,0.0
dish_1558116201,205,4,train,91.0,17.563,313.586,24.843,3.822,91.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.563,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558116260,205,4,train,75.0,16.8,433.575,37.125,17.025,0.0,0.0,0.0,0.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.8
dish_1558116298,205,4,train,368.0,35.868,854.691,62.372,48.107,91.0,0.0,0.0,0.0,202.0,0.0,0.0,75.0,17.563,0.0,0.0,0.0,1.505,0.0,0.0,16.8
dish_1558116399,205,4,train,25.0,0.55,7.0,0.1,1.325,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.55,0.0,0.0,0.0
dish_1558116486,205,4,train,57.0,0.399,39.33,0.114,10.26,0.0,0.0,0.0,0.0,57.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.399,0.0,0.0,0.0
dish_1558122583,206,1,train,61.0,1.22,46.97,0.061,10.37,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.22,0.0,0.0,0.0,0.0
dish_1558372525,208,0,train,42.0,9.408,242.802,20.79,9.534,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.408
dish_1558372650,209,4,train,83.0,0.581,57.27,0.166,14.94,0.0,0.0,0.0,0.0,83.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.581,0.0,0.0,0.0
dish_1558372695,209,4,train,43.0,8.299,148.178,11.739,1.806,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.299,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558373639,212,3,train,20.0,0.44,5.6,0.08,1.06,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.44,0.0,0.0,0.0
dish_1558373687,212,3,train,89.0,17.177,306.694,24.297,3.738,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.177,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558377105,216,4,train,113.0,21.809,389.398,30.849,4.746,113.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.809,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558379128,218,2,train,53.0,0.371,36.57,0.106,9.54,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.371,0.0,0.0,0.0
dish_1558379182,218,2,train,56.0,12.544,323.736,27.72,12.712,0.0,0.0,0.0,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.544
dish_1558379273,218,2,train,66.0,12.738,227.436,18.018,2.772,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.738,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558379324,218,2,train,48.0,0.384,16.32,0.096,3.84,0.0,0.0,0.0,0.0,48.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.384,0.0,0.0,0.0
dish_1558379391,218,2,train,223.0,26.037,604.062,45.94,28.864,66.0,0.0,0.0,0.0,101.0,0.0,0.0,56.0,12.738,0.0,0.0,0.0,0.755,0.0,0.0,12.544
dish_1558380014,221,2,train,134.0,0.938,92.46,0.268,24.12,0.0,0.0,0.0,0.0,134.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.938,0.0,0.0,0.0
dish_1558380058,221,2,train,36.0,8.064,208.116,17.82,8.172,0.0,0.0,0.0,0.0,0.0,0.0,0.0,36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.064
dish_1558380804,223,3,train,27.0,0.594,7.56,0.108,1.431,0.0,0.0,0.0,0.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.594,0.0,0.0,0.0
dish_1558381032,224,4,train,20.0,4.48,115.62,9.9,4.54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.48
dish_1558381084,224,4,train,53.0,0.371,36.57,0.106,9.54,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.371,0.0,0.0,0.0
dish_1558381143,224,4,train,53.0,10.229,182.638,14.469,2.226,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.229,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558381206,224,4,train,153.0,15.674,342.388,24.583,17.737,53.0,0.0,0.0,0.0,80.0,0.0,0.0,20.0,10.229,0.0,0.0,0.0,0.965,0.0,0.0,4.48
dish_1558381665,225,1,train,51.0,1.224,17.85,0.204,3.57,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.224,0.0,0.0,0.0
dish_1558381708,225,1,train,53.0,0.954,16.43,0.053,3.71,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.954,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558381742,225,1,train,68.0,0.544,78.2,7.276,4.284,0.0,0.0,0.0,0.0,68.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.544,0.0,0.0,0.0
dish_1558381819,225,1,train,65.0,0.455,22.1,0.455,4.42,0.0,0.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.455,0.0,0.0,0.0
dish_1558381840,225,1,train,69.0,21.39,113.85,2.484,0.0,69.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.39,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558381860,225,1,train,59.0,1.18,94.4,8.85,5.31,0.0,0.0,0.0,0.0,59.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.18,0.0,0.0,0.0
dish_1558381943,225,1,train,60.0,1.74,13.8,0.24,2.16,0.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.74,0.0,0.0,0.0
dish_1558382263,226,2,train,425.0,27.487,356.63,19.562,23.454,69.0,53.0,0.0,0.0,303.0,0.0,0.0,0.0,21.39,0.954,0.0,0.0,5.143,0.0,0.0,0.0
dish_1558458737,229,1,train,32.0,0.864,41.6,0.096,8.96,0.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.864,0.0,0.0,0.0,0.0
dish_1558458847,229,1,train,143.0,0.429,74.36,0.286,20.02,0.0,0.0,0.0,0.0,143.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.429,0.0,0.0,0.0
dish_1558458903,229,1,train,20.0,4.48,115.62,9.9,4.54,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.48
dish_1558459020,229,1,train,61.0,0.549,10.98,0.122,2.379,0.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.549,0.0,0.0,0.0
dish_1558459059,229,1,train,42.0,0.294,28.98,0.084,7.56,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.294,0.0,0.0,0.0
dish_1558459115,229,1,train,298.0,6.616,271.54,10.488,43.459,0.0,0.0,0.0,32.0,246.0,0.0,0.0,20.0,0.0,0.0,0.0,0.864,1.272,0.0,0.0,4.48
dish_1558459945,231,3,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558459994,231,3,train,37.0,0.259,25.53,0.074,6.66,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.259,0.0,0.0,0.0
dish_1558460061,231,3,train,59.0,0.413,40.71,0.118,10.62,0.0,0.0,0.0,0.0,59.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.413,0.0,0.0,0.0
dish_1558460138,231,3,train,56.0,0.504,10.08,0.112,2.184,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.504,0.0,0.0,0.0
dish_1558460908,232,1,train,302.0,1.834,152.0,0.62,39.632,0.0,0.0,0.0,0.0,302.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.834,0.0,0.0,0.0
dish_1558461607,234,2,test,127.0,0.889,87.63,0.254,22.86,0.0,0.0,0.0,0.0,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.889,0.0,0.0,0.0
dish_1558461692,234,2,test,40.0,8.96,231.24,19.8,9.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.96
dish_1558461760,234,2,test,61.0,0.549,10.98,0.122,2.379,0.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.549,0.0,0.0,0.0
dish_1558461792,234,2,test,233.0,10.543,331.0,20.196,34.499,0.0,0.0,0.0,0.0,193.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,1.583,0.0,0.0,8.96
dish_1558462106,235,0,train,37.0,0.259,25.53,0.074,6.66,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.259,0.0,0.0,0.0
dish_1558462531,237,4,train,72.0,0.648,12.96,0.144,2.808,0.0,0.0,0.0,0.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.648,0.0,0.0,0.0
dish_1558463113,238,3,train,139.0,1.743,63.58,0.321,14.696,0.0,0.0,0.0,17.0,122.0,0.0,0.0,0.0,0.0,0.0,0.0,0.459,1.284,0.0,0.0,0.0
dish_1558463362,239,1,train,116.0,1.044,20.88,0.232,4.524,0.0,0.0,0.0,0.0,116.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.044,0.0,0.0,0.0
dish_1558463429,239,1,train,61.0,0.427,42.09,0.122,10.98,0.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.427,0.0,0.0,0.0
dish_1558463562,240,2,train,64.0,14.336,369.984,31.68,14.528,0.0,0.0,0.0,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.336
dish_1558463690,241,1,train,241.0,15.807,432.954,32.034,30.032,0.0,0.0,0.0,0.0,177.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,1.471,0.0,0.0,14.336
dish_1558468285,242,4,train,44.0,0.968,12.32,0.176,2.332,0.0,0.0,0.0,0.0,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.968,0.0,0.0,0.0
dish_1558468401,242,4,train,47.0,0.94,75.2,7.05,4.23,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.94,0.0,0.0,0.0
dish_1558468435,242,4,train,80.0,1.92,28.0,0.32,5.6,0.0,0.0,0.0,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.92,0.0,0.0,0.0
dish_1558468494,242,4,train,77.0,0.693,15.785,0.231,2.618,0.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.693,0.0,0.0,0.0
dish_1558468547,242,4,train,34.0,0.306,6.97,0.102,1.156,0.0,0.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.306,0.0,0.0,0.0
dish_1558468589,242,4,train,55.0,1.705,12.1,0.165,1.815,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.705,0.0,0.0,0.0
dish_1558468632,242,4,train,103.0,31.93,169.95,3.708,0.0,103.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,31.93,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558468731,242,4,train,440.0,38.462,320.325,11.752,17.751,103.0,0.0,0.0,0.0,337.0,0.0,0.0,0.0,31.93,0.0,0.0,0.0,6.532,0.0,0.0,0.0
dish_1558544427,243,0,train,103.0,12.875,160.165,10.815,1.236,103.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.875,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558544482,243,0,train,67.0,0.469,38.19,0.201,9.38,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.469,0.0,0.0,0.0
dish_1558544663,244,0,test,208.0,4.992,141.44,2.912,24.96,0.0,0.0,0.0,208.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.992,0.0,0.0,0.0,0.0
dish_1558544870,245,4,train,170.0,13.344,198.355,11.016,10.616,103.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,12.875,0.0,0.0,0.0,0.469,0.0,0.0,0.0
dish_1558545738,246,3,test,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558546918,249,1,train,54.0,1.458,70.2,0.162,15.12,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.458,0.0,0.0,0.0,0.0
dish_1558547008,249,1,train,96.0,1.92,73.92,0.096,16.32,0.0,0.0,0.0,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.92,0.0,0.0,0.0,0.0
dish_1558547124,249,1,train,25.0,5.6,144.525,12.375,5.675,0.0,0.0,0.0,0.0,0.0,0.0,0.0,25.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.6
dish_1558547205,249,1,train,186.0,9.297,291.175,12.677,37.511,0.0,0.0,0.0,150.0,11.0,0.0,0.0,25.0,0.0,0.0,0.0,3.378,0.319,0.0,0.0,5.6
dish_1558548666,250,2,train,75.0,2.025,97.5,0.225,21.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.025,0.0,0.0,0.0,0.0
dish_1558549008,251,0,train,115.0,1.035,20.7,0.23,4.485,0.0,0.0,0.0,0.0,115.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.035,0.0,0.0,0.0
dish_1558549086,251,0,train,95.0,1.9,73.15,0.095,16.15,0.0,0.0,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.9,0.0,0.0,0.0,0.0
dish_1558549150,251,0,train,66.0,0.462,45.54,0.132,11.88,0.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1558549214,251,0,train,361.0,5.712,239.19,0.722,53.875,0.0,0.0,0.0,170.0,191.0,0.0,0.0,0.0,0.0,0.0,0.0,3.925,1.787,0.0,0.0,0.0
dish_1558549605,253,1,test,75.0,2.025,97.5,0.225,21.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.025,0.0,0.0,0.0,0.0
dish_1558549806,254,0,test,203.0,28.747,781.958,58.675,48.146,0.0,0.0,0.0,75.0,10.0,0.0,0.0,118.0,0.0,0.0,0.0,2.025,0.29,0.0,0.0,26.432
dish_1558628515,257,4,train,56.0,0.392,31.92,0.168,7.84,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.392,0.0,0.0,0.0
dish_1558628711,258,4,train,129.0,1.677,129.064,0.129,30.573,0.0,0.0,0.0,129.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.677,0.0,0.0,0.0,0.0
dish_1558628760,258,4,train,198.0,6.879,231.314,5.757,38.595,13.0,0.0,0.0,129.0,56.0,0.0,0.0,0.0,4.81,0.0,0.0,1.677,0.392,0.0,0.0,0.0
dish_1558628888,259,4,train,72.0,9.0,111.96,7.56,0.864,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1558628921,259,4,train,60.0,0.54,24.6,0.12,6.0,0.0,0.0,0.0,0.0,60.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.54,0.0,0.0,0.0
dish_1558629013,259,4,train,80.0,0.56,26.4,0.24,6.4,0.0,0.0,0.0,0.0,80.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.56,0.0,0.0,0.0
dish_1558629041,259,4,train,226.0,10.954,172.004,8.396,13.712,72.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,9.0,0.0,0.0,0.0,1.954,0.0,0.0,0.0
dish_1558629398,260,4,train,64.0,0.448,21.76,0.448,4.352,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.448,0.0,0.0,0.0
dish_1558629444,260,4,train,39.0,8.736,225.459,19.305,8.853,0.0,0.0,0.0,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.736
dish_1558629488,260,4,train,64.0,1.152,16.0,0.192,3.2,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.152,0.0,0.0,0.0
dish_1558629517,260,4,train,167.0,10.336,263.219,19.945,16.405,0.0,0.0,0.0,0.0,128.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,1.6,0.0,0.0,8.736
dish_1558629578,260,4,train,90.0,1.62,22.5,0.27,4.5,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.62,0.0,0.0,0.0
dish_1558629677,260,4,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558629715,260,4,train,247.0,2.376,100.54,0.614,25.175,0.0,0.0,0.0,0.0,247.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.376,0.0,0.0,0.0
dish_1558629818,260,4,train,27.0,6.048,156.087,13.365,6.129,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.048
dish_1558629878,260,4,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558630450,262,2,train,125.0,0.875,86.25,0.25,22.5,0.0,0.0,0.0,0.0,125.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.875,0.0,0.0,0.0
dish_1558630488,262,2,train,88.0,0.704,29.92,0.176,7.04,0.0,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.704,0.0,0.0,0.0
dish_1558630600,262,2,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558630649,262,2,train,355.0,2.005,190.01,0.71,49.42,0.0,0.0,0.0,0.0,355.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.005,0.0,0.0,0.0
dish_1558630737,262,2,train,89.0,1.602,22.25,0.267,4.45,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.602,0.0,0.0,0.0
dish_1558630814,262,2,train,34.0,7.616,196.554,16.83,7.718,0.0,0.0,0.0,0.0,0.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.616
dish_1558630867,262,2,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558630932,262,2,train,265.0,9.644,292.644,17.381,32.048,0.0,0.0,0.0,0.0,231.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0,2.028,0.0,0.0,7.616
dish_1558631825,263,3,train,112.0,3.472,118.832,2.688,24.64,0.0,0.0,0.0,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.472,0.0,0.0,0.0,0.0
dish_1558631872,263,3,train,89.0,0.801,16.02,0.178,3.471,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.801,0.0,0.0,0.0
dish_1558631904,263,3,train,154.0,0.462,80.08,0.308,21.56,0.0,0.0,0.0,0.0,154.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1558631932,263,3,train,355.0,4.735,214.932,3.174,49.671,0.0,0.0,0.0,112.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,3.472,1.263,0.0,0.0,0.0
dish_1558632445,265,3,train,91.0,20.384,526.071,45.045,20.657,0.0,0.0,0.0,0.0,0.0,0.0,0.0,91.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,20.384
dish_1558632475,265,3,train,112.0,3.472,118.832,2.688,24.64,0.0,0.0,0.0,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.472,0.0,0.0,0.0,0.0
dish_1558632921,267,2,train,155.0,0.465,80.6,0.31,21.7,0.0,0.0,0.0,0.0,155.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.465,0.0,0.0,0.0
dish_1558632959,267,2,train,89.0,0.801,16.02,0.178,3.471,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.801,0.0,0.0,0.0
dish_1558632990,267,2,train,101.0,1.818,25.25,0.303,5.05,0.0,0.0,0.0,0.0,101.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.818,0.0,0.0,0.0
dish_1558633158,268,2,train,85.0,0.765,34.85,0.17,8.5,0.0,0.0,0.0,0.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.765,0.0,0.0,0.0
dish_1558633221,268,2,train,449.0,4.438,176.879,1.417,42.901,0.0,0.0,0.0,19.0,430.0,0.0,0.0,0.0,0.0,0.0,0.0,0.589,3.849,0.0,0.0,0.0
dish_1558635400,269,0,test,100.0,1.8,25.0,0.3,5.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.8,0.0,0.0,0.0
dish_1558635489,269,0,test,74.0,2.294,78.514,1.776,16.28,0.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.294,0.0,0.0,0.0,0.0
dish_1558635523,269,0,test,43.0,1.161,55.9,0.129,12.04,0.0,0.0,0.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.161,0.0,0.0,0.0,0.0
dish_1558635546,269,0,test,117.0,0.351,60.84,0.234,16.38,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.351,0.0,0.0,0.0
dish_1558635618,269,0,test,334.0,5.606,220.254,2.439,49.7,0.0,0.0,0.0,117.0,217.0,0.0,0.0,0.0,0.0,0.0,0.0,3.455,2.151,0.0,0.0,0.0
dish_1558635933,270,1,train,98.0,0.784,33.32,0.196,7.84,0.0,0.0,0.0,0.0,98.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.784,0.0,0.0,0.0
dish_1558635994,270,1,train,136.0,0.952,93.84,0.272,24.48,0.0,0.0,0.0,0.0,136.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.952,0.0,0.0,0.0
dish_1558636028,270,1,train,262.0,4.341,199.348,5.379,38.543,0.0,0.0,0.0,19.0,234.0,0.0,0.0,9.0,0.0,0.0,0.0,0.589,1.736,0.0,0.0,2.016
dish_1558636763,272,1,train,117.0,0.351,60.84,0.234,16.38,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.351,0.0,0.0,0.0
dish_1558636813,272,1,train,88.0,0.616,60.72,0.176,15.84,0.0,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.616,0.0,0.0,0.0
dish_1558636902,272,1,train,95.0,0.855,38.95,0.19,9.5,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.855,0.0,0.0,0.0
dish_1558636928,272,1,train,89.0,0.801,16.02,0.178,3.471,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.801,0.0,0.0,0.0
dish_1558636980,272,1,train,397.0,4.415,222.778,4.738,47.007,0.0,0.0,0.0,0.0,389.0,0.0,0.0,8.0,0.0,0.0,0.0,0.0,2.623,0.0,0.0,1.792
dish_1558638137,275,2,train,117.0,0.351,60.84,0.234,16.38,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.351,0.0,0.0,0.0
dish_1558638163,275,2,train,99.0,0.891,40.59,0.198,9.9,0.0,0.0,0.0,0.0,99.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.891,0.0,0.0,0.0
dish_1558638216,275,2,train,43.0,1.161,55.9,0.129,12.04,0.0,0.0,0.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.161,0.0,0.0,0.0,0.0
dish_1558638292,275,2,train,356.0,3.179,190.31,0.755,46.08,0.0,0.0,0.0,43.0,313.0,0.0,0.0,0.0,0.0,0.0,0.0,1.161,2.018,0.0,0.0,0.0
dish_1558717604,279,3,test,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558717886,280,1,train,142.0,0.426,73.84,0.284,19.88,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.426,0.0,0.0,0.0
dish_1558718009,280,1,train,76.0,0.608,25.84,0.152,6.08,0.0,0.0,0.0,0.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.608,0.0,0.0,0.0
dish_1558718051,280,1,train,97.0,1.746,24.25,0.291,4.85,0.0,0.0,0.0,0.0,97.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.746,0.0,0.0,0.0
dish_1558718088,280,1,train,405.0,5.57,219.42,2.887,50.61,0.0,0.0,0.0,90.0,315.0,0.0,0.0,0.0,0.0,0.0,0.0,2.79,2.78,0.0,0.0,0.0
dish_1558718587,281,0,test,41.0,0.738,10.25,0.123,2.05,0.0,0.0,0.0,0.0,41.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.738,0.0,0.0,0.0
dish_1558718610,281,0,test,107.0,3.317,113.527,2.568,23.54,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.317,0.0,0.0,0.0,0.0
dish_1558718649,281,0,test,76.0,0.608,25.84,0.152,6.08,0.0,0.0,0.0,0.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.608,0.0,0.0,0.0
dish_1558718680,281,0,test,23.0,5.152,132.963,11.385,5.221,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.152
dish_1558723414,286,0,train,49.0,1.519,51.989,1.176,10.78,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.519,0.0,0.0,0.0,0.0
dish_1558723449,286,0,train,143.0,0.429,74.36,0.286,20.02,0.0,0.0,0.0,0.0,143.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.429,0.0,0.0,0.0
dish_1558723475,286,0,train,67.0,0.536,22.78,0.134,5.36,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.536,0.0,0.0,0.0
dish_1558723512,286,0,train,39.0,0.351,15.99,0.078,3.9,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.351,0.0,0.0,0.0
dish_1558723537,286,0,train,58.0,1.044,14.5,0.174,2.9,0.0,0.0,0.0,0.0,58.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.044,0.0,0.0,0.0
dish_1558723568,286,0,train,356.0,3.879,179.619,1.848,42.96,0.0,0.0,0.0,49.0,307.0,0.0,0.0,0.0,0.0,0.0,0.0,1.519,2.36,0.0,0.0,0.0
dish_1558723639,286,0,train,41.0,0.738,10.25,0.123,2.05,0.0,0.0,0.0,0.0,41.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.738,0.0,0.0,0.0
dish_1558723671,286,0,train,93.0,0.837,38.13,0.186,9.3,0.0,0.0,0.0,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.837,0.0,0.0,0.0
dish_1558723695,286,0,train,90.0,2.79,95.49,2.16,19.8,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.79,0.0,0.0,0.0,0.0
dish_1558723719,286,0,train,40.0,8.96,231.24,19.8,9.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.96
dish_1558723742,286,0,train,264.0,13.325,375.11,22.269,40.23,0.0,0.0,0.0,90.0,134.0,0.0,0.0,40.0,0.0,0.0,0.0,2.79,1.575,0.0,0.0,8.96
dish_1558723777,286,0,train,143.0,0.429,74.36,0.286,20.02,0.0,0.0,0.0,0.0,143.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.429,0.0,0.0,0.0
dish_1558723818,286,0,train,107.0,3.317,113.527,2.568,23.54,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.317,0.0,0.0,0.0,0.0
dish_1558723847,286,0,train,96.0,21.504,554.976,47.52,21.792,0.0,0.0,0.0,0.0,0.0,0.0,0.0,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.504
dish_1558723866,286,0,train,59.0,1.062,14.75,0.177,2.95,0.0,0.0,0.0,0.0,59.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.062,0.0,0.0,0.0
dish_1558723894,286,0,train,405.0,26.312,757.613,50.551,68.302,0.0,0.0,0.0,107.0,202.0,0.0,0.0,96.0,0.0,0.0,0.0,3.317,1.491,0.0,0.0,21.504
dish_1558723926,286,0,train,107.0,0.963,19.26,0.214,4.173,0.0,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.963,0.0,0.0,0.0
dish_1558723960,286,0,train,52.0,0.936,13.0,0.156,2.6,0.0,0.0,0.0,0.0,52.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.936,0.0,0.0,0.0
dish_1558723982,286,0,train,51.0,11.424,294.831,25.245,11.577,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.424
dish_1558724005,286,0,train,93.0,0.744,31.62,0.186,7.44,0.0,0.0,0.0,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.744,0.0,0.0,0.0
dish_1558724031,286,0,train,303.0,14.067,358.711,25.801,25.79,0.0,0.0,0.0,0.0,252.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,2.643,0.0,0.0,11.424
dish_1558725576,288,0,train,79.0,1.422,19.75,0.237,3.95,0.0,0.0,0.0,0.0,79.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.422,0.0,0.0,0.0
dish_1558725621,288,0,train,223.0,15.818,349.22,25.269,23.76,0.0,0.0,0.0,0.0,173.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,4.618,0.0,0.0,11.2
dish_1558725702,288,0,train,117.0,3.978,50.31,0.351,10.53,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.978,0.0,0.0,0.0
dish_1558725734,288,0,train,75.0,0.675,30.75,0.15,7.5,0.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.675,0.0,0.0,0.0
dish_1558725760,288,0,train,161.0,0.483,83.72,0.322,22.54,0.0,0.0,0.0,0.0,161.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.483,0.0,0.0,0.0
dish_1558725786,288,0,train,107.0,3.317,113.527,2.568,23.54,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.317,0.0,0.0,0.0,0.0
dish_1558725829,288,0,train,460.0,8.453,278.307,3.391,64.11,0.0,0.0,0.0,107.0,353.0,0.0,0.0,0.0,0.0,0.0,0.0,3.317,5.136,0.0,0.0,0.0
dish_1559059924,289,4,train,20.0,7.4,108.2,8.4,0.28,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559059954,289,4,train,20.0,7.4,108.2,8.4,0.28,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559059992,289,4,train,94.0,0.94,71.44,0.0,16.92,0.0,0.0,0.0,94.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.94,0.0,0.0,0.0,0.0
dish_1559060055,289,4,train,81.0,0.567,26.73,0.243,6.48,0.0,0.0,0.0,0.0,81.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.567,0.0,0.0,0.0
dish_1559060106,289,4,train,195.0,8.907,206.37,8.643,23.68,20.0,0.0,0.0,94.0,81.0,0.0,0.0,0.0,7.4,0.0,0.0,0.94,0.567,0.0,0.0,0.0
dish_1559061675,290,1,train,83.0,0.498,24.9,0.166,6.308,0.0,0.0,0.0,0.0,83.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.498,0.0,0.0,0.0
dish_1559061752,290,1,train,124.0,15.5,192.82,13.02,1.488,124.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,15.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559062005,291,0,train,30.0,8.73,113.91,8.43,0.93,30.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.73,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559062051,291,0,train,74.0,0.74,56.24,0.0,13.32,0.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.74,0.0,0.0,0.0,0.0
dish_1559062082,291,0,train,311.0,25.468,387.87,21.616,22.046,154.0,0.0,0.0,74.0,83.0,0.0,0.0,0.0,24.23,0.0,0.0,0.74,0.498,0.0,0.0,0.0
dish_1559157454,293,2,test,28.0,0.672,34.565,3.284,1.127,0.0,0.0,0.0,0.0,23.63,3.69,0.0,0.68,0.0,0.0,0.0,0.0,0.562,0.008,0.0,0.102
dish_1559157919,295,1,test,39.0,0.991,37.095,3.328,1.523,0.0,0.0,0.0,0.0,34.63,3.69,0.0,0.68,0.0,0.0,0.0,0.0,0.881,0.008,0.0,0.102
dish_1559172356,297,0,train,176.0,32.684,313.104,15.885,6.929,132.16,0.0,0.0,26.43,5.45,11.96,0.0,0.0,31.718,0.0,0.0,0.687,0.147,0.132,0.0,0.0
dish_1559232986,298,4,train,35.0,12.95,189.35,14.7,0.49,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.95,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559233055,298,4,train,39.0,2.379,25.194,1.326,1.248,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.379,0.0,0.0,0.0
dish_1559233102,298,4,train,124.0,12.4,183.52,13.64,1.984,124.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559233144,298,4,train,198.0,27.729,398.064,29.666,3.722,159.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,25.35,0.0,0.0,0.0,2.379,0.0,0.0,0.0
dish_1559233445,299,4,test,49.0,0.245,24.5,0.049,6.37,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.245,0.0,0.0,0.0
dish_1559233570,300,4,test,51.0,0.357,16.83,0.153,4.08,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.357,0.0,0.0,0.0
dish_1559233715,300,4,test,110.0,1.1,83.6,0.0,19.8,0.0,0.0,0.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.1,0.0,0.0,0.0,0.0
dish_1559233756,300,4,test,222.0,6.142,189.85,5.242,30.418,12.0,0.0,0.0,110.0,100.0,0.0,0.0,0.0,4.44,0.0,0.0,1.1,0.602,0.0,0.0,0.0
dish_1559238435,302,3,train,202.0,0.808,115.544,0.202,30.704,0.0,0.0,0.0,0.0,202.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.808,0.0,0.0,0.0
dish_1559238481,302,3,train,127.0,1.143,59.69,0.127,15.24,0.0,0.0,0.0,0.0,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.143,0.0,0.0,0.0
dish_1559238566,302,3,train,109.0,0.763,62.13,0.327,15.26,0.0,0.0,0.0,0.0,109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.763,0.0,0.0,0.0
dish_1559238608,302,3,train,438.0,2.714,237.364,0.656,61.204,0.0,0.0,0.0,0.0,438.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.714,0.0,0.0,0.0
dish_1559239151,304,3,test,40.0,1.16,9.2,0.16,1.44,0.0,0.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.16,0.0,0.0,0.0
dish_1559239256,304,3,test,128.0,1.152,60.16,0.128,15.36,0.0,0.0,0.0,0.0,128.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.152,0.0,0.0,0.0
dish_1559239369,304,3,test,276.0,3.068,130.92,0.612,31.92,0.0,0.0,0.0,0.0,276.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.068,0.0,0.0,0.0
dish_1559239753,305,3,train,54.0,0.48,26.546,0.639,5.327,0.0,0.0,0.0,0.0,53.23,0.77,0.0,0.0,0.0,0.0,0.0,0.0,0.479,0.001,0.0,0.0
dish_1559239912,306,2,train,128.0,1.152,60.16,0.128,15.36,0.0,0.0,0.0,0.0,128.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.152,0.0,0.0,0.0
dish_1559239985,306,2,train,203.0,0.812,116.116,0.203,30.856,0.0,0.0,0.0,0.0,203.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.812,0.0,0.0,0.0
dish_1559240089,306,2,train,385.0,2.444,202.822,0.97,51.543,0.0,0.0,0.0,0.0,384.23,0.77,0.0,0.0,0.0,0.0,0.0,0.0,2.443,0.001,0.0,0.0
dish_1559240256,307,2,train,54.0,1.296,18.9,0.216,3.78,0.0,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.296,0.0,0.0,0.0
dish_1559242568,311,1,train,203.0,0.812,116.116,0.203,30.856,0.0,0.0,0.0,0.0,203.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.812,0.0,0.0,0.0
dish_1559243139,312,0,train,53.0,0.471,26.054,0.627,5.228,0.0,0.0,0.0,0.0,52.25,0.75,0.0,0.0,0.0,0.0,0.0,0.0,0.47,0.001,0.0,0.0
dish_1559243704,313,4,train,203.0,0.812,116.116,0.203,30.856,0.0,0.0,0.0,0.0,203.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.812,0.0,0.0,0.0
dish_1559243887,314,0,train,50.0,11.2,289.05,24.75,11.35,0.0,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.2
dish_1559243974,314,0,train,306.0,12.483,431.22,25.58,47.434,0.0,0.0,0.0,0.0,255.25,0.75,0.0,50.0,0.0,0.0,0.0,0.0,1.282,0.001,0.0,11.2
dish_1559244305,315,4,train,64.0,3.483,113.066,7.045,9.833,0.0,32.06,0.0,0.0,11.22,20.72,0.0,0.0,0.0,2.885,0.0,0.0,0.301,0.297,0.0,0.0
dish_1559244417,315,4,train,59.0,1.416,72.833,6.921,2.376,0.0,0.0,0.0,0.0,49.8,7.78,0.0,1.42,0.0,0.0,0.0,0.0,1.184,0.017,0.0,0.215
dish_1559244546,316,1,train,281.0,75.775,469.942,13.549,6.331,241.67,0.0,0.0,24.17,9.36,5.8,0.0,0.0,74.917,0.0,0.0,0.628,0.229,0.0,0.0,0.0
dish_1559244598,316,1,train,51.0,2.601,250.92,12.75,33.15,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.601
dish_1559244649,316,1,train,455.0,83.275,906.761,40.265,51.689,241.67,32.06,0.0,24.17,70.38,34.3,0.0,52.42,74.917,2.885,0.0,0.628,1.714,0.314,0.0,2.816
dish_1559319826,319,0,train,79.0,0.553,45.03,0.237,11.06,0.0,0.0,0.0,0.0,79.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.553,0.0,0.0,0.0
dish_1559319971,319,0,train,96.0,0.96,72.96,0.0,17.28,0.0,0.0,0.0,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.96,0.0,0.0,0.0,0.0
dish_1559320019,319,0,train,205.0,6.408,196.82,5.714,30.732,13.0,0.0,0.0,96.0,96.0,0.0,0.0,0.0,4.81,0.0,0.0,0.96,0.638,0.0,0.0,0.0
dish_1559332795,321,3,train,39.0,0.531,39.368,3.721,1.442,0.0,0.0,0.0,0.0,33.2,5.8,0.0,0.0,0.0,0.0,0.0,0.0,0.529,0.002,0.0,0.0
dish_1559332822,321,3,train,408.0,51.691,540.587,28.622,15.933,203.49,0.0,0.0,40.7,139.6,24.21,0.0,0.0,48.838,0.0,0.0,1.058,1.589,0.206,0.0,0.0
dish_1559585533,324,3,train,37.0,0.888,12.95,0.148,2.59,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.888,0.0,0.0,0.0
dish_1559593646,326,2,train,37.0,1.123,55.386,4.106,4.059,0.0,0.0,0.0,5.52,21.14,9.42,0.0,0.92,0.0,0.0,0.0,0.243,0.626,0.08,0.0,0.175
dish_1559593722,326,2,train,106.0,2.779,140.563,12.2,6.837,0.0,0.0,0.0,5.52,79.38,18.52,0.0,2.58,0.0,0.0,0.0,0.243,2.01,0.1,0.0,0.426
dish_1559593746,326,2,train,186.0,3.499,156.963,12.44,9.557,0.0,0.0,0.0,5.52,159.38,18.52,0.0,2.58,0.0,0.0,0.0,0.243,2.73,0.1,0.0,0.426
dish_1559593794,326,2,train,380.0,40.379,450.561,25.55,14.105,149.23,0.0,0.0,20.44,187.92,19.08,0.0,3.33,35.666,0.0,0.0,0.899,3.286,0.1,0.0,0.427
dish_1559674074,329,2,train,53.0,5.83,140.98,5.141,17.49,0.0,0.0,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.83,0.0
dish_1559674689,330,4,train,177.0,0.885,63.72,0.177,15.93,0.0,0.0,0.0,0.0,177.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.885,0.0,0.0,0.0
dish_1559674781,330,4,train,203.0,1.067,81.66,0.229,20.61,0.0,0.0,0.0,0.0,203.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.067,0.0,0.0,0.0
dish_1559674866,330,4,train,303.0,4.267,125.66,2.329,24.91,0.0,0.0,0.0,0.0,303.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.267,0.0,0.0,0.0
dish_1559675101,331,3,train,31.0,1.054,13.33,0.093,2.79,0.0,0.0,0.0,0.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.054,0.0,0.0,0.0
dish_1559675146,331,3,train,45.0,1.18,19.07,0.121,4.19,0.0,0.0,0.0,0.0,45.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.18,0.0,0.0,0.0
dish_1559675178,331,3,train,100.0,1.675,28.97,0.231,6.335,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.675,0.0,0.0,0.0
dish_1559675234,331,3,train,120.0,2.075,36.97,0.271,8.135,0.0,0.0,0.0,0.0,120.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.075,0.0,0.0,0.0
dish_1559675268,331,3,train,144.0,2.315,39.37,0.271,8.615,0.0,0.0,0.0,0.0,144.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.315,0.0,0.0,0.0
dish_1559678083,332,1,train,66.0,0.999,75.812,7.073,2.976,0.0,0.0,0.0,0.0,55.43,10.57,0.0,0.0,0.0,0.0,0.0,0.0,0.999,0.0,0.0,0.0
dish_1559678104,332,1,train,135.0,2.655,99.962,7.349,7.806,0.0,0.0,0.0,0.0,124.43,10.57,0.0,0.0,0.0,0.0,0.0,0.0,2.655,0.0,0.0,0.0
dish_1559678127,332,1,train,170.0,3.005,108.642,7.419,9.871,0.0,0.0,0.0,0.0,159.43,10.57,0.0,0.0,0.0,0.0,0.0,0.0,3.005,0.0,0.0,0.0
dish_1559678146,332,1,train,284.0,26.294,348.577,21.193,13.254,87.05,0.0,0.0,11.61,173.94,11.4,0.0,0.0,22.633,0.0,0.0,0.406,3.254,0.0,0.0,0.0
dish_1559678330,333,0,train,150.0,3.199,169.694,16.187,5.27,0.0,0.0,0.0,0.0,105.39,44.61,0.0,0.0,0.0,0.0,0.0,0.0,2.594,0.605,0.0,0.0
dish_1559678367,333,0,train,215.0,4.759,192.444,16.447,9.82,0.0,0.0,0.0,0.0,170.39,44.61,0.0,0.0,0.0,0.0,0.0,0.0,4.154,0.605,0.0,0.0
dish_1559678405,333,0,train,386.0,39.692,552.346,37.108,14.895,130.58,0.0,0.0,17.41,192.15,45.86,0.0,0.0,33.95,0.0,0.0,0.609,4.528,0.605,0.0,0.0
dish_1559678457,333,0,train,465.0,43.01,624.236,37.503,29.984,130.58,0.0,0.0,96.41,192.15,45.86,0.0,0.0,33.95,0.0,0.0,3.927,4.528,0.605,0.0,0.0
dish_1559678629,334,2,train,80.0,9.622,89.377,8.446,6.144,49.19,0.0,0.0,9.84,8.36,12.61,0.0,0.0,9.051,0.0,0.0,0.266,0.168,0.137,0.0,0.0
dish_1559678651,334,2,train,122.0,10.966,107.857,9.328,7.95,49.19,0.0,0.0,9.84,50.36,12.61,0.0,0.0,9.051,0.0,0.0,0.266,1.512,0.137,0.0,0.0
dish_1559751710,337,4,train,66.0,0.462,37.62,0.198,9.24,0.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1559751744,337,4,train,188.0,15.712,227.33,13.008,10.704,122.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,15.25,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1559751775,337,4,train,254.0,16.768,309.236,13.14,30.108,122.0,0.0,0.0,66.0,66.0,0.0,0.0,0.0,15.25,0.0,0.0,1.056,0.462,0.0,0.0,0.0
dish_1559751801,337,4,train,343.0,29.228,451.636,22.04,32.778,211.0,0.0,0.0,66.0,66.0,0.0,0.0,0.0,27.71,0.0,0.0,1.056,0.462,0.0,0.0,0.0
dish_1559838279,338,1,train,114.0,3.702,103.7,3.678,14.952,8.0,0.0,0.0,0.0,106.0,0.0,0.0,0.0,2.96,0.0,0.0,0.0,0.742,0.0,0.0,0.0
dish_1559838402,339,0,train,233.0,5.249,222.76,3.797,43.155,8.0,0.0,0.0,119.0,106.0,0.0,0.0,0.0,2.96,0.0,0.0,1.547,0.742,0.0,0.0,0.0
dish_1559841023,340,2,train,46.0,6.44,73.6,4.6,1.38,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.44,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1559841061,340,2,train,99.0,7.182,96.39,4.865,6.256,46.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,6.44,0.0,0.0,0.0,0.742,0.0,0.0,0.0
dish_1559841091,340,2,train,166.0,8.053,163.423,4.932,22.135,46.0,0.0,0.0,67.0,53.0,0.0,0.0,0.0,6.44,0.0,0.0,0.871,0.742,0.0,0.0,0.0
dish_1559841127,340,2,train,285.0,19.953,339.543,18.022,24.039,165.0,0.0,0.0,67.0,53.0,0.0,0.0,0.0,18.34,0.0,0.0,0.871,0.742,0.0,0.0,0.0
dish_1559841163,340,2,train,335.0,21.153,357.044,18.222,27.539,165.0,0.0,0.0,67.0,103.0,0.0,0.0,0.0,18.34,0.0,0.0,0.871,1.942,0.0,0.0,0.0
dish_1559841781,342,0,train,43.0,0.866,45.02,0.411,10.49,0.0,0.0,0.0,5.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.6,0.266,0.0,0.0,0.0
dish_1559841808,342,0,train,96.0,1.29,73.11,0.57,17.38,0.0,0.0,0.0,5.0,38.0,0.0,0.0,53.0,0.0,0.0,0.0,0.6,0.266,0.0,0.0,0.424
dish_1559841870,342,0,train,201.0,0.804,114.972,0.201,30.552,0.0,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.804,0.0,0.0,0.0
dish_1559841891,342,0,train,272.0,1.301,163.962,0.343,43.332,0.0,0.0,0.0,0.0,272.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.301,0.0,0.0,0.0
dish_1559841908,342,0,train,304.0,2.069,175.162,0.471,45.572,0.0,0.0,0.0,0.0,304.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.069,0.0,0.0,0.0
dish_1559841950,342,0,test,32.0,0.768,11.2,0.128,2.24,0.0,0.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,0.0
dish_1559841979,342,0,test,85.0,1.192,39.29,0.287,9.13,0.0,0.0,0.0,0.0,32.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,0.424
dish_1559842047,342,0,test,285.0,1.992,153.69,0.487,39.53,0.0,0.0,0.0,0.0,232.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,1.568,0.0,0.0,0.424
dish_1559843030,344,1,train,77.0,0.539,25.41,0.231,6.16,0.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.539,0.0,0.0,0.0
dish_1559843052,344,1,train,130.0,0.963,53.5,0.39,13.05,0.0,0.0,0.0,0.0,77.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.539,0.0,0.0,0.424
dish_1559843185,345,3,train,161.0,1.707,64.35,0.514,15.22,0.0,0.0,0.0,0.0,108.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,1.283,0.0,0.0,0.424
dish_1559843222,345,3,train,229.0,2.183,111.27,0.65,27.46,0.0,0.0,0.0,0.0,176.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,1.759,0.0,0.0,0.424
dish_1559843335,345,3,train,77.0,0.539,25.41,0.231,6.16,0.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.539,0.0,0.0,0.0
dish_1559843492,346,4,train,95.0,0.971,31.71,0.303,7.42,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.971,0.0,0.0,0.0
dish_1559843690,347,0,train,134.0,1.751,61.74,0.342,14.05,0.0,0.0,0.0,39.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.78,0.971,0.0,0.0,0.0
dish_1559843715,347,0,train,334.0,2.551,176.14,0.542,44.45,0.0,0.0,0.0,39.0,295.0,0.0,0.0,0.0,0.0,0.0,0.0,0.78,1.771,0.0,0.0,0.0
dish_1559844928,350,3,train,86.0,1.72,66.22,0.086,14.62,0.0,0.0,0.0,86.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.72,0.0,0.0,0.0,0.0
dish_1559844951,350,3,train,122.0,9.784,274.336,17.906,22.792,0.0,0.0,0.0,86.0,0.0,0.0,0.0,36.0,0.0,0.0,0.0,1.72,0.0,0.0,0.0,8.064
dish_1559845018,350,3,train,175.0,10.208,302.426,18.065,29.682,0.0,0.0,0.0,86.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,1.72,0.0,0.0,0.0,8.488
dish_1559845046,350,3,train,207.0,10.976,313.626,18.193,31.922,0.0,0.0,0.0,86.0,32.0,0.0,0.0,89.0,0.0,0.0,0.0,1.72,0.768,0.0,0.0,8.488
dish_1559845745,352,4,train,322.0,3.224,208.142,0.322,51.122,0.0,0.0,0.0,121.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,2.42,0.804,0.0,0.0,0.0
dish_1559845767,352,4,train,375.0,3.648,236.232,0.481,58.012,0.0,0.0,0.0,121.0,201.0,0.0,0.0,53.0,0.0,0.0,0.0,2.42,0.804,0.0,0.0,0.424
dish_1559846034,353,2,train,32.0,0.768,11.2,0.128,2.24,0.0,0.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,0.0
dish_1559846073,353,2,train,85.0,1.192,39.29,0.287,9.13,0.0,0.0,0.0,0.0,32.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,0.424
dish_1559846099,353,2,train,114.0,7.688,206.939,14.642,15.713,0.0,0.0,0.0,0.0,32.0,0.0,0.0,82.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,6.92
dish_1559846311,354,3,train,201.0,0.804,114.972,0.201,30.552,0.0,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.804,0.0,0.0,0.0
dish_1559846334,354,3,train,233.0,1.572,126.172,0.329,32.792,0.0,0.0,0.0,0.0,233.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.572,0.0,0.0,0.0
dish_1559846363,354,3,train,286.0,1.996,154.262,0.488,39.682,0.0,0.0,0.0,0.0,233.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,1.572,0.0,0.0,0.424
dish_1559846450,354,3,train,201.0,0.804,114.972,0.201,30.552,0.0,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.804,0.0,0.0,0.0
dish_1559846504,354,3,train,233.0,1.572,126.172,0.329,32.792,0.0,0.0,0.0,0.0,233.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.572,0.0,0.0,0.0
dish_1559846528,354,3,train,267.0,9.188,322.726,17.159,40.51,0.0,0.0,0.0,0.0,233.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0,1.572,0.0,0.0,7.616
dish_1559846585,354,3,train,201.0,0.804,114.972,0.201,30.552,0.0,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.804,0.0,0.0,0.0
dish_1559846688,354,3,test,73.0,1.292,82.832,7.752,3.111,0.0,0.0,0.0,0.0,61.38,11.62,0.0,0.0,0.0,0.0,0.0,0.0,1.283,0.009,0.0,0.0
dish_1559846825,355,3,train,95.0,10.45,252.7,9.215,31.35,0.0,0.0,0.0,0.0,0.0,0.0,95.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.45,0.0
dish_1559848857,356,4,train,53.0,0.424,28.09,0.159,6.89,0.0,0.0,0.0,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.424
dish_1559848882,356,4,train,254.0,1.228,143.062,0.36,37.442,0.0,0.0,0.0,0.0,201.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.804,0.0,0.0,0.424
dish_1559848929,356,4,train,321.0,1.697,189.292,0.494,49.502,0.0,0.0,0.0,0.0,268.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,1.273,0.0,0.0,0.424
dish_1559849020,356,4,test,67.0,0.469,46.23,0.134,12.06,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.469,0.0,0.0,0.0
dish_1559849053,356,4,test,125.0,13.461,381.528,28.844,25.226,0.0,0.0,0.0,0.0,67.0,0.0,0.0,58.0,0.0,0.0,0.0,0.0,0.469,0.0,0.0,12.992
dish_1559849097,356,4,test,325.0,14.261,495.928,29.044,55.626,0.0,0.0,0.0,0.0,267.0,0.0,0.0,58.0,0.0,0.0,0.0,0.0,1.269,0.0,0.0,12.992
dish_1559850283,358,1,test,47.0,10.528,271.707,23.265,10.669,0.0,0.0,0.0,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.528
dish_1559850314,358,1,test,100.0,11.005,296.617,23.318,17.029,0.0,0.0,0.0,0.0,53.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.477,0.0,0.0,10.528
dish_1559850338,358,1,test,130.0,11.725,307.117,23.438,19.129,0.0,0.0,0.0,0.0,83.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,1.197,0.0,0.0,10.528
dish_1559925083,359,3,train,81.0,4.623,125.136,4.271,16.967,10.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,3.7,0.0,0.0,0.923,0.0,0.0,0.0,0.0
dish_1559925174,359,3,train,155.0,1.085,88.35,0.465,21.7,0.0,0.0,0.0,0.0,155.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.085,0.0,0.0,0.0
dish_1559932674,360,2,train,124.0,1.116,58.28,0.124,14.88,0.0,0.0,0.0,0.0,124.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.116,0.0,0.0,0.0
dish_1559933793,361,0,train,111.0,3.716,56.677,1.42,9.843,0.0,0.0,0.0,0.0,109.23,1.77,0.0,0.0,0.0,0.0,0.0,0.0,3.714,0.003,0.0,0.0
dish_1559933818,361,0,train,154.0,5.035,115.067,6.58,12.201,0.0,0.0,3.75,0.0,142.31,7.93,0.0,0.0,0.0,0.0,0.713,0.0,4.301,0.02,0.0,0.0
dish_1559933834,361,0,train,458.0,54.541,775.029,51.821,22.732,187.08,37.42,3.75,0.0,178.32,49.56,0.0,1.87,44.898,3.367,0.713,0.0,5.162,0.394,0.0,0.006
dish_1559934477,362,1,test,57.0,1.824,25.08,1.197,2.451,0.0,0.0,0.0,0.0,57.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.824,0.0,0.0,0.0
dish_1559934645,363,4,train,56.0,1.792,24.64,1.176,2.408,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.792,0.0,0.0,0.0
dish_1559934679,363,4,train,146.0,18.901,160.845,7.258,4.518,69.23,0.0,0.0,6.92,69.24,0.26,0.0,0.35,16.546,0.0,0.0,0.305,2.05,0.0,0.0,0.0
dish_1559934708,363,4,train,178.0,19.765,202.445,7.354,13.478,69.23,0.0,0.0,38.92,69.24,0.26,0.0,0.35,16.546,0.0,0.0,1.169,2.05,0.0,0.0,0.0
dish_1560203400,364,2,test,104.0,12.48,293.28,12.48,33.28,0.0,0.0,0.0,0.0,0.0,0.0,104.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.48,0.0
dish_1560203424,364,2,test,138.0,14.588,451.754,22.272,50.212,0.0,0.0,0.0,0.0,0.0,0.0,104.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,12.48,2.108
dish_1560270537,365,3,train,48.0,0.24,24.0,0.048,6.24,0.0,0.0,0.0,0.0,48.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.24,0.0,0.0,0.0
dish_1560270585,365,3,train,114.0,0.636,43.8,0.18,11.256,0.0,0.0,0.0,0.0,114.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.636,0.0,0.0,0.0
dish_1560270630,365,3,train,215.0,1.646,120.56,0.18,29.436,0.0,0.0,0.0,101.0,114.0,0.0,0.0,0.0,0.0,0.0,0.0,1.01,0.636,0.0,0.0,0.0
dish_1560356314,366,2,train,27.0,0.576,30.545,2.914,0.949,0.0,0.0,0.0,0.0,18.97,8.03,0.0,0.0,0.0,0.0,0.0,0.0,0.467,0.109,0.0,0.0
dish_1560356337,366,2,train,119.0,2.048,144.717,3.098,27.997,0.0,0.0,0.0,92.0,18.97,8.03,0.0,0.0,0.0,0.0,0.0,1.472,0.467,0.109,0.0,0.0
dish_1560356355,366,2,train,148.0,12.778,301.607,15.278,28.403,29.0,0.0,0.0,92.0,18.97,8.03,0.0,0.0,10.73,0.0,0.0,1.472,0.467,0.109,0.0,0.0
dish_1560357792,370,1,train,50.0,3.95,145.15,7.15,16.35,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.95,0.0,0.0,0.0,0.0
dish_1560357832,370,1,train,82.0,4.632,181.351,10.603,17.474,0.0,0.0,0.0,50.0,22.48,9.52,0.0,0.0,0.0,0.0,0.0,3.95,0.553,0.129,0.0,0.0
dish_1560366389,372,2,train,38.0,0.912,46.909,4.457,1.53,0.0,0.0,0.0,0.0,32.07,5.01,0.0,0.92,0.0,0.0,0.0,0.0,0.762,0.011,0.0,0.138
dish_1560366410,372,2,train,96.0,2.188,58.741,4.515,3.792,0.0,0.0,0.0,0.0,90.07,5.01,0.0,0.92,0.0,0.0,0.0,0.0,2.038,0.011,0.0,0.138
dish_1560366438,372,2,train,136.0,2.58,71.933,4.986,6.106,0.0,0.0,0.0,0.0,129.29,5.8,0.0,0.92,0.0,0.0,0.0,0.0,2.431,0.011,0.0,0.138
dish_1560366460,372,2,train,255.0,27.041,330.741,20.444,9.966,91.89,0.0,0.0,12.25,139.7,10.24,0.0,0.92,23.892,0.0,0.0,0.319,2.656,0.036,0.0,0.138
dish_1560366963,373,0,train,42.0,0.378,7.56,0.084,1.638,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.378,0.0,0.0,0.0
dish_1560366998,373,0,train,131.0,0.823,39.6,0.173,9.648,0.0,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.823,0.0,0.0,0.0
dish_1560367085,373,0,train,183.0,4.983,77.56,2.669,10.636,0.0,52.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0,4.16,0.0,0.0,0.823,0.0,0.0,0.0
dish_1560367116,373,0,train,211.0,9.206,108.41,6.257,12.152,21.92,52.0,0.0,4.38,132.21,0.49,0.0,0.0,4.033,4.16,0.0,0.169,0.844,0.0,0.0,0.0
dish_1560367153,373,0,train,254.0,10.152,117.182,6.3,13.829,21.92,52.0,0.0,4.38,175.21,0.49,0.0,0.0,4.033,4.16,0.0,0.169,1.79,0.0,0.0,0.0
dish_1560367193,373,0,train,317.0,12.105,131.042,6.489,15.908,21.92,52.0,0.0,4.38,238.21,0.49,0.0,0.0,4.033,4.16,0.0,0.169,3.743,0.0,0.0,0.0
dish_1560367258,373,0,train,341.0,12.48,157.992,8.975,17.012,21.92,52.0,0.0,4.38,258.53,4.17,0.0,0.0,4.033,4.16,0.0,0.169,4.118,0.0,0.0,0.0
dish_1560367296,373,0,train,362.0,13.027,184.383,11.225,18.36,21.92,52.0,0.0,4.38,275.33,7.38,0.0,0.99,4.033,4.16,0.0,0.169,4.471,0.006,0.0,0.188
dish_1560367333,373,0,train,427.0,33.177,291.633,13.565,18.36,86.92,52.0,0.0,4.38,275.33,7.38,0.0,0.99,24.183,4.16,0.0,0.169,4.471,0.006,0.0,0.188
dish_1560367676,374,1,train,198.0,40.699,430.621,25.719,6.424,152.9,0.0,0.0,20.39,17.33,7.39,0.0,0.0,39.753,0.0,0.0,0.53,0.376,0.041,0.0,0.0
dish_1560367698,374,1,train,295.0,43.318,556.721,26.01,33.584,152.9,0.0,0.0,117.39,17.33,7.39,0.0,0.0,39.753,0.0,0.0,3.149,0.376,0.041,0.0,0.0
dish_1560367715,374,1,train,305.0,44.518,597.531,26.68,40.814,152.9,0.0,0.0,117.39,17.33,7.39,0.0,10.0,39.753,0.0,0.0,3.149,0.376,0.041,0.0,1.2
dish_1560367733,374,1,train,388.0,46.178,661.441,26.763,54.924,152.9,0.0,0.0,200.39,17.33,7.39,0.0,10.0,39.753,0.0,0.0,4.809,0.376,0.041,0.0,1.2
dish_1560367904,375,1,test,69.0,10.406,76.023,8.841,3.735,54.01,0.0,0.0,10.8,2.97,1.22,0.0,0.0,9.938,0.0,0.0,0.417,0.051,0.0,0.0,0.0
dish_1560367926,375,1,test,194.0,24.156,408.523,20.966,44.985,54.01,0.0,0.0,10.8,2.97,1.22,125.0,0.0,9.938,0.0,0.0,0.417,0.051,0.0,13.75,0.0
dish_1560367952,375,1,test,221.0,24.939,414.733,21.074,45.957,54.01,0.0,0.0,10.8,29.97,1.22,125.0,0.0,9.938,0.0,0.0,0.417,0.834,0.0,13.75,0.0
dish_1560367980,375,1,test,254.0,25.665,421.465,21.107,47.244,54.01,0.0,0.0,10.8,62.97,1.22,125.0,0.0,9.938,0.0,0.0,0.417,1.56,0.0,13.75,0.0
dish_1560368019,375,1,test,301.0,26.126,436.966,21.66,49.963,54.01,0.0,0.0,10.8,109.05,2.14,125.0,0.0,9.938,0.0,0.0,0.417,2.021,0.0,13.75,0.0
dish_1560368110,375,1,train,89.0,0.801,16.02,0.178,3.471,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.801,0.0,0.0,0.0
dish_1560368138,375,1,train,123.0,1.855,23.5,0.28,4.593,0.0,0.0,0.0,0.0,123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.855,0.0,0.0,0.0
dish_1560368162,375,1,train,178.0,2.394,41.639,0.927,7.774,0.0,0.0,0.0,0.0,176.92,1.08,0.0,0.0,0.0,0.0,0.0,0.0,2.394,0.0,0.0,0.0
dish_1560368944,378,2,train,66.0,13.566,143.54,8.573,2.141,50.97,0.0,0.0,6.8,5.78,2.46,0.0,0.0,13.251,0.0,0.0,0.177,0.125,0.014,0.0,0.0
dish_1560368972,378,2,train,107.0,14.673,196.84,8.696,13.621,50.97,0.0,0.0,47.8,5.78,2.46,0.0,0.0,13.251,0.0,0.0,1.284,0.125,0.014,0.0,0.0
dish_1560368990,378,2,train,119.0,14.986,211.921,9.982,14.392,50.97,0.0,0.0,47.8,15.38,4.3,0.0,0.56,13.251,0.0,0.0,1.284,0.327,0.017,0.0,0.107
dish_1560369008,378,2,train,146.0,15.85,223.801,10.549,15.553,50.97,0.0,0.0,47.8,42.38,4.3,0.0,0.56,13.251,0.0,0.0,1.284,1.191,0.017,0.0,0.107
dish_1560369080,378,2,train,49.0,0.99,55.051,5.117,2.069,0.0,0.0,0.0,0.0,41.09,7.91,0.0,0.0,0.0,0.0,0.0,0.0,0.979,0.011,0.0,0.0
dish_1560369102,378,2,train,121.0,2.574,69.739,5.189,4.877,0.0,0.0,0.0,0.0,113.09,7.91,0.0,0.0,0.0,0.0,0.0,0.0,2.563,0.011,0.0,0.0
dish_1560369122,378,2,train,183.0,2.884,92.059,5.251,10.457,0.0,0.0,0.0,0.0,175.09,7.91,0.0,0.0,0.0,0.0,0.0,0.0,2.873,0.011,0.0,0.0
dish_1560369146,378,2,train,359.0,39.061,474.833,28.112,16.167,135.91,0.0,0.0,18.12,190.49,14.48,0.0,0.0,35.336,0.0,0.0,0.471,3.207,0.048,0.0,0.0
dish_1560369255,378,2,train,61.0,12.539,132.666,7.924,1.979,47.1,0.0,0.0,6.28,5.34,2.28,0.0,0.0,12.247,0.0,0.0,0.163,0.116,0.013,0.0,0.0
dish_1560369273,378,2,train,221.0,36.668,308.952,28.424,10.64,172.35,0.0,0.0,31.33,12.23,5.09,0.0,0.0,35.292,0.0,0.0,1.13,0.233,0.013,0.0,0.0
dish_1560369294,378,2,train,342.0,39.33,333.636,28.545,15.359,172.35,0.0,0.0,31.33,133.23,5.09,0.0,0.0,35.292,0.0,0.0,1.13,2.895,0.013,0.0,0.0
dish_1560369313,378,2,train,387.0,40.545,392.136,28.68,27.959,172.35,0.0,0.0,76.33,133.23,5.09,0.0,0.0,35.292,0.0,0.0,2.345,2.895,0.013,0.0,0.0
dish_1560369403,378,2,train,405.0,43.26,411.968,30.987,28.933,186.44,0.0,0.0,79.15,134.0,5.41,0.0,0.0,37.885,0.0,0.0,2.454,2.909,0.013,0.0,0.0
dish_1560369421,378,2,train,411.0,43.319,413.947,31.057,29.28,186.44,0.0,0.0,79.15,139.88,5.53,0.0,0.0,37.885,0.0,0.0,2.454,2.968,0.013,0.0,0.0
dish_1560370114,379,0,train,39.0,0.858,7.956,0.039,1.521,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.858,0.0,0.0,0.0
dish_1560370137,379,0,train,104.0,2.552,89.644,7.003,5.694,0.0,0.0,0.0,0.0,91.0,9.94,0.0,3.06,0.0,0.0,0.0,0.0,1.953,0.018,0.0,0.581
dish_1560441270,381,0,train,46.0,0.835,24.54,0.063,5.304,0.0,0.0,0.0,29.0,17.0,0.0,0.0,0.0,0.0,0.0,0.0,0.58,0.255,0.0,0.0,0.0
dish_1560441288,381,0,train,72.0,10.455,165.2,10.983,5.668,26.0,0.0,0.0,29.0,17.0,0.0,0.0,0.0,9.62,0.0,0.0,0.58,0.255,0.0,0.0,0.0
dish_1560441311,381,0,train,93.0,11.379,190.4,11.382,10.078,26.0,0.0,0.0,50.0,17.0,0.0,0.0,0.0,9.62,0.0,0.0,1.504,0.255,0.0,0.0,0.0
dish_1560441339,381,0,train,154.0,17.479,280.68,18.092,11.054,87.0,0.0,0.0,50.0,17.0,0.0,0.0,0.0,15.72,0.0,0.0,1.504,0.255,0.0,0.0,0.0
dish_1560442226,383,3,train,147.0,1.029,83.79,0.441,20.58,0.0,0.0,0.0,0.0,147.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.029,0.0,0.0,0.0
dish_1560442258,383,3,train,185.0,15.089,289.37,16.401,21.112,38.0,0.0,0.0,0.0,147.0,0.0,0.0,0.0,14.06,0.0,0.0,0.0,1.029,0.0,0.0,0.0
dish_1560442279,383,3,train,222.0,16.717,333.77,17.104,28.882,38.0,0.0,0.0,37.0,147.0,0.0,0.0,0.0,14.06,0.0,0.0,1.628,1.029,0.0,0.0,0.0
dish_1560442303,383,3,train,297.0,24.217,444.77,25.354,30.082,113.0,0.0,0.0,37.0,147.0,0.0,0.0,0.0,21.56,0.0,0.0,1.628,1.029,0.0,0.0,0.0
dish_1560442374,383,3,train,233.0,1.165,90.637,0.233,21.203,0.0,0.0,0.0,0.0,0.0,0.0,0.0,233.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.165
dish_1560442450,383,3,train,149.0,14.9,87.761,0.596,5.364,0.0,0.0,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.9,0.0,0.0,0.0,0.0,0.0
dish_1560443437,384,0,train,204.0,1.428,116.28,0.612,28.56,0.0,0.0,0.0,0.0,204.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.428,0.0,0.0,0.0
dish_1560443474,384,0,train,317.0,1.993,172.78,0.725,43.25,0.0,0.0,0.0,0.0,317.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.993,0.0,0.0,0.0
dish_1560443537,384,0,train,411.0,2.651,237.64,0.913,60.17,0.0,0.0,0.0,0.0,411.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.651,0.0,0.0,0.0
dish_1560443565,384,0,train,475.0,2.971,260.68,0.977,65.93,0.0,0.0,0.0,0.0,475.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.971,0.0,0.0,0.0
dish_1560443607,384,0,train,706.0,26.071,380.8,1.439,67.547,231.0,0.0,0.0,0.0,475.0,0.0,0.0,0.0,23.1,0.0,0.0,0.0,2.971,0.0,0.0,0.0
dish_1560454488,387,4,train,42.0,0.635,48.244,4.501,1.894,0.0,0.0,0.0,0.0,35.27,6.73,0.0,0.0,0.0,0.0,0.0,0.0,0.635,0.0,0.0,0.0
dish_1560454513,387,4,train,62.0,0.995,54.444,4.521,3.294,0.0,20.0,0.0,0.0,35.27,6.73,0.0,0.0,0.0,0.36,0.0,0.0,0.635,0.0,0.0,0.0
dish_1560454539,387,4,train,119.0,3.845,122.274,4.749,17.544,0.0,20.0,0.0,57.0,35.27,6.73,0.0,0.0,0.0,0.36,0.0,2.85,0.635,0.0,0.0,0.0
dish_1560454562,387,4,train,299.0,26.887,318.822,26.924,24.929,117.84,20.0,0.0,80.57,51.77,20.57,0.0,8.25,21.682,0.36,0.0,3.651,0.922,0.189,0.0,0.082
dish_1560456305,389,3,test,21.0,0.547,26.392,2.25,1.348,0.0,0.0,0.0,0.0,16.8,3.21,0.0,0.99,0.0,0.0,0.0,0.0,0.354,0.006,0.0,0.188
dish_1560456326,389,3,test,138.0,29.742,206.873,7.522,3.792,92.17,0.0,0.0,9.22,30.4,5.23,0.0,0.99,28.573,0.0,0.0,0.387,0.587,0.007,0.0,0.188
dish_1560456348,389,3,test,149.0,29.94,210.283,7.533,4.562,92.17,11.0,0.0,9.22,30.4,5.23,0.0,0.99,28.573,0.198,0.0,0.387,0.587,0.007,0.0,0.188
dish_1560456370,389,3,test,186.0,30.939,258.383,7.644,14.922,92.17,11.0,0.0,46.22,30.4,5.23,0.0,0.99,28.573,0.198,0.0,1.386,0.587,0.007,0.0,0.188
dish_1560456794,390,2,train,188.0,24.066,205.283,23.161,7.713,123.08,0.0,0.0,24.62,17.23,14.46,0.0,8.62,22.646,0.0,0.0,0.837,0.3,0.197,0.0,0.086
dish_1560456814,390,2,train,283.0,28.816,318.333,23.541,31.463,123.08,0.0,0.0,119.62,17.23,14.46,0.0,8.62,22.646,0.0,0.0,5.587,0.3,0.197,0.0,0.086
dish_1560456833,390,2,train,378.0,31.096,351.583,23.921,38.113,123.08,0.0,0.0,119.62,112.23,14.46,0.0,8.62,22.646,0.0,0.0,5.587,2.58,0.197,0.0,0.086
dish_1560456853,390,2,train,501.0,35.032,405.703,26.504,43.402,123.08,0.0,0.0,119.62,235.23,14.46,0.0,8.62,22.646,0.0,0.0,5.587,6.516,0.197,0.0,0.086
dish_1560527070,392,2,train,69.0,1.38,53.13,0.069,11.73,0.0,0.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.38,0.0,0.0,0.0,0.0
dish_1560527105,392,2,train,104.0,2.126,92.725,3.846,12.96,0.0,0.0,0.0,69.0,24.59,10.41,0.0,0.0,0.0,0.0,0.0,1.38,0.605,0.141,0.0,0.0
dish_1560527140,392,2,train,138.0,14.706,276.665,18.126,13.436,34.0,0.0,0.0,69.0,24.59,10.41,0.0,0.0,12.58,0.0,0.0,1.38,0.605,0.141,0.0,0.0
dish_1560527168,392,2,train,219.0,15.273,322.835,18.369,24.776,34.0,0.0,0.0,69.0,105.59,10.41,0.0,0.0,12.58,0.0,0.0,1.38,1.172,0.141,0.0,0.0
dish_1560527191,392,2,train,318.0,34.38,663.989,45.396,28.934,133.0,0.0,0.0,69.0,105.59,10.41,0.0,0.0,31.687,0.0,0.0,1.38,1.172,0.141,0.0,0.0
dish_1560527225,392,2,train,503.0,52.88,937.789,65.746,31.894,318.0,0.0,0.0,69.0,105.59,10.41,0.0,0.0,50.187,0.0,0.0,1.38,1.172,0.141,0.0,0.0
dish_1560527745,393,0,train,149.0,14.9,87.761,0.596,5.364,0.0,0.0,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.9,0.0,0.0,0.0,0.0,0.0
dish_1560527950,394,4,train,77.0,0.462,23.1,0.154,5.852,0.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1560529804,395,1,train,51.0,0.306,15.3,0.102,3.876,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.306,0.0,0.0,0.0
dish_1560529825,395,1,train,117.0,0.768,52.92,0.3,13.116,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.768,0.0,0.0,0.0
dish_1560529844,395,1,train,168.0,1.278,91.68,0.3,22.296,0.0,0.0,0.0,51.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.51,0.768,0.0,0.0,0.0
dish_1560541680,397,1,train,77.0,1.848,95.053,9.032,3.1,0.0,0.0,0.0,0.0,64.99,10.15,0.0,1.86,0.0,0.0,0.0,0.0,1.545,0.023,0.0,0.28
dish_1560541706,397,1,train,140.0,2.982,110.803,9.221,6.25,0.0,0.0,0.0,0.0,127.99,10.15,0.0,1.86,0.0,0.0,0.0,0.0,2.679,0.023,0.0,0.28
dish_1560541760,397,1,train,372.0,63.607,465.584,16.74,13.393,192.03,4.8,0.0,19.2,141.43,12.67,0.0,1.86,59.53,0.086,0.0,0.518,3.091,0.1,0.0,0.28
dish_1560541793,397,1,train,462.0,64.406,509.826,17.805,22.271,192.03,4.8,0.0,19.2,230.16,13.95,0.0,1.86,59.53,0.086,0.0,0.518,3.889,0.102,0.0,0.28
dish_1560788338,400,0,train,121.0,12.1,71.269,0.484,4.356,0.0,0.0,121.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.1,0.0,0.0,0.0,0.0,0.0
dish_1560788349,400,0,train,134.0,13.4,78.926,0.536,4.824,0.0,0.0,134.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.4,0.0,0.0,0.0,0.0,0.0
dish_1560788384,400,0,train,242.0,12.947,111.199,0.847,14.036,0.0,0.0,121.0,0.0,121.0,0.0,0.0,0.0,0.0,0.0,12.1,0.0,0.847,0.0,0.0,0.0
dish_1560798656,401,4,train,108.0,12.096,287.172,10.152,36.18,0.0,0.0,0.0,0.0,0.0,0.0,108.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.096,0.0
dish_1560798707,401,4,train,139.0,12.84,298.022,10.276,38.35,0.0,0.0,0.0,0.0,31.0,0.0,108.0,0.0,0.0,0.0,0.0,0.0,0.744,0.0,12.096,0.0
dish_1560798744,401,4,train,220.0,19.32,357.152,14.164,39.889,0.0,81.0,0.0,0.0,31.0,0.0,108.0,0.0,0.0,6.48,0.0,0.0,0.744,0.0,12.096,0.0
dish_1560798785,401,4,train,310.0,20.94,379.652,14.434,44.389,0.0,81.0,0.0,0.0,121.0,0.0,108.0,0.0,0.0,6.48,0.0,0.0,2.364,0.0,12.096,0.0
dish_1560798815,401,4,train,341.0,22.862,404.964,15.424,46.988,0.0,81.0,2.53,0.0,145.24,4.04,108.0,0.19,0.0,6.48,0.97,0.0,3.164,0.129,12.096,0.023
dish_1560798854,401,4,train,372.0,23.492,455.507,19.444,50.674,0.0,81.0,2.53,0.0,171.39,7.31,108.0,1.77,0.0,6.48,0.97,0.0,3.493,0.129,12.096,0.324
dish_1560798928,401,4,train,399.0,23.907,485.038,21.99,52.309,0.0,81.0,2.53,0.0,194.46,11.24,108.0,1.77,0.0,6.48,0.97,0.0,3.902,0.135,12.096,0.324
dish_1560798960,401,4,train,429.0,24.173,493.721,22.375,53.315,0.0,81.0,2.53,0.0,224.05,11.65,108.0,1.77,0.0,6.48,0.97,0.0,4.168,0.135,12.096,0.324
dish_1560799113,402,3,train,37.0,4.07,98.42,3.589,12.21,0.0,0.0,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.07,0.0
dish_1560799277,403,4,train,73.0,4.934,111.02,3.733,14.73,0.0,0.0,0.0,0.0,36.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.864,0.0,4.07,0.0
dish_1560799368,403,4,train,53.0,10.821,108.403,6.013,1.962,40.61,0.0,0.0,5.42,6.63,0.34,0.0,0.0,10.559,0.0,0.0,0.146,0.115,0.0,0.0,0.0
dish_1560799407,403,4,train,122.0,12.223,220.902,14.961,10.166,40.61,0.0,0.0,5.42,64.83,7.61,0.0,3.53,10.559,0.0,0.0,0.146,0.847,0.0,0.0,0.67
dish_1560799558,404,0,train,74.0,8.14,196.84,7.178,24.42,0.0,0.0,0.0,0.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,8.14,0.0
dish_1560799590,404,0,train,163.0,9.948,298.019,16.461,28.492,0.0,0.0,0.0,0.0,74.24,14.76,74.0,0.0,0.0,0.0,0.0,0.0,1.766,0.042,8.14,0.0
dish_1560799615,404,0,train,283.0,11.013,332.75,18.0,32.516,0.0,0.0,0.0,0.0,192.61,16.39,74.0,0.0,0.0,0.0,0.0,0.0,2.831,0.042,8.14,0.0
dish_1560799647,404,0,train,474.0,50.008,723.41,39.669,39.588,146.36,0.0,0.0,19.51,216.51,17.61,74.0,0.0,38.054,0.0,0.0,0.527,3.245,0.042,8.14,0.0
dish_1560802730,409,3,train,64.0,16.083,101.702,2.802,2.27,48.46,1.1,1.21,5.39,6.71,1.13,0.0,0.0,15.071,0.099,0.465,0.269,0.17,0.009,0.0,0.0
dish_1560802764,409,3,train,99.0,16.394,111.832,3.251,3.444,48.46,1.1,1.21,5.39,41.23,1.61,0.0,0.0,15.071,0.099,0.465,0.269,0.481,0.009,0.0,0.0
dish_1560802853,409,3,train,118.0,16.907,136.532,3.308,8.764,48.46,1.1,1.21,24.39,41.23,1.61,0.0,0.0,15.071,0.099,0.465,0.782,0.481,0.009,0.0,0.0
dish_1560961385,414,2,train,293.0,12.781,249.52,11.831,23.794,104.0,0.0,0.0,75.0,114.0,0.0,0.0,0.0,10.4,0.0,0.0,0.75,1.631,0.0,0.0,0.0
dish_1560974707,416,4,train,60.0,2.993,55.025,2.004,7.089,0.0,10.38,3.11,7.26,36.84,2.14,0.26,0.0,0.0,0.924,0.778,0.355,0.881,0.05,0.005,0.0
dish_1560974769,416,4,train,3051.0,83.75,3943.325,10.977,844.569,0.0,10.38,3.11,2998.26,36.84,2.14,0.26,0.0,0.0,0.924,0.778,81.112,0.881,0.05,0.005,0.0
dish_1560975346,417,1,train,224.0,41.013,326.339,14.925,4.475,168.34,0.0,0.0,11.47,34.19,0.74,9.26,0.0,40.234,0.0,0.0,0.172,0.44,0.0,0.167,0.0
dish_1560976044,419,0,train,137.0,5.754,124.67,0.685,26.167,0.0,0.0,0.0,137.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.754,0.0,0.0,0.0,0.0
dish_1561060925,422,3,train,156.0,8.247,93.915,4.284,9.304,0.0,65.0,0.0,0.0,89.54,1.46,0.0,0.0,0.0,5.2,0.0,0.0,3.045,0.002,0.0,0.0
dish_1561060976,422,3,train,266.0,28.171,265.507,11.455,14.357,79.78,65.0,0.0,15.96,102.81,2.25,0.2,0.0,19.148,5.2,0.0,0.415,3.403,0.002,0.004,0.0
dish_1561061123,423,4,train,357.0,31.094,399.431,20.405,26.884,79.78,65.0,0.0,26.57,142.6,42.84,0.2,0.0,19.148,5.2,0.0,1.688,4.129,0.926,0.004,0.0
dish_1561061593,424,4,train,52.0,3.224,42.459,1.66,4.359,0.0,0.0,4.24,0.0,40.67,6.78,0.0,0.32,0.0,0.0,1.627,0.0,1.342,0.217,0.0,0.038
dish_1561061630,424,4,train,150.0,6.371,186.684,11.299,17.851,0.0,0.0,4.24,11.43,83.53,50.49,0.0,0.32,0.0,0.0,1.627,1.371,2.124,1.211,0.0,0.038
dish_1561061658,424,4,train,227.0,8.949,226.0,12.284,24.679,0.0,0.0,4.24,11.43,159.29,51.72,0.0,0.32,0.0,0.0,1.627,1.371,4.7,1.213,0.0,0.038
dish_1561061687,424,4,train,408.0,41.734,508.348,24.083,32.992,131.28,0.0,4.24,37.68,181.12,53.04,0.33,0.32,31.507,0.0,1.627,2.054,5.289,1.213,0.006,0.038
dish_1561062171,425,3,train,58.0,7.957,101.143,4.821,5.761,25.42,0.0,0.0,20.34,9.47,2.26,0.0,0.51,6.61,0.0,0.0,1.037,0.256,0.053,0.0,0.001
dish_1561062237,425,3,train,110.0,17.376,182.259,8.211,8.149,63.14,0.0,0.0,27.88,15.74,2.63,0.09,0.51,15.662,0.0,0.0,1.233,0.426,0.053,0.002,0.001
dish_1561062286,425,3,train,167.0,18.915,256.359,8.382,24.109,63.14,0.0,0.0,84.88,15.74,2.63,0.09,0.51,15.662,0.0,0.0,2.772,0.426,0.053,0.002,0.001
dish_1561062320,425,3,train,202.0,20.039,307.868,11.825,28.928,63.14,0.0,0.0,88.96,31.05,18.25,0.09,0.51,15.662,0.0,0.0,3.262,0.705,0.408,0.002,0.001
dish_1561062555,426,2,train,32.0,1.028,47.094,3.148,4.405,0.0,0.0,0.0,3.73,13.99,14.27,0.0,0.0,0.0,0.0,0.0,0.448,0.255,0.325,0.0,0.0
dish_1561062586,426,2,train,89.0,2.567,121.194,3.319,20.365,0.0,0.0,0.0,60.73,13.99,14.27,0.0,0.0,0.0,0.0,0.0,1.987,0.255,0.325,0.0,0.0
dish_1561134102,427,2,train,81.0,0.486,24.3,0.162,6.156,0.0,0.0,0.0,0.0,81.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.486,0.0,0.0,0.0
dish_1561134128,427,2,train,141.0,0.906,58.5,0.342,14.556,0.0,0.0,0.0,0.0,141.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.906,0.0,0.0,0.0
dish_1561134156,427,2,train,220.0,1.696,118.54,0.342,28.776,0.0,0.0,0.0,79.0,141.0,0.0,0.0,0.0,0.0,0.0,0.0,0.79,0.906,0.0,0.0,0.0
dish_1561145732,428,3,train,67.0,0.402,19.698,0.067,5.025,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.402,0.0,0.0,0.0
dish_1561146758,429,1,train,93.0,9.393,232.5,1.395,45.57,0.0,0.0,0.0,93.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.393,0.0,0.0,0.0,0.0
dish_1561146781,429,1,train,223.0,9.783,300.1,1.655,63.77,0.0,0.0,0.0,93.0,130.0,0.0,0.0,0.0,0.0,0.0,0.0,9.393,0.39,0.0,0.0,0.0
dish_1561147944,430,1,test,34.0,0.82,42.371,3.643,2.259,0.0,0.0,0.0,0.0,28.11,4.48,0.0,1.41,0.0,0.0,0.0,0.0,0.499,0.007,0.0,0.315
dish_1561147967,430,1,test,85.0,6.532,177.98,8.437,19.344,0.0,0.0,0.0,0.0,28.11,4.48,51.0,1.41,0.0,0.0,0.0,0.0,0.499,0.007,5.712,0.315
dish_1561147997,430,1,test,133.0,7.06,185.18,8.629,20.64,0.0,0.0,0.0,0.0,76.11,4.48,51.0,1.41,0.0,0.0,0.0,0.0,1.027,0.007,5.712,0.315
dish_1561148021,430,1,test,216.0,22.204,361.765,19.949,23.612,59.25,0.0,0.0,7.61,89.77,6.96,51.0,1.41,14.545,0.0,0.0,0.206,1.377,0.049,5.712,0.315
dish_1561148042,430,1,test,245.0,22.726,370.755,19.978,25.642,59.25,29.0,0.0,7.61,89.77,6.96,51.0,1.41,14.545,0.522,0.0,0.206,1.377,0.049,5.712,0.315
dish_1561148303,431,0,train,45.0,5.04,119.655,4.23,15.075,0.0,0.0,0.0,0.0,0.0,0.0,45.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.04,0.0
dish_1561148336,431,0,train,68.0,5.455,128.099,4.566,16.327,0.0,0.0,0.0,0.0,18.05,4.95,45.0,0.0,0.0,0.0,0.0,0.0,0.279,0.136,5.04,0.0
dish_1561148481,432,2,train,140.0,18.592,281.281,14.386,18.905,51.4,0.0,0.0,6.61,29.9,7.1,45.0,0.0,12.618,0.0,0.0,0.178,0.583,0.173,5.04,0.0
dish_1561404438,435,2,train,169.0,32.328,366.069,23.857,6.258,123.51,0.0,0.0,24.7,19.76,1.03,0.0,0.0,30.877,0.0,0.0,1.037,0.413,0.0,0.0,0.0
dish_1561404458,435,2,train,259.0,33.127,410.312,24.922,15.135,123.51,0.0,0.0,24.7,108.49,2.3,0.0,0.0,30.877,0.0,0.0,1.037,1.212,0.001,0.0,0.0
dish_1561404485,435,2,train,349.0,34.533,511.373,34.245,19.276,123.51,0.0,0.0,24.7,184.69,16.1,0.0,0.0,30.877,0.0,0.0,1.037,2.618,0.001,0.0,0.0
dish_1561404505,435,2,train,442.0,36.765,543.923,34.617,25.786,123.51,0.0,0.0,24.7,277.69,16.1,0.0,0.0,30.877,0.0,0.0,1.037,4.85,0.001,0.0,0.0
dish_1561404924,436,1,train,39.0,2.418,31.844,1.245,3.269,0.0,0.0,3.18,0.0,30.5,5.08,0.0,0.24,0.0,0.0,1.22,0.0,1.007,0.163,0.0,0.029
dish_1561404956,436,1,train,82.0,13.654,97.601,2.639,4.593,35.59,0.89,3.18,3.56,32.99,5.55,0.0,0.24,11.034,0.016,1.22,0.096,1.083,0.177,0.0,0.029
dish_1561405008,436,1,train,134.0,14.763,156.428,8.25,6.42,35.59,0.89,3.18,3.56,69.53,21.01,0.0,0.24,11.034,0.016,1.22,0.096,1.982,0.387,0.0,0.029
dish_1561405034,436,1,train,153.0,14.915,178.278,10.283,7.617,35.59,0.89,3.18,3.56,88.53,21.01,0.0,0.24,11.034,0.016,1.22,0.096,2.134,0.387,0.0,0.029
dish_1561405332,437,0,test,53.0,1.272,18.55,0.212,3.71,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.272,0.0,0.0,0.0
dish_1561405356,437,0,test,176.0,33.414,206.645,4.198,7.497,101.81,2.55,0.0,10.18,60.13,1.34,0.0,0.0,31.561,0.046,0.0,0.275,1.49,0.041,0.0,0.0
dish_1561405389,437,0,test,223.0,34.986,265.983,8.393,12.105,101.81,11.97,0.0,10.18,90.99,8.05,0.0,0.0,31.561,0.894,0.0,0.275,2.158,0.098,0.0,0.0
dish_1561405411,437,0,test,282.0,37.228,332.063,8.511,25.675,101.81,11.97,0.0,69.18,90.99,8.05,0.0,0.0,31.561,0.894,0.0,2.517,2.158,0.098,0.0,0.0
dish_1561406762,439,1,train,35.0,0.311,17.205,0.414,3.452,0.0,0.0,0.0,0.0,34.5,0.5,0.0,0.0,0.0,0.0,0.0,0.0,0.311,0.0,0.0,0.0
dish_1561406789,439,1,train,164.0,3.794,184.905,0.801,39.572,0.0,0.0,0.0,129.0,34.5,0.5,0.0,0.0,0.0,0.0,0.0,3.483,0.311,0.0,0.0,0.0
dish_1561406817,439,1,train,243.0,6.796,273.385,0.959,57.742,0.0,0.0,0.0,208.0,34.5,0.5,0.0,0.0,0.0,0.0,0.0,6.485,0.311,0.0,0.0,0.0
dish_1561407208,440,4,train,29.0,7.578,44.348,0.94,0.893,24.0,0.6,0.0,2.4,1.68,0.32,0.0,0.0,7.441,0.011,0.0,0.065,0.051,0.01,0.0,0.0
dish_1561407249,440,4,train,56.0,7.993,73.879,3.486,2.528,24.0,0.6,0.0,2.4,24.75,4.25,0.0,0.0,7.441,0.011,0.0,0.065,0.461,0.015,0.0,0.0
dish_1561407283,440,4,train,86.0,8.26,88.627,3.841,5.487,24.0,0.6,0.0,2.4,54.32,4.67,0.0,0.0,7.441,0.011,0.0,0.065,0.727,0.016,0.0,0.0
dish_1561407312,440,4,train,121.0,9.59,127.827,3.911,13.537,24.0,0.6,0.0,37.4,54.32,4.67,0.0,0.0,7.441,0.011,0.0,1.395,0.727,0.016,0.0,0.0
dish_1561480391,441,3,train,113.0,0.791,64.41,0.339,15.82,0.0,0.0,0.0,0.0,113.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.791,0.0,0.0,0.0
dish_1561480414,441,3,train,129.0,5.447,125.162,4.835,16.316,16.0,0.0,0.0,0.0,113.0,0.0,0.0,0.0,4.656,0.0,0.0,0.0,0.791,0.0,0.0,0.0
dish_1561480439,441,3,train,189.0,5.747,155.162,4.895,24.116,16.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,4.656,0.0,0.0,0.0,1.091,0.0,0.0,0.0
dish_1561480460,441,3,train,277.0,14.547,285.402,14.575,25.524,104.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,13.456,0.0,0.0,0.0,1.091,0.0,0.0,0.0
dish_1561491121,443,2,train,28.0,0.566,31.457,2.924,1.183,0.0,0.0,0.0,0.0,23.48,4.52,0.0,0.0,0.0,0.0,0.0,0.0,0.559,0.007,0.0,0.0
dish_1561491154,443,2,train,158.0,18.301,166.137,18.456,5.614,90.59,0.0,0.0,18.12,40.92,8.37,0.0,0.0,16.669,0.0,0.0,0.761,0.865,0.007,0.0,0.0
dish_1561491180,443,2,train,217.0,21.251,236.347,18.692,20.364,90.59,0.0,0.0,77.12,40.92,8.37,0.0,0.0,16.669,0.0,0.0,3.711,0.865,0.007,0.0,0.0
dish_1561491210,443,2,train,276.0,22.313,251.097,18.869,23.314,90.59,0.0,0.0,77.12,99.92,8.37,0.0,0.0,16.669,0.0,0.0,3.711,1.927,0.007,0.0,0.0
dish_1561492161,444,1,train,43.0,0.869,48.31,4.49,1.816,0.0,0.0,0.0,0.0,36.06,6.94,0.0,0.0,0.0,0.0,0.0,0.0,0.859,0.01,0.0,0.0
dish_1561492182,444,1,train,171.0,6.885,250.166,7.306,40.088,0.0,0.0,0.0,0.0,36.06,6.94,128.0,0.0,0.0,0.0,0.0,0.0,0.859,0.01,6.016,0.0
dish_1561492204,444,1,train,334.0,37.193,488.636,15.686,49.121,92.89,0.0,0.0,23.22,75.54,14.34,128.0,0.0,28.797,0.0,0.0,0.813,1.549,0.019,6.016,0.0
dish_1561492228,444,1,train,488.0,58.203,648.18,34.086,54.371,200.21,0.0,0.0,44.69,96.2,18.9,128.0,0.0,48.543,0.0,0.0,1.714,1.911,0.019,6.016,0.0
dish_1561493975,446,0,train,106.0,2.544,130.852,12.434,4.268,0.0,0.0,0.0,0.0,89.46,13.98,0.0,2.56,0.0,0.0,0.0,0.0,2.127,0.031,0.0,0.386
dish_1561494001,446,0,train,216.0,3.52,162.688,13.845,7.957,0.0,0.0,0.0,0.0,197.97,15.47,0.0,2.56,0.0,0.0,0.0,0.0,3.103,0.031,0.0,0.386
dish_1561494032,446,0,train,307.0,5.158,185.438,14.118,12.507,0.0,0.0,0.0,0.0,288.97,15.47,0.0,2.56,0.0,0.0,0.0,0.0,4.741,0.031,0.0,0.386
dish_1561567053,447,4,train,87.0,0.522,26.1,0.174,6.612,0.0,0.0,0.0,0.0,87.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.522,0.0,0.0,0.0
dish_1561567086,447,4,train,181.0,2.026,142.754,0.362,34.248,0.0,0.0,0.0,94.0,87.0,0.0,0.0,0.0,0.0,0.0,0.0,1.504,0.522,0.0,0.0,0.0
dish_1561574215,448,1,train,134.0,2.221,93.235,7.88,4.701,0.0,0.0,5.7,0.0,116.89,11.4,0.0,0.0,0.0,0.0,0.787,0.0,1.434,0.0,0.0,0.0
dish_1561574600,450,3,train,100.0,0.98,32.98,1.176,5.784,0.0,0.0,0.0,0.0,98.04,1.96,0.0,0.0,0.0,0.0,0.0,0.0,0.98,0.0,0.0,0.0
dish_1561574623,450,3,train,152.0,13.903,117.603,4.01,7.02,40.49,0.0,0.0,4.05,102.24,5.0,0.0,0.23,12.551,0.0,0.0,0.17,1.118,0.053,0.0,0.011
dish_1561574783,451,4,test,24.0,0.576,29.627,2.815,0.966,0.0,0.0,0.0,0.0,20.26,3.17,0.0,0.58,0.0,0.0,0.0,0.0,0.482,0.007,0.0,0.087
dish_1561574815,451,4,test,89.0,1.653,74.853,6.638,3.247,0.0,0.0,2.77,0.0,76.96,8.7,0.0,0.58,0.0,0.0,0.382,0.0,1.177,0.007,0.0,0.087
dish_1561574848,451,4,test,155.0,18.055,182.258,10.234,4.816,51.39,0.0,2.77,5.14,82.29,12.55,0.0,0.87,15.93,0.0,0.382,0.216,1.352,0.074,0.0,0.102
dish_1561574974,452,1,train,152.0,1.49,50.13,1.788,8.792,0.0,0.0,0.0,0.0,149.02,2.98,0.0,0.0,0.0,0.0,0.0,0.0,1.49,0.0,0.0,0.0
dish_1561575033,452,1,train,339.0,47.963,354.445,11.979,13.237,145.6,0.0,0.0,14.56,164.13,13.9,0.0,0.82,45.135,0.0,0.0,0.612,1.986,0.189,0.0,0.041
dish_1561575068,452,1,train,436.0,49.57,421.937,17.683,16.641,145.6,0.0,4.13,14.56,248.74,22.16,0.0,0.82,45.135,0.0,0.57,0.612,3.024,0.189,0.0,0.041
dish_1561575170,452,1,train,171.0,3.078,42.75,0.513,8.55,0.0,0.0,0.0,0.0,171.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.078,0.0,0.0,0.0
dish_1561575192,452,1,train,236.0,4.638,122.989,8.138,11.167,0.0,0.0,0.0,0.0,225.86,8.57,0.0,1.57,0.0,0.0,0.0,0.0,4.382,0.019,0.0,0.237
dish_1561575274,452,1,train,145.0,29.622,305.18,17.52,4.303,110.72,0.0,0.0,14.76,18.45,1.06,0.0,0.0,28.788,0.0,0.0,0.517,0.317,0.0,0.0,0.0
dish_1561575300,452,1,train,199.0,30.702,326.78,17.628,9.163,110.72,0.0,0.0,14.76,72.45,1.06,0.0,0.0,28.788,0.0,0.0,0.517,1.397,0.0,0.0,0.0
dish_1561575327,452,1,train,289.0,32.592,453.68,23.748,25.363,110.72,0.0,0.0,104.76,72.45,1.06,0.0,0.0,28.788,0.0,0.0,2.407,1.397,0.0,0.0,0.0
dish_1561575347,452,1,train,327.0,36.772,554.76,27.434,37.903,110.72,0.0,0.0,104.76,72.45,1.06,38.0,0.0,28.788,0.0,0.0,2.407,1.397,0.0,4.18,0.0
dish_1561575410,452,1,train,64.0,1.152,16.0,0.192,3.2,0.0,0.0,0.0,0.0,64.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.152,0.0,0.0,0.0
dish_1561575445,452,1,train,123.0,1.73,35.458,0.886,6.613,0.0,0.0,0.0,0.0,121.84,1.16,0.0,0.0,0.0,0.0,0.0,0.0,1.73,0.0,0.0,0.0
dish_1561575474,452,1,train,183.0,2.99,120.058,4.966,17.413,0.0,0.0,0.0,60.0,121.84,1.16,0.0,0.0,0.0,0.0,0.0,1.26,1.73,0.0,0.0,0.0
dish_1561575506,452,1,train,393.0,45.891,562.043,30.34,23.645,160.36,0.0,0.0,81.38,148.57,2.69,0.0,0.0,41.693,0.0,0.0,2.008,2.19,0.0,0.0,0.0
dish_1561575613,452,1,train,70.0,1.68,86.411,8.211,2.818,0.0,0.0,0.0,0.0,59.08,9.23,0.0,1.69,0.0,0.0,0.0,0.0,1.404,0.021,0.0,0.255
dish_1561575661,452,1,train,228.0,33.958,418.952,27.302,7.508,120.65,0.0,0.0,16.09,79.19,10.39,0.0,1.69,31.369,0.0,0.0,0.563,1.75,0.021,0.0,0.255
dish_1561575688,452,1,train,362.0,36.772,607.892,36.414,31.628,120.65,0.0,0.0,150.09,79.19,10.39,0.0,1.69,31.369,0.0,0.0,3.377,1.75,0.021,0.0,0.255
dish_1561576212,454,0,train,103.0,2.163,145.23,7.004,18.54,0.0,0.0,0.0,103.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.163,0.0,0.0,0.0,0.0
dish_1561576257,454,0,train,134.0,2.411,180.88,10.321,20.493,0.0,0.0,0.0,103.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,2.163,0.248,0.0,0.0,0.0
dish_1561576295,454,0,train,232.0,4.035,249.067,16.084,23.931,0.0,0.0,4.17,103.0,116.49,8.34,0.0,0.0,0.0,0.0,0.575,2.163,1.297,0.0,0.0,0.0
dish_1561576320,454,0,train,383.0,41.561,494.798,24.313,27.521,117.57,0.0,4.17,114.76,128.69,17.16,0.0,0.66,36.446,0.0,0.575,2.657,1.697,0.153,0.0,0.033
dish_1561577549,456,2,train,49.0,5.39,130.34,4.753,16.17,0.0,0.0,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.39,0.0
dish_1561577573,456,2,train,346.0,66.064,755.433,40.639,24.985,226.79,0.0,0.0,30.24,37.8,2.17,49.0,0.0,58.965,0.0,0.0,1.058,0.65,0.0,5.39,0.0
dish_1561577598,456,2,train,393.0,67.192,813.452,46.152,26.877,226.79,0.0,0.0,30.24,77.47,8.37,49.0,1.13,58.965,0.0,0.0,1.058,1.593,0.014,5.39,0.171
dish_1561577678,456,2,train,104.0,21.246,218.888,12.566,3.087,79.41,0.0,0.0,10.59,13.24,0.76,0.0,0.0,20.648,0.0,0.0,0.371,0.228,0.0,0.0,0.0
dish_1561577701,456,2,train,263.0,24.108,258.638,13.043,11.037,79.41,0.0,0.0,10.59,172.24,0.76,0.0,0.0,20.648,0.0,0.0,0.371,3.09,0.0,0.0,0.0
dish_1561577731,456,2,train,334.0,26.309,274.258,13.256,13.38,79.41,0.0,0.0,10.59,243.24,0.76,0.0,0.0,20.648,0.0,0.0,0.371,5.291,0.0,0.0,0.0
dish_1561577848,456,2,test,254.0,63.123,413.348,13.842,6.038,197.76,0.0,0.0,19.78,20.52,14.83,0.0,1.11,61.306,0.0,0.0,0.831,0.673,0.257,0.0,0.056
dish_1561577947,456,2,test,424.0,67.203,623.204,33.783,12.883,197.76,0.0,0.0,19.78,164.0,37.25,0.0,5.21,61.306,0.0,0.0,0.831,4.084,0.307,0.0,0.675
dish_1561580278,459,4,train,53.0,1.696,23.32,1.113,2.279,0.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.696,0.0,0.0,0.0
dish_1561580322,459,4,train,76.0,2.662,44.25,1.228,6.672,0.0,0.0,0.0,23.0,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.966,1.696,0.0,0.0,0.0
dish_1561580370,459,4,train,124.0,5.41,126.796,4.964,16.201,0.0,0.0,0.71,58.29,59.35,5.65,0.0,0.0,0.0,0.0,0.176,2.766,1.784,0.683,0.0,0.0
dish_1561580406,459,4,train,171.0,17.09,203.281,7.525,17.319,36.59,0.0,0.71,61.95,63.15,8.39,0.0,0.21,11.344,0.0,0.176,2.92,1.909,0.731,0.0,0.01
dish_1561580431,459,4,train,197.0,22.402,258.003,10.666,18.09,56.45,0.0,0.71,64.6,66.46,8.58,0.0,0.21,16.506,0.0,0.176,3.012,1.966,0.731,0.0,0.01
dish_1561580552,460,4,train,89.0,1.602,22.25,0.267,4.45,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.602,0.0,0.0,0.0
dish_1561580608,460,4,train,81.0,1.458,20.25,0.243,4.05,0.0,0.0,0.0,0.0,81.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.458,0.0,0.0,0.0
dish_1561580628,460,4,train,156.0,2.701,72.434,4.654,6.681,0.0,0.0,3.19,0.0,146.43,6.38,0.0,0.0,0.0,0.0,0.44,0.0,2.261,0.0,0.0,0.0
dish_1561580659,460,4,train,480.0,68.891,754.354,43.802,16.297,247.41,0.0,3.19,32.99,187.66,8.75,0.0,0.0,64.326,0.0,0.44,1.155,2.97,0.0,0.0,0.0
dish_1561654707,462,3,train,135.0,13.5,199.8,14.85,2.16,135.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1561654732,462,3,train,191.0,14.06,242.36,14.85,12.24,135.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,13.5,0.0,0.0,0.56,0.0,0.0,0.0,0.0
dish_1561654756,462,3,train,225.0,15.216,256.98,14.952,15.3,135.0,0.0,0.0,56.0,34.0,0.0,0.0,0.0,13.5,0.0,0.0,0.56,1.156,0.0,0.0,0.0
dish_1561661806,464,2,train,89.0,16.236,183.476,8.4,9.825,59.47,0.0,0.0,8.5,10.62,10.41,0.0,0.0,15.463,0.0,0.0,0.229,0.263,0.28,0.0,0.0
dish_1561661850,464,2,train,264.0,35.486,648.976,25.375,67.575,59.47,0.0,0.0,8.5,10.62,10.41,175.0,0.0,15.463,0.0,0.0,0.229,0.263,0.28,19.25,0.0
dish_1561662358,466,3,train,91.0,2.821,96.551,2.184,20.02,0.0,0.0,0.0,91.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.821,0.0,0.0,0.0,0.0
dish_1561662458,466,3,train,119.0,3.202,124.815,4.856,21.055,0.0,0.0,0.0,91.0,23.84,4.16,0.0,0.0,0.0,0.0,0.0,2.821,0.38,0.002,0.0,0.0
dish_1561662501,466,3,train,165.0,6.882,158.396,7.064,21.929,0.0,46.0,0.0,91.0,23.84,4.16,0.0,0.0,0.0,3.68,0.0,2.821,0.38,0.002,0.0,0.0
dish_1561662532,466,3,train,191.0,7.35,166.456,7.09,23.749,0.0,72.0,0.0,91.0,23.84,4.16,0.0,0.0,0.0,4.148,0.0,2.821,0.38,0.002,0.0,0.0
dish_1561662562,466,3,train,289.0,25.228,368.485,16.34,34.568,65.49,72.0,0.0,100.36,35.53,15.62,0.0,0.0,17.027,4.148,0.0,3.074,0.67,0.309,0.0,0.0
dish_1561663786,469,3,train,44.0,1.364,46.684,1.056,9.68,0.0,0.0,0.0,44.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.364,0.0,0.0,0.0,0.0
dish_1561663817,469,3,train,86.0,2.099,95.495,4.994,12.892,0.0,0.0,0.0,47.98,33.11,4.9,0.0,0.0,0.0,0.0,0.0,1.503,0.596,0.0,0.0,0.0
dish_1561663855,469,3,train,139.0,15.287,179.138,7.395,14.41,41.94,0.0,0.0,52.18,38.93,5.95,0.0,0.0,13.001,0.0,0.0,1.612,0.673,0.0,0.0,0.0
dish_1561663891,469,3,train,172.0,21.307,247.169,10.51,18.053,63.99,0.0,0.0,55.33,42.87,9.81,0.0,0.0,18.735,0.0,0.0,1.698,0.771,0.104,0.0,0.0
dish_1561663914,469,3,train,205.0,21.901,257.399,10.543,20.363,63.99,33.0,0.0,55.33,42.87,9.81,0.0,0.0,18.735,0.594,0.0,1.698,0.771,0.104,0.0,0.0
dish_1561663990,469,3,train,115.0,28.615,181.49,5.211,3.294,91.0,0.0,0.0,9.1,12.63,2.27,0.0,0.0,28.21,0.0,0.0,0.237,0.168,0.0,0.0,0.0
dish_1561664013,469,3,train,163.0,37.371,280.443,9.742,8.593,123.08,0.0,0.0,13.68,18.35,7.89,0.0,0.0,36.55,0.0,0.0,0.36,0.31,0.151,0.0,0.0
dish_1561664042,469,3,train,209.0,37.831,285.043,9.742,9.513,123.08,0.0,0.0,13.68,64.35,7.89,0.0,0.0,36.55,0.0,0.0,0.36,0.77,0.151,0.0,0.0
dish_1561664061,469,3,train,269.0,38.911,303.643,9.802,13.713,123.08,60.0,0.0,13.68,64.35,7.89,0.0,0.0,36.55,1.08,0.0,0.36,0.77,0.151,0.0,0.0
dish_1561664977,470,2,train,47.0,0.846,14.57,0.047,3.29,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.846,0.0,0.0,0.0,0.0,0.0,0.0
dish_1561665024,470,2,train,121.0,2.141,100.571,6.985,8.949,0.0,47.0,0.0,7.02,58.34,8.64,0.0,0.0,0.0,0.846,0.0,0.246,1.05,0.0,0.0,0.0
dish_1561665065,470,2,train,303.0,47.427,387.798,15.232,14.161,144.02,47.0,0.0,21.42,78.32,12.24,0.0,0.0,44.645,0.846,0.0,0.62,1.316,0.0,0.0,0.0
dish_1561665378,471,1,train,70.0,2.17,74.27,1.68,15.4,0.0,0.0,0.0,70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.17,0.0,0.0,0.0,0.0
dish_1561665422,471,1,train,385.0,80.549,571.394,15.954,24.422,249.26,0.0,0.0,94.93,34.58,6.23,0.0,0.0,77.27,0.0,0.0,2.818,0.461,0.0,0.0,0.0
dish_1561665452,471,1,train,431.0,81.354,624.854,20.266,27.939,249.26,0.0,0.0,99.29,70.85,11.6,0.0,0.0,77.27,0.0,0.0,2.971,1.113,0.0,0.0,0.0
dish_1561665626,472,3,train,49.0,1.519,51.989,1.176,10.78,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.519,0.0,0.0,0.0,0.0
dish_1561665671,472,3,train,89.0,5.919,158.389,5.056,23.98,0.0,0.0,0.0,49.0,0.0,0.0,40.0,0.0,0.0,0.0,0.0,1.519,0.0,0.0,4.4,0.0
dish_1561665706,472,3,train,228.0,18.429,386.349,8.67,61.51,0.0,139.0,0.0,49.0,0.0,0.0,40.0,0.0,0.0,12.51,0.0,1.519,0.0,0.0,4.4,0.0
dish_1561665747,472,3,train,314.0,19.934,486.296,16.733,68.086,0.0,139.0,0.0,57.16,67.8,10.04,40.0,0.0,0.0,12.51,0.0,1.804,1.22,0.0,4.4,0.0
dish_1561666619,473,1,train,52.0,0.364,35.88,0.104,9.36,0.0,0.0,0.0,0.0,52.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.364,0.0,0.0,0.0
dish_1561666640,473,1,train,90.0,0.668,48.8,0.18,12.4,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.668,0.0,0.0,0.0
dish_1561666761,474,4,train,43.0,4.816,114.337,4.042,14.405,0.0,0.0,0.0,0.0,0.0,0.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.816,0.0
dish_1561666794,474,4,train,97.0,6.544,138.097,5.176,16.727,0.0,0.0,0.0,0.0,54.0,0.0,43.0,0.0,0.0,0.0,0.0,0.0,1.728,0.0,4.816,0.0
dish_1561666821,474,4,train,170.0,7.538,211.787,12.142,19.426,0.0,0.0,0.0,0.0,116.15,10.85,43.0,0.0,0.0,0.0,0.0,0.0,2.718,0.004,4.816,0.0
dish_1561666849,474,4,train,298.0,11.506,347.595,15.214,47.586,0.0,0.0,0.0,128.0,116.15,10.85,43.0,0.0,0.0,0.0,0.0,3.968,2.718,0.004,4.816,0.0
dish_1561666875,474,4,train,359.0,26.684,443.863,17.978,49.333,48.27,0.0,0.0,132.83,122.84,12.06,43.0,0.0,14.963,0.0,0.0,4.093,2.807,0.004,4.816,0.0
dish_1561666897,474,4,train,402.0,34.529,532.509,22.037,54.08,77.0,0.0,0.0,136.93,127.97,17.09,43.0,0.0,22.435,0.0,0.0,4.204,2.935,0.139,4.816,0.0
dish_1561666923,474,4,train,425.0,34.943,539.639,22.06,55.69,77.0,23.0,0.0,136.93,127.97,17.09,43.0,0.0,22.435,0.414,0.0,4.204,2.935,0.139,4.816,0.0
dish_1561667207,475,0,test,121.0,22.073,249.445,11.421,13.358,80.86,0.0,0.0,11.55,14.44,14.15,0.0,0.0,21.023,0.0,0.0,0.312,0.358,0.38,0.0,0.0
dish_1561667238,475,0,test,165.0,23.481,268.805,12.345,15.25,80.86,0.0,0.0,11.55,58.44,14.15,0.0,0.0,21.023,0.0,0.0,0.312,1.766,0.38,0.0,0.0
dish_1561667430,476,0,train,166.0,30.283,342.214,15.668,18.326,110.93,0.0,0.0,15.85,19.81,19.41,0.0,0.0,28.842,0.0,0.0,0.428,0.491,0.521,0.0,0.0
dish_1561736715,477,4,train,109.0,10.9,161.32,11.99,1.744,109.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.9,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1561736736,477,4,train,145.0,24.22,356.08,27.11,2.248,145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,24.22,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1561737776,479,3,train,139.0,12.955,375.3,11.12,56.017,0.0,0.0,27.8,111.2,0.0,0.0,0.0,0.0,0.0,0.0,1.724,11.231,0.0,0.0,0.0,0.0
dish_1561737839,479,3,train,370.0,2.59,210.9,1.11,51.8,0.0,0.0,0.0,0.0,370.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.59,0.0,0.0,0.0
dish_1561739576,481,0,train,84.0,0.84,63.84,0.0,15.12,0.0,0.0,0.0,84.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.84,0.0,0.0,0.0,0.0
dish_1561739623,481,0,train,348.0,7.176,156.24,1.056,33.6,0.0,0.0,0.0,84.0,264.0,0.0,0.0,0.0,0.0,0.0,0.0,0.84,6.336,0.0,0.0,0.0
dish_1561739649,481,0,train,388.0,8.456,208.234,3.843,39.756,0.0,0.0,0.0,90.45,294.97,2.58,0.0,0.0,0.0,0.0,0.0,1.485,6.971,0.0,0.0,0.0
dish_1561739719,481,0,train,637.0,10.199,350.164,4.59,74.616,0.0,0.0,0.0,90.45,543.97,2.58,0.0,0.0,0.0,0.0,0.0,1.485,8.714,0.0,0.0,0.0
dish_1561739760,481,0,train,937.0,12.599,452.164,5.19,98.616,0.0,0.0,0.0,90.45,843.97,2.58,0.0,0.0,0.0,0.0,0.0,1.485,11.114,0.0,0.0,0.0
dish_1561739805,481,0,train,1102.0,29.099,696.364,23.34,101.256,165.0,0.0,0.0,90.45,843.97,2.58,0.0,0.0,16.5,0.0,0.0,1.485,11.114,0.0,0.0,0.0
dish_1561748274,486,4,train,260.0,48.04,558.515,28.201,27.626,141.4,0.0,42.42,56.56,17.32,1.06,1.24,0.0,33.795,0.0,10.605,3.394,0.224,0.0,0.022,0.0
dish_1561748862,487,2,train,87.0,16.156,154.773,7.852,3.425,65.33,0.0,0.0,13.07,2.69,5.91,0.0,0.0,15.679,0.0,0.0,0.34,0.073,0.065,0.0,0.0
dish_1561748950,487,2,train,205.0,16.746,197.253,7.97,14.045,65.33,0.0,0.0,13.07,120.69,5.91,0.0,0.0,15.679,0.0,0.0,0.34,0.663,0.065,0.0,0.0
dish_1561749016,487,2,train,272.0,17.901,280.118,15.774,17.086,65.33,0.0,0.0,13.07,175.62,17.99,0.0,0.0,15.679,0.0,0.0,0.34,1.795,0.087,0.0,0.0
dish_1561749061,487,2,train,360.0,19.745,302.86,16.033,21.472,65.33,0.0,0.0,13.07,207.29,23.4,0.0,50.92,15.679,0.0,0.0,0.34,2.418,0.189,0.0,1.12
dish_1561749348,488,2,train,93.0,17.271,165.447,8.394,3.661,69.83,0.0,0.0,13.97,2.88,6.32,0.0,0.0,16.76,0.0,0.0,0.363,0.078,0.07,0.0,0.0
dish_1561749392,488,2,train,210.0,38.889,416.779,21.084,16.093,133.46,0.0,19.09,39.42,10.68,6.8,0.56,0.0,31.968,0.0,4.772,1.89,0.179,0.07,0.01,0.0
dish_1562007739,499,3,train,65.0,2.08,28.6,1.365,2.795,0.0,0.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.08,0.0,0.0,0.0
dish_1562007763,499,3,train,298.0,11.84,290.7,6.344,47.407,0.0,0.0,0.0,198.51,96.27,3.23,0.0,0.0,0.0,0.0,0.0,8.734,3.105,0.0,0.0,0.0
dish_1562007789,499,3,train,322.0,12.643,302.955,6.651,49.535,0.0,0.0,0.0,198.51,119.88,3.61,0.0,0.0,0.0,0.0,0.0,8.734,3.908,0.001,0.0,0.0
dish_1562007817,499,3,train,391.0,26.163,444.121,15.102,51.506,50.24,0.0,0.0,205.21,130.01,5.54,0.0,0.0,13.063,0.0,0.0,9.016,4.066,0.017,0.0,0.0
dish_1562007842,499,3,train,434.0,27.797,492.281,15.188,61.396,50.24,0.0,0.0,248.21,130.01,5.54,0.0,0.0,13.063,0.0,0.0,10.65,4.066,0.017,0.0,0.0
dish_1562007866,499,3,train,465.0,28.076,497.861,15.25,62.605,50.24,0.0,0.0,248.21,161.01,5.54,0.0,0.0,13.063,0.0,0.0,10.65,4.345,0.017,0.0,0.0
dish_1562007904,499,3,train,489.0,28.268,525.461,17.818,64.117,50.24,0.0,0.0,248.21,185.01,5.54,0.0,0.0,13.063,0.0,0.0,10.65,4.537,0.017,0.0,0.0
dish_1562008394,500,2,train,24.0,0.24,2.4,0.0,0.48,0.0,0.0,0.0,0.0,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.24,0.0,0.0,0.0
dish_1562008429,500,2,train,69.0,9.057,94.465,5.511,1.765,32.77,0.0,0.0,4.37,30.61,1.26,0.0,0.0,8.519,0.0,0.0,0.183,0.343,0.011,0.0,0.0
dish_1562008476,500,2,train,120.0,10.765,120.506,6.164,6.288,32.77,0.0,0.0,4.37,80.79,2.07,0.0,0.0,8.519,0.0,0.0,0.183,2.049,0.012,0.0,0.0
dish_1562008517,500,2,train,370.0,21.236,401.73,11.506,54.154,32.77,0.0,0.0,217.36,114.34,5.53,0.0,0.0,8.519,0.0,0.0,9.555,3.15,0.012,0.0,0.0
dish_1562008549,500,2,train,405.0,22.356,417.13,12.241,55.659,32.77,0.0,0.0,217.36,149.34,5.53,0.0,0.0,8.519,0.0,0.0,9.555,4.27,0.012,0.0,0.0
dish_1562008695,501,3,train,72.0,1.296,18.0,0.216,3.6,0.0,0.0,0.0,0.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.296,0.0,0.0,0.0
dish_1562008738,501,3,train,109.0,2.136,64.067,4.61,5.137,0.0,0.0,0.0,0.0,103.28,4.83,0.0,0.89,0.0,0.0,0.0,0.0,2.046,0.01,0.0,0.08
dish_1562008780,501,3,train,195.0,4.773,180.846,14.93,9.854,0.0,0.0,7.51,0.0,169.45,17.15,0.0,0.89,0.0,0.0,1.427,0.0,3.221,0.045,0.0,0.08
dish_1562008813,501,3,train,225.0,6.03,214.593,15.571,15.598,0.0,0.0,7.51,25.56,173.48,17.57,0.0,0.89,0.0,0.0,1.427,1.125,3.353,0.045,0.0,0.08
dish_1562008842,501,3,train,294.0,19.549,355.76,24.022,17.568,50.24,0.0,7.51,32.26,183.61,19.49,0.0,0.89,13.063,0.0,1.427,1.406,3.512,0.062,0.0,0.08
dish_1562008936,501,3,test,131.0,4.192,57.64,2.751,5.633,0.0,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.192,0.0,0.0,0.0
dish_1562008979,501,3,test,290.0,35.345,382.937,22.225,10.174,115.78,0.0,0.0,15.44,154.35,4.44,0.0,0.0,30.102,0.0,0.0,0.648,4.557,0.039,0.0,0.0
dish_1562009006,501,3,test,323.0,38.975,470.717,25.426,21.064,115.78,0.0,0.0,15.44,154.35,4.44,33.0,0.0,30.102,0.0,0.0,0.648,4.557,0.039,3.63,0.0
dish_1562009098,501,3,train,96.0,7.68,70.08,4.608,1.824,0.0,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.68,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562009120,501,3,train,146.0,9.354,95.61,5.248,6.258,0.0,96.0,0.0,0.0,49.2,0.8,0.0,0.0,0.0,7.68,0.0,0.0,1.673,0.001,0.0,0.0
dish_1562009148,501,3,train,210.0,21.894,226.547,13.086,8.085,46.6,96.0,0.0,6.21,58.6,2.59,0.0,0.0,12.117,7.68,0.0,0.261,1.82,0.017,0.0,0.0
dish_1562009182,501,3,train,274.0,23.348,306.23,20.687,10.745,46.6,96.0,0.0,6.21,112.7,10.95,0.0,1.54,12.117,7.68,0.0,0.261,3.117,0.035,0.0,0.138
dish_1562009314,502,3,train,195.0,6.529,99.568,2.495,17.292,0.0,0.0,0.0,0.0,191.88,3.12,0.0,0.0,0.0,0.0,0.0,0.0,6.524,0.005,0.0,0.0
dish_1562010821,505,1,train,134.0,26.255,274.149,16.412,3.827,97.57,0.0,0.0,13.01,19.68,3.74,0.0,0.0,25.369,0.0,0.0,0.546,0.307,0.033,0.0,0.0
dish_1562010865,505,1,train,216.0,28.118,376.243,26.15,7.234,97.57,0.0,0.0,13.01,89.0,14.45,0.0,1.97,25.369,0.0,0.0,0.546,1.97,0.055,0.0,0.177
dish_1562010891,505,1,train,341.0,28.743,438.743,26.275,23.484,97.57,0.0,0.0,13.01,214.0,14.45,0.0,1.97,25.369,0.0,0.0,0.546,2.595,0.055,0.0,0.177
dish_1562010929,505,1,train,405.0,29.311,457.266,27.096,25.63,97.57,0.0,0.0,13.01,277.13,15.32,0.0,1.97,25.369,0.0,0.0,0.546,3.163,0.055,0.0,0.177
dish_1562011727,506,0,train,182.0,35.66,372.352,22.291,5.197,132.52,0.0,0.0,17.67,26.73,5.08,0.0,0.0,34.456,0.0,0.0,0.742,0.417,0.044,0.0,0.0
dish_1562011765,506,0,train,283.0,39.041,423.923,23.583,14.154,132.52,0.0,0.0,17.67,126.11,6.7,0.0,0.0,34.456,0.0,0.0,0.742,3.796,0.047,0.0,0.0
dish_1562011794,506,0,train,364.0,42.119,514.643,23.745,32.784,132.52,0.0,0.0,98.67,126.11,6.7,0.0,0.0,34.456,0.0,0.0,3.82,3.796,0.047,0.0,0.0
dish_1562012048,507,1,train,248.0,10.388,278.974,5.299,47.484,0.0,0.0,0.0,211.29,33.28,3.43,0.0,0.0,0.0,0.0,0.0,9.297,1.091,0.0,0.0,0.0
dish_1562012076,507,1,train,438.0,15.518,525.974,5.869,100.684,0.0,0.0,0.0,401.29,33.28,3.43,0.0,0.0,0.0,0.0,0.0,14.427,1.091,0.0,0.0,0.0
dish_1562085163,508,0,train,144.0,18.99,279.93,21.11,2.27,144.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,18.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562085185,508,0,train,227.0,21.148,550.51,39.37,28.83,144.0,0.0,0.0,83.0,0.0,0.0,0.0,0.0,18.99,0.0,0.0,2.158,0.0,0.0,0.0,0.0
dish_1562086037,510,4,train,121.0,0.847,68.97,0.363,16.94,0.0,0.0,0.0,0.0,121.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.847,0.0,0.0,0.0
dish_1562086067,510,4,train,209.0,2.255,178.178,0.539,42.812,0.0,0.0,0.0,88.0,121.0,0.0,0.0,0.0,0.0,0.0,0.0,1.408,0.847,0.0,0.0,0.0
dish_1562086116,510,4,train,285.0,9.855,290.658,8.899,44.028,76.0,0.0,0.0,88.0,121.0,0.0,0.0,0.0,7.6,0.0,0.0,1.408,0.847,0.0,0.0,0.0
dish_1562094621,512,3,train,34.0,0.612,10.54,0.034,2.38,0.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.612,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562094694,512,3,train,142.0,2.246,134.596,11.607,7.25,0.0,34.0,0.0,0.0,90.71,17.29,0.0,0.0,0.0,0.612,0.0,0.0,1.634,0.0,0.0,0.0
dish_1562094730,512,3,train,336.0,39.126,428.194,24.716,11.798,149.23,34.0,0.0,14.92,119.25,17.85,0.0,0.75,35.666,0.612,0.0,0.657,2.19,0.0,0.0,0.001
dish_1562095283,514,4,train,197.0,5.269,229.517,19.747,11.773,0.0,0.0,0.0,0.0,166.83,30.17,0.0,0.0,0.0,0.0,0.0,0.0,5.225,0.044,0.0,0.0
dish_1562095340,514,4,train,381.0,25.509,718.957,37.595,72.493,0.0,0.0,0.0,0.0,166.83,30.17,184.0,0.0,0.0,0.0,0.0,0.0,5.225,0.044,20.24,0.0
dish_1562096343,515,0,train,62.0,0.62,6.2,0.0,1.24,0.0,0.0,0.0,0.0,62.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.62,0.0,0.0,0.0
dish_1562096372,515,0,train,287.0,43.393,346.714,15.204,6.515,173.08,0.0,0.0,17.31,95.1,0.65,0.0,0.87,41.365,0.0,0.0,0.762,1.265,0.0,0.0,0.001
dish_1562096401,515,0,train,352.0,44.562,366.864,15.269,11.065,173.08,65.0,0.0,17.31,95.1,0.65,0.0,0.87,41.365,1.17,0.0,0.762,1.265,0.0,0.0,0.001
dish_1562096512,515,0,train,25.0,0.492,30.34,2.739,1.373,0.0,0.0,0.0,0.0,20.97,4.03,0.0,0.0,0.0,0.0,0.0,0.0,0.492,0.0,0.0,0.0
dish_1562096542,515,0,train,50.0,0.942,38.09,2.764,3.123,0.0,25.0,0.0,0.0,20.97,4.03,0.0,0.0,0.0,0.45,0.0,0.0,0.492,0.0,0.0,0.0
dish_1562096584,515,0,train,90.0,2.651,94.991,4.667,11.279,0.0,25.0,0.0,38.83,20.97,5.2,0.0,0.0,0.0,0.45,0.0,1.709,0.492,0.0,0.0,0.0
dish_1562096608,515,0,train,163.0,13.095,176.712,13.218,14.912,49.87,36.97,0.0,38.83,29.94,7.38,0.0,0.0,9.176,1.517,0.0,1.709,0.691,0.001,0.0,0.0
dish_1562096633,515,0,train,237.0,27.162,288.703,18.219,16.647,106.79,36.97,0.0,44.53,40.83,7.6,0.0,0.28,22.781,1.517,0.0,1.959,0.903,0.001,0.0,0.0
dish_1562096907,516,0,train,266.0,38.057,297.78,31.159,13.238,181.73,43.61,0.0,0.0,32.71,7.95,0.0,0.0,33.437,3.889,0.0,0.0,0.726,0.005,0.0,0.0
dish_1562096933,516,0,train,314.0,38.921,312.66,31.207,16.598,181.73,91.61,0.0,0.0,32.71,7.95,0.0,0.0,33.437,4.753,0.0,0.0,0.726,0.005,0.0,0.0
dish_1562096973,516,0,train,350.0,39.411,349.0,34.643,17.929,181.73,91.61,0.0,0.0,63.36,13.3,0.0,0.0,33.437,4.753,0.0,0.0,1.214,0.007,0.0,0.0
dish_1562097001,516,0,train,418.0,46.891,529.88,41.239,40.369,181.73,91.61,0.0,0.0,63.36,13.3,68.0,0.0,33.437,4.753,0.0,0.0,1.214,0.007,7.48,0.0
dish_1562097624,517,2,train,68.0,2.773,77.586,2.311,12.61,0.0,0.0,0.0,66.02,0.0,1.98,0.0,0.0,0.0,0.0,0.0,2.773,0.0,0.0,0.0,0.0
dish_1562097653,517,2,train,102.0,3.375,116.166,5.921,14.058,0.0,0.0,0.0,66.02,28.59,7.39,0.0,0.0,0.0,0.0,0.0,2.773,0.598,0.004,0.0,0.0
dish_1562097673,517,2,train,307.0,32.704,345.657,29.935,24.261,140.05,33.61,0.0,66.02,53.8,13.52,0.0,0.0,25.769,2.997,0.0,2.773,1.157,0.008,0.0,0.0
dish_1562169729,521,4,train,50.0,1.674,25.53,0.64,4.434,0.0,0.0,0.0,0.0,49.2,0.8,0.0,0.0,0.0,0.0,0.0,0.0,1.673,0.001,0.0,0.0
dish_1562169762,521,4,train,210.0,17.674,262.33,18.24,6.994,160.0,0.0,0.0,0.0,49.2,0.8,0.0,0.0,16.0,0.0,0.0,0.0,1.673,0.001,0.0,0.0
dish_1562169818,521,4,train,275.0,19.754,346.82,22.769,16.997,160.0,0.0,0.0,10.48,99.52,4.99,0.0,0.0,16.0,0.0,0.0,1.048,2.704,0.001,0.0,0.0
dish_1562169840,521,4,train,389.0,41.756,739.664,53.891,21.785,274.0,0.0,0.0,10.48,99.52,4.99,0.0,0.0,38.002,0.0,0.0,1.048,2.704,0.001,0.0,0.0
dish_1562170163,522,3,train,107.0,0.535,53.5,0.107,13.91,0.0,0.0,0.0,0.0,107.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.535,0.0,0.0,0.0
dish_1562170236,522,3,train,333.0,2.069,169.36,0.737,42.478,0.0,0.0,0.0,0.0,333.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.069,0.0,0.0,0.0
dish_1562170331,522,3,train,400.0,3.476,263.83,5.293,54.538,0.0,0.0,0.0,67.0,333.0,0.0,0.0,0.0,0.0,0.0,0.0,1.407,2.069,0.0,0.0,0.0
dish_1562170356,522,3,train,432.0,12.788,385.334,14.285,55.53,32.0,0.0,0.0,67.0,333.0,0.0,0.0,0.0,9.312,0.0,0.0,1.407,2.069,0.0,0.0,0.0
dish_1562180025,526,1,train,256.0,59.066,798.914,55.476,12.931,175.68,0.0,27.94,0.0,2.62,35.79,0.0,13.97,56.622,0.0,0.922,0.0,0.098,0.0,0.0,1.425
dish_1562180089,526,1,train,437.0,60.673,851.299,57.797,19.001,175.68,0.0,27.94,0.0,181.16,38.25,0.0,13.97,56.622,0.0,0.922,0.0,1.705,0.0,0.0,1.425
dish_1562180157,526,1,train,557.0,63.838,1009.177,72.163,25.457,175.68,0.0,27.94,0.0,281.36,54.95,0.0,17.08,56.622,0.0,0.922,0.0,4.145,0.029,0.0,2.121
dish_1562181217,527,0,train,199.0,41.38,546.484,36.503,11.24,126.35,0.0,25.27,0.0,2.37,32.38,0.0,12.63,39.168,0.0,0.834,0.0,0.089,0.0,0.0,1.289
dish_1562181247,527,0,train,220.0,49.15,660.094,45.323,11.534,147.35,0.0,25.27,0.0,2.37,32.38,0.0,12.63,46.938,0.0,0.834,0.0,0.089,0.0,0.0,1.289
dish_1562181285,527,0,train,324.0,51.892,796.922,57.775,17.129,147.35,0.0,25.27,0.0,89.2,46.85,0.0,15.33,46.938,0.0,0.834,0.0,2.203,0.025,0.0,1.892
dish_1562183065,528,1,train,75.0,2.325,16.5,0.225,2.475,0.0,0.0,0.0,0.0,75.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.325,0.0,0.0,0.0
dish_1562183096,528,1,train,95.0,2.805,41.189,2.571,3.28,0.0,0.0,0.0,0.0,91.88,2.64,0.0,0.48,0.0,0.0,0.0,0.0,2.726,0.006,0.0,0.073
dish_1562183125,528,1,train,131.0,3.754,88.552,6.881,5.217,0.0,0.0,0.0,0.0,121.94,7.65,0.0,1.41,0.0,0.0,0.0,0.0,3.458,0.014,0.0,0.282
dish_1562183151,528,1,train,195.0,4.906,108.392,6.945,9.697,0.0,64.0,0.0,0.0,121.94,7.65,0.0,1.41,0.0,1.152,0.0,0.0,3.458,0.014,0.0,0.282
dish_1562183175,528,1,train,324.0,7.615,290.282,15.717,32.917,0.0,64.0,0.0,129.0,121.94,7.65,0.0,1.41,0.0,1.152,0.0,2.709,3.458,0.014,0.0,0.282
dish_1562183201,528,1,train,425.0,18.725,558.942,25.514,66.247,0.0,64.0,0.0,129.0,121.94,7.65,101.0,1.41,0.0,1.152,0.0,2.709,3.458,0.014,11.11,0.282
dish_1562184604,529,2,train,78.0,19.5,211.38,14.82,0.0,78.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562184628,529,2,train,121.0,35.41,444.01,32.88,0.602,121.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.41,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562184663,529,2,train,271.0,56.024,600.54,50.941,5.703,226.22,0.0,0.0,21.04,19.27,4.47,0.0,0.0,54.77,0.0,0.0,0.884,0.371,0.0,0.0,0.0
dish_1562184705,529,2,train,376.0,58.229,635.19,51.203,13.053,226.22,52.5,0.0,21.04,71.77,4.47,0.0,0.0,54.77,0.945,0.0,0.884,1.631,0.0,0.0,0.0
dish_1562603813,532,1,train,146.0,3.796,475.96,32.12,46.72,0.0,0.0,0.0,146.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.796,0.0,0.0,0.0,0.0
dish_1562603842,532,1,train,237.0,6.708,594.245,38.461,60.725,0.0,0.0,0.0,160.68,70.45,5.87,0.0,0.0,0.0,0.0,0.0,5.264,1.444,0.0,0.0,0.0
dish_1562603867,532,1,train,369.0,32.184,1049.117,74.497,66.269,132.0,0.0,0.0,160.68,70.45,5.87,0.0,0.0,25.476,0.0,0.0,5.264,1.444,0.0,0.0,0.0
dish_1562603895,532,1,train,517.0,46.984,1268.157,90.777,68.637,280.0,0.0,0.0,160.68,70.45,5.87,0.0,0.0,40.276,0.0,0.0,5.264,1.444,0.0,0.0,0.0
dish_1562605315,533,0,train,147.0,14.7,217.56,16.17,2.352,147.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562605352,533,0,train,195.0,16.307,242.069,16.784,6.608,147.0,0.0,0.0,0.0,47.23,0.77,0.0,0.0,14.7,0.0,0.0,0.0,1.606,0.001,0.0,0.0
dish_1562605384,533,0,train,262.0,16.961,306.193,18.418,18.374,147.0,0.0,0.0,65.37,47.23,2.4,0.0,0.0,14.7,0.0,0.0,0.654,1.606,0.001,0.0,0.0
dish_1562605411,533,0,train,363.0,17.567,336.493,18.62,26.05,147.0,0.0,0.0,65.37,148.23,2.4,0.0,0.0,14.7,0.0,0.0,0.654,2.212,0.001,0.0,0.0
dish_1562611522,534,0,train,50.0,5.6,132.95,4.7,16.75,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.6,0.0
dish_1562611580,534,0,train,50.0,6.0,141.0,6.0,16.0,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.0,0.0
dish_1562611620,534,0,train,76.0,11.093,199.845,9.792,17.089,19.87,0.0,0.0,2.65,3.31,0.17,50.0,0.0,4.968,0.0,0.0,0.072,0.054,0.0,6.0,0.0
dish_1562611649,534,0,train,146.0,12.508,278.489,17.102,20.045,19.87,0.0,0.0,2.65,62.01,11.46,50.0,0.0,4.968,0.0,0.0,0.072,1.452,0.016,6.0,0.0
dish_1562611680,534,0,train,256.0,16.028,326.889,19.412,24.775,19.87,0.0,0.0,2.65,172.01,11.46,50.0,0.0,4.968,0.0,0.0,0.072,4.972,0.016,6.0,0.0
dish_1562615283,540,4,train,78.0,15.28,176.536,11.376,3.266,59.62,0.0,0.0,7.95,9.94,0.5,0.0,0.0,14.904,0.0,0.0,0.215,0.161,0.0,0.0,0.0
dish_1562615324,540,4,train,136.0,16.498,258.316,15.32,13.706,59.62,0.0,0.0,65.95,9.94,0.5,0.0,0.0,14.904,0.0,0.0,1.433,0.161,0.0,0.0,0.0
dish_1562615360,540,4,train,191.0,17.891,331.482,20.402,20.034,59.62,0.0,1.17,65.95,29.78,34.49,0.0,0.0,14.904,0.0,0.222,1.433,0.486,0.846,0.0,0.0
dish_1562615388,540,4,train,273.0,19.859,360.182,20.73,25.774,59.62,0.0,1.17,65.95,111.78,34.49,0.0,0.0,14.904,0.0,0.222,1.433,2.454,0.846,0.0,0.0
dish_1562617370,541,4,train,55.0,6.6,155.1,6.6,17.6,0.0,0.0,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.6,0.0
dish_1562617395,541,4,train,125.0,8.84,185.9,8.07,20.61,0.0,0.0,0.0,0.0,70.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,2.24,0.0,6.6,0.0
dish_1562617416,541,4,train,168.0,9.872,200.95,8.242,23.62,0.0,0.0,0.0,0.0,113.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,3.272,0.0,6.6,0.0
dish_1562617446,541,4,train,232.0,11.216,291.19,12.594,35.14,0.0,0.0,0.0,64.0,113.0,0.0,55.0,0.0,0.0,0.0,0.0,1.344,3.272,0.0,6.6,0.0
dish_1562617470,541,4,train,321.0,29.482,440.867,19.671,37.177,75.11,0.0,0.0,71.51,117.79,1.6,55.0,0.0,17.95,0.0,0.0,1.539,3.392,0.0,6.6,0.0
dish_1562617492,541,4,train,373.0,39.668,558.558,27.255,39.354,114.85,0.0,0.0,76.81,124.41,1.93,55.0,0.0,27.887,0.0,0.0,1.682,3.5,0.0,6.6,0.0
dish_1562617703,542,2,test,110.0,4.51,179.63,6.49,26.95,0.0,0.0,110.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.51,0.0,0.0,0.0,0.0,0.0
dish_1562617939,543,3,test,133.0,7.239,234.966,14.641,20.434,0.0,66.62,0.0,0.0,23.32,43.06,0.0,0.0,0.0,5.996,0.0,0.0,0.626,0.616,0.0,0.0
dish_1562618000,543,3,test,448.0,59.16,685.994,36.104,30.821,206.75,66.62,0.0,20.68,104.79,49.16,0.0,0.0,49.414,5.996,0.0,0.538,2.596,0.616,0.0,0.0
dish_1562686492,544,1,train,100.0,14.0,160.0,10.0,3.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562686520,544,1,train,226.0,26.6,346.48,23.86,5.016,226.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,26.6,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562686548,544,1,train,280.0,27.464,413.494,23.968,20.892,226.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,26.6,0.0,0.0,0.864,0.0,0.0,0.0,0.0
dish_1562686577,544,1,train,531.0,29.221,556.564,24.721,56.032,226.0,0.0,0.0,54.0,251.0,0.0,0.0,0.0,26.6,0.0,0.0,0.864,1.757,0.0,0.0,0.0
dish_1562686664,544,1,train,209.0,1.463,119.13,0.627,29.26,0.0,0.0,0.0,0.0,209.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.463,0.0,0.0,0.0
dish_1562687246,545,4,train,61.0,0.601,115.422,11.9,2.317,0.0,0.0,0.0,0.0,42.15,18.85,0.0,0.0,0.0,0.0,0.0,0.0,0.574,0.027,0.0,0.0
dish_1562687274,545,4,train,141.0,1.161,161.022,12.14,13.517,0.0,0.0,0.0,0.0,122.15,18.85,0.0,0.0,0.0,0.0,0.0,0.0,1.134,0.027,0.0,0.0
dish_1562687299,545,4,train,196.0,1.436,180.822,12.195,18.467,0.0,0.0,0.0,0.0,177.15,18.85,0.0,0.0,0.0,0.0,0.0,0.0,1.409,0.027,0.0,0.0
dish_1562687527,546,2,train,210.0,3.396,203.222,13.595,18.887,14.0,0.0,0.0,0.0,177.15,18.85,0.0,0.0,1.96,0.0,0.0,0.0,1.409,0.027,0.0,0.0
dish_1562687554,546,2,train,281.0,9.005,409.335,23.748,42.104,14.0,0.0,0.0,71.0,177.15,18.85,0.0,0.0,1.96,0.0,0.0,5.609,1.409,0.027,0.0,0.0
dish_1562699570,553,3,train,22.0,0.572,24.42,0.198,5.06,0.0,0.0,0.0,22.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.572,0.0,0.0,0.0,0.0
dish_1562699612,553,3,train,74.0,2.793,98.391,2.672,15.662,0.0,0.0,0.0,72.49,0.0,1.51,0.0,0.0,0.0,0.0,0.0,2.793,0.0,0.0,0.0,0.0
dish_1562699652,553,3,train,103.0,3.569,132.178,5.579,17.395,0.0,0.0,0.0,72.49,24.56,5.96,0.0,0.0,0.0,0.0,0.0,2.793,0.769,0.006,0.0,0.0
dish_1562699699,553,3,train,217.0,26.858,372.113,19.353,20.778,87.05,0.0,0.0,84.09,39.07,6.79,0.0,0.0,22.633,0.0,0.0,3.2,1.019,0.006,0.0,0.0
dish_1562700885,555,2,train,117.0,2.106,29.25,0.351,5.85,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.106,0.0,0.0,0.0
dish_1562700906,555,2,train,222.0,3.038,59.639,1.697,9.372,0.0,0.0,0.0,0.0,220.58,1.42,0.0,0.0,0.0,0.0,0.0,0.0,3.038,0.0,0.0,0.0
dish_1562700931,555,2,train,312.0,4.614,164.235,10.135,16.254,0.0,0.0,0.0,8.54,291.53,11.93,0.0,0.0,0.0,0.0,0.0,0.299,4.315,0.0,0.0,0.0
dish_1562700957,555,2,train,424.0,20.524,286.934,23.181,21.41,75.68,18.16,0.0,12.32,301.18,16.66,0.0,0.0,13.924,1.514,0.0,0.427,4.628,0.03,0.0,0.0
dish_1562701956,556,4,train,38.0,0.342,6.84,0.076,1.482,0.0,0.0,0.0,0.0,38.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.342,0.0,0.0,0.0
dish_1562702007,556,4,train,100.0,0.892,24.784,0.871,3.561,0.0,0.0,0.0,0.0,99.16,0.84,0.0,0.0,0.0,0.0,0.0,0.0,0.892,0.0,0.0,0.0
dish_1562702044,556,4,train,177.0,2.278,44.034,1.102,7.411,0.0,0.0,0.0,0.0,176.16,0.84,0.0,0.0,0.0,0.0,0.0,0.0,2.278,0.0,0.0,0.0
dish_1562702091,556,4,train,214.0,2.838,86.535,5.067,9.08,0.0,0.0,0.0,0.0,207.23,6.77,0.0,0.0,0.0,0.0,0.0,0.0,2.838,0.0,0.0,0.0
dish_1562702132,556,4,train,359.0,23.436,245.386,21.956,15.756,97.97,23.51,0.0,4.9,219.73,12.89,0.0,0.0,18.027,1.959,0.0,0.167,3.244,0.039,0.0,0.0
dish_1562702355,557,3,train,99.0,1.641,68.883,5.822,3.473,0.0,0.0,4.21,0.0,86.36,8.43,0.0,0.0,0.0,0.0,0.581,0.0,1.06,0.0,0.0,0.0
dish_1562702379,557,3,train,227.0,3.945,100.883,6.206,9.873,0.0,0.0,4.21,0.0,214.36,8.43,0.0,0.0,0.0,0.0,0.581,0.0,3.364,0.0,0.0,0.0
dish_1562702412,557,3,train,285.0,6.423,183.389,8.965,21.699,0.0,0.0,4.21,56.31,214.36,10.11,0.0,0.0,0.0,0.0,0.581,2.478,3.364,0.0,0.0,0.0
dish_1562702441,557,3,train,371.0,6.939,209.189,9.137,28.235,0.0,0.0,4.21,56.31,300.36,10.11,0.0,0.0,0.0,0.0,0.581,2.478,3.88,0.0,0.0,0.0
dish_1562702469,557,3,train,529.0,39.216,541.73,28.228,32.924,120.65,0.0,4.21,72.4,320.47,11.27,0.0,0.0,31.369,0.0,0.581,3.041,4.225,0.0,0.0,0.0
dish_1562702813,558,3,train,59.0,0.978,41.051,3.47,2.07,0.0,0.0,2.51,0.0,51.47,5.02,0.0,0.0,0.0,0.0,0.346,0.0,0.631,0.0,0.0,0.0
dish_1562702853,558,3,train,169.0,23.45,272.567,16.761,5.335,84.0,0.0,2.51,11.2,65.47,5.83,0.0,0.0,21.839,0.0,0.346,0.392,0.872,0.0,0.0,0.0
dish_1562702892,558,3,train,338.0,47.457,457.711,36.445,13.116,198.19,27.41,2.51,16.91,80.03,12.96,0.0,0.0,42.85,2.284,0.346,0.586,1.345,0.046,0.0,0.0
dish_1562704528,560,2,train,56.0,6.272,148.904,5.264,18.76,0.0,0.0,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.272,0.0
dish_1562704564,560,2,train,124.0,20.164,292.023,13.48,20.778,51.92,0.0,0.0,6.92,8.65,0.5,56.0,0.0,13.5,0.0,0.0,0.242,0.149,0.0,6.272,0.0
dish_1562704616,560,2,train,213.0,20.954,317.781,14.622,23.763,51.92,0.0,0.0,6.92,96.45,1.7,56.0,0.0,13.5,0.0,0.0,0.242,0.939,0.0,6.272,0.0
dish_1562704640,560,2,train,275.0,22.07,333.281,14.808,26.863,51.92,0.0,0.0,6.92,158.45,1.7,56.0,0.0,13.5,0.0,0.0,0.242,2.055,0.0,6.272,0.0
dish_1562704665,560,2,train,399.0,26.038,387.841,17.412,32.195,51.92,0.0,0.0,6.92,282.45,1.7,56.0,0.0,13.5,0.0,0.0,0.242,6.023,0.0,6.272,0.0
dish_1562774099,561,0,test,40.0,1.28,51.994,2.787,6.156,0.0,0.0,0.0,6.45,30.97,2.58,0.0,0.0,0.0,0.0,0.0,0.645,0.635,0.0,0.0,0.0
dish_1562774129,561,0,test,77.0,1.539,73.084,2.898,11.336,0.0,0.0,0.0,6.45,67.97,2.58,0.0,0.0,0.0,0.0,0.0,0.645,0.894,0.0,0.0,0.0
dish_1562774159,561,0,test,129.0,1.955,90.764,3.002,15.496,0.0,0.0,0.0,6.45,119.97,2.58,0.0,0.0,0.0,0.0,0.0,0.645,1.31,0.0,0.0,0.0
dish_1562774180,561,0,test,181.0,3.047,164.084,6.538,24.856,0.0,0.0,0.0,58.45,119.97,2.58,0.0,0.0,0.0,0.0,0.0,1.737,1.31,0.0,0.0,0.0
dish_1562774206,561,0,test,272.0,12.147,298.764,16.548,26.312,91.0,0.0,0.0,58.45,119.97,2.58,0.0,0.0,9.1,0.0,0.0,1.737,1.31,0.0,0.0,0.0
dish_1562774980,563,3,train,201.0,4.824,136.68,2.814,24.12,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.824,0.0,0.0,0.0,0.0
dish_1562775047,563,3,train,461.0,6.644,284.88,3.594,60.52,0.0,0.0,0.0,201.0,260.0,0.0,0.0,0.0,0.0,0.0,0.0,4.824,1.82,0.0,0.0,0.0
dish_1562775110,563,3,train,282.0,1.41,101.52,0.282,25.38,0.0,0.0,0.0,0.0,282.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.41,0.0,0.0,0.0
dish_1562775212,563,3,train,136.0,3.264,92.48,1.904,16.32,0.0,0.0,0.0,136.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.264,0.0,0.0,0.0,0.0
dish_1562775241,563,3,train,220.0,3.852,140.36,2.156,28.08,0.0,0.0,0.0,136.0,84.0,0.0,0.0,0.0,0.0,0.0,0.0,3.264,0.588,0.0,0.0,0.0
dish_1562775275,563,3,train,318.0,4.636,173.68,2.352,35.92,0.0,0.0,0.0,136.0,182.0,0.0,0.0,0.0,0.0,0.0,0.0,3.264,1.372,0.0,0.0,0.0
dish_1562775356,563,3,train,120.0,12.0,177.6,13.2,1.92,120.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562775373,563,3,train,148.0,22.36,329.08,24.96,2.312,148.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,22.36,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562775456,563,3,train,66.0,0.66,50.16,0.0,11.88,0.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.66,0.0,0.0,0.0,0.0
dish_1562775488,563,3,train,145.0,8.56,91.24,0.158,12.433,79.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,7.9,0.0,0.0,0.66,0.0,0.0,0.0,0.0
dish_1562775510,563,3,train,190.0,10.067,114.217,0.734,16.423,79.0,0.0,0.0,66.0,44.28,0.72,0.0,0.0,7.9,0.0,0.0,0.66,1.506,0.001,0.0,0.0
dish_1562775549,563,3,train,252.0,16.887,174.977,3.4,18.531,79.0,0.0,62.0,66.0,44.28,0.72,0.0,0.0,7.9,0.0,6.82,0.66,1.506,0.001,0.0,0.0
dish_1562775574,563,3,train,387.0,17.562,242.477,3.535,36.081,79.0,0.0,62.0,66.0,179.28,0.72,0.0,0.0,7.9,0.0,6.82,0.66,2.181,0.001,0.0,0.0
dish_1562776428,564,2,test,66.0,2.21,33.7,0.844,5.853,0.0,0.0,0.0,0.0,64.94,1.06,0.0,0.0,0.0,0.0,0.0,0.0,2.208,0.002,0.0,0.0
dish_1562776465,564,2,test,212.0,3.634,173.432,4.405,31.492,0.0,0.0,0.0,142.44,64.94,4.62,0.0,0.0,0.0,0.0,0.0,1.424,2.208,0.002,0.0,0.0
dish_1562776497,564,2,test,292.0,11.634,291.832,13.205,32.772,80.0,0.0,0.0,142.44,64.94,4.62,0.0,0.0,8.0,0.0,0.0,1.424,2.208,0.002,0.0,0.0
dish_1562777000,565,1,train,132.0,13.2,195.36,14.52,2.112,132.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562777037,565,1,train,171.0,14.506,215.274,15.019,5.57,132.0,0.0,0.0,0.0,38.38,0.62,0.0,0.0,13.2,0.0,0.0,0.0,1.305,0.001,0.0,0.0
dish_1562777068,565,1,train,236.0,14.961,236.724,15.214,10.77,132.0,0.0,0.0,0.0,103.38,0.62,0.0,0.0,13.2,0.0,0.0,0.0,1.76,0.001,0.0,0.0
dish_1562777099,565,1,train,306.0,15.644,303.719,16.921,23.063,132.0,0.0,0.0,68.29,103.38,2.33,0.0,0.0,13.2,0.0,0.0,0.683,1.76,0.001,0.0,0.0
dish_1562784242,568,1,train,32.0,3.52,85.12,3.104,10.56,0.0,0.0,0.0,0.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.52,0.0
dish_1562784266,568,1,train,76.0,4.884,131.804,4.16,20.24,0.0,0.0,0.0,44.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,1.364,0.0,0.0,3.52,0.0
dish_1562784307,568,1,train,157.0,5.678,158.518,5.113,24.925,0.0,0.0,0.0,44.0,79.41,1.59,32.0,0.0,0.0,0.0,0.0,1.364,0.794,0.0,3.52,0.0
dish_1562784347,568,1,train,264.0,9.115,315.988,15.638,39.656,0.0,0.0,0.0,56.48,126.2,49.32,32.0,0.0,0.0,0.0,0.0,2.861,1.648,1.086,3.52,0.0
dish_1562784388,568,1,train,358.0,11.465,465.636,20.244,63.72,0.0,0.0,0.0,56.48,126.2,49.32,32.0,94.0,0.0,0.0,0.0,2.861,1.648,1.086,3.52,2.35
dish_1562785132,569,1,train,192.0,28.956,211.543,24.601,10.393,150.29,0.0,0.0,30.06,8.27,3.38,0.0,0.0,27.654,0.0,0.0,1.16,0.141,0.0,0.0,0.0
dish_1562785159,569,1,train,315.0,32.892,265.663,27.184,15.682,150.29,0.0,0.0,30.06,131.27,3.38,0.0,0.0,27.654,0.0,0.0,1.16,4.077,0.0,0.0,0.0
dish_1562785184,569,1,train,390.0,33.627,290.398,28.066,20.02,150.29,0.0,0.0,30.06,204.8,4.85,0.0,0.0,27.654,0.0,0.0,1.16,4.813,0.0,0.0,0.0
dish_1562785328,570,1,train,91.0,2.923,133.923,8.951,12.528,0.0,0.0,0.0,10.61,39.8,40.59,0.0,0.0,0.0,0.0,0.0,1.273,0.726,0.923,0.0,0.0
dish_1562785369,570,1,train,232.0,31.905,440.578,27.266,17.102,108.88,0.0,0.0,25.13,52.14,45.85,0.0,0.0,28.309,0.0,0.0,1.651,0.993,0.952,0.0,0.0
dish_1562786103,571,1,test,57.0,2.337,93.081,3.363,13.965,0.0,0.0,57.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.337,0.0,0.0,0.0,0.0,0.0
dish_1562789207,575,4,train,55.0,6.16,146.245,5.17,18.425,0.0,0.0,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.16,0.0
dish_1562789240,575,4,train,154.0,26.51,361.556,18.03,21.637,76.45,0.0,0.0,10.19,8.66,3.69,55.0,0.0,19.876,0.0,0.0,0.265,0.188,0.02,6.16,0.0
dish_1562789268,575,4,train,350.0,32.586,569.512,22.734,64.757,76.45,0.0,0.0,206.19,8.66,3.69,55.0,0.0,19.876,0.0,0.0,6.341,0.188,0.02,6.16,0.0
dish_1562789301,575,4,train,435.0,35.306,606.912,24.519,68.412,76.45,0.0,0.0,206.19,93.66,3.69,55.0,0.0,19.876,0.0,0.0,6.341,2.908,0.02,6.16,0.0
dish_1562789328,575,4,train,503.0,40.746,656.552,27.783,69.704,76.45,68.0,0.0,206.19,93.66,3.69,55.0,0.0,19.876,5.44,0.0,6.341,2.908,0.02,6.16,0.0
dish_1562790170,576,3,train,84.0,2.016,103.694,9.853,3.382,0.0,0.0,0.0,0.0,70.9,11.08,0.0,2.03,0.0,0.0,0.0,0.0,1.685,0.025,0.0,0.306
dish_1562790224,576,3,train,116.0,6.848,312.75,30.397,7.83,0.0,0.0,0.0,0.0,70.9,11.08,0.0,34.03,0.0,0.0,0.0,0.0,1.685,0.025,0.0,5.138
dish_1562790259,576,3,train,207.0,27.414,518.592,43.956,7.921,91.0,0.0,0.0,0.0,70.9,11.08,0.0,34.03,20.566,0.0,0.0,0.0,1.685,0.025,0.0,5.138
dish_1562790295,576,3,train,268.0,28.012,538.71,44.674,11.45,91.0,0.0,0.0,0.0,130.7,12.27,0.0,34.03,20.566,0.0,0.0,0.0,2.283,0.025,0.0,5.138
dish_1562790321,576,3,train,354.0,37.472,767.47,53.016,39.83,91.0,0.0,0.0,0.0,130.7,12.27,86.0,34.03,20.566,0.0,0.0,0.0,2.283,0.025,9.46,5.138
dish_1562790775,577,0,test,40.0,0.394,75.686,7.803,1.52,0.0,0.0,0.0,0.0,27.64,12.36,0.0,0.0,0.0,0.0,0.0,0.0,0.376,0.018,0.0,0.0
dish_1562790824,577,0,test,368.0,49.86,437.072,49.83,19.274,256.75,0.0,0.0,51.35,41.76,18.14,0.0,0.0,47.242,0.0,0.0,1.982,0.618,0.018,0.0,0.0
dish_1562790855,577,0,test,507.0,78.432,739.376,67.885,23.783,364.09,0.0,0.0,65.66,53.92,23.33,0.0,0.0,75.15,0.0,0.0,2.354,0.881,0.047,0.0,0.0
dish_1562859116,578,0,train,98.0,9.8,145.04,10.78,1.568,98.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562860045,579,0,train,123.0,19.05,280.29,21.28,1.918,123.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.05,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562860073,579,0,train,259.0,20.002,357.81,21.688,20.958,123.0,0.0,0.0,0.0,136.0,0.0,0.0,0.0,19.05,0.0,0.0,0.0,0.952,0.0,0.0,0.0
dish_1562860700,580,4,train,47.0,0.235,23.5,0.047,6.11,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.235,0.0,0.0,0.0
dish_1562860723,580,4,train,160.0,1.026,87.91,0.386,21.93,0.0,0.0,0.0,0.0,160.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.026,0.0,0.0,0.0
dish_1562861103,581,4,train,50.0,0.493,94.608,9.754,1.899,0.0,0.0,0.0,0.0,34.55,15.45,0.0,0.0,0.0,0.0,0.0,0.0,0.47,0.022,0.0,0.0
dish_1562861125,581,4,train,235.0,18.993,190.808,10.124,3.194,185.0,0.0,0.0,0.0,34.55,15.45,0.0,0.0,18.5,0.0,0.0,0.0,0.47,0.022,0.0,0.0
dish_1562861174,581,4,train,274.0,33.423,401.798,26.504,3.74,224.0,0.0,0.0,0.0,34.55,15.45,0.0,0.0,32.93,0.0,0.0,0.0,0.47,0.022,0.0,0.0
dish_1562862493,582,3,train,134.0,13.4,198.32,14.74,2.144,134.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.4,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562862550,582,3,train,257.0,18.83,364.52,19.19,29.103,134.0,0.0,10.6,42.4,70.0,0.0,0.0,0.0,13.4,0.0,0.657,4.282,0.49,0.0,0.0,0.0
dish_1562871731,584,4,train,131.0,1.965,17.03,0.262,2.882,0.0,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.965,0.0,0.0,0.0
dish_1562872150,585,3,train,34.0,1.054,7.48,0.102,1.122,0.0,0.0,0.0,0.0,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.054,0.0,0.0,0.0
dish_1562872182,585,3,train,296.0,4.984,41.54,0.626,6.886,0.0,0.0,0.0,0.0,296.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.984,0.0,0.0,0.0
dish_1562872223,585,3,train,468.0,47.903,306.863,8.377,10.478,135.5,0.0,0.0,13.55,315.99,2.96,0.0,0.0,42.005,0.0,0.0,0.569,5.327,0.002,0.0,0.0
dish_1562872365,586,2,train,76.0,1.14,9.88,0.152,1.672,0.0,0.0,0.0,0.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.14,0.0,0.0,0.0
dish_1562872396,586,2,train,186.0,28.588,179.563,5.109,3.969,86.66,0.0,0.0,8.67,88.78,1.9,0.0,0.0,26.864,0.0,0.0,0.364,1.359,0.001,0.0,0.0
dish_1562872432,586,2,train,257.0,31.286,259.083,5.251,20.299,86.66,0.0,0.0,79.67,88.78,1.9,0.0,0.0,26.864,0.0,0.0,3.062,1.359,0.001,0.0,0.0
dish_1562872470,586,2,train,282.0,34.486,286.382,8.331,21.325,103.02,0.0,0.0,82.94,91.07,3.82,0.0,1.15,29.875,0.0,0.0,3.173,1.399,0.027,0.0,0.011
dish_1562872498,586,2,train,320.0,35.153,306.265,9.318,23.918,103.02,37.05,0.0,82.94,91.07,4.77,0.0,1.15,29.875,0.667,0.0,3.173,1.399,0.027,0.0,0.011
dish_1562872662,587,2,train,122.0,15.617,133.216,15.03,5.005,79.87,0.0,0.0,15.97,11.18,9.38,0.0,5.59,14.696,0.0,0.0,0.543,0.194,0.128,0.0,0.056
dish_1562872746,587,2,train,203.0,17.217,242.364,23.448,12.913,79.87,0.0,0.0,15.97,79.29,22.28,0.0,5.59,14.696,0.0,0.0,0.543,1.767,0.156,0.0,0.056
dish_1562872914,588,0,train,124.0,15.873,135.4,15.276,5.087,81.18,0.0,0.0,16.24,11.36,9.54,0.0,5.68,14.937,0.0,0.0,0.552,0.198,0.13,0.0,0.057
dish_1562872945,588,0,train,234.0,20.053,258.6,15.496,30.387,81.18,0.0,0.0,126.24,11.36,9.54,0.0,5.68,14.937,0.0,0.0,4.732,0.198,0.13,0.0,0.057
dish_1562872981,588,0,train,288.0,20.539,268.32,15.604,32.493,81.18,0.0,0.0,126.24,65.36,9.54,0.0,5.68,14.937,0.0,0.0,4.732,0.684,0.13,0.0,0.057
dish_1562873002,588,0,train,307.0,20.873,278.261,16.098,33.79,81.18,18.52,0.0,126.24,65.36,10.01,0.0,5.68,14.937,0.333,0.0,4.732,0.684,0.13,0.0,0.057
dish_1562873028,588,0,train,320.0,20.964,280.341,16.124,34.232,81.18,18.52,0.0,126.24,78.36,10.01,0.0,5.68,14.937,0.333,0.0,4.732,0.775,0.13,0.0,0.057
dish_1562873155,589,0,train,79.0,1.386,41.337,2.052,5.392,0.0,77.03,0.0,0.0,0.0,1.98,0.0,0.0,0.0,1.386,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562873187,589,0,train,255.0,4.026,64.217,2.404,9.264,0.0,77.03,0.0,0.0,176.0,1.98,0.0,0.0,0.0,1.386,0.0,0.0,2.64,0.0,0.0,0.0
dish_1562873224,589,0,train,436.0,27.196,261.857,24.703,16.69,118.49,77.03,0.0,23.7,192.59,15.9,0.0,8.29,21.803,1.386,0.0,0.806,2.929,0.19,0.0,0.083
dish_1562873264,589,0,train,595.0,33.334,503.6,35.515,66.934,118.49,77.03,0.0,182.7,192.59,15.9,0.0,8.29,21.803,1.386,0.0,6.943,2.929,0.19,0.0,0.083
dish_1562873416,590,2,train,119.0,3.808,52.36,2.499,5.117,0.0,0.0,0.0,0.0,119.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.808,0.0,0.0,0.0
dish_1562873495,590,2,train,152.0,7.504,140.107,5.601,16.172,0.0,0.0,0.0,0.0,119.0,0.0,33.0,0.0,0.0,0.0,0.0,0.0,3.808,0.0,3.696,0.0
dish_1562873544,590,2,train,201.0,13.776,193.612,11.638,18.182,32.08,0.0,0.0,6.42,123.49,3.77,33.0,2.25,5.902,0.0,0.0,0.218,3.886,0.051,3.696,0.022
dish_1562874203,593,1,train,332.0,42.499,362.522,40.901,13.621,217.35,0.0,0.0,43.47,30.43,25.54,0.0,15.21,39.992,0.0,0.0,1.478,0.529,0.348,0.0,0.152
dish_1562874250,593,1,train,393.0,43.57,394.44,42.486,17.784,217.35,59.48,0.0,43.47,30.43,27.06,0.0,15.21,39.992,1.071,0.0,1.478,0.529,0.348,0.0,0.152
dish_1562874282,593,1,train,469.0,46.002,427.88,44.082,21.052,217.35,59.48,0.0,43.47,106.43,27.06,0.0,15.21,39.992,1.071,0.0,1.478,2.961,0.348,0.0,0.152
dish_1562875833,595,4,train,143.0,18.305,156.146,17.617,5.867,93.62,0.0,0.0,18.72,13.11,11.0,0.0,6.55,17.226,0.0,0.0,0.637,0.228,0.15,0.0,0.066
dish_1562875870,595,4,train,233.0,21.725,256.946,17.797,26.567,93.62,0.0,0.0,108.72,13.11,11.0,0.0,6.55,17.226,0.0,0.0,4.057,0.228,0.15,0.0,0.066
dish_1562875898,595,4,train,342.0,23.36,271.116,18.015,28.965,93.62,0.0,0.0,108.72,122.11,11.0,0.0,6.55,17.226,0.0,0.0,4.057,1.863,0.15,0.0,0.066
dish_1562875940,595,4,train,418.0,26.294,386.667,23.183,52.981,93.62,0.0,0.0,184.72,122.11,11.0,0.0,6.55,17.226,0.0,0.0,6.99,1.863,0.15,0.0,0.066
dish_1562945131,597,1,train,67.0,1.569,37.324,1.896,4.576,0.0,0.0,0.0,0.0,65.37,1.63,0.0,0.0,0.0,0.0,0.0,0.0,1.569,0.0,0.0,0.0
dish_1562945186,597,1,train,219.0,13.362,191.531,8.581,17.336,0.0,0.0,135.57,0.0,65.37,1.63,0.0,16.43,0.0,0.0,9.001,0.0,1.569,0.0,0.0,2.792
dish_1562945332,598,3,train,325.0,23.962,246.651,8.793,18.078,106.0,0.0,135.57,0.0,65.37,1.63,0.0,16.43,10.6,0.0,9.001,0.0,1.569,0.0,0.0,2.792
dish_1562945393,598,3,train,401.0,24.703,319.388,10.647,31.425,106.0,0.0,135.57,74.15,65.37,3.49,0.0,16.43,10.6,0.0,9.001,0.741,1.569,0.0,0.0,2.792
dish_1562945445,598,3,train,463.0,25.137,362.168,10.771,42.585,106.0,0.0,135.57,74.15,127.37,3.49,0.0,16.43,10.6,0.0,9.001,0.741,2.003,0.0,0.0,2.792
dish_1562945499,598,3,train,625.0,26.271,454.508,11.257,65.265,106.0,0.0,135.57,74.15,289.37,3.49,0.0,16.43,10.6,0.0,9.001,0.741,3.137,0.0,0.0,2.792
dish_1562945779,599,1,train,29.0,0.928,37.695,2.021,4.463,0.0,0.0,0.0,4.68,22.45,1.87,0.0,0.0,0.0,0.0,0.0,0.468,0.46,0.0,0.0,0.0
dish_1562945803,599,1,train,64.0,1.103,50.295,2.056,7.613,0.0,0.0,0.0,4.68,57.45,1.87,0.0,0.0,0.0,0.0,0.0,0.468,0.635,0.0,0.0,0.0
dish_1562945823,599,1,train,99.0,1.383,62.195,2.126,10.413,0.0,0.0,0.0,4.68,92.45,1.87,0.0,0.0,0.0,0.0,0.0,0.468,0.915,0.0,0.0,0.0
dish_1562945851,599,1,train,150.0,1.74,91.265,2.279,17.553,0.0,0.0,0.0,4.68,143.45,1.87,0.0,0.0,0.0,0.0,0.0,0.468,1.272,0.0,0.0,0.0
dish_1562945879,599,1,train,234.0,5.94,174.425,3.119,34.353,0.0,0.0,0.0,4.68,143.45,1.87,0.0,84.0,0.0,0.0,0.0,0.468,1.272,0.0,0.0,4.2
dish_1562945900,599,1,train,281.0,6.927,240.695,6.315,42.813,0.0,0.0,0.0,51.68,143.45,1.87,0.0,84.0,0.0,0.0,0.0,1.455,1.272,0.0,0.0,4.2
dish_1562946006,599,1,train,224.0,32.072,535.984,41.592,6.288,224.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,32.072,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562946075,599,1,train,297.0,32.51,557.884,41.738,11.836,224.0,0.0,0.0,0.0,73.0,0.0,0.0,0.0,32.072,0.0,0.0,0.0,0.438,0.0,0.0,0.0
dish_1562946106,599,1,train,359.0,33.006,578.964,41.862,16.796,224.0,0.0,0.0,0.0,135.0,0.0,0.0,0.0,32.072,0.0,0.0,0.0,0.934,0.0,0.0,0.0
dish_1562947020,600,3,train,333.0,35.679,533.117,40.119,5.28,317.14,0.0,15.86,0.0,0.0,0.0,0.0,0.0,31.714,0.0,3.964,0.0,0.0,0.0,0.0,0.0
dish_1562947376,601,3,train,73.0,0.365,36.5,0.073,9.49,0.0,0.0,0.0,0.0,73.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.365,0.0,0.0,0.0
dish_1562947409,601,3,train,190.0,1.184,103.19,0.424,25.87,0.0,0.0,0.0,0.0,190.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.184,0.0,0.0,0.0
dish_1562947436,601,3,train,242.0,11.22,282.382,14.62,28.054,52.0,0.0,0.0,0.0,190.0,0.0,0.0,0.0,10.036,0.0,0.0,0.0,1.184,0.0,0.0,0.0
dish_1562947469,601,3,train,320.0,12.858,392.362,19.924,42.094,52.0,0.0,0.0,78.0,190.0,0.0,0.0,0.0,10.036,0.0,0.0,1.638,1.184,0.0,0.0,0.0
dish_1562947503,601,3,train,408.0,21.658,438.122,20.1,42.71,140.0,0.0,0.0,78.0,190.0,0.0,0.0,0.0,18.836,0.0,0.0,1.638,1.184,0.0,0.0,0.0
dish_1562947598,601,3,train,258.0,2.064,87.72,0.516,20.64,0.0,0.0,0.0,0.0,258.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.064,0.0,0.0,0.0
dish_1562947885,602,2,train,313.0,2.191,215.97,0.626,56.34,0.0,0.0,0.0,0.0,313.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.191,0.0,0.0,0.0
dish_1562949205,603,0,train,57.0,4.8,80.22,5.13,3.768,12.0,0.0,0.0,0.0,45.0,0.0,0.0,0.0,4.44,0.0,0.0,0.0,0.36,0.0,0.0,0.0
dish_1562949242,603,0,train,119.0,5.405,139.559,6.642,14.656,12.0,0.0,0.0,60.49,45.0,1.51,0.0,0.0,4.44,0.0,0.0,0.605,0.36,0.0,0.0,0.0
dish_1562949467,604,0,train,72.0,1.054,24.656,1.897,1.545,0.0,0.0,0.0,0.0,70.24,1.76,0.0,0.0,0.0,0.0,0.0,0.0,1.054,0.0,0.0,0.0
dish_1562949494,604,0,train,149.0,1.593,68.546,2.128,12.325,0.0,0.0,0.0,0.0,147.24,1.76,0.0,0.0,0.0,0.0,0.0,0.0,1.593,0.0,0.0,0.0
dish_1562949541,604,0,train,374.0,31.733,494.226,33.718,16.525,225.0,0.0,0.0,0.0,147.24,1.76,0.0,0.0,30.14,0.0,0.0,0.0,1.593,0.0,0.0,0.0
dish_1562950018,605,4,train,142.0,2.982,200.22,9.656,25.56,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.982,0.0,0.0,0.0,0.0
dish_1562950057,605,4,train,223.0,11.082,320.1,18.566,26.856,81.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,8.1,0.0,0.0,2.982,0.0,0.0,0.0,0.0
dish_1562950091,605,4,train,290.0,24.013,550.982,36.857,29.67,148.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,21.031,0.0,0.0,2.982,0.0,0.0,0.0,0.0
dish_1562950895,606,4,train,144.0,1.405,137.819,3.512,25.288,0.0,0.0,0.0,140.49,0.0,3.51,0.0,0.0,0.0,0.0,0.0,1.405,0.0,0.0,0.0,0.0
dish_1562950934,606,4,train,464.0,33.405,304.219,4.152,27.528,320.0,0.0,0.0,140.49,0.0,3.51,0.0,0.0,32.0,0.0,0.0,1.405,0.0,0.0,0.0,0.0
dish_1562956486,607,3,train,142.0,11.36,103.66,6.816,2.698,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.36,0.0,0.0,0.0,0.0,0.0,0.0
dish_1562956510,607,3,train,199.0,21.758,221.167,12.196,8.991,38.09,142.0,0.0,5.44,6.8,6.67,0.0,0.0,9.904,11.36,0.0,0.147,0.169,0.179,0.0,0.0
dish_1562960437,610,3,train,69.0,1.449,97.29,4.692,12.42,0.0,0.0,0.0,69.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.449,0.0,0.0,0.0,0.0
dish_1562960472,610,3,train,166.0,4.553,139.97,6.729,16.591,0.0,0.0,0.0,69.0,97.0,0.0,0.0,0.0,0.0,0.0,0.0,1.449,3.104,0.0,0.0,0.0
dish_1562960520,610,3,train,303.0,29.545,422.399,19.66,31.716,91.55,0.0,0.0,82.08,113.35,16.02,0.0,0.0,23.803,0.0,0.0,1.802,3.509,0.43,0.0,0.0
dish_1562961588,611,2,train,67.0,7.37,178.22,6.499,22.11,0.0,0.0,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.37,0.0
dish_1562961609,611,2,train,186.0,29.079,423.542,17.731,35.247,79.52,0.0,0.0,11.36,14.2,13.92,67.0,0.0,20.676,0.0,0.0,0.307,0.352,0.374,7.37,0.0
dish_1562961650,611,2,train,312.0,45.516,555.713,32.363,40.293,162.97,0.0,0.0,28.05,33.18,20.8,67.0,0.0,36.03,0.0,0.0,1.041,0.634,0.441,7.37,0.0
dish_1563206562,616,1,train,68.0,0.834,98.209,3.913,15.204,0.0,0.0,0.0,64.15,0.0,3.85,0.0,0.0,0.0,0.0,0.0,0.834,0.0,0.0,0.0,0.0
dish_1563206582,616,1,train,320.0,26.034,229.249,4.417,16.968,252.0,0.0,0.0,64.15,0.0,3.85,0.0,0.0,25.2,0.0,0.0,0.834,0.0,0.0,0.0,0.0
dish_1563206601,616,1,train,337.0,30.981,293.798,9.194,17.495,269.0,0.0,0.0,64.15,0.0,3.85,0.0,0.0,30.147,0.0,0.0,0.834,0.0,0.0,0.0,0.0
dish_1563208094,618,2,train,111.0,1.361,160.311,6.388,24.818,0.0,0.0,0.0,104.72,0.0,6.28,0.0,0.0,0.0,0.0,0.0,1.361,0.0,0.0,0.0,0.0
dish_1563208129,618,2,train,291.0,19.361,426.711,26.188,27.698,180.0,0.0,0.0,104.72,0.0,6.28,0.0,0.0,18.0,0.0,0.0,1.361,0.0,0.0,0.0,0.0
dish_1563216625,621,2,train,109.0,29.255,171.535,4.27,2.408,89.53,0.0,2.24,8.95,7.83,0.45,0.0,0.0,27.843,0.0,0.859,0.376,0.176,0.0,0.0,0.0
dish_1563216659,621,2,train,173.0,29.823,190.058,5.091,4.555,89.53,0.0,2.24,8.95,70.97,1.32,0.0,0.0,27.843,0.0,0.859,0.376,0.744,0.0,0.0,0.0
dish_1563216689,621,2,train,302.0,32.919,235.208,5.607,13.585,89.53,0.0,2.24,8.95,199.97,1.32,0.0,0.0,27.843,0.0,0.859,0.376,3.84,0.0,0.0,0.0
dish_1563216717,621,2,train,332.0,33.069,246.008,5.637,16.285,89.53,0.0,2.24,8.95,229.97,1.32,0.0,0.0,27.843,0.0,0.859,0.376,3.99,0.0,0.0,0.0
dish_1563218408,624,1,test,87.0,10.44,245.34,10.44,27.84,0.0,0.0,0.0,0.0,0.0,0.0,87.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.44,0.0
dish_1563218959,625,1,train,131.0,26.745,267.94,14.862,4.85,100.38,0.0,0.0,13.38,16.4,0.84,0.0,0.0,26.1,0.0,0.0,0.361,0.284,0.0,0.0,0.0
dish_1563218987,625,1,train,172.0,31.665,383.56,19.782,17.97,100.38,0.0,0.0,13.38,16.4,0.84,41.0,0.0,26.1,0.0,0.0,0.361,0.284,0.0,4.92,0.0
dish_1563219009,625,1,train,202.0,31.875,388.36,19.842,18.99,100.38,0.0,0.0,13.38,46.4,0.84,41.0,0.0,26.1,0.0,0.0,0.361,0.494,0.0,4.92,0.0
dish_1563219035,625,1,train,221.0,32.027,410.21,21.875,20.187,100.38,0.0,0.0,13.38,65.4,0.84,41.0,0.0,26.1,0.0,0.0,0.361,0.646,0.0,4.92,0.0
dish_1563219056,625,1,train,258.0,32.36,416.87,21.949,21.63,100.38,0.0,0.0,13.38,102.4,0.84,41.0,0.0,26.1,0.0,0.0,0.361,0.979,0.0,4.92,0.0
dish_1563220767,627,1,train,130.0,26.541,265.894,14.749,4.813,99.62,0.0,0.0,13.28,16.27,0.83,0.0,0.0,25.9,0.0,0.0,0.359,0.282,0.0,0.0,0.0
dish_1563220802,627,1,train,252.0,29.469,308.594,15.237,13.353,99.62,0.0,0.0,13.28,138.27,0.83,0.0,0.0,25.9,0.0,0.0,0.359,3.21,0.0,0.0,0.0
dish_1563221495,629,4,train,164.0,31.254,365.27,13.375,29.243,64.07,0.0,1.6,6.41,5.61,0.32,86.0,0.0,19.924,0.0,0.615,0.269,0.126,0.0,10.32,0.0
dish_1563221519,629,4,train,206.0,32.262,379.97,13.543,32.183,64.07,0.0,1.6,6.41,47.61,0.32,86.0,0.0,19.924,0.0,0.615,0.269,1.134,0.0,10.32,0.0
dish_1563221562,629,4,train,260.0,32.742,395.599,14.236,33.994,64.07,0.0,1.6,6.41,100.87,1.05,86.0,0.0,19.924,0.0,0.615,0.269,1.613,0.0,10.32,0.0
dish_1563221602,629,4,train,306.0,33.676,447.894,19.034,36.099,64.07,0.0,1.6,6.41,139.24,8.68,86.0,0.0,19.924,0.0,0.615,0.269,2.526,0.022,10.32,0.0
dish_1563221640,629,4,train,364.0,34.722,469.186,19.881,39.257,64.07,0.0,1.6,6.41,184.76,21.17,86.0,0.0,19.924,0.0,0.615,0.269,3.229,0.364,10.32,0.0
dish_1563222745,630,3,train,48.0,5.76,135.36,5.76,15.36,0.0,0.0,0.0,0.0,0.0,0.0,48.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.76,0.0
dish_1563222775,630,3,train,75.0,11.272,190.584,8.823,16.36,20.69,0.0,0.0,2.76,3.38,0.17,48.0,0.0,5.379,0.0,0.0,0.074,0.059,0.0,5.76,0.0
dish_1563222800,630,3,train,146.0,30.328,302.318,11.604,17.928,79.01,0.0,1.46,8.59,8.48,0.46,48.0,0.0,23.516,0.0,0.56,0.319,0.173,0.0,5.76,0.0
dish_1563222830,630,3,train,211.0,31.888,325.068,11.864,22.478,79.01,0.0,1.46,8.59,73.48,0.46,48.0,0.0,23.516,0.0,0.56,0.319,1.733,0.0,5.76,0.0
dish_1563222850,630,3,train,232.0,32.434,348.378,12.053,27.308,79.01,0.0,1.46,29.59,73.48,0.46,48.0,0.0,23.516,0.0,0.56,0.865,1.733,0.0,5.76,0.0
dish_1563295016,632,0,test,79.0,6.965,127.88,7.621,8.182,18.0,0.0,0.0,0.0,61.0,0.0,0.0,0.0,6.66,0.0,0.0,0.0,0.305,0.0,0.0,0.0
dish_1563295044,632,0,test,158.0,7.934,241.975,12.167,25.845,18.0,0.0,0.0,74.53,61.0,4.47,0.0,0.0,6.66,0.0,0.0,0.969,0.305,0.0,0.0,0.0
dish_1563302315,634,2,train,63.0,1.953,13.86,0.189,2.079,0.0,0.0,0.0,0.0,63.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.953,0.0,0.0,0.0
dish_1563302345,634,2,train,215.0,43.001,265.855,7.567,5.093,130.02,0.0,0.0,13.0,69.22,2.76,0.0,0.0,40.306,0.0,0.0,0.546,2.149,0.0,0.0,0.0
dish_1563302391,634,2,train,311.0,45.017,401.215,14.095,22.373,130.02,0.0,0.0,109.0,69.22,2.76,0.0,0.0,40.306,0.0,0.0,2.562,2.149,0.0,0.0,0.0
dish_1563302420,634,2,train,381.0,45.639,435.626,14.924,29.278,130.02,0.0,0.0,109.0,138.22,3.75,0.0,0.0,40.306,0.0,0.0,2.562,2.77,0.001,0.0,0.0
dish_1563302447,634,2,train,440.0,48.123,561.417,27.386,31.362,130.02,0.0,6.03,109.0,174.63,16.31,0.0,4.02,40.306,0.0,1.265,2.562,3.373,0.01,0.0,0.607
dish_1563303298,635,0,train,103.0,27.816,170.76,5.0,2.042,88.1,0.0,0.0,8.81,4.21,1.87,0.0,0.0,27.312,0.0,0.0,0.37,0.133,0.0,0.0,0.0
dish_1563303350,635,0,train,167.0,28.384,202.221,5.757,8.355,88.1,0.0,0.0,8.81,67.31,2.78,0.0,0.0,27.312,0.0,0.0,0.37,0.701,0.001,0.0,0.0
dish_1563303386,635,0,train,218.0,29.455,274.131,9.225,17.535,88.1,0.0,0.0,59.81,67.31,2.78,0.0,0.0,27.312,0.0,0.0,1.441,0.701,0.001,0.0,0.0
dish_1563306392,640,0,train,288.0,77.775,477.465,13.98,5.71,246.35,0.0,0.0,24.64,11.78,5.23,0.0,0.0,76.369,0.0,0.0,1.035,0.372,0.0,0.0,0.0
dish_1563307488,641,4,train,247.0,50.928,538.035,30.636,9.611,190.24,0.0,0.0,25.37,25.37,4.76,1.27,0.0,49.463,0.0,0.0,0.685,0.654,0.103,0.023,0.0
dish_1563307519,641,4,train,335.0,51.72,574.115,30.812,18.411,190.24,0.0,0.0,25.37,113.37,4.76,1.27,0.0,49.463,0.0,0.0,0.685,1.446,0.103,0.023,0.0
dish_1563307575,641,4,train,419.0,53.94,671.082,38.148,25.062,190.24,0.0,0.0,42.34,166.61,18.54,1.27,0.0,49.463,0.0,0.0,1.432,2.783,0.238,0.023,0.0
dish_1563308323,642,1,train,66.0,0.594,27.06,0.132,6.6,0.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.594,0.0,0.0,0.0
dish_1563308353,642,1,train,140.0,1.319,51.465,1.003,10.88,0.0,0.0,0.0,0.0,138.55,1.45,0.0,0.0,0.0,0.0,0.0,0.0,1.319,0.0,0.0,0.0
dish_1563308386,642,1,train,214.0,2.873,155.805,6.035,24.2,0.0,0.0,0.0,74.0,138.55,1.45,0.0,0.0,0.0,0.0,0.0,1.554,1.319,0.0,0.0,0.0
dish_1563308428,642,1,train,329.0,26.585,406.308,20.298,28.675,88.58,0.0,0.0,85.81,150.36,3.67,0.59,0.0,23.03,0.0,0.0,1.873,1.624,0.048,0.011,0.0
dish_1563309189,643,2,train,33.0,0.264,37.95,3.531,2.079,0.0,0.0,0.0,0.0,33.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.264,0.0,0.0,0.0
dish_1563309237,643,2,train,62.0,0.844,49.55,3.589,4.689,0.0,0.0,0.0,0.0,62.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.844,0.0,0.0,0.0
dish_1563309285,643,2,train,128.0,1.768,59.45,3.721,6.603,0.0,0.0,0.0,0.0,128.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.768,0.0,0.0,0.0
dish_1563309315,643,2,train,462.0,91.966,613.177,19.934,13.226,285.7,0.0,0.0,28.57,141.66,6.07,0.0,0.0,88.567,0.0,0.0,1.2,2.199,0.0,0.0,0.0
dish_1563309353,643,2,train,529.0,92.569,625.237,20.068,15.839,285.7,0.0,0.0,28.57,208.66,6.07,0.0,0.0,88.567,0.0,0.0,1.2,2.802,0.0,0.0,0.0
dish_1563309398,643,2,train,604.0,93.244,655.987,20.218,23.339,285.7,0.0,0.0,28.57,283.66,6.07,0.0,0.0,88.567,0.0,0.0,1.2,3.477,0.0,0.0,0.0
dish_1563389042,652,0,train,62.0,0.551,30.478,0.734,6.116,0.0,0.0,0.0,0.0,61.12,0.88,0.0,0.0,0.0,0.0,0.0,0.0,0.55,0.001,0.0,0.0
dish_1563389079,652,0,train,119.0,1.919,50.428,0.962,10.106,0.0,0.0,0.0,0.0,118.12,0.88,0.0,0.0,0.0,0.0,0.0,0.0,1.918,0.001,0.0,0.0
dish_1563389114,652,0,train,148.0,2.578,86.534,4.405,11.311,0.0,0.0,0.0,0.0,142.64,4.67,0.0,0.7,0.0,0.0,0.0,0.0,2.506,0.009,0.0,0.063
dish_1563389153,652,0,train,224.0,11.698,346.454,5.393,69.071,0.0,0.0,0.0,76.0,142.64,4.67,0.0,0.7,0.0,0.0,0.0,9.12,2.506,0.009,0.0,0.063
dish_1563389199,652,0,train,284.0,16.498,390.254,8.273,70.211,0.0,60.0,0.0,76.0,142.64,4.67,0.0,0.7,0.0,4.8,0.0,9.12,2.506,0.009,0.0,0.063
dish_1563391333,654,3,train,81.0,0.719,23.443,1.039,2.717,0.0,0.0,0.0,0.0,79.9,1.1,0.0,0.0,0.0,0.0,0.0,0.0,0.719,0.0,0.0,0.0
dish_1563391375,654,3,train,139.0,8.285,84.284,7.774,5.039,38.41,0.0,0.0,7.68,88.64,4.27,0.0,0.0,7.068,0.0,0.0,0.338,0.849,0.031,0.0,0.0
dish_1563391413,654,3,train,237.0,26.228,227.057,14.304,6.997,112.06,0.0,0.0,12.7,103.6,4.59,4.05,0.0,24.67,0.0,0.0,0.413,1.042,0.031,0.073,0.0
dish_1563391647,655,2,train,22.0,0.5,27.391,2.613,0.914,0.0,0.0,0.0,0.0,18.6,2.87,0.0,0.53,0.0,0.0,0.0,0.0,0.446,0.006,0.0,0.048
dish_1563391681,655,2,train,143.0,16.284,154.317,16.664,5.76,80.13,0.0,0.0,16.03,36.83,9.48,0.0,0.53,14.744,0.0,0.0,0.705,0.717,0.07,0.0,0.048
dish_1563391708,655,2,train,205.0,23.104,319.237,22.678,26.22,80.13,0.0,0.0,16.03,36.83,9.48,62.0,0.53,14.744,0.0,0.0,0.705,0.717,0.07,6.82,0.048
dish_1563391922,656,1,train,104.0,1.525,24.44,0.381,4.541,0.0,0.0,0.0,0.0,104.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.525,0.0,0.0,0.0
dish_1563391959,656,1,train,315.0,29.051,245.774,24.885,12.991,139.74,0.0,0.0,27.95,135.79,11.53,0.0,0.0,25.711,0.0,0.0,1.23,1.998,0.112,0.0,0.0
dish_1563391989,656,1,train,464.0,56.331,462.848,34.812,15.968,251.71,0.0,0.0,35.58,158.54,12.02,6.16,0.0,52.474,0.0,0.0,1.344,2.291,0.112,0.111,0.0
dish_1563395341,659,0,train,54.0,6.048,143.586,5.076,18.09,0.0,0.0,0.0,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.048,0.0
dish_1563395374,659,0,train,84.0,6.315,158.333,5.431,21.049,0.0,0.0,0.0,0.0,29.57,0.43,54.0,0.0,0.0,0.0,0.0,0.0,0.266,0.0,6.048,0.0
dish_1563395415,659,0,train,187.0,7.825,182.538,5.809,25.547,0.0,0.0,0.0,0.0,132.57,0.43,54.0,0.0,0.0,0.0,0.0,0.0,1.777,0.0,6.048,0.0
dish_1563395449,659,0,train,222.0,9.571,214.636,6.977,29.682,0.0,6.05,1.82,4.24,154.07,1.67,54.15,0.0,0.0,0.539,0.454,0.207,2.291,0.029,6.051,0.0
dish_1563395480,659,0,train,329.0,29.162,370.521,14.107,31.82,80.41,6.05,1.82,9.72,170.4,2.03,58.57,0.0,19.219,0.539,0.454,0.289,2.501,0.029,6.13,0.0
dish_1563395510,659,0,train,372.0,34.771,415.627,19.1,33.542,108.89,6.05,1.82,15.41,176.88,4.37,58.57,0.0,24.459,0.539,0.454,0.54,2.597,0.052,6.13,0.0
dish_1563465805,661,4,train,23.0,0.483,32.43,1.564,4.14,0.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.483,0.0,0.0,0.0,0.0
dish_1563465847,661,4,train,182.0,16.383,267.75,19.054,6.684,159.0,0.0,0.0,23.0,0.0,0.0,0.0,0.0,15.9,0.0,0.0,0.483,0.0,0.0,0.0,0.0
dish_1563476267,666,2,train,51.0,3.162,237.711,14.688,25.398,0.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.162
dish_1563476323,666,2,train,339.0,55.328,686.971,33.461,38.625,208.88,0.0,0.0,41.78,34.73,2.09,0.52,51.0,50.132,0.0,0.0,1.086,0.938,0.0,0.009,3.162
dish_1563476408,666,2,train,513.0,79.2,990.4,47.925,55.908,285.16,0.0,0.0,102.8,63.14,8.86,0.52,52.53,69.964,0.0,0.0,4.198,1.706,0.159,0.009,3.164
dish_1563476463,666,2,train,542.0,82.448,1067.511,50.651,65.623,285.16,0.0,0.0,102.8,63.14,8.86,29.52,52.53,69.964,0.0,0.0,4.198,1.706,0.159,3.257,3.164
dish_1563476551,666,2,train,750.0,85.498,1116.391,51.414,74.706,285.16,0.0,0.0,102.8,271.14,8.86,29.52,52.53,69.964,0.0,0.0,4.198,4.757,0.159,3.257,3.164
dish_1563476577,666,2,train,857.0,87.672,1238.034,62.574,79.601,285.16,0.0,0.0,102.8,360.39,26.61,29.52,52.53,69.964,0.0,0.0,4.198,6.88,0.209,3.257,3.164
dish_1563476646,666,2,train,218.0,29.909,380.158,18.122,21.653,95.56,0.0,0.0,76.45,35.6,8.48,0.0,1.91,24.846,0.0,0.0,3.899,0.963,0.199,0.0,0.002
dish_1563477659,667,0,train,52.0,1.741,26.551,0.665,4.611,0.0,0.0,0.0,0.0,51.17,0.83,0.0,0.0,0.0,0.0,0.0,0.0,1.74,0.001,0.0,0.0
dish_1563477701,667,0,train,150.0,19.492,179.425,7.053,9.112,71.08,0.0,0.0,14.22,62.99,1.54,0.18,0.0,17.059,0.0,0.0,0.37,2.059,0.001,0.003,0.0
dish_1563477736,667,0,train,234.0,20.962,277.047,14.929,15.535,71.08,0.0,0.0,22.18,129.21,11.35,0.18,0.0,17.059,0.0,0.0,0.648,3.25,0.001,0.003,0.0
dish_1563478751,668,4,test,545.0,74.772,950.395,45.304,54.133,238.9,0.0,0.0,191.12,88.99,21.2,0.0,4.78,62.115,0.0,0.0,9.747,2.408,0.497,0.0,0.005
dish_1563479526,669,1,train,51.0,1.291,67.845,4.713,5.868,0.0,0.0,1.08,0.0,18.4,31.52,0.0,0.0,0.0,0.0,0.206,0.0,0.301,0.785,0.0,0.0
dish_1563479568,669,1,train,82.0,2.329,83.674,5.11,8.617,0.0,0.0,1.08,0.0,48.9,32.02,0.0,0.0,0.0,0.0,0.206,0.0,1.338,0.785,0.0,0.0
dish_1563479596,669,1,train,151.0,3.778,104.374,5.351,12.757,0.0,0.0,1.08,0.0,117.9,32.02,0.0,0.0,0.0,0.0,0.206,0.0,2.787,0.785,0.0,0.0
dish_1563479625,669,1,train,214.0,15.189,202.649,9.458,15.65,45.69,0.0,1.08,9.14,125.5,32.47,0.11,0.0,10.966,0.0,0.206,0.238,2.992,0.785,0.002,0.0
dish_1563479655,669,1,train,282.0,24.519,321.231,15.11,22.405,75.5,0.0,1.08,32.99,136.6,35.12,0.11,0.6,18.717,0.0,0.206,1.454,3.293,0.847,0.002,0.001
dish_1563480476,670,0,train,332.0,60.135,517.897,21.641,15.248,240.8,0.0,0.0,48.16,40.03,2.41,0.6,0.0,57.791,0.0,0.0,1.252,1.081,0.0,0.011,0.0
dish_1563480505,670,0,train,475.0,79.754,767.267,33.528,29.452,303.48,0.0,0.0,98.31,63.38,7.97,0.6,1.25,74.09,0.0,0.0,3.81,1.713,0.13,0.011,0.001
dish_1563480534,670,0,train,576.0,81.875,797.567,33.882,35.512,303.48,0.0,0.0,98.31,164.38,7.97,0.6,1.25,74.09,0.0,0.0,3.81,3.834,0.13,0.011,0.001
dish_1563480608,670,0,train,650.0,83.223,886.946,40.421,42.645,303.48,0.0,0.0,98.31,228.21,18.14,0.6,1.25,74.09,0.0,0.0,3.81,5.166,0.145,0.011,0.001
dish_1563481030,671,3,train,129.0,2.348,155.81,11.4,12.434,0.0,0.0,0.0,0.0,111.27,17.73,0.0,0.0,0.0,0.0,0.0,0.0,2.323,0.026,0.0,0.0
dish_1563481061,671,3,train,350.0,32.669,541.2,29.771,34.385,96.88,0.0,0.0,77.5,147.35,26.33,0.0,1.94,25.188,0.0,0.0,3.953,3.299,0.227,0.0,0.002
dish_1563550584,672,2,train,145.0,14.5,75.4,0.29,1.015,145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.5,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563550614,672,2,train,205.0,15.085,132.824,1.753,11.552,145.0,0.0,0.0,58.54,0.0,1.46,0.0,0.0,14.5,0.0,0.0,0.585,0.0,0.0,0.0,0.0
dish_1563551360,674,1,train,218.0,21.8,322.64,23.98,3.488,218.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563551397,674,1,train,302.0,24.612,365.531,25.055,10.937,218.0,0.0,0.0,0.0,82.66,1.34,0.0,0.0,21.8,0.0,0.0,0.0,2.81,0.002,0.0,0.0
dish_1563552016,675,3,train,174.0,10.739,493.589,19.477,69.953,0.0,0.0,0.0,135.94,0.0,38.06,0.0,0.0,0.0,0.0,0.0,10.739,0.0,0.0,0.0,0.0
dish_1563553735,676,4,train,88.0,2.816,114.386,6.132,13.543,0.0,0.0,0.0,14.19,68.13,5.68,0.0,0.0,0.0,0.0,0.0,1.419,1.397,0.0,0.0,0.0
dish_1563553756,676,4,train,124.0,16.136,309.146,21.252,14.047,36.0,0.0,0.0,14.19,68.13,5.68,0.0,0.0,13.32,0.0,0.0,1.419,1.397,0.0,0.0,0.0
dish_1563553780,676,4,train,167.0,25.768,557.729,42.537,23.808,36.0,0.0,0.0,14.19,68.13,5.68,0.0,43.0,13.32,0.0,0.0,1.419,1.397,0.0,0.0,9.632
dish_1563555072,677,3,train,112.0,11.2,165.76,12.32,1.792,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563555100,677,3,train,186.0,11.792,190.92,12.468,7.712,112.0,0.0,0.0,0.0,74.0,0.0,0.0,0.0,11.2,0.0,0.0,0.0,0.592,0.0,0.0,0.0
dish_1563555160,677,3,train,287.0,13.774,268.833,14.444,21.797,112.0,0.0,0.0,57.56,115.33,2.11,0.0,0.0,11.2,0.0,0.0,0.576,1.997,0.001,0.0,0.0
dish_1563563240,679,2,train,42.0,0.336,48.3,4.494,2.646,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.336,0.0,0.0,0.0
dish_1563563264,679,2,train,163.0,27.682,322.002,22.523,2.767,121.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,27.346,0.0,0.0,0.0,0.336,0.0,0.0,0.0
dish_1563563289,679,2,train,218.0,28.177,331.902,22.633,4.912,121.0,0.0,0.0,0.0,97.0,0.0,0.0,0.0,27.346,0.0,0.0,0.0,0.831,0.0,0.0,0.0
dish_1563563317,679,2,train,252.0,28.483,345.842,22.701,8.312,121.0,0.0,0.0,0.0,131.0,0.0,0.0,0.0,27.346,0.0,0.0,0.0,1.137,0.0,0.0,0.0
dish_1563563348,679,2,train,294.0,29.323,362.642,22.785,12.092,121.0,0.0,0.0,0.0,173.0,0.0,0.0,0.0,27.346,0.0,0.0,0.0,1.977,0.0,0.0,0.0
dish_1563563378,679,2,train,351.0,29.95,385.442,22.842,17.222,121.0,0.0,0.0,0.0,230.0,0.0,0.0,0.0,27.346,0.0,0.0,0.0,2.604,0.0,0.0,0.0
dish_1563563467,679,2,train,119.0,1.741,40.75,3.135,2.554,0.0,0.0,0.0,0.0,116.1,2.9,0.0,0.0,0.0,0.0,0.0,0.0,1.741,0.0,0.0,0.0
dish_1563563492,679,2,train,176.0,8.011,192.37,8.664,21.364,0.0,0.0,0.0,0.0,116.1,2.9,57.0,0.0,0.0,0.0,0.0,0.0,1.741,0.0,6.27,0.0
dish_1563563527,679,2,train,302.0,25.201,322.906,23.718,25.66,87.8,0.0,0.0,17.56,133.0,6.63,57.0,0.0,16.156,0.0,0.0,0.738,2.038,0.0,6.27,0.0
dish_1563564676,680,4,train,62.0,3.844,50.624,1.979,5.198,0.0,0.0,5.05,0.0,48.49,8.08,0.0,0.38,0.0,0.0,1.94,0.0,1.6,0.259,0.0,0.045
dish_1563564707,680,4,train,150.0,6.572,143.992,4.091,24.558,0.0,0.0,5.05,88.0,48.49,8.08,0.0,0.38,0.0,0.0,1.94,2.728,1.6,0.259,0.0,0.045
dish_1563564760,680,4,train,243.0,23.54,341.852,16.775,27.887,66.39,0.0,5.05,96.53,63.79,10.85,0.0,0.38,16.298,0.0,1.94,2.958,1.993,0.306,0.0,0.045
dish_1563564797,680,4,train,310.0,32.681,411.264,24.78,30.171,113.08,0.0,5.05,105.87,72.78,12.84,0.0,0.38,24.889,0.0,1.94,3.351,2.15,0.306,0.0,0.045
dish_1563564828,680,4,train,355.0,34.571,452.214,25.005,38.766,113.08,0.0,5.05,150.87,72.78,12.84,0.0,0.38,24.889,0.0,1.94,5.241,2.15,0.306,0.0,0.045
dish_1563565683,682,4,train,63.0,8.658,65.743,7.585,2.142,44.19,0.0,0.0,8.84,8.09,1.88,0.0,0.0,8.131,0.0,0.0,0.371,0.156,0.0,0.0,0.0
dish_1563565720,682,4,train,149.0,11.324,84.663,7.843,4.98,44.19,0.0,0.0,8.84,94.09,1.88,0.0,0.0,8.131,0.0,0.0,0.371,2.822,0.0,0.0,0.0
dish_1563565761,682,4,train,159.0,11.434,88.663,7.853,5.88,44.19,0.0,0.0,8.84,104.09,1.88,0.0,0.0,8.131,0.0,0.0,0.371,2.932,0.0,0.0,0.0
dish_1563565805,682,4,train,360.0,14.376,157.493,13.148,10.194,44.19,0.0,0.0,8.84,300.19,6.78,0.0,0.0,8.131,0.0,0.0,0.371,5.873,0.0,0.0,0.0
dish_1563567599,685,3,train,156.0,3.544,194.227,18.526,6.482,0.0,0.0,0.0,0.0,131.87,20.38,0.0,3.75,0.0,0.0,0.0,0.0,3.163,0.043,0.0,0.337
dish_1563567626,685,3,train,248.0,6.396,291.839,20.734,26.722,0.0,0.0,0.0,92.0,131.87,20.38,0.0,3.75,0.0,0.0,0.0,2.852,3.163,0.043,0.0,0.337
dish_1563567663,685,3,train,461.0,35.668,514.112,46.38,33.965,149.41,0.0,0.0,121.88,159.23,26.73,0.0,3.75,27.491,0.0,0.0,4.107,3.689,0.043,0.0,0.337
dish_1563567722,685,3,train,576.0,40.357,645.324,50.288,55.29,149.41,0.0,0.0,233.53,159.23,30.08,0.0,3.75,27.491,0.0,0.0,8.796,3.689,0.043,0.0,0.337
dish_1563567978,686,2,train,114.0,15.667,118.963,13.726,3.876,79.96,0.0,0.0,15.99,14.64,3.4,0.0,0.0,14.714,0.0,0.0,0.672,0.282,0.0,0.0,0.0
dish_1563568003,686,2,train,135.0,15.974,126.154,14.279,4.327,79.96,0.0,0.0,15.99,35.13,3.91,0.0,0.0,14.714,0.0,0.0,0.672,0.589,0.0,0.0,0.0
dish_1563568025,686,2,train,247.0,19.446,244.986,16.967,28.967,79.96,0.0,0.0,127.99,35.13,3.91,0.0,0.0,14.714,0.0,0.0,4.144,0.589,0.0,0.0,0.0
dish_1563568118,686,2,train,134.0,18.416,139.834,16.134,4.556,93.99,0.0,0.0,18.8,17.21,3.99,0.0,0.0,17.295,0.0,0.0,0.79,0.331,0.0,0.0,0.0
dish_1563568148,686,2,train,225.0,19.747,170.996,18.531,6.51,93.99,0.0,0.0,18.8,105.99,6.21,0.0,0.0,17.295,0.0,0.0,0.79,1.663,0.0,0.0,0.0
dish_1563568263,686,2,train,45.0,5.04,119.655,4.23,15.075,0.0,0.0,0.0,0.0,0.0,0.0,45.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.04,0.0
dish_1563568292,686,2,train,193.0,9.628,276.683,7.782,47.635,0.0,0.0,0.0,148.0,0.0,0.0,45.0,0.0,0.0,0.0,0.0,4.588,0.0,0.0,5.04,0.0
dish_1563568319,686,2,train,347.0,37.726,604.323,28.785,53.149,109.94,0.0,0.0,162.13,25.34,4.59,45.0,0.0,26.988,0.0,0.0,4.969,0.65,0.079,5.04,0.0
dish_1563568338,686,2,train,371.0,38.705,631.707,29.601,57.599,109.94,0.0,0.0,185.43,25.34,5.29,45.0,0.0,26.988,0.0,0.0,5.948,0.65,0.079,5.04,0.0
dish_1563568357,686,2,train,448.0,42.896,767.74,38.077,69.429,109.94,38.57,0.0,185.43,38.84,30.22,45.0,0.0,26.988,3.472,0.0,5.948,1.012,0.436,5.04,0.0
dish_1563811412,687,4,train,217.0,21.7,112.84,0.434,1.519,217.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563811439,687,4,train,289.0,22.276,137.32,0.578,7.279,217.0,0.0,0.0,0.0,72.0,0.0,0.0,0.0,21.7,0.0,0.0,0.0,0.576,0.0,0.0,0.0
dish_1563811463,687,4,train,376.0,23.125,220.585,2.7,22.557,217.0,0.0,0.0,84.88,72.0,2.12,0.0,0.0,21.7,0.0,0.0,0.849,0.576,0.0,0.0,0.0
dish_1563811490,687,4,train,390.0,25.827,268.829,6.522,23.145,231.0,0.0,0.0,84.88,72.0,2.12,0.0,0.0,24.402,0.0,0.0,0.849,0.576,0.0,0.0,0.0
dish_1563811515,687,4,train,491.0,27.342,298.119,6.724,29.811,231.0,0.0,0.0,84.88,72.0,103.12,0.0,0.0,24.402,0.0,0.0,0.849,0.576,1.515,0.0,0.0
dish_1563811611,687,4,train,132.0,13.2,195.36,14.52,2.112,132.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563811639,687,4,train,198.0,13.662,217.14,14.718,7.392,132.0,0.0,0.0,0.0,66.0,0.0,0.0,0.0,13.2,0.0,0.0,0.0,0.462,0.0,0.0,0.0
dish_1563811662,687,4,train,254.0,15.537,245.734,15.434,12.358,132.0,0.0,0.0,0.0,121.1,0.9,0.0,0.0,13.2,0.0,0.0,0.0,2.336,0.001,0.0,0.0
dish_1563811686,687,4,train,331.0,16.288,319.428,17.313,25.88,132.0,0.0,0.0,75.12,121.1,2.77,0.0,0.0,13.2,0.0,0.0,0.751,2.336,0.001,0.0,0.0
dish_1563812508,688,2,train,217.0,21.7,112.84,0.434,1.519,217.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563812535,688,2,train,266.0,35.959,298.893,14.203,3.038,266.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.959,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563812570,688,2,train,320.0,37.687,369.084,17.966,11.349,266.0,0.0,0.0,8.71,41.81,3.48,0.0,0.0,35.959,0.0,0.0,0.871,0.857,0.0,0.0,0.0
dish_1563812615,688,2,train,386.0,39.897,402.784,18.81,17.201,266.0,0.0,0.0,8.71,106.75,4.54,0.0,0.0,35.959,0.0,0.0,0.871,3.065,0.002,0.0,0.0
dish_1563820775,691,2,train,162.0,2.376,38.07,0.594,7.074,0.0,0.0,0.0,0.0,162.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.376,0.0,0.0,0.0
dish_1563820806,691,2,train,273.0,31.382,207.814,4.191,10.491,91.88,2.3,0.0,9.19,168.43,1.21,0.0,0.0,28.482,0.041,0.0,0.248,2.573,0.037,0.0,0.0
dish_1563820834,691,2,train,311.0,35.562,308.894,7.877,23.031,91.88,2.3,0.0,9.19,168.43,1.21,38.0,0.0,28.482,0.041,0.0,0.248,2.573,0.037,4.18,0.0
dish_1563822546,692,1,train,88.0,2.816,38.72,1.848,3.784,0.0,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.816,0.0,0.0,0.0
dish_1563822597,692,1,train,176.0,4.107,59.4,2.171,7.627,0.0,0.0,0.0,0.0,176.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.107,0.0,0.0,0.0
dish_1563822636,692,1,train,258.0,19.792,237.019,13.746,10.663,59.93,0.0,0.0,11.99,185.59,0.5,0.0,0.0,14.982,0.0,0.0,0.503,4.307,0.0,0.0,0.0
dish_1563822663,692,1,train,317.0,22.034,303.099,13.864,24.233,59.93,0.0,0.0,70.99,185.59,0.5,0.0,0.0,14.982,0.0,0.0,2.745,4.307,0.0,0.0,0.0
dish_1563823814,694,0,train,85.0,0.588,25.586,0.92,4.275,0.0,0.0,0.0,0.0,0.0,1.36,0.0,83.64,0.0,0.0,0.0,0.0,0.0,0.002,0.0,0.585
dish_1563823894,694,0,train,400.0,30.525,368.341,20.784,18.219,100.12,0.0,0.0,20.02,80.07,13.14,0.0,186.64,25.03,0.0,0.0,0.841,1.594,0.208,0.0,2.852
dish_1563827060,697,1,train,55.0,6.16,146.245,5.17,18.425,0.0,0.0,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.16,0.0
dish_1563827086,697,1,train,175.0,10.0,199.045,7.69,23.585,0.0,0.0,0.0,0.0,120.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,3.84,0.0,6.16,0.0
dish_1563827110,697,1,train,217.0,10.21,220.045,7.732,29.045,0.0,0.0,0.0,0.0,162.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,4.05,0.0,6.16,0.0
dish_1563827132,697,1,train,302.0,11.457,240.02,8.044,32.757,0.0,0.0,0.0,0.0,247.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,5.297,0.0,6.16,0.0
dish_1563827162,697,1,train,354.0,25.045,319.54,9.729,34.358,43.04,1.08,0.0,4.3,250.01,0.56,55.0,0.0,13.343,0.019,0.0,0.116,5.389,0.017,6.16,0.0
dish_1563827188,697,1,train,377.0,29.445,369.36,12.976,35.209,59.85,1.08,0.0,7.67,252.7,0.7,55.0,0.0,17.545,0.019,0.0,0.257,5.445,0.017,6.16,0.0
dish_1563827425,698,1,train,138.0,4.416,60.72,2.898,5.934,0.0,0.0,0.0,0.0,138.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.416,0.0,0.0,0.0
dish_1563827454,698,1,train,248.0,6.029,86.57,3.301,10.737,0.0,0.0,0.0,0.0,248.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.029,0.0,0.0,0.0
dish_1563827482,698,1,train,621.0,77.38,894.522,55.956,24.549,272.59,0.0,0.0,54.52,291.62,2.27,0.0,0.0,68.149,0.0,0.0,2.29,6.942,0.0,0.0,0.0
dish_1563897047,699,4,test,127.0,12.7,66.04,0.254,0.889,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,12.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1563897087,699,4,test,231.0,15.196,102.44,0.67,8.169,127.0,0.0,0.0,0.0,104.0,0.0,0.0,0.0,12.7,0.0,0.0,0.0,2.496,0.0,0.0,0.0
dish_1563897338,700,3,train,88.0,0.704,29.92,0.176,7.04,0.0,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.704,0.0,0.0,0.0
dish_1563897369,700,3,train,179.0,1.592,117.014,2.396,23.02,0.0,0.0,0.0,88.78,88.0,2.22,0.0,0.0,0.0,0.0,0.0,0.888,0.704,0.0,0.0,0.0
dish_1563897395,700,3,train,233.0,2.888,135.914,2.612,26.8,0.0,0.0,0.0,88.78,142.0,2.22,0.0,0.0,0.0,0.0,0.0,0.888,2.0,0.0,0.0,0.0
dish_1563897439,700,3,train,578.0,37.388,315.314,3.302,29.215,345.0,0.0,0.0,88.78,142.0,2.22,0.0,0.0,34.5,0.0,0.0,0.888,2.0,0.0,0.0,0.0
dish_1563897492,700,3,train,662.0,38.648,339.674,3.47,34.759,345.0,0.0,0.0,88.78,142.0,86.22,0.0,0.0,34.5,0.0,0.0,0.888,2.0,1.26,0.0,0.0
dish_1563900120,703,0,train,49.0,0.478,46.897,1.195,8.605,0.0,0.0,0.0,47.8,0.0,1.2,0.0,0.0,0.0,0.0,0.0,0.478,0.0,0.0,0.0,0.0
dish_1563900150,703,0,train,154.0,10.978,202.297,12.745,10.285,105.0,0.0,0.0,47.8,0.0,1.2,0.0,0.0,10.5,0.0,0.0,0.478,0.0,0.0,0.0,0.0
dish_1563900172,703,0,train,255.0,11.786,236.637,12.947,18.365,105.0,0.0,0.0,47.8,101.0,1.2,0.0,0.0,10.5,0.0,0.0,0.478,0.808,0.0,0.0,0.0
dish_1563900205,703,0,train,301.0,12.863,262.262,14.249,21.506,105.0,0.0,0.0,47.8,145.88,2.32,0.0,0.0,10.5,0.0,0.0,0.478,1.885,0.0,0.0,0.0
dish_1563908923,704,0,train,112.0,3.584,49.28,2.352,4.816,0.0,0.0,0.0,0.0,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.584,0.0,0.0,0.0
dish_1563908957,704,0,train,159.0,4.409,73.873,3.573,8.024,0.0,45.83,0.0,0.0,112.0,1.18,0.0,0.0,0.0,0.825,0.0,0.0,3.584,0.0,0.0,0.0
dish_1563908996,704,0,train,293.0,29.325,269.916,10.462,15.449,76.37,45.83,0.0,19.09,144.46,7.26,0.0,0.0,23.674,0.825,0.0,0.668,4.151,0.007,0.0,0.0
dish_1563909951,706,3,train,188.0,3.664,180.603,15.631,9.267,0.0,49.73,0.0,0.0,114.89,23.39,0.0,0.0,0.0,0.895,0.0,0.0,2.737,0.032,0.0,0.0
dish_1563909982,706,3,train,318.0,27.836,370.794,22.315,16.471,74.09,49.73,0.0,18.52,146.38,29.29,0.0,0.0,22.967,0.895,0.0,0.648,3.287,0.039,0.0,0.0
dish_1563910106,707,4,train,84.0,1.529,101.458,7.423,8.097,0.0,0.0,0.0,0.0,72.45,11.55,0.0,0.0,0.0,0.0,0.0,0.0,1.512,0.017,0.0,0.0
dish_1563910137,707,4,train,141.0,2.53,131.283,8.904,11.987,0.0,55.58,0.0,0.0,72.45,12.97,0.0,0.0,0.0,1.0,0.0,0.0,1.512,0.017,0.0,0.0
dish_1563910171,707,4,train,423.0,53.608,571.184,27.286,24.939,204.53,55.58,0.0,40.91,106.46,15.02,0.51,0.0,49.088,1.0,0.0,1.064,2.431,0.017,0.009,0.0
dish_1563910276,707,4,train,241.0,43.652,375.943,15.709,11.069,174.8,0.0,0.0,34.96,29.06,1.75,0.44,0.0,41.951,0.0,0.0,0.909,0.785,0.0,0.008,0.0
dish_1563910530,708,3,train,214.0,38.762,333.825,13.949,9.829,155.21,0.0,0.0,31.04,25.8,1.55,0.39,0.0,37.251,0.0,0.0,0.807,0.697,0.0,0.007,0.0
dish_1563910639,708,3,train,447.0,80.965,697.289,29.137,20.53,324.21,0.0,0.0,64.84,53.9,3.24,0.81,0.0,77.81,0.0,0.0,1.686,1.455,0.0,0.015,0.0
dish_1563910676,708,3,train,501.0,82.057,757.957,34.776,22.811,324.21,0.0,0.0,64.84,99.18,11.96,0.81,0.0,77.81,0.0,0.0,1.686,2.534,0.013,0.015,0.0
dish_1563910708,708,3,train,555.0,83.004,786.213,36.179,26.496,324.21,52.65,0.0,64.84,99.18,13.31,0.81,0.0,77.81,0.948,0.0,1.686,2.534,0.013,0.015,0.0
dish_1563911156,709,2,train,96.0,17.85,140.449,4.935,5.32,54.71,0.0,0.0,13.68,23.25,4.36,0.0,0.0,16.96,0.0,0.0,0.479,0.406,0.005,0.0,0.0
dish_1563911185,709,2,train,133.0,18.5,159.809,5.896,7.845,54.71,36.08,0.0,13.68,23.25,5.28,0.0,0.0,16.96,0.649,0.0,0.479,0.406,0.005,0.0,0.0
dish_1563911215,709,2,train,170.0,20.239,218.158,6.71,18.908,54.71,36.08,0.0,13.68,23.25,5.28,37.0,0.0,16.96,0.649,0.0,0.479,0.406,0.005,1.739,0.0
dish_1563911255,709,2,train,235.0,21.956,293.192,12.388,24.054,54.71,36.08,0.0,26.81,64.45,15.95,37.0,0.0,16.96,0.649,0.0,1.056,1.441,0.11,1.739,0.0
dish_1563986093,713,2,train,215.0,13.409,239.03,14.341,14.228,128.0,0.0,0.0,0.0,87.0,0.0,0.0,0.0,12.8,0.0,0.0,0.0,0.609,0.0,0.0,0.0
dish_1563996251,714,4,train,158.0,2.844,39.5,0.474,7.9,0.0,0.0,0.0,0.0,158.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.844,0.0,0.0,0.0
dish_1563996279,714,4,train,278.0,3.91,98.49,1.894,19.737,0.0,0.0,0.0,0.0,276.3,1.7,0.0,0.0,0.0,0.0,0.0,0.0,3.909,0.001,0.0,0.0
dish_1563996308,714,4,train,659.0,98.595,718.512,22.656,28.794,296.64,0.0,0.0,29.66,307.08,23.95,0.0,1.67,91.959,0.0,0.0,1.246,4.918,0.387,0.0,0.084
dish_1563997327,716,3,train,120.0,3.164,157.878,14.367,6.456,0.0,0.0,0.0,0.0,100.19,16.7,0.0,3.11,0.0,0.0,0.0,0.0,2.44,0.029,0.0,0.696
dish_1563997386,716,3,train,238.0,4.213,215.885,15.763,18.095,0.0,0.0,0.0,0.0,216.52,18.37,0.0,3.11,0.0,0.0,0.0,0.0,3.487,0.03,0.0,0.696
dish_1563997418,716,3,train,400.0,34.667,608.704,43.176,24.549,118.03,0.0,0.0,23.61,225.37,23.98,0.0,9.01,29.508,0.0,0.0,0.614,3.689,0.03,0.0,0.826
dish_1563998210,717,1,test,91.0,22.615,148.089,4.959,2.163,70.85,0.0,0.0,7.09,7.35,5.31,0.0,0.4,21.964,0.0,0.0,0.298,0.241,0.092,0.0,0.02
dish_1563998244,717,1,test,170.0,24.274,259.479,10.331,16.383,70.85,0.0,0.0,86.09,7.35,5.31,0.0,0.4,21.964,0.0,0.0,1.957,0.241,0.092,0.0,0.02
dish_1563998279,717,1,test,296.0,47.961,565.005,31.652,21.403,162.65,0.0,0.0,104.45,14.24,9.67,0.0,4.99,44.915,0.0,0.0,2.434,0.399,0.092,0.0,0.121
dish_1563998323,717,1,test,574.0,52.965,634.505,32.486,35.303,162.65,0.0,0.0,104.45,292.24,9.67,0.0,4.99,44.915,0.0,0.0,2.434,5.403,0.092,0.0,0.121
dish_1563998513,718,1,test,599.0,54.24,757.505,38.736,51.553,162.65,0.0,0.0,104.45,292.24,9.67,0.0,29.99,44.915,0.0,0.0,2.434,5.403,0.092,0.0,1.396
dish_1564000202,719,0,train,149.0,16.688,396.191,14.006,49.915,0.0,0.0,0.0,0.0,0.0,0.0,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.688,0.0
dish_1564000417,720,0,train,128.0,14.336,340.352,12.032,42.88,0.0,0.0,0.0,0.0,0.0,0.0,128.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.336,0.0
dish_1564000459,720,0,train,217.0,36.454,485.186,16.882,44.996,69.29,0.0,0.0,6.93,7.19,5.2,128.0,0.39,21.481,0.0,0.0,0.291,0.236,0.09,14.336,0.02
dish_1564000490,720,0,train,325.0,39.91,532.706,19.15,49.64,69.29,0.0,0.0,6.93,115.19,5.2,128.0,0.39,21.481,0.0,0.0,0.291,3.692,0.09,14.336,0.02
dish_1564000536,720,0,train,397.0,43.829,659.906,27.076,60.702,69.29,36.07,0.0,6.93,127.81,28.51,128.0,0.39,21.481,3.246,0.0,0.291,4.031,0.424,14.336,0.02
dish_1564082212,723,2,train,425.0,100.637,707.941,23.037,18.468,326.83,0.0,0.0,34.89,47.4,15.89,0.0,0.0,98.744,0.0,0.0,0.914,0.737,0.242,0.0,0.0
dish_1564082323,723,2,train,518.0,103.613,748.861,24.99,22.467,326.83,0.0,0.0,34.89,140.4,15.89,0.0,0.0,98.744,0.0,0.0,0.914,3.713,0.242,0.0,0.0
dish_1564082462,724,4,train,609.0,108.823,905.354,32.072,40.533,326.83,0.0,1.34,101.8,152.44,26.59,0.0,0.0,98.744,0.0,0.335,4.327,3.88,1.537,0.0,0.0
dish_1564083480,725,2,train,133.0,21.992,259.702,18.263,2.427,96.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,21.696,0.0,0.0,0.0,0.296,0.0,0.0,0.0
dish_1564083508,725,2,train,225.0,22.82,276.262,18.447,6.015,96.0,0.0,0.0,0.0,129.0,0.0,0.0,0.0,21.696,0.0,0.0,0.0,1.124,0.0,0.0,0.0
dish_1564083534,725,2,train,307.0,24.46,309.062,18.611,13.395,96.0,0.0,0.0,0.0,211.0,0.0,0.0,0.0,21.696,0.0,0.0,0.0,2.764,0.0,0.0,0.0
dish_1564086602,726,4,train,49.0,5.488,130.291,4.606,16.415,0.0,0.0,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.488,0.0
dish_1564086624,726,4,train,131.0,8.03,217.293,6.574,34.455,0.0,0.0,0.0,82.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,2.542,0.0,0.0,5.488,0.0
dish_1564086658,726,4,train,230.0,11.198,260.853,8.653,38.712,0.0,0.0,0.0,82.0,99.0,0.0,49.0,0.0,0.0,0.0,0.0,2.542,3.168,0.0,5.488,0.0
dish_1564086709,726,4,train,307.0,15.606,393.27,14.645,53.999,0.0,0.0,1.13,138.62,109.19,9.06,49.0,0.0,0.0,0.0,0.283,5.429,3.31,1.096,5.488,0.0
dish_1564086748,726,4,train,339.0,23.569,443.772,16.095,54.915,25.32,0.0,1.13,141.15,112.7,9.69,49.0,0.0,7.85,0.0,0.283,5.495,3.356,1.096,5.488,0.0
dish_1564086778,726,4,train,437.0,41.446,645.801,25.345,65.734,90.81,0.0,1.13,150.51,124.4,21.15,49.0,0.0,24.877,0.0,0.283,5.748,3.646,1.404,5.488,0.0
dish_1564160043,728,1,test,98.0,9.898,245.0,1.47,48.02,0.0,0.0,0.0,98.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.898,0.0,0.0,0.0,0.0
dish_1564160161,728,1,train,97.0,9.797,242.5,1.455,47.53,0.0,0.0,0.0,97.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.797,0.0,0.0,0.0,0.0
dish_1564169197,730,4,train,50.0,5.5,116.45,6.5,8.95,0.0,0.0,0.0,0.0,0.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.5,0.0
dish_1564169254,730,4,train,201.0,6.406,161.75,6.802,20.426,0.0,0.0,0.0,0.0,151.0,0.0,50.0,0.0,0.0,0.0,0.0,0.0,0.906,0.0,5.5,0.0
dish_1564169298,730,4,train,299.0,8.46,187.076,7.091,25.31,0.0,0.0,0.0,0.0,186.27,6.03,50.0,56.71,0.0,0.0,0.0,0.0,1.599,0.113,5.5,1.248
dish_1564169340,730,4,train,518.0,48.925,657.518,30.844,48.58,119.1,0.0,35.73,47.64,200.86,6.92,51.04,56.71,28.466,0.0,8.933,2.858,1.788,0.113,5.519,1.248
dish_1564169878,732,0,train,341.0,63.006,732.513,36.986,36.232,185.45,0.0,55.64,74.18,22.72,1.39,1.62,0.0,44.323,0.0,13.909,4.451,0.294,0.0,0.029,0.0
dish_1564169932,732,0,train,473.0,65.773,766.627,37.375,42.811,185.45,0.0,55.64,74.18,70.22,9.51,1.62,76.38,44.323,0.0,13.909,4.451,1.228,0.153,0.029,1.68
dish_1564169958,732,0,train,563.0,67.093,787.776,37.705,46.741,185.45,0.0,55.64,74.18,160.22,9.51,1.62,76.38,44.323,0.0,13.909,4.451,2.548,0.153,0.029,1.68
dish_1564170010,732,0,train,831.0,116.863,1264.548,61.894,57.292,386.69,0.0,55.64,114.43,168.52,27.72,1.62,76.38,92.62,0.0,13.909,5.497,2.772,0.354,0.029,1.68
dish_1564170071,732,0,train,35.0,0.385,14.0,0.035,3.15,0.0,0.0,0.0,0.0,35.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.385,0.0,0.0,0.0
dish_1564170098,732,0,train,67.0,0.641,50.8,3.459,5.166,0.0,0.0,0.0,0.0,67.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.641,0.0,0.0,0.0
dish_1564170141,732,0,train,133.0,1.235,62.68,3.591,7.74,0.0,0.0,0.0,0.0,133.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.235,0.0,0.0,0.0
dish_1564170173,732,0,train,176.0,2.095,79.88,3.677,11.61,0.0,0.0,0.0,0.0,176.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.095,0.0,0.0,0.0
dish_1564170283,732,0,train,301.0,55.616,646.588,32.648,31.982,163.7,0.0,49.11,65.48,20.05,1.23,1.43,0.0,39.124,0.0,12.277,3.929,0.26,0.0,0.026,0.0
dish_1564170474,733,0,train,168.0,2.464,39.48,0.616,7.336,0.0,0.0,0.0,0.0,168.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.464,0.0,0.0,0.0
dish_1564170542,733,0,train,440.0,52.976,523.368,25.166,18.044,204.24,0.0,0.0,40.85,176.43,18.48,0.0,0.0,49.018,0.0,0.0,1.062,2.692,0.204,0.0,0.0
dish_1564170836,734,3,train,135.0,25.07,240.165,12.185,5.315,101.37,0.0,0.0,20.27,4.18,9.17,0.0,0.0,24.329,0.0,0.0,0.527,0.113,0.101,0.0,0.0
dish_1564170880,734,3,train,230.0,28.944,348.557,15.413,22.931,101.37,0.0,0.0,112.51,4.18,11.94,0.0,0.0,24.329,0.0,0.0,4.401,0.113,0.101,0.0,0.0
dish_1564170923,734,3,train,319.0,31.214,378.817,15.813,28.583,101.37,0.0,0.0,112.51,93.18,11.94,0.0,0.0,24.329,0.0,0.0,4.401,2.383,0.101,0.0,0.0
dish_1564170967,734,3,train,410.0,32.548,400.202,16.147,32.557,101.37,0.0,0.0,112.51,184.18,11.94,0.0,0.0,24.329,0.0,0.0,4.401,3.717,0.101,0.0,0.0
dish_1564171888,735,2,train,194.0,36.027,345.126,17.51,7.638,145.67,0.0,0.0,29.13,6.01,13.18,0.0,0.0,34.962,0.0,0.0,0.757,0.162,0.146,0.0,0.0
dish_1564171935,735,2,train,309.0,40.716,476.338,21.418,28.963,145.67,0.0,0.0,140.79,6.01,16.53,0.0,0.0,34.962,0.0,0.0,5.447,0.162,0.146,0.0,0.0
dish_1564171970,735,2,train,396.0,42.54,498.821,21.674,33.299,145.67,0.0,0.0,140.79,37.32,21.88,0.0,50.34,34.962,0.0,0.0,5.447,0.778,0.246,0.0,1.108
dish_1564171996,735,2,train,484.0,43.831,519.501,21.997,37.141,145.67,0.0,0.0,140.79,125.32,21.88,0.0,50.34,34.962,0.0,0.0,5.447,2.068,0.246,0.0,1.108
dish_1564428404,741,1,train,104.0,1.598,113.751,9.808,6.299,0.0,0.0,0.0,0.0,88.85,15.15,0.0,0.0,0.0,0.0,0.0,0.0,1.576,0.022,0.0,0.0
dish_1564428444,741,1,train,277.0,4.712,157.001,10.327,14.949,0.0,0.0,0.0,0.0,261.85,15.15,0.0,0.0,0.0,0.0,0.0,0.0,4.69,0.022,0.0,0.0
dish_1564428475,741,1,train,370.0,8.608,261.616,12.315,32.755,0.0,0.0,0.0,79.23,274.32,16.44,0.0,0.0,0.0,0.0,0.0,3.486,5.099,0.022,0.0,0.0
dish_1564428515,741,1,train,479.0,29.964,484.618,25.664,35.868,79.37,0.0,0.0,89.82,290.33,19.48,0.0,0.0,20.636,0.0,0.0,3.931,5.349,0.048,0.0,0.0
dish_1564429702,743,2,train,158.0,2.844,39.5,0.474,7.9,0.0,0.0,0.0,0.0,158.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.844,0.0,0.0,0.0
dish_1564429744,743,2,train,374.0,11.892,282.477,5.089,49.257,0.0,0.0,0.0,184.03,186.98,2.99,0.0,0.0,0.0,0.0,0.0,8.097,3.794,0.0,0.0,0.0
dish_1564429808,743,2,train,422.0,21.296,380.68,10.968,50.628,34.95,0.0,0.0,188.69,194.03,4.33,0.0,0.0,9.087,0.0,0.0,8.293,3.905,0.012,0.0,0.0
dish_1564429843,743,2,train,474.0,21.758,395.73,11.635,52.372,34.95,0.0,0.0,188.69,245.33,5.04,0.0,0.0,9.087,0.0,0.0,8.293,4.366,0.012,0.0,0.0
dish_1564429894,743,2,train,566.0,25.254,498.77,11.819,73.532,34.95,0.0,0.0,280.69,245.33,5.04,0.0,0.0,9.087,0.0,0.0,11.789,4.366,0.012,0.0,0.0
dish_1564503219,746,3,train,242.0,29.06,213.86,8.008,1.82,242.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,29.06,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564516732,749,4,train,54.0,0.48,26.546,0.639,5.327,0.0,0.0,0.0,0.0,53.23,0.77,0.0,0.0,0.0,0.0,0.0,0.0,0.479,0.001,0.0,0.0
dish_1564516760,749,4,train,329.0,52.757,442.729,19.221,11.774,211.54,0.0,0.0,21.15,93.69,1.56,0.0,1.06,50.558,0.0,0.0,0.931,1.267,0.001,0.0,0.001
dish_1564518850,751,2,train,119.0,4.879,194.327,7.021,29.155,0.0,0.0,119.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.879,0.0,0.0,0.0,0.0,0.0
dish_1564588483,752,1,train,51.0,9.843,175.746,13.923,2.142,51.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.843,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564588533,752,1,train,167.0,21.443,236.066,14.155,2.954,167.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.443,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564588567,752,1,train,312.0,35.943,450.666,30.105,5.274,312.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.943,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564589308,754,3,train,161.0,3.381,227.01,10.948,28.98,0.0,0.0,0.0,161.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.381,0.0,0.0,0.0,0.0
dish_1564589366,754,3,train,348.0,22.081,324.25,11.322,30.289,187.0,0.0,0.0,161.0,0.0,0.0,0.0,0.0,18.7,0.0,0.0,3.381,0.0,0.0,0.0,0.0
dish_1564601470,756,0,train,73.0,0.511,15.841,0.073,3.723,0.0,0.0,0.0,0.0,0.0,0.0,0.0,73.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.511
dish_1564601528,756,0,train,163.0,2.401,142.741,6.193,19.923,0.0,0.0,0.0,90.0,0.0,0.0,0.0,73.0,0.0,0.0,0.0,1.89,0.0,0.0,0.0,0.511
dish_1564601569,756,0,train,291.0,6.687,208.098,7.831,31.273,0.0,0.0,0.0,90.0,125.95,2.05,0.0,73.0,0.0,0.0,0.0,1.89,4.282,0.003,0.0,0.511
dish_1564601600,756,0,train,434.0,36.422,600.798,34.062,39.351,90.79,0.0,18.16,90.0,127.66,25.31,0.0,82.08,28.146,0.0,0.599,1.89,4.346,0.003,0.0,1.437
dish_1564601638,756,0,train,541.0,51.127,712.456,46.945,42.989,165.85,0.0,18.16,105.01,141.4,28.5,0.0,82.08,41.956,0.0,0.599,2.52,4.611,0.003,0.0,1.437
dish_1564675380,759,2,test,23.0,6.693,87.331,6.463,0.713,23.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.693,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564675411,759,2,test,308.0,35.193,235.531,7.033,2.708,308.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,35.193,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564685144,760,0,train,50.0,0.877,26.163,1.299,3.413,0.0,48.75,0.0,0.0,0.0,1.25,0.0,0.0,0.0,0.877,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564685183,760,0,train,150.0,2.24,127.107,10.841,7.11,0.0,48.75,0.0,0.0,85.13,16.12,0.0,0.0,0.0,0.877,0.0,0.0,1.356,0.006,0.0,0.0
dish_1564685229,760,0,train,347.0,41.905,591.435,41.459,14.73,154.41,48.75,0.0,20.59,97.48,25.77,0.0,0.0,38.602,0.877,0.0,0.721,1.603,0.101,0.0,0.0
dish_1564685360,761,0,train,129.0,25.974,304.052,20.049,4.99,101.11,0.0,0.0,13.48,8.09,6.32,0.0,0.0,25.278,0.0,0.0,0.472,0.162,0.062,0.0,0.0
dish_1564685406,761,0,train,192.0,26.832,367.647,26.061,7.319,101.11,0.0,0.0,13.48,61.72,15.69,0.0,0.0,25.278,0.0,0.0,0.472,1.016,0.066,0.0,0.0
dish_1564685450,761,0,train,288.0,37.392,623.008,35.373,38.999,101.11,0.0,0.0,13.48,61.72,15.69,96.0,0.0,25.278,0.0,0.0,0.472,1.016,0.066,10.56,0.0
dish_1564685482,761,0,train,337.0,40.43,851.397,49.485,63.401,101.11,0.0,0.0,13.48,61.72,15.69,96.0,49.0,25.278,0.0,0.0,0.472,1.016,0.066,10.56,3.038
dish_1564686657,762,4,train,105.0,11.76,279.195,9.87,35.175,0.0,0.0,0.0,0.0,0.0,0.0,105.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.76,0.0
dish_1564686702,762,4,train,130.0,11.96,307.945,12.545,36.75,0.0,0.0,0.0,0.0,25.0,0.0,105.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,11.76,0.0
dish_1564686740,762,4,train,221.0,13.2,399.805,21.228,40.114,0.0,0.0,0.0,0.0,102.47,13.53,105.0,0.0,0.0,0.0,0.0,0.0,1.434,0.005,11.76,0.0
dish_1564686791,762,4,train,343.0,51.142,601.349,25.62,40.114,122.0,0.0,0.0,0.0,102.47,13.53,105.0,0.0,37.942,0.0,0.0,0.0,1.434,0.005,11.76,0.0
dish_1564686832,762,4,train,392.0,54.18,829.738,39.732,64.516,122.0,0.0,0.0,0.0,102.47,13.53,105.0,49.0,37.942,0.0,0.0,0.0,1.434,0.005,11.76,3.038
dish_1564686855,762,4,train,435.0,56.846,1030.161,52.116,85.93,122.0,0.0,0.0,0.0,102.47,13.53,105.0,92.0,37.942,0.0,0.0,0.0,1.434,0.005,11.76,5.704
dish_1564686966,762,4,test,65.0,13.088,153.205,10.102,2.514,50.95,0.0,0.0,6.79,4.08,3.18,0.0,0.0,12.737,0.0,0.0,0.238,0.082,0.031,0.0,0.0
dish_1564687013,762,4,test,224.0,32.211,330.841,26.888,14.725,148.72,0.0,0.0,26.35,20.7,28.24,0.0,0.0,30.727,0.0,0.0,0.766,0.416,0.303,0.0,0.0
dish_1564687046,762,4,test,308.0,33.686,374.794,29.07,20.458,148.72,81.9,0.0,26.35,20.7,30.34,0.0,0.0,30.727,1.474,0.0,0.766,0.416,0.303,0.0,0.0
dish_1564687090,762,4,test,416.0,38.089,498.019,32.74,40.486,148.72,81.9,0.0,131.2,20.7,33.48,0.0,0.0,30.727,1.474,0.0,5.17,0.416,0.303,0.0,0.0
dish_1564690038,764,1,train,81.0,7.29,92.34,0.324,16.2,0.0,81.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.29,0.0,0.0,0.0,0.0,0.0,0.0
dish_1564690085,764,1,train,143.0,9.364,170.616,5.857,22.279,0.0,93.43,0.0,0.0,40.71,8.86,0.0,0.0,0.0,8.409,0.0,0.0,0.881,0.075,0.0,0.0
dish_1564690138,764,1,train,181.0,10.031,190.499,6.844,24.873,0.0,130.48,0.0,0.0,40.71,9.81,0.0,0.0,0.0,9.076,0.0,0.0,0.881,0.075,0.0,0.0
dish_1564690167,764,1,train,207.0,15.266,251.781,10.885,25.879,20.38,130.48,0.0,2.72,42.34,11.08,0.0,0.0,5.095,9.076,0.0,0.095,0.913,0.087,0.0,0.0
dish_1564690194,764,1,train,430.0,42.088,500.919,34.428,43.005,157.5,130.48,0.0,30.14,65.65,46.22,0.0,0.0,30.326,9.076,0.0,0.836,1.382,0.468,0.0,0.0
dish_1564691380,765,4,train,111.0,2.903,96.261,2.743,18.35,0.0,39.0,0.0,71.0,0.0,1.0,0.0,0.0,0.0,0.702,0.0,2.201,0.0,0.0,0.0,0.0
dish_1564691420,765,4,train,226.0,4.469,212.347,13.717,22.602,0.0,39.0,0.0,71.0,97.9,18.1,0.0,0.0,0.0,0.702,0.0,2.201,1.56,0.007,0.0,0.0
dish_1564691450,765,4,train,366.0,32.658,542.327,35.475,28.017,109.73,39.0,0.0,85.63,106.68,24.96,0.0,0.0,27.433,0.702,0.0,2.713,1.735,0.074,0.0,0.0
dish_1564773826,767,3,test,56.0,0.504,10.08,0.112,2.184,0.0,0.0,0.0,0.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.504,0.0,0.0,0.0
dish_1564773847,767,3,test,161.0,17.603,238.028,15.738,5.821,64.62,12.92,0.0,0.0,68.44,14.38,0.0,0.65,15.508,1.163,0.0,0.0,0.801,0.129,0.0,0.002
dish_1564773876,767,3,test,219.0,18.995,258.328,15.97,9.881,64.62,12.92,0.0,0.0,126.44,14.38,0.0,0.65,15.508,1.163,0.0,0.0,2.193,0.129,0.0,0.002
dish_1564773903,767,3,test,336.0,19.58,316.828,16.087,25.091,64.62,12.92,0.0,0.0,243.44,14.38,0.0,0.65,15.508,1.163,0.0,0.0,2.778,0.129,0.0,0.002
dish_1564773942,767,3,test,442.0,52.546,491.94,19.903,25.091,170.62,12.92,0.0,0.0,243.44,14.38,0.0,0.65,48.474,1.163,0.0,0.0,2.778,0.129,0.0,0.002
dish_1564773976,767,3,test,463.0,52.714,516.089,22.15,26.414,170.62,12.92,0.0,0.0,264.44,14.38,0.0,0.65,48.474,1.163,0.0,0.0,2.946,0.129,0.0,0.002
dish_1564774773,768,1,train,90.0,2.107,50.137,2.546,6.146,0.0,0.0,0.0,0.0,87.8,2.2,0.0,0.0,0.0,0.0,0.0,0.0,2.107,0.0,0.0,0.0
dish_1564774805,768,1,train,353.0,44.937,621.091,41.685,15.257,161.85,32.37,0.0,0.0,118.96,38.21,0.0,1.62,38.843,2.913,0.0,0.0,2.852,0.324,0.0,0.005
dish_1564777933,770,4,train,39.0,4.368,103.701,3.666,13.065,0.0,0.0,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.368,0.0
dish_1564777960,770,4,train,147.0,26.431,331.008,16.715,16.27,82.47,0.0,0.0,11.0,13.74,0.79,39.0,0.0,21.442,0.0,0.0,0.385,0.236,0.0,4.368,0.0
dish_1564777986,770,4,train,221.0,28.164,372.231,18.809,21.324,82.47,0.0,0.0,11.0,85.94,2.6,39.0,0.0,21.442,0.0,0.0,0.385,1.969,0.0,4.368,0.0
dish_1564778030,770,4,train,283.0,30.774,504.418,31.905,23.514,82.47,0.0,6.33,11.0,124.2,15.79,39.0,4.22,21.442,0.0,1.33,0.385,2.602,0.01,4.368,0.637
dish_1564778057,770,4,train,377.0,31.991,575.185,33.987,35.539,82.47,0.0,6.33,47.24,175.61,22.13,39.0,4.22,21.442,0.0,1.33,0.711,3.42,0.083,4.368,0.637
dish_1565024675,775,3,train,34.0,9.894,129.098,9.554,1.054,34.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.894,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565024707,775,3,train,149.0,21.394,299.298,22.204,2.894,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.394,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565024741,775,3,train,192.0,22.297,359.928,25.128,10.634,149.0,0.0,0.0,43.0,0.0,0.0,0.0,0.0,21.394,0.0,0.0,0.903,0.0,0.0,0.0,0.0
dish_1565030350,776,0,test,62.0,0.845,62.586,5.916,2.292,0.0,0.0,0.0,0.0,52.78,9.22,0.0,0.0,0.0,0.0,0.0,0.0,0.841,0.004,0.0,0.0
dish_1565030391,776,0,test,313.0,52.598,609.334,37.048,12.059,193.32,0.0,0.0,25.78,78.56,14.05,1.29,0.0,50.264,0.0,0.0,0.696,1.506,0.108,0.023,0.0
dish_1565032680,778,2,train,208.0,4.109,280.282,21.616,20.305,0.0,0.0,0.0,0.0,174.89,33.11,0.0,0.0,0.0,0.0,0.0,0.0,4.038,0.071,0.0,0.0
dish_1565032714,778,2,train,330.0,5.192,315.591,23.18,24.397,0.0,0.0,0.0,0.0,295.23,34.77,0.0,0.0,0.0,0.0,0.0,0.0,5.121,0.071,0.0,0.0
dish_1565032748,778,2,train,435.0,26.842,544.31,36.204,28.482,80.87,0.0,0.0,10.78,306.02,36.79,0.54,0.0,21.027,0.0,0.0,0.291,5.399,0.115,0.01,0.0
dish_1565033085,779,2,train,55.0,1.21,15.4,0.22,2.915,0.0,0.0,0.0,0.0,55.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.21,0.0,0.0,0.0
dish_1565033110,779,2,train,147.0,3.418,47.6,0.588,9.355,0.0,0.0,0.0,0.0,147.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.418,0.0,0.0,0.0
dish_1565033139,779,2,train,274.0,4.434,193.65,14.177,17.356,0.0,0.0,0.0,0.0,274.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.434,0.0,0.0,0.0
dish_1565033189,779,2,train,507.0,76.664,578.1,22.565,17.356,233.0,0.0,0.0,0.0,274.0,0.0,0.0,0.0,72.23,0.0,0.0,0.0,4.434,0.0,0.0,0.0
dish_1565033220,779,2,train,578.0,76.664,908.605,59.343,19.202,233.0,0.0,0.0,0.0,274.0,71.0,0.0,0.0,72.23,0.0,0.0,0.0,4.434,0.0,0.0,0.0
dish_1565033265,779,2,train,625.0,76.664,1324.085,106.343,19.202,233.0,0.0,0.0,0.0,274.0,118.0,0.0,0.0,72.23,0.0,0.0,0.0,4.434,0.0,0.0,0.0
dish_1565034234,780,1,train,175.0,3.15,43.75,0.525,8.75,0.0,0.0,0.0,0.0,175.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.15,0.0,0.0,0.0
dish_1565034281,780,1,train,306.0,5.901,228.46,9.433,32.33,0.0,0.0,0.0,131.0,175.0,0.0,0.0,0.0,0.0,0.0,0.0,2.751,3.15,0.0,0.0,0.0
dish_1565034330,780,1,train,483.0,42.396,614.016,31.386,39.217,136.33,0.0,0.0,149.18,193.18,3.41,0.91,0.0,35.445,0.0,0.0,3.242,3.619,0.074,0.016,0.0
dish_1565035715,782,0,test,39.0,0.429,15.6,0.039,3.51,0.0,0.0,0.0,0.0,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.429,0.0,0.0,0.0
dish_1565035746,782,0,test,149.0,2.409,43.1,0.369,9.01,0.0,0.0,0.0,0.0,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.409,0.0,0.0,0.0
dish_1565035774,782,0,test,185.0,3.129,57.5,0.441,12.25,0.0,0.0,0.0,0.0,185.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.129,0.0,0.0,0.0
dish_1565035802,782,0,test,250.0,3.714,69.2,0.571,14.785,0.0,0.0,0.0,0.0,250.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.714,0.0,0.0,0.0
dish_1565035830,782,0,test,414.0,54.718,340.128,6.475,14.785,164.0,0.0,0.0,0.0,250.0,0.0,0.0,0.0,51.004,0.0,0.0,0.0,3.714,0.0,0.0,0.0
dish_1565035858,782,0,test,531.0,55.42,375.228,6.709,23.677,164.0,0.0,0.0,0.0,367.0,0.0,0.0,0.0,51.004,0.0,0.0,0.0,4.416,0.0,0.0,0.0
dish_1565118999,789,1,train,288.0,4.773,200.387,16.937,10.105,0.0,0.0,12.26,0.0,251.23,24.51,0.0,0.0,0.0,0.0,1.691,0.0,3.082,0.0,0.0,0.0
dish_1565119418,790,4,train,122.0,13.664,324.398,11.468,40.87,0.0,0.0,0.0,0.0,0.0,0.0,122.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.664,0.0
dish_1565119439,790,4,train,189.0,27.351,465.412,19.563,42.858,51.16,0.0,0.0,6.82,8.53,0.49,122.0,0.0,13.302,0.0,0.0,0.239,0.147,0.0,13.664,0.0
dish_1565119464,790,4,train,223.0,28.042,504.065,23.11,44.414,51.16,0.0,0.0,6.82,36.89,6.13,122.0,0.0,13.302,0.0,0.0,0.239,0.821,0.016,13.664,0.0
dish_1565119493,790,4,train,280.0,29.866,529.145,24.307,46.865,51.16,0.0,0.0,6.82,93.89,6.13,122.0,0.0,13.302,0.0,0.0,0.239,2.645,0.016,13.664,0.0
dish_1565119642,791,3,train,78.0,1.989,26.52,0.351,4.953,0.0,0.0,0.0,0.0,78.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.989,0.0,0.0,0.0
dish_1565119669,791,3,train,229.0,3.272,55.965,1.03,10.842,0.0,0.0,0.0,0.0,229.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.272,0.0,0.0,0.0
dish_1565119698,791,3,train,308.0,19.411,222.235,10.576,13.187,60.32,0.0,0.0,8.04,239.05,0.58,0.0,0.0,15.684,0.0,0.0,0.282,3.445,0.0,0.0,0.0
dish_1565119739,791,3,train,461.0,25.65,396.804,15.775,41.558,60.32,0.0,0.0,156.59,239.05,5.03,0.0,0.0,15.684,0.0,0.0,6.52,3.445,0.0,0.0,0.0
dish_1565196260,795,0,train,143.0,14.3,74.36,0.286,1.001,143.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.3,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565203015,796,4,train,397.0,107.056,663.94,19.142,8.944,341.43,0.0,0.0,34.14,13.23,8.19,0.0,0.0,105.844,0.0,0.0,0.888,0.324,0.0,0.0,0.0
dish_1565203094,796,4,train,485.0,116.736,898.02,27.678,37.984,341.43,0.0,0.0,34.14,13.23,8.19,88.0,0.0,105.844,0.0,0.0,0.888,0.324,0.0,9.68,0.0
dish_1565204855,797,0,train,105.0,3.515,53.613,1.343,9.311,0.0,0.0,0.0,0.0,103.32,1.68,0.0,0.0,0.0,0.0,0.0,0.0,3.513,0.003,0.0,0.0
dish_1565204891,797,0,train,314.0,16.297,302.886,9.081,40.93,38.33,0.0,0.0,81.76,183.04,10.88,0.0,0.0,9.16,0.0,0.0,2.208,4.927,0.003,0.0,0.0
dish_1565204931,797,0,train,515.0,22.729,391.326,13.302,49.573,38.33,0.0,0.0,81.76,384.04,10.88,0.0,0.0,9.16,0.0,0.0,2.208,11.359,0.003,0.0,0.0
dish_1565208562,799,4,train,92.0,2.944,40.48,1.932,3.956,0.0,0.0,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.944,0.0,0.0,0.0
dish_1565208611,799,4,train,157.0,5.12,73.669,2.764,9.72,0.0,0.0,0.0,0.0,155.96,1.04,0.0,0.0,0.0,0.0,0.0,0.0,5.119,0.002,0.0,0.0
dish_1565208637,799,4,train,305.0,14.171,250.188,8.243,32.11,27.14,0.0,0.0,57.9,212.41,7.55,0.0,0.0,6.486,0.0,0.0,1.563,6.12,0.002,0.0,0.0
dish_1565378680,802,3,train,81.0,3.303,92.419,2.752,15.02,0.0,0.0,0.0,78.64,0.0,2.36,0.0,0.0,0.0,0.0,0.0,3.303,0.0,0.0,0.0,0.0
dish_1565378731,802,3,train,207.0,4.422,154.358,4.243,27.449,0.0,0.0,0.0,78.64,124.21,4.14,0.0,0.0,0.0,0.0,0.0,3.303,1.118,0.002,0.0,0.0
dish_1565378770,802,3,train,287.0,19.016,319.28,11.794,36.281,53.46,0.0,0.0,86.28,133.76,13.5,0.0,0.0,13.9,0.0,0.0,3.509,1.355,0.253,0.0,0.0
dish_1565379251,803,2,train,67.0,0.595,32.936,0.793,6.609,0.0,0.0,0.0,0.0,66.05,0.95,0.0,0.0,0.0,0.0,0.0,0.0,0.594,0.001,0.0,0.0
dish_1565379320,803,2,train,247.0,33.432,404.011,17.782,26.481,120.29,0.0,0.0,17.18,87.53,22.0,0.0,0.0,31.274,0.0,0.0,0.464,1.127,0.566,0.0,0.0
dish_1565379377,803,2,train,397.0,38.232,470.011,20.932,32.931,120.29,0.0,0.0,17.18,237.53,22.0,0.0,0.0,31.274,0.0,0.0,0.464,5.927,0.566,0.0,0.0
dish_1565379788,804,1,train,101.0,0.897,49.65,1.195,9.963,0.0,0.0,0.0,0.0,99.57,1.43,0.0,0.0,0.0,0.0,0.0,0.0,0.896,0.001,0.0,0.0
dish_1565379827,804,1,train,235.0,2.632,150.531,4.163,27.105,0.0,0.0,0.0,51.66,172.87,10.47,0.0,0.0,0.0,0.0,0.0,0.465,2.061,0.106,0.0,0.0
dish_1565379868,804,1,train,434.0,38.934,560.775,22.946,49.074,132.98,0.0,0.0,70.66,196.61,33.74,0.0,0.0,34.576,0.0,0.0,0.978,2.65,0.731,0.0,0.0
dish_1565627751,806,3,train,72.0,0.702,68.909,1.756,12.644,0.0,0.0,0.0,70.24,0.0,1.76,0.0,0.0,0.0,0.0,0.0,0.702,0.0,0.0,0.0,0.0
dish_1565627800,806,3,train,261.0,19.602,167.189,2.134,13.967,189.0,0.0,0.0,70.24,0.0,1.76,0.0,0.0,18.9,0.0,0.0,0.702,0.0,0.0,0.0,0.0
dish_1565638287,809,1,train,175.0,35.728,357.935,19.854,6.479,134.1,0.0,0.0,17.88,21.9,1.12,0.0,0.0,34.866,0.0,0.0,0.483,0.38,0.0,0.0,0.0
dish_1565638323,809,1,train,274.0,38.302,467.825,20.745,29.249,134.1,0.0,0.0,116.88,21.9,1.12,0.0,0.0,34.866,0.0,0.0,3.057,0.38,0.0,0.0,0.0
dish_1565638365,809,1,train,389.0,41.299,612.35,33.066,36.632,134.1,0.0,0.0,116.88,113.9,18.71,0.0,5.41,34.866,0.0,0.0,3.057,2.317,0.031,0.0,1.028
dish_1565638478,809,1,test,91.0,1.597,47.616,2.364,6.211,0.0,88.72,0.0,0.0,0.0,2.27,0.0,0.0,0.0,1.597,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565638515,809,1,test,165.0,16.705,198.971,10.759,8.95,56.7,88.72,0.0,7.56,9.26,2.75,0.0,0.0,14.743,1.597,0.0,0.204,0.16,0.0,0.0,0.0
dish_1565638550,809,1,test,215.0,30.124,277.657,12.718,10.055,97.77,88.72,1.03,11.67,12.86,2.95,0.0,0.0,27.515,1.597,0.394,0.377,0.241,0.0,0.0,0.0
dish_1565638599,809,1,test,279.0,32.858,368.698,15.762,23.104,97.77,88.72,1.03,73.8,12.86,4.82,0.0,0.0,27.515,1.597,0.394,3.111,0.241,0.0,0.0,0.0
dish_1565640195,810,1,train,54.0,1.08,21.6,0.108,4.86,0.0,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.08,0.0,0.0,0.0
dish_1565640227,810,1,train,165.0,14.4,334.62,13.428,40.38,0.0,0.0,0.0,0.0,54.0,0.0,111.0,0.0,0.0,0.0,0.0,0.0,1.08,0.0,13.32,0.0
dish_1565640272,810,1,train,405.0,16.56,377.82,13.908,49.74,0.0,0.0,0.0,0.0,294.0,0.0,111.0,0.0,0.0,0.0,0.0,0.0,3.24,0.0,13.32,0.0
dish_1565640365,810,1,train,623.0,75.069,720.89,22.448,54.557,179.06,0.0,4.48,17.91,309.67,0.9,111.0,0.0,55.686,0.0,1.719,0.752,3.592,0.0,13.32,0.0
dish_1565640427,810,1,train,42.0,2.604,195.762,12.096,20.916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.604
dish_1565640528,810,1,train,105.0,1.893,38.547,1.534,5.718,0.0,0.0,0.0,0.0,82.39,22.61,0.0,0.0,0.0,0.0,0.0,0.0,1.273,0.621,0.0,0.0
dish_1565640549,810,1,train,139.0,2.641,45.483,1.568,7.044,0.0,0.0,0.0,0.0,116.39,22.61,0.0,0.0,0.0,0.0,0.0,0.0,2.021,0.621,0.0,0.0
dish_1565640576,810,1,train,267.0,28.774,307.287,16.09,11.783,98.08,0.0,0.0,13.08,132.42,23.42,0.0,0.0,25.502,0.0,0.0,0.353,2.298,0.621,0.0,0.0
dish_1565640601,810,1,train,333.0,46.488,411.152,18.676,13.241,152.29,0.0,1.36,18.5,137.16,23.69,0.0,0.0,42.361,0.0,0.52,0.581,2.405,0.621,0.0,0.0
dish_1565641844,812,2,train,92.0,10.304,244.628,8.648,30.82,0.0,0.0,0.0,0.0,0.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.304,0.0
dish_1565641885,812,2,train,139.0,11.338,254.216,8.695,32.653,0.0,0.0,0.0,0.0,47.0,0.0,92.0,0.0,0.0,0.0,0.0,0.0,1.034,0.0,10.304,0.0
dish_1565641915,812,2,train,190.0,13.517,326.765,11.121,43.051,0.0,0.0,0.0,49.51,47.0,1.49,92.0,0.0,0.0,0.0,0.0,2.179,1.034,0.0,10.304,0.0
dish_1565641943,812,2,train,307.0,44.918,510.89,15.704,45.636,96.1,0.0,2.4,59.12,55.41,1.97,92.0,0.0,29.887,0.0,0.923,2.582,1.223,0.0,10.304,0.0
dish_1565721800,814,0,train,171.0,1.454,33.345,0.769,6.669,0.0,0.0,0.0,0.0,171.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.454,0.0,0.0,0.0
dish_1565721828,814,0,train,239.0,2.881,129.225,5.394,18.909,0.0,0.0,0.0,68.0,171.0,0.0,0.0,0.0,0.0,0.0,0.0,1.428,1.454,0.0,0.0,0.0
dish_1565721855,814,0,train,450.0,59.863,479.034,15.636,23.093,180.49,0.0,0.0,86.05,179.63,3.84,0.0,0.0,55.951,0.0,0.0,2.186,1.726,0.0,0.0,0.0
dish_1565724050,815,3,train,72.0,1.382,84.723,7.981,3.011,0.0,0.0,0.0,0.0,59.52,12.48,0.0,0.0,0.0,0.0,0.0,0.0,1.347,0.035,0.0,0.0
dish_1565724087,815,3,train,160.0,25.147,230.615,12.253,4.756,75.27,0.0,0.0,7.53,63.12,14.07,0.0,0.0,23.335,0.0,0.0,0.316,1.461,0.035,0.0,0.0
dish_1565724196,815,3,train,82.0,1.92,45.68,2.32,5.6,0.0,0.0,0.0,0.0,80.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,1.92,0.0,0.0,0.0
dish_1565724235,815,3,train,209.0,36.217,256.229,8.485,8.118,108.63,0.0,0.0,10.86,85.19,4.31,0.0,0.0,33.677,0.0,0.0,0.456,2.084,0.0,0.0,0.0
dish_1565724269,815,3,train,311.0,47.437,527.549,18.379,41.778,108.63,0.0,0.0,10.86,85.19,4.31,102.0,0.0,33.677,0.0,0.0,0.456,2.084,0.0,11.22,0.0
dish_1565724509,816,1,train,158.0,1.343,30.81,0.711,6.162,0.0,0.0,0.0,0.0,158.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.343,0.0,0.0,0.0
dish_1565724552,816,1,train,298.0,4.621,108.8,4.672,15.723,0.0,0.0,0.0,0.0,294.59,3.41,0.0,0.0,0.0,0.0,0.0,0.0,4.621,0.0,0.0,0.0
dish_1565724588,816,1,train,410.0,6.973,266.72,12.288,35.883,0.0,0.0,0.0,112.0,294.59,3.41,0.0,0.0,0.0,0.0,0.0,2.352,4.621,0.0,0.0,0.0
dish_1565724620,816,1,train,535.0,40.73,473.953,18.356,38.361,106.92,0.0,0.0,122.69,299.7,5.69,0.0,0.0,33.146,0.0,0.0,2.801,4.782,0.0,0.0,0.0
dish_1565726812,818,2,train,65.0,1.522,36.21,1.839,4.439,0.0,0.0,0.0,0.0,63.41,1.59,0.0,0.0,0.0,0.0,0.0,0.0,1.522,0.0,0.0,0.0
dish_1565726848,818,2,train,121.0,2.597,102.105,8.047,6.781,0.0,0.0,0.0,0.0,109.71,11.29,0.0,0.0,0.0,0.0,0.0,0.0,2.57,0.027,0.0,0.0
dish_1565726887,818,2,train,172.0,3.668,174.015,11.515,15.961,0.0,0.0,0.0,51.0,109.71,11.29,0.0,0.0,0.0,0.0,0.0,1.071,2.57,0.027,0.0,0.0
dish_1565726915,818,2,train,251.0,4.369,196.88,12.528,18.61,0.0,0.0,0.0,51.0,187.64,12.36,0.0,0.0,0.0,0.0,0.0,1.071,3.271,0.027,0.0,0.0
dish_1565726949,818,2,train,372.0,29.088,451.547,27.148,22.201,92.4,0.0,0.0,63.32,203.04,13.25,0.0,0.0,24.023,0.0,0.0,1.502,3.536,0.027,0.0,0.0
dish_1565728388,820,3,train,47.0,5.264,124.973,4.418,15.745,0.0,0.0,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.264,0.0
dish_1565728415,820,3,train,80.0,12.006,194.428,8.405,16.724,25.2,0.0,0.0,3.36,4.2,0.24,47.0,0.0,6.552,0.0,0.0,0.118,0.072,0.0,5.264,0.0
dish_1565728453,820,3,train,140.0,28.209,293.9,11.318,17.914,76.52,0.0,0.0,8.49,6.65,1.33,47.0,0.0,22.462,0.0,0.0,0.333,0.15,0.0,5.264,0.0
dish_1565728494,820,3,train,163.0,28.692,326.33,12.882,22.054,76.52,0.0,0.0,31.49,6.65,1.33,47.0,0.0,22.462,0.0,0.0,0.816,0.15,0.0,5.264,0.0
dish_1565728562,820,3,train,262.0,29.571,354.982,14.151,25.374,76.52,0.0,0.0,31.49,104.31,2.67,47.0,0.0,22.462,0.0,0.0,0.816,1.029,0.0,5.264,0.0
dish_1565728591,820,3,train,322.0,30.976,388.407,15.849,29.472,76.52,0.0,0.0,31.49,162.85,4.14,47.0,0.0,22.462,0.0,0.0,0.816,2.434,0.0,5.264,0.0
dish_1565728634,820,3,train,418.0,31.456,436.407,15.945,41.952,76.52,0.0,0.0,31.49,258.85,4.14,47.0,0.0,22.462,0.0,0.0,0.816,2.914,0.0,5.264,0.0
dish_1565728684,820,3,train,464.0,32.248,493.299,21.303,44.04,76.52,0.0,0.0,31.49,296.56,12.43,47.0,0.0,22.462,0.0,0.0,0.816,3.691,0.015,5.264,0.0
dish_1565797645,821,0,train,70.0,0.42,21.0,0.14,5.32,0.0,0.0,0.0,0.0,70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.42,0.0,0.0,0.0
dish_1565797671,821,0,train,102.0,0.644,39.24,0.236,9.8,0.0,0.0,0.0,0.0,102.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.644,0.0,0.0,0.0
dish_1565797699,821,0,train,171.0,2.852,128.929,5.044,20.419,0.0,0.0,0.0,11.13,155.42,4.45,0.0,0.0,0.0,0.0,0.0,1.113,1.739,0.0,0.0,0.0
dish_1565797723,821,0,train,221.0,7.852,202.929,10.544,21.219,50.0,0.0,0.0,11.13,155.42,4.45,0.0,0.0,5.0,0.0,0.0,1.113,1.739,0.0,0.0,0.0
dish_1565798531,822,4,train,61.0,0.831,61.576,5.821,2.255,0.0,0.0,0.0,0.0,51.93,9.07,0.0,0.0,0.0,0.0,0.0,0.0,0.827,0.004,0.0,0.0
dish_1565798558,822,4,train,109.0,1.955,88.316,7.179,5.533,0.0,0.0,0.0,0.0,98.76,10.24,0.0,0.0,0.0,0.0,0.0,0.0,1.951,0.004,0.0,0.0
dish_1565798591,822,4,train,224.0,13.455,148.116,7.409,6.338,115.0,0.0,0.0,0.0,98.76,10.24,0.0,0.0,11.5,0.0,0.0,0.0,1.951,0.004,0.0,0.0
dish_1565798617,822,4,train,311.0,14.064,197.706,7.67,18.518,115.0,0.0,0.0,0.0,185.76,10.24,0.0,0.0,11.5,0.0,0.0,0.0,2.56,0.004,0.0,0.0
dish_1565808943,824,0,train,74.0,0.666,13.32,0.148,2.886,0.0,0.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.666,0.0,0.0,0.0
dish_1565808975,824,0,train,130.0,1.282,35.72,0.204,7.926,0.0,0.0,0.0,0.0,130.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.282,0.0,0.0,0.0
dish_1565809033,824,0,train,433.0,62.692,341.42,1.63,16.746,205.0,0.0,0.0,0.0,228.0,0.0,0.0,0.0,59.45,0.0,0.0,0.0,3.242,0.0,0.0,0.0
dish_1565809080,824,0,train,485.0,63.16,362.74,1.734,21.946,205.0,0.0,0.0,0.0,280.0,0.0,0.0,0.0,59.45,0.0,0.0,0.0,3.71,0.0,0.0,0.0
dish_1565809120,824,0,train,502.0,63.228,365.63,1.734,22.898,205.0,0.0,0.0,0.0,297.0,0.0,0.0,0.0,59.45,0.0,0.0,0.0,3.778,0.0,0.0,0.0
dish_1565811287,827,4,train,127.0,1.905,16.51,0.254,2.794,0.0,0.0,0.0,0.0,127.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.905,0.0,0.0,0.0
dish_1565811335,827,4,train,219.0,5.656,121.479,3.38,19.854,0.0,0.0,0.0,89.32,127.0,2.68,0.0,0.0,0.0,0.0,0.0,3.751,1.905,0.0,0.0,0.0
dish_1565811390,827,4,train,266.0,14.385,205.092,7.622,21.705,35.29,0.0,0.0,96.38,128.46,5.87,0.0,0.0,8.47,0.0,0.0,3.935,1.944,0.035,0.0,0.0
dish_1565811423,827,4,train,290.0,18.779,240.057,9.221,22.184,53.33,0.0,0.0,97.61,132.12,5.95,0.99,0.0,12.781,0.0,0.0,3.953,1.992,0.035,0.018,0.0
dish_1565811529,827,4,train,104.0,2.102,116.842,10.86,4.392,0.0,0.0,0.0,0.0,87.22,16.78,0.0,0.0,0.0,0.0,0.0,0.0,2.077,0.024,0.0,0.0
dish_1565811588,827,4,train,263.0,31.213,348.484,21.454,7.569,119.49,0.0,0.0,8.14,111.49,17.31,6.57,0.0,28.559,0.0,0.0,0.122,2.39,0.024,0.118,0.0
dish_1565812116,828,4,train,36.0,0.864,44.44,4.223,1.449,0.0,0.0,0.0,0.0,30.38,4.75,0.0,0.87,0.0,0.0,0.0,0.0,0.722,0.011,0.0,0.131
dish_1565812146,828,4,train,93.0,1.149,72.94,4.28,8.859,0.0,0.0,0.0,0.0,87.38,4.75,0.0,0.87,0.0,0.0,0.0,0.0,1.007,0.011,0.0,0.131
dish_1565812186,828,4,train,266.0,32.824,324.978,15.807,12.316,130.01,0.0,0.0,8.86,113.79,5.32,7.15,0.87,31.073,0.0,0.0,0.133,1.347,0.011,0.129,0.131
dish_1565894858,831,3,test,27.0,1.377,132.84,6.75,17.55,0.0,0.0,0.0,0.0,0.0,0.0,0.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.377
dish_1565894889,831,3,test,74.0,6.641,257.813,11.168,33.295,0.0,0.0,0.0,0.0,0.0,0.0,47.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,5.264,1.377
dish_1565894910,831,3,test,179.0,18.191,537.113,21.353,67.945,0.0,0.0,0.0,0.0,0.0,0.0,152.0,27.0,0.0,0.0,0.0,0.0,0.0,0.0,16.814,1.377
dish_1565897569,834,2,train,150.0,28.693,324.914,21.175,5.554,109.62,0.0,0.0,21.92,17.54,0.91,0.0,0.0,27.406,0.0,0.0,0.921,0.367,0.0,0.0,0.0
dish_1565897617,834,2,train,211.0,29.764,356.832,22.759,9.717,109.62,59.48,0.0,21.92,17.54,2.44,0.0,0.0,27.406,1.071,0.0,0.921,0.367,0.0,0.0,0.0
dish_1565897680,834,2,train,405.0,37.931,770.449,63.737,16.571,109.62,59.48,19.81,21.92,137.24,43.72,0.0,13.21,27.406,1.071,4.161,0.921,2.348,0.031,0.0,1.994
dish_1565972591,837,3,train,122.0,17.08,195.2,12.2,3.66,122.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.08,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565972628,837,3,train,359.0,40.78,545.96,38.27,7.452,359.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,40.78,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1565972674,837,3,train,426.0,41.693,613.593,44.663,9.929,359.0,0.0,0.0,0.0,57.04,9.96,0.0,0.0,40.78,0.0,0.0,0.0,0.909,0.004,0.0,0.0
dish_1565972715,837,3,train,510.0,43.167,657.546,46.845,15.662,359.0,81.9,0.0,0.0,57.04,12.06,0.0,0.0,40.78,1.474,0.0,0.0,0.909,0.004,0.0,0.0
dish_1565981769,839,2,train,47.0,0.418,23.104,0.556,4.636,0.0,0.0,0.0,0.0,46.33,0.67,0.0,0.0,0.0,0.0,0.0,0.0,0.417,0.001,0.0,0.0
dish_1565981802,839,2,train,152.0,8.818,99.754,5.596,6.631,0.0,105.0,0.0,0.0,46.33,0.67,0.0,0.0,0.0,8.4,0.0,0.0,0.417,0.001,0.0,0.0
dish_1565981848,839,2,train,198.0,15.093,147.41,11.092,8.199,32.06,105.0,0.0,6.41,52.5,2.03,0.0,0.0,5.898,8.4,0.0,0.269,0.525,0.001,0.0,0.0
dish_1565981898,839,2,train,290.0,17.301,260.98,21.884,11.903,32.06,105.0,0.0,6.41,130.15,14.16,0.0,2.22,5.898,8.4,0.0,0.269,2.371,0.028,0.0,0.335
dish_1565984245,840,1,train,78.0,0.693,38.344,0.923,7.694,0.0,0.0,0.0,0.0,76.89,1.11,0.0,0.0,0.0,0.0,0.0,0.0,0.692,0.001,0.0,0.0
dish_1565984299,840,1,train,298.0,4.653,93.344,1.583,18.694,0.0,0.0,0.0,0.0,296.89,1.11,0.0,0.0,0.0,0.0,0.0,0.0,4.652,0.001,0.0,0.0
dish_1565984345,840,1,train,353.0,14.688,210.358,9.084,20.663,39.26,0.0,0.0,5.05,305.95,2.75,0.0,0.0,9.638,0.0,0.0,0.136,4.884,0.029,0.0,0.0
dish_1565984434,840,1,train,430.0,17.977,319.892,12.747,36.362,39.26,0.0,0.0,79.8,305.95,4.99,0.0,0.0,9.638,0.0,0.0,3.426,4.884,0.029,0.0,0.0
dish_1566317711,850,0,train,182.0,4.368,123.76,2.548,21.84,0.0,0.0,0.0,182.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.368,0.0,0.0,0.0,0.0
dish_1566317742,850,0,train,254.0,5.376,154.72,2.908,28.464,0.0,0.0,0.0,182.0,72.0,0.0,0.0,0.0,0.0,0.0,0.0,4.368,1.008,0.0,0.0,0.0
dish_1566317775,850,0,train,410.0,7.248,237.4,4.0,47.184,0.0,0.0,0.0,182.0,228.0,0.0,0.0,0.0,0.0,0.0,0.0,4.368,2.88,0.0,0.0,0.0
dish_1566317801,850,0,train,420.0,7.258,275.4,4.0,56.984,0.0,0.0,0.0,182.0,228.0,10.0,0.0,0.0,0.0,0.0,0.0,4.368,2.88,0.01,0.0,0.0
dish_1566329203,852,1,train,377.0,6.786,94.25,1.131,18.85,0.0,0.0,0.0,0.0,377.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.786,0.0,0.0,0.0
dish_1566329234,852,1,train,457.0,23.129,262.625,10.797,21.224,61.09,0.0,0.0,8.15,387.18,0.59,0.0,0.0,15.883,0.0,0.0,0.285,6.961,0.0,0.0,0.0
dish_1566329257,852,1,train,537.0,29.529,321.025,14.637,22.744,61.09,80.0,0.0,8.15,387.18,0.59,0.0,0.0,15.883,6.4,0.0,0.285,6.961,0.0,0.0,0.0
dish_1566330399,853,0,train,126.0,25.74,265.191,15.224,3.74,96.21,0.0,0.0,12.83,16.04,0.92,0.0,0.0,25.016,0.0,0.0,0.449,0.276,0.0,0.0,0.0
dish_1566330428,853,0,train,228.0,27.576,290.691,15.53,8.84,96.21,0.0,0.0,12.83,118.04,0.92,0.0,0.0,25.016,0.0,0.0,0.449,2.112,0.0,0.0,0.0
dish_1566330466,853,0,train,283.0,30.161,377.426,16.74,25.285,96.21,0.0,0.0,12.83,118.04,0.92,55.0,0.0,25.016,0.0,0.0,0.449,2.112,0.0,2.585,0.0
dish_1566331801,854,4,train,203.0,37.746,296.991,10.436,11.249,115.69,0.0,0.0,28.92,49.17,9.22,0.0,0.0,35.864,0.0,0.0,1.012,0.859,0.011,0.0,0.0
dish_1566331828,854,4,train,306.0,41.195,349.583,11.754,20.383,115.69,0.0,0.0,28.92,150.52,10.87,0.0,0.0,35.864,0.0,0.0,1.012,4.305,0.013,0.0,0.0
dish_1566331868,854,4,train,372.0,44.297,453.665,13.206,40.117,115.69,0.0,0.0,28.92,150.52,10.87,66.0,0.0,35.864,0.0,0.0,1.012,4.305,0.013,3.102,0.0
dish_1566331914,854,4,train,481.0,46.927,589.502,24.885,47.359,115.69,0.0,0.0,28.92,240.65,25.23,66.0,4.51,35.864,0.0,0.0,1.012,5.905,0.034,3.102,1.009
dish_1566333118,856,0,train,90.0,10.08,239.31,8.46,30.15,0.0,0.0,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.08,0.0
dish_1566333167,856,0,train,140.0,20.294,344.544,14.501,31.634,38.18,0.0,0.0,5.09,6.36,0.37,90.0,0.0,9.927,0.0,0.0,0.178,0.109,0.0,10.08,0.0
dish_1566333194,856,0,train,220.0,35.17,461.585,18.614,36.067,83.77,0.0,0.0,16.49,25.74,4.0,90.0,0.0,24.06,0.0,0.0,0.577,0.448,0.004,10.08,0.0
dish_1566333228,856,0,train,304.0,37.858,498.545,20.378,39.679,83.77,0.0,0.0,16.49,109.74,4.0,90.0,0.0,24.06,0.0,0.0,0.577,3.136,0.004,10.08,0.0
dish_1566333255,856,0,train,372.0,39.082,515.545,20.582,43.079,83.77,0.0,0.0,16.49,177.74,4.0,90.0,0.0,24.06,0.0,0.0,0.577,4.36,0.004,10.08,0.0
dish_1566333291,856,0,train,439.0,41.325,549.756,21.439,49.02,83.77,0.0,0.0,16.49,243.67,5.07,90.0,0.0,24.06,0.0,0.0,0.577,6.602,0.006,10.08,0.0
dish_1566333324,856,0,train,476.0,43.064,608.105,22.253,60.083,83.77,0.0,0.0,16.49,243.67,5.07,127.0,0.0,24.06,0.0,0.0,0.577,6.602,0.006,11.819,0.0
dish_1566414225,859,3,test,88.0,3.759,125.182,4.186,17.942,0.0,0.0,0.0,85.44,0.0,2.56,0.0,0.0,0.0,0.0,0.0,3.759,0.0,0.0,0.0,0.0
dish_1566414291,859,3,test,244.0,7.035,345.142,14.794,46.022,0.0,0.0,0.0,241.44,0.0,2.56,0.0,0.0,0.0,0.0,0.0,7.035,0.0,0.0,0.0,0.0
dish_1566414342,859,3,test,552.0,83.578,846.367,31.579,53.343,239.81,0.0,0.0,265.42,24.88,20.55,0.0,1.35,74.34,0.0,0.0,8.042,0.816,0.312,0.0,0.068
dish_1566414412,859,3,test,658.0,85.335,920.12,37.812,57.062,239.81,0.0,4.51,265.42,117.35,29.57,0.0,1.35,74.34,0.0,0.622,8.042,1.951,0.312,0.0,0.068
dish_1566415520,861,4,train,143.0,2.51,74.825,3.714,9.76,0.0,139.43,0.0,0.0,0.0,3.58,0.0,0.0,0.0,2.51,0.0,0.0,0.0,0.0,0.0,0.0
dish_1566415558,861,4,train,234.0,14.924,169.101,14.587,12.862,63.41,139.43,0.0,12.68,12.21,6.27,0.0,0.0,11.668,2.51,0.0,0.533,0.214,0.0,0.0,0.0
dish_1566415591,861,4,train,316.0,16.844,214.781,16.907,18.462,63.41,139.43,0.0,12.68,92.21,8.27,0.0,0.0,11.668,2.51,0.0,0.533,2.134,0.0,0.0,0.0
dish_1566419510,865,1,train,103.0,11.536,273.877,9.682,34.505,0.0,0.0,0.0,0.0,0.0,0.0,103.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.536,0.0
dish_1566419546,865,1,train,177.0,13.904,306.437,11.236,37.687,0.0,0.0,0.0,0.0,74.0,0.0,103.0,0.0,0.0,0.0,0.0,0.0,2.368,0.0,11.536,0.0
dish_1566419577,865,1,train,210.0,14.483,323.704,12.093,39.939,0.0,32.17,0.0,0.0,74.0,0.82,103.0,0.0,0.0,0.579,0.0,0.0,2.368,0.0,11.536,0.0
dish_1566419603,865,1,train,252.0,15.365,382.924,14.949,47.499,0.0,32.17,0.0,42.0,74.0,0.82,103.0,0.0,0.0,0.579,0.0,0.882,2.368,0.0,11.536,0.0
dish_1566419635,865,1,train,389.0,49.412,605.872,22.415,50.756,106.67,32.17,0.0,52.67,85.07,8.82,103.0,0.6,33.067,0.579,0.0,1.33,2.731,0.139,11.536,0.03
dish_1566499564,867,3,train,37.0,0.296,42.55,3.959,2.331,0.0,0.0,0.0,0.0,37.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.296,0.0,0.0,0.0
dish_1566499633,867,3,train,139.0,3.024,161.386,14.184,8.426,0.0,0.0,0.0,0.0,123.38,15.62,0.0,0.0,0.0,0.0,0.0,0.0,3.001,0.023,0.0,0.0
dish_1566499686,867,3,train,388.0,48.448,674.707,37.686,35.916,166.4,0.0,0.0,23.77,153.09,44.74,0.0,0.0,43.263,0.0,0.0,0.642,3.738,0.805,0.0,0.0
dish_1566502000,869,2,train,184.0,2.693,63.009,4.847,3.949,0.0,0.0,0.0,0.0,179.51,4.49,0.0,0.0,0.0,0.0,0.0,0.0,2.693,0.0,0.0,0.0
dish_1566502033,869,2,train,253.0,4.832,136.218,6.503,19.129,0.0,0.0,0.0,69.0,179.51,4.49,0.0,0.0,0.0,0.0,0.0,2.139,2.693,0.0,0.0,0.0
dish_1566502061,869,2,train,385.0,28.912,408.339,18.962,33.702,88.21,0.0,0.0,81.6,195.26,19.92,0.0,0.0,22.935,0.0,0.0,2.479,3.083,0.415,0.0,0.0
dish_1566502271,870,2,train,359.0,37.94,296.817,10.505,13.685,96.54,0.0,0.0,9.65,250.39,2.41,0.0,0.0,29.927,0.0,0.0,0.251,7.762,0.0,0.0,0.0
dish_1566502315,870,2,train,495.0,62.75,577.185,23.342,28.699,187.42,0.0,0.0,22.64,266.62,18.32,0.0,0.0,53.556,0.0,0.0,0.602,8.165,0.427,0.0,0.0
dish_1566587102,872,1,test,146.0,27.147,213.599,7.506,8.091,83.21,0.0,0.0,20.8,35.36,6.63,0.0,0.0,25.794,0.0,0.0,0.728,0.618,0.008,0.0,0.0
dish_1566587143,872,1,test,234.0,28.907,354.399,20.706,16.011,83.21,0.0,0.0,20.8,123.36,6.63,0.0,0.0,25.794,0.0,0.0,0.728,2.378,0.008,0.0,0.0
dish_1566587182,872,1,test,377.0,31.372,531.261,37.362,22.502,83.21,0.0,0.0,20.8,240.59,32.4,0.0,0.0,25.794,0.0,0.0,0.728,4.795,0.055,0.0,0.0
dish_1566587689,873,1,train,270.0,18.58,196.847,7.475,15.383,54.82,0.0,0.0,10.96,199.26,4.96,0.0,0.0,13.156,0.0,0.0,0.285,5.085,0.055,0.0,0.0
dish_1566587714,873,1,train,373.0,37.732,347.537,12.771,21.091,113.52,0.0,0.0,25.64,224.21,9.64,0.0,0.0,31.353,0.0,0.0,0.799,5.52,0.06,0.0,0.0
dish_1566589933,875,2,test,122.0,9.76,89.06,5.856,2.318,0.0,122.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.76,0.0,0.0,0.0,0.0,0.0,0.0
dish_1566589967,875,2,test,322.0,46.948,381.662,16.138,13.401,113.98,122.0,0.0,28.5,48.44,9.08,0.0,0.0,35.334,9.76,0.0,0.997,0.846,0.011,0.0,0.0
dish_1566590007,875,2,test,457.0,61.798,740.762,29.233,57.951,113.98,122.0,0.0,28.5,48.44,9.08,135.0,0.0,35.334,9.76,0.0,0.997,0.846,0.011,14.85,0.0
dish_1566590056,875,2,test,519.0,62.927,815.648,34.712,63.927,113.98,122.0,0.0,28.5,101.92,17.61,135.0,0.0,35.334,9.76,0.0,0.997,1.963,0.023,14.85,0.0
dish_1566590536,876,0,train,51.0,0.453,25.071,0.603,5.031,0.0,0.0,0.0,0.0,50.28,0.72,0.0,0.0,0.0,0.0,0.0,0.0,0.452,0.001,0.0,0.0
dish_1566590566,876,0,train,142.0,17.352,186.96,8.817,8.613,68.33,0.0,0.0,13.67,53.1,6.91,0.0,0.0,16.399,0.0,0.0,0.355,0.529,0.069,0.0,0.0
dish_1566590946,877,1,train,228.0,42.341,405.612,20.579,8.976,171.2,0.0,0.0,34.24,7.06,15.49,0.0,0.0,41.089,0.0,0.0,0.89,0.191,0.171,0.0,0.0
dish_1566590987,877,1,train,370.0,86.361,639.912,25.691,8.976,313.2,0.0,0.0,34.24,7.06,15.49,0.0,0.0,85.109,0.0,0.0,0.89,0.191,0.171,0.0,0.0
dish_1566591048,877,1,train,481.0,102.963,876.202,42.709,12.671,410.2,0.0,0.0,34.24,21.06,15.49,0.0,0.0,101.599,0.0,0.0,0.89,0.303,0.171,0.0,0.0
dish_1566591080,877,1,train,549.0,103.567,909.63,43.513,19.379,410.2,0.0,0.0,34.24,88.1,16.46,0.0,0.0,101.599,0.0,0.0,0.89,0.906,0.172,0.0,0.0
dish_1566592456,878,4,train,126.0,4.032,55.44,2.646,5.418,0.0,0.0,0.0,0.0,126.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.032,0.0,0.0,0.0
dish_1566835603,879,0,train,66.0,0.644,63.167,1.61,11.59,0.0,0.0,0.0,64.39,0.0,1.61,0.0,0.0,0.0,0.0,0.0,0.644,0.0,0.0,0.0,0.0
dish_1566835641,879,0,train,546.0,48.644,312.767,2.57,14.95,480.0,0.0,0.0,64.39,0.0,1.61,0.0,0.0,48.0,0.0,0.0,0.644,0.0,0.0,0.0,0.0
dish_1566835701,879,0,train,744.0,51.614,370.187,2.966,28.018,480.0,0.0,0.0,64.39,0.0,199.61,0.0,0.0,48.0,0.0,0.0,0.644,0.0,2.97,0.0,0.0
dish_1566844766,881,4,train,164.0,5.248,72.16,3.444,7.052,0.0,0.0,0.0,0.0,164.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.248,0.0,0.0,0.0
dish_1566844803,881,4,train,227.0,12.808,249.82,11.004,27.212,0.0,0.0,0.0,0.0,164.0,0.0,63.0,0.0,0.0,0.0,0.0,0.0,5.248,0.0,7.56,0.0
dish_1566844850,881,4,train,345.0,35.928,491.235,25.456,30.582,85.92,0.0,0.0,11.46,181.33,3.29,63.0,0.0,22.34,0.0,0.0,0.481,5.519,0.029,7.56,0.0
dish_1566846997,882,4,train,168.0,32.917,343.71,20.576,4.798,122.33,0.0,0.0,16.31,24.67,4.69,0.0,0.0,31.806,0.0,0.0,0.685,0.385,0.041,0.0,0.0
dish_1566847043,882,4,train,483.0,35.713,434.877,24.615,15.362,122.33,0.0,0.0,16.31,335.4,8.96,0.0,0.0,31.806,0.0,0.0,0.685,3.182,0.041,0.0,0.0
dish_1566847431,883,3,train,90.0,3.77,101.241,1.923,17.232,0.0,0.0,0.0,76.68,12.08,1.25,0.0,0.0,0.0,0.0,0.0,3.374,0.396,0.0,0.0,0.0
dish_1566847454,883,3,train,210.0,27.282,346.747,16.62,20.659,87.38,0.0,0.0,88.33,29.7,4.6,0.0,0.0,22.718,0.0,0.0,3.863,0.671,0.029,0.0,0.0
dish_1566847483,883,3,train,314.0,29.644,476.232,28.971,24.98,87.38,0.0,0.0,88.33,117.61,18.18,0.0,2.5,22.718,0.0,0.0,3.863,2.78,0.058,0.0,0.225
dish_1566849208,884,3,train,70.0,1.54,19.6,0.28,3.71,0.0,0.0,0.0,0.0,70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.54,0.0,0.0,0.0
dish_1566849233,884,3,train,117.0,2.48,94.8,7.33,7.94,0.0,0.0,0.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.48,0.0,0.0,0.0
dish_1566849269,884,3,train,153.0,16.304,249.852,17.626,9.452,0.0,0.0,36.0,0.0,117.0,0.0,0.0,0.0,0.0,0.0,13.824,0.0,2.48,0.0,0.0,0.0
dish_1566849295,884,3,train,239.0,16.992,348.752,26.828,14.87,0.0,0.0,36.0,0.0,203.0,0.0,0.0,0.0,0.0,0.0,13.824,0.0,3.168,0.0,0.0,0.0
dish_1566849327,884,3,train,362.0,55.122,551.702,31.256,14.87,123.0,0.0,36.0,0.0,203.0,0.0,0.0,0.0,38.13,0.0,13.824,0.0,3.168,0.0,0.0,0.0
dish_1566849387,884,3,train,517.0,60.082,619.902,34.511,21.535,123.0,0.0,36.0,0.0,358.0,0.0,0.0,0.0,38.13,0.0,13.824,0.0,8.128,0.0,0.0,0.0
dish_1566851115,886,0,train,70.0,1.872,81.554,7.017,4.183,0.0,0.0,0.0,0.0,59.28,10.72,0.0,0.0,0.0,0.0,0.0,0.0,1.857,0.016,0.0,0.0
dish_1566851155,886,0,train,260.0,9.831,295.284,11.077,40.562,0.0,0.0,0.0,161.87,84.77,13.35,0.0,0.0,0.0,0.0,0.0,7.122,2.693,0.016,0.0,0.0
dish_1566851198,886,0,train,342.0,25.897,463.047,21.12,42.904,59.71,0.0,0.0,169.84,96.82,15.64,0.0,0.0,15.524,0.0,0.0,7.457,2.881,0.035,0.0,0.0
dish_1566851222,886,0,train,375.0,29.527,550.828,24.321,53.794,59.71,0.0,0.0,169.84,96.82,15.64,33.0,0.0,15.524,0.0,0.0,7.457,2.881,0.035,3.63,0.0
dish_1566931116,890,4,train,142.0,2.556,35.5,0.426,7.1,0.0,0.0,0.0,0.0,142.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.556,0.0,0.0,0.0
dish_1566931140,890,4,train,199.0,3.696,126.7,8.976,12.23,0.0,0.0,0.0,0.0,199.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.696,0.0,0.0,0.0
dish_1566931179,890,4,train,263.0,4.956,204.369,15.988,15.746,0.0,0.0,0.0,0.0,252.67,10.33,0.0,0.0,0.0,0.0,0.0,0.0,4.956,0.0,0.0,0.0
dish_1566931217,890,4,train,378.0,21.409,333.109,29.459,21.469,78.57,18.86,0.0,0.0,266.81,13.77,0.0,0.0,14.456,1.681,0.0,0.0,5.269,0.002,0.0,0.0
dish_1566933180,892,3,train,96.0,19.702,161.45,7.634,2.197,81.01,0.0,0.0,8.1,5.16,1.72,0.0,0.0,19.362,0.0,0.0,0.211,0.13,0.0,0.0,0.0
dish_1566933226,892,3,train,171.0,22.552,245.45,7.784,19.447,81.01,0.0,0.0,83.1,5.16,1.72,0.0,0.0,19.362,0.0,0.0,3.061,0.13,0.0,0.0,0.0
dish_1566933401,893,2,train,79.0,1.156,27.053,2.081,1.696,0.0,0.0,0.0,0.0,77.07,1.93,0.0,0.0,0.0,0.0,0.0,0.0,1.156,0.0,0.0,0.0
dish_1566933454,893,2,train,154.0,3.665,121.741,8.775,9.05,0.0,15.04,0.0,0.0,126.32,12.64,0.0,0.0,0.0,1.353,0.0,0.0,2.222,0.09,0.0,0.0
dish_1566933498,893,2,train,284.0,28.378,318.483,17.559,12.098,100.0,15.04,0.0,10.0,145.45,13.02,0.0,0.5,23.9,1.353,0.0,0.44,2.594,0.09,0.0,0.001
dish_1566934057,894,3,train,193.0,2.824,66.091,5.084,4.142,0.0,0.0,0.0,0.0,188.29,4.71,0.0,0.0,0.0,0.0,0.0,0.0,2.824,0.0,0.0,0.0
dish_1566934085,894,3,train,259.0,4.012,82.591,5.282,7.442,0.0,0.0,0.0,0.0,254.29,4.71,0.0,0.0,0.0,0.0,0.0,0.0,4.012,0.0,0.0,0.0
dish_1566934120,894,3,train,316.0,14.598,183.994,10.427,9.686,42.8,0.0,0.0,8.56,256.06,8.58,0.0,0.0,10.272,0.0,0.0,0.223,4.06,0.043,0.0,0.0
dish_1566934152,894,3,train,396.0,29.806,305.065,15.832,11.562,104.34,0.0,0.0,14.71,267.83,8.81,0.0,0.31,24.98,0.0,0.0,0.493,4.289,0.043,0.0,0.0
dish_1566936266,896,2,train,146.0,27.755,220.956,9.866,3.423,112.31,0.0,0.0,11.23,21.48,0.42,0.0,0.56,26.842,0.0,0.0,0.494,0.418,0.0,0.0,0.001
dish_1566936297,896,2,train,266.0,44.923,355.292,23.923,9.395,194.29,19.68,0.0,11.23,36.24,4.01,0.0,0.56,41.926,1.754,0.0,0.494,0.746,0.002,0.0,0.001
dish_1566936323,896,2,train,318.0,45.859,368.292,24.079,11.995,194.29,19.68,0.0,11.23,88.24,4.01,0.0,0.56,41.926,1.754,0.0,0.494,1.682,0.002,0.0,0.001
dish_1566936352,896,2,train,368.0,47.759,424.292,24.179,23.495,194.29,19.68,0.0,61.23,88.24,4.01,0.0,0.56,41.926,1.754,0.0,2.394,1.682,0.002,0.0,0.001
dish_1566937992,897,1,train,104.0,11.648,276.536,9.776,34.84,0.0,0.0,0.0,0.0,0.0,0.0,104.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.648,0.0
dish_1566938026,897,1,train,219.0,33.51,450.576,17.547,37.536,88.46,0.0,0.0,8.85,16.92,0.33,104.0,0.44,21.142,0.0,0.0,0.389,0.33,0.0,11.648,0.0
dish_1566938059,897,1,train,256.0,34.051,463.247,18.522,38.33,88.46,0.0,0.0,8.85,53.02,1.23,104.0,0.44,21.142,0.0,0.0,0.389,0.871,0.0,11.648,0.0
dish_1566938088,897,1,train,381.0,36.301,494.497,18.897,44.58,88.46,0.0,0.0,8.85,178.02,1.23,104.0,0.44,21.142,0.0,0.0,0.389,3.121,0.0,11.648,0.0
dish_1567019993,899,1,train,55.0,1.841,28.083,0.704,4.877,0.0,0.0,0.0,0.0,54.12,0.88,0.0,0.0,0.0,0.0,0.0,0.0,1.84,0.001,0.0,0.0
dish_1567020054,899,1,train,145.0,14.21,122.001,11.54,7.937,63.13,0.0,0.0,12.63,65.68,3.56,0.0,0.0,11.616,0.0,0.0,0.53,2.062,0.001,0.0,0.0
dish_1567020091,899,1,train,238.0,16.442,236.805,22.449,11.682,63.13,0.0,0.0,12.63,144.17,15.83,0.0,2.24,11.616,0.0,0.0,0.53,3.928,0.029,0.0,0.339
dish_1567021799,901,0,train,76.0,1.824,93.818,8.915,3.06,0.0,0.0,0.0,0.0,64.14,10.02,0.0,1.83,0.0,0.0,0.0,0.0,1.525,0.022,0.0,0.277
dish_1567021831,901,0,train,228.0,22.713,252.436,27.216,8.229,106.62,0.0,0.0,21.32,83.67,14.55,0.0,1.83,19.618,0.0,0.0,0.896,1.9,0.022,0.0,0.277
dish_1567021864,901,0,train,262.0,26.453,342.876,30.514,19.449,106.62,0.0,0.0,21.32,83.67,14.55,34.0,1.83,19.618,0.0,0.0,0.896,1.9,0.022,3.74,0.277
dish_1567024066,904,2,train,86.0,9.632,228.674,8.084,28.81,0.0,0.0,0.0,0.0,0.0,0.0,86.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.632,0.0
dish_1567024112,904,2,train,116.0,15.335,274.076,10.111,29.513,23.08,0.0,0.0,2.31,4.41,0.09,86.0,0.12,5.515,0.0,0.0,0.102,0.086,0.0,9.632,0.0
dish_1567024135,904,2,train,147.0,15.986,317.786,12.219,35.093,23.08,0.0,0.0,33.31,4.41,0.09,86.0,0.12,5.515,0.0,0.0,0.753,0.086,0.0,9.632,0.0
dish_1567024187,904,2,train,235.0,16.734,334.946,12.615,38.525,23.08,0.0,0.0,33.31,92.41,0.09,86.0,0.12,5.515,0.0,0.0,0.753,0.834,0.0,9.632,0.0
dish_1567024224,904,2,train,315.0,18.174,354.946,12.855,42.525,23.08,0.0,0.0,33.31,172.41,0.09,86.0,0.12,5.515,0.0,0.0,0.753,2.274,0.0,9.632,0.0
dish_1567024252,904,2,train,454.0,22.622,512.016,25.921,49.058,23.08,0.0,0.0,33.31,172.41,0.09,86.0,139.12,5.515,0.0,0.0,0.753,2.274,0.0,9.632,4.448
dish_1567106594,905,1,train,88.0,0.792,15.84,0.176,3.432,0.0,0.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.792,0.0,0.0,0.0
dish_1567106620,905,1,train,146.0,1.952,39.04,0.292,8.652,0.0,0.0,0.0,0.0,146.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.952,0.0,0.0,0.0
dish_1567106646,905,1,train,201.0,2.557,61.04,0.347,13.602,0.0,0.0,0.0,0.0,201.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.557,0.0,0.0,0.0
dish_1567106668,905,1,train,248.0,2.98,80.31,0.441,18.302,0.0,0.0,0.0,0.0,248.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.98,0.0,0.0,0.0
dish_1567106693,905,1,train,417.0,55.37,359.16,6.525,18.302,169.0,0.0,0.0,0.0,248.0,0.0,0.0,0.0,52.39,0.0,0.0,0.0,2.98,0.0,0.0,0.0
dish_1567106794,905,1,train,48.0,2.021,102.338,10.139,1.696,0.0,0.0,4.9,0.0,29.62,10.21,0.0,3.27,0.0,0.0,1.029,0.0,0.49,0.008,0.0,0.493
dish_1567106825,905,1,train,104.0,13.461,220.201,16.905,3.358,42.76,0.0,4.9,5.7,36.74,10.62,0.0,3.27,11.118,0.0,1.029,0.2,0.613,0.008,0.0,0.493
dish_1567106870,905,1,train,211.0,25.231,504.821,27.284,38.668,42.76,0.0,4.9,5.7,36.74,10.62,107.0,3.27,11.118,0.0,1.029,0.2,0.613,0.008,11.77,0.493
dish_1567107839,907,3,test,156.0,18.763,174.284,16.469,11.981,95.93,0.0,0.0,19.19,16.31,24.58,0.0,0.0,17.65,0.0,0.0,0.518,0.328,0.267,0.0,0.0
dish_1567107989,908,2,train,111.0,3.713,140.139,9.907,10.884,0.0,22.26,0.0,0.0,72.89,15.86,0.0,0.0,0.0,2.003,0.0,0.0,1.577,0.134,0.0,0.0
dish_1567108037,908,2,train,365.0,55.603,674.731,40.597,18.422,193.95,22.26,0.0,25.86,105.21,17.72,0.0,0.0,50.428,2.003,0.0,0.905,2.133,0.134,0.0,0.0
dish_1567110519,909,3,train,100.0,1.362,100.945,9.542,3.697,0.0,0.0,0.0,0.0,85.13,14.87,0.0,0.0,0.0,0.0,0.0,0.0,1.356,0.006,0.0,0.0
dish_1567110556,909,3,train,150.0,2.24,127.107,10.841,7.11,0.0,48.75,0.0,0.0,85.13,16.12,0.0,0.0,0.0,0.877,0.0,0.0,1.356,0.006,0.0,0.0
dish_1567110631,909,3,train,260.0,6.725,252.614,14.579,27.508,0.0,48.75,0.0,106.8,85.13,19.32,0.0,0.0,0.0,0.877,0.0,4.485,1.356,0.006,0.0,0.0
dish_1567110665,909,3,train,532.0,39.44,556.495,43.295,48.397,167.26,48.75,0.0,140.25,113.56,62.18,0.0,0.0,30.775,0.877,0.0,5.389,1.928,0.471,0.0,0.0
dish_1567110756,909,3,train,42.0,4.62,111.72,4.074,13.86,0.0,0.0,0.0,0.0,0.0,0.0,42.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.62,0.0
dish_1567110783,909,3,train,210.0,38.941,465.308,24.373,18.846,128.29,0.0,0.0,17.1,21.38,1.23,42.0,0.0,33.354,0.0,0.0,0.599,0.368,0.0,4.62,0.0
dish_1567110811,909,3,train,282.0,39.921,537.988,31.243,21.508,128.29,0.0,0.0,17.1,82.68,11.93,42.0,0.0,33.354,0.0,0.0,0.599,1.344,0.004,4.62,0.0
dish_1567525637,910,4,train,81.0,0.798,153.265,15.801,3.077,0.0,0.0,0.0,0.0,55.96,25.04,0.0,0.0,0.0,0.0,0.0,0.0,0.762,0.036,0.0,0.0
dish_1567525712,910,4,train,184.0,6.916,510.545,40.461,32.977,10.0,0.0,0.0,93.0,55.96,25.04,0.0,0.0,3.7,0.0,0.0,2.418,0.762,0.036,0.0,0.0
dish_1567525744,910,4,train,298.0,18.316,679.265,53.001,34.801,124.0,0.0,0.0,93.0,55.96,25.04,0.0,0.0,15.1,0.0,0.0,2.418,0.762,0.036,0.0,0.0
dish_1567537185,912,1,train,185.0,38.145,402.982,22.946,7.199,142.49,0.0,0.0,19.0,19.0,3.56,0.95,0.0,37.047,0.0,0.0,0.513,0.49,0.077,0.017,0.0
dish_1567537212,912,1,train,238.0,39.386,432.507,24.445,10.818,142.49,0.0,0.0,19.0,70.71,4.85,0.95,0.0,37.047,0.0,0.0,0.513,1.731,0.077,0.017,0.0
dish_1567537252,912,1,train,303.0,39.963,464.46,25.214,17.23,142.49,0.0,0.0,19.0,134.78,5.78,0.95,0.0,37.047,0.0,0.0,0.513,2.308,0.078,0.017,0.0
dish_1567537280,912,1,train,404.0,51.275,733.019,34.708,51.065,142.49,0.0,0.0,19.0,134.78,5.78,101.95,0.0,37.047,0.0,0.0,0.513,2.308,0.078,11.329,0.0
dish_1567537352,912,1,train,73.0,1.21,50.792,4.293,2.561,0.0,0.0,3.11,0.0,63.68,6.21,0.0,0.0,0.0,0.0,0.429,0.0,0.781,0.0,0.0,0.0
dish_1567537422,912,1,train,203.0,3.267,109.319,6.288,12.189,0.0,0.0,3.11,0.0,192.07,7.82,0.0,0.0,0.0,0.0,0.429,0.0,2.839,0.0,0.0,0.0
dish_1567537457,912,1,train,361.0,12.93,297.765,12.137,36.092,28.97,0.0,3.11,61.81,252.34,14.78,0.0,0.0,6.925,0.0,0.429,1.669,3.908,0.0,0.0,0.0
dish_1567538263,913,1,train,56.0,1.873,70.701,4.998,5.491,0.0,11.23,0.0,0.0,36.77,8.0,0.0,0.0,0.0,1.011,0.0,0.0,0.796,0.067,0.0,0.0
dish_1567538303,913,1,train,307.0,53.626,617.449,36.13,15.258,193.32,11.23,0.0,25.78,62.55,12.83,1.29,0.0,50.264,1.011,0.0,0.696,1.461,0.172,0.023,0.0
dish_1567612954,915,0,train,155.0,1.512,148.346,3.78,27.22,0.0,0.0,0.0,151.22,0.0,3.78,0.0,0.0,0.0,0.0,0.0,1.512,0.0,0.0,0.0,0.0
dish_1567612986,915,0,train,524.0,38.412,340.226,4.518,29.803,369.0,0.0,0.0,151.22,0.0,3.78,0.0,0.0,36.9,0.0,0.0,1.512,0.0,0.0,0.0,0.0
dish_1567613019,915,0,train,645.0,39.138,376.526,4.76,38.999,369.0,0.0,0.0,151.22,121.0,3.78,0.0,0.0,36.9,0.0,0.0,1.512,0.726,0.0,0.0,0.0
dish_1567613057,915,0,train,749.0,40.698,406.686,4.968,45.863,369.0,0.0,0.0,151.22,121.0,107.78,0.0,0.0,36.9,0.0,0.0,1.512,0.726,1.56,0.0,0.0
dish_1567628069,917,4,train,73.0,2.345,107.433,7.18,10.05,0.0,0.0,0.0,8.51,31.92,32.56,0.0,0.0,0.0,0.0,0.0,1.022,0.582,0.741,0.0,0.0
dish_1567628099,917,4,train,132.0,3.849,127.493,7.446,13.796,0.0,0.0,0.0,8.51,90.92,32.56,0.0,0.0,0.0,0.0,0.0,1.022,2.087,0.741,0.0,0.0
dish_1567628157,917,4,train,355.0,37.48,373.191,36.019,25.867,174.56,0.0,0.0,43.43,100.52,36.49,0.0,0.0,32.119,0.0,0.0,2.369,2.251,0.741,0.0,0.0
dish_1567628193,917,4,train,439.0,41.068,492.683,40.015,42.993,174.56,0.0,0.0,124.98,100.52,38.94,0.0,0.0,32.119,0.0,0.0,5.958,2.251,0.741,0.0,0.0
dish_1567629081,918,0,train,90.0,10.08,239.31,8.46,30.15,0.0,0.0,0.0,0.0,0.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.08,0.0
dish_1567629106,918,0,train,211.0,13.952,292.55,11.001,35.353,0.0,0.0,0.0,0.0,121.0,0.0,90.0,0.0,0.0,0.0,0.0,0.0,3.872,0.0,10.08,0.0
dish_1567785130,921,3,train,59.0,1.888,76.69,4.111,9.08,0.0,0.0,0.0,9.52,45.68,3.81,0.0,0.0,0.0,0.0,0.0,0.952,0.936,0.0,0.0,0.0
dish_1567785165,921,3,train,116.0,12.889,273.112,19.672,11.474,57.0,0.0,0.0,9.52,45.68,3.81,0.0,0.0,11.001,0.0,0.0,0.952,0.936,0.0,0.0,0.0
dish_1567785223,921,3,train,261.0,27.389,487.712,35.622,13.794,202.0,0.0,0.0,9.52,45.68,3.81,0.0,0.0,25.501,0.0,0.0,0.952,0.936,0.0,0.0,0.0
dish_1567785622,922,4,test,54.0,0.378,30.78,0.162,7.56,0.0,0.0,0.0,0.0,54.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.378,0.0,0.0,0.0
dish_1567785654,922,4,test,96.0,0.588,51.78,0.204,13.02,0.0,0.0,0.0,0.0,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.588,0.0,0.0,0.0
dish_1567785709,922,4,test,138.0,1.002,131.251,8.397,14.615,0.0,0.0,0.0,0.0,125.02,12.98,0.0,0.0,0.0,0.0,0.0,0.0,0.983,0.019,0.0,0.0
dish_1567785739,922,4,test,177.0,1.382,168.576,9.349,21.464,0.0,0.0,0.0,38.05,125.02,13.93,0.0,0.0,0.0,0.0,0.0,0.38,0.983,0.019,0.0,0.0
dish_1567785778,922,4,test,310.0,11.701,303.508,15.198,32.63,0.0,0.0,118.63,38.05,125.02,13.93,0.0,14.37,0.0,0.0,7.876,0.38,0.983,0.019,0.0,2.443
dish_1567789154,923,3,train,111.0,21.423,382.506,30.303,4.662,111.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,21.423,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1567789190,923,3,train,177.0,22.073,507.388,43.178,7.169,111.0,0.0,0.0,0.0,45.6,20.4,0.0,0.0,21.423,0.0,0.0,0.0,0.621,0.03,0.0,0.0
dish_1567789230,923,3,train,300.0,34.373,689.428,56.708,9.137,234.0,0.0,0.0,0.0,45.6,20.4,0.0,0.0,33.723,0.0,0.0,0.0,0.621,0.03,0.0,0.0
dish_1567789256,923,3,train,319.0,41.403,792.218,64.688,9.403,253.0,0.0,0.0,0.0,45.6,20.4,0.0,0.0,40.753,0.0,0.0,0.0,0.621,0.03,0.0,0.0
dish_1567801468,925,4,train,196.0,6.272,86.24,4.116,8.428,0.0,0.0,0.0,0.0,196.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,6.272,0.0,0.0,0.0
dish_1567801520,925,4,train,600.0,79.972,919.097,42.248,53.029,269.98,0.0,0.0,38.57,244.21,47.25,0.0,0.0,70.194,0.0,0.0,1.041,7.468,1.269,0.0,0.0
dish_1568044496,927,2,train,118.0,0.708,35.4,0.236,8.968,0.0,0.0,0.0,0.0,118.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.708,0.0,0.0,0.0
dish_1568044528,927,2,train,233.0,1.83,145.463,3.041,29.163,0.0,0.0,0.0,112.2,118.0,2.8,0.0,0.0,0.0,0.0,0.0,1.122,0.708,0.0,0.0,0.0
dish_1568044581,927,2,train,696.0,48.13,386.223,3.967,32.404,463.0,0.0,0.0,112.2,118.0,2.8,0.0,0.0,46.3,0.0,0.0,1.122,0.708,0.0,0.0,0.0
dish_1568044634,927,2,train,789.0,49.525,413.193,4.153,38.542,463.0,0.0,0.0,112.2,118.0,95.8,0.0,0.0,46.3,0.0,0.0,1.122,0.708,1.395,0.0,0.0
dish_1568055963,929,1,train,32.0,0.256,36.8,3.424,2.016,0.0,0.0,0.0,0.0,32.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.256,0.0,0.0,0.0
dish_1568055986,929,1,train,76.0,0.74,54.4,3.468,5.976,0.0,0.0,0.0,0.0,76.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.74,0.0,0.0,0.0
dish_1568056019,929,1,train,131.0,1.235,64.3,3.578,8.121,0.0,0.0,0.0,0.0,131.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.235,0.0,0.0,0.0
dish_1568056046,929,1,train,189.0,1.757,88.08,3.694,13.921,0.0,0.0,0.0,0.0,189.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.757,0.0,0.0,0.0
dish_1568056217,930,3,train,427.0,75.537,480.78,12.262,13.921,238.0,0.0,0.0,0.0,189.0,0.0,0.0,0.0,73.78,0.0,0.0,0.0,1.757,0.0,0.0,0.0
dish_1568056258,930,3,train,485.0,77.393,506.358,13.48,16.415,238.0,0.0,0.0,0.0,189.0,58.0,0.0,0.0,73.78,0.0,0.0,0.0,1.757,1.856,0.0,0.0
dish_1568305198,942,2,test,82.0,23.862,311.354,23.042,2.542,82.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,23.862,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1568305225,942,2,test,144.0,24.473,428.668,35.137,4.897,82.0,0.0,0.0,0.0,42.84,19.16,0.0,0.0,23.862,0.0,0.0,0.0,0.583,0.028,0.0,0.0
dish_1568305257,942,2,test,420.0,52.073,837.148,65.497,9.313,358.0,0.0,0.0,0.0,42.84,19.16,0.0,0.0,51.462,0.0,0.0,0.0,0.583,0.028,0.0,0.0
dish_1568313487,943,3,train,49.0,1.078,13.72,0.196,2.597,0.0,0.0,0.0,0.0,49.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.078,0.0,0.0,0.0
dish_1568313527,943,3,train,71.0,1.518,48.92,3.496,4.577,0.0,0.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.518,0.0,0.0,0.0
dish_1568313575,943,3,train,112.0,2.789,57.94,3.619,5.93,0.0,0.0,0.0,0.0,112.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.789,0.0,0.0,0.0
dish_1568313640,943,3,train,145.0,3.152,71.14,3.652,8.9,0.0,0.0,0.0,0.0,145.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.152,0.0,0.0,0.0
dish_1568313676,943,3,train,214.0,24.542,184.99,6.136,8.9,69.0,0.0,0.0,0.0,145.0,0.0,0.0,0.0,21.39,0.0,0.0,0.0,3.152,0.0,0.0,0.0
dish_1568319725,947,0,train,59.0,0.581,111.637,11.51,2.241,0.0,0.0,0.0,0.0,40.76,18.24,0.0,0.0,0.0,0.0,0.0,0.0,0.555,0.026,0.0,0.0
dish_1568319769,947,0,train,151.0,10.701,356.357,20.434,32.601,0.0,0.0,0.0,0.0,40.76,18.24,92.0,0.0,0.0,0.0,0.0,0.0,0.555,0.026,10.12,0.0
dish_1568319838,947,0,train,312.0,39.863,607.506,30.928,39.996,116.77,0.0,0.0,23.35,60.18,19.4,92.29,0.0,28.025,0.0,0.0,0.607,1.079,0.026,10.125,0.0
dish_1568319899,947,0,train,377.0,41.033,627.656,30.993,44.546,116.77,65.0,0.0,23.35,60.18,19.4,92.29,0.0,28.025,1.17,0.0,0.607,1.079,0.026,10.125,0.0
dish_1568320296,948,2,train,85.0,9.52,226.015,7.99,28.475,0.0,0.0,0.0,0.0,0.0,0.0,85.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.52,0.0
dish_1568320338,948,2,train,163.0,25.455,390.181,17.415,30.79,59.56,0.0,0.0,7.94,9.93,0.57,85.0,0.0,15.486,0.0,0.0,0.278,0.171,0.0,9.52,0.0
dish_1568320383,948,2,train,196.0,26.864,437.124,18.984,37.518,59.56,0.0,0.0,39.98,9.93,1.53,85.0,0.0,15.486,0.0,0.0,1.688,0.171,0.0,9.52,0.0
dish_1568320427,948,2,train,240.0,27.656,450.764,19.028,40.598,59.56,44.0,0.0,39.98,9.93,1.53,85.0,0.0,15.486,0.792,0.0,1.688,0.171,0.0,9.52,0.0
dish_1568320470,948,2,train,341.0,29.33,521.039,24.968,44.142,59.56,44.0,4.3,39.98,98.03,10.13,85.0,0.0,15.486,0.792,0.593,1.688,1.252,0.0,9.52,0.0
dish_1568404850,950,2,train,100.0,4.7,157.7,2.2,29.9,0.0,0.0,0.0,0.0,0.0,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.7,0.0
dish_1568404889,950,2,train,219.0,6.24,247.288,4.835,45.123,0.0,0.0,0.0,45.88,65.09,8.03,100.0,0.0,0.0,0.0,0.0,0.413,1.035,0.093,4.7,0.0
dish_1568405183,951,2,train,64.0,1.365,72.403,6.906,2.248,0.0,0.0,0.0,0.0,44.97,19.03,0.0,0.0,0.0,0.0,0.0,0.0,1.107,0.258,0.0,0.0
dish_1568405214,951,2,train,152.0,6.821,144.256,9.715,9.625,0.0,0.0,7.17,0.0,113.79,30.5,0.0,0.54,0.0,0.0,2.753,0.0,3.378,0.625,0.0,0.065
dish_1568405273,951,2,train,288.0,32.077,386.2,21.99,14.98,102.12,0.0,7.17,20.42,118.0,39.75,0.0,0.54,24.509,0.0,2.753,0.531,3.492,0.727,0.0,0.065
dish_1568405306,951,2,train,336.0,35.917,421.24,24.294,15.892,102.12,48.0,7.17,20.42,118.0,39.75,0.0,0.54,24.509,3.84,2.753,0.531,3.492,0.727,0.0,0.065
dish_1568405405,951,2,train,74.0,1.578,83.716,7.985,2.6,0.0,0.0,0.0,0.0,51.99,22.01,0.0,0.0,0.0,0.0,0.0,0.0,1.28,0.298,0.0,0.0
dish_1568405430,951,2,train,121.0,2.187,119.099,9.026,8.612,0.0,0.0,0.0,18.12,77.7,25.18,0.0,0.0,0.0,0.0,0.0,0.163,1.688,0.335,0.0,0.0
dish_1568405466,951,2,train,189.0,2.527,153.099,9.094,17.452,0.0,0.0,0.0,18.12,145.7,25.18,0.0,0.0,0.0,0.0,0.0,0.163,2.028,0.335,0.0,0.0
dish_1568405499,951,2,train,419.0,45.239,562.269,29.854,26.507,172.71,0.0,0.0,52.66,152.83,40.81,0.0,0.0,41.449,0.0,0.0,1.061,2.221,0.508,0.0,0.0
dish_1568649312,952,4,train,78.0,0.546,44.46,0.234,10.92,0.0,0.0,0.0,0.0,78.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.546,0.0,0.0,0.0
dish_1568649354,952,4,train,129.0,1.74,72.871,1.677,14.403,0.0,0.0,0.0,0.0,127.76,1.24,0.0,0.0,0.0,0.0,0.0,0.0,1.74,0.0,0.0,0.0
dish_1568649387,952,4,train,192.0,2.513,163.858,5.302,28.489,0.0,0.0,0.0,59.43,127.76,4.81,0.0,0.0,0.0,0.0,0.0,0.773,1.74,0.0,0.0,0.0
dish_1568649430,952,4,train,327.0,16.013,363.658,20.152,30.649,135.0,0.0,0.0,59.43,127.76,4.81,0.0,0.0,13.5,0.0,0.0,0.773,1.74,0.0,0.0,0.0
dish_1568660810,953,2,train,47.0,5.64,132.54,5.64,15.04,0.0,0.0,0.0,0.0,0.0,0.0,47.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.64,0.0
dish_1568664931,954,3,train,60.0,1.28,67.878,6.475,2.108,0.0,0.0,0.0,0.0,42.16,17.84,0.0,0.0,0.0,0.0,0.0,0.0,1.038,0.242,0.0,0.0
dish_1568664972,954,3,train,221.0,31.216,303.422,14.752,11.03,91.75,0.0,0.0,22.94,81.15,25.15,0.0,0.0,28.444,0.0,0.0,0.803,1.719,0.251,0.0,0.0
dish_1568665012,954,3,train,268.0,32.79,327.421,15.353,15.197,91.75,0.0,0.0,22.94,127.4,25.91,0.0,0.0,28.444,0.0,0.0,0.803,3.291,0.252,0.0,0.0
dish_1568665052,954,3,train,331.0,33.349,345.654,16.161,17.31,91.75,0.0,0.0,22.94,189.55,26.76,0.0,0.0,28.444,0.0,0.0,0.803,3.851,0.252,0.0,0.0
dish_1571935085,960,1,sem_split,85.0,8.118,126.067,5.88,10.653,0.0,0.0,23.4,35.1,26.51,0.0,0.0,0.0,0.0,0.0,6.551,1.228,0.338,0.0,0.0,0.0
dish_1571935895,961,0,sem_split,222.0,5.328,150.96,3.108,26.64,0.0,0.0,0.0,222.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.328,0.0,0.0,0.0,0.0
dish_1571945939,964,4,sem_split,97.0,12.433,236.421,16.825,9.119,43.18,0.0,0.0,14.47,17.43,21.92,0.0,0.0,10.364,0.0,0.0,1.722,0.128,0.219,0.0,0.0
dish_1572021459,967,1,sem_split,108.0,11.232,166.32,12.636,0.648,108.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.232,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1572021481,967,1,sem_split,140.0,11.587,182.827,13.086,3.76,108.0,0.0,0.77,0.0,30.84,0.39,0.0,0.0,11.232,0.0,0.077,0.0,0.278,0.0,0.0,0.0
dish_1572021506,967,1,sem_split,193.0,14.078,266.408,14.252,19.607,108.0,0.0,0.77,0.0,30.84,0.39,53.0,0.0,11.232,0.0,0.077,0.0,0.278,0.0,2.491,0.0
dish_1572032323,971,3,sem_split,75.0,2.677,92.946,3.43,15.704,0.0,0.0,1.77,71.0,0.0,2.23,0.0,0.0,0.0,0.0,0.444,2.201,0.0,0.032,0.0,0.0
dish_1572279989,975,1,sem_split,243.0,1.701,80.19,0.729,19.44,0.0,0.0,0.0,0.0,243.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.701,0.0,0.0,0.0
dish_1572280018,975,1,sem_split,261.0,22.402,125.715,0.522,4.393,217.5,0.0,0.0,0.0,0.0,43.5,0.0,0.0,21.75,0.0,0.0,0.0,0.0,0.652,0.0,0.0
dish_1572365610,978,2,sem_split,167.0,3.507,235.47,11.356,30.06,0.0,0.0,0.0,167.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.507,0.0,0.0,0.0,0.0
dish_1572453962,982,0,sem_split,261.0,22.402,125.715,0.522,4.393,217.5,0.0,0.0,0.0,0.0,43.5,0.0,0.0,21.75,0.0,0.0,0.0,0.0,0.652,0.0,0.0
dish_1572454041,982,0,sem_split,233.0,1.631,76.89,0.699,18.64,0.0,0.0,0.0,0.0,233.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.631,0.0,0.0,0.0
dish_1572538087,986,1,sem_split,334.0,28.668,160.877,0.668,5.622,278.33,0.0,0.0,0.0,0.0,55.67,0.0,0.0,27.833,0.0,0.0,0.0,0.0,0.835,0.0,0.0
dish_1572538461,987,0,sem_split,167.0,16.7,247.16,18.37,2.672,167.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,16.7,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1572540136,988,2,sem_split,46.0,0.322,15.18,0.138,3.68,0.0,0.0,0.0,0.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.322,0.0,0.0,0.0
dish_1572540173,988,2,sem_split,130.0,2.086,133.62,5.85,18.8,0.0,0.0,0.0,84.0,46.0,0.0,0.0,0.0,0.0,0.0,0.0,1.764,0.322,0.0,0.0,0.0
dish_1572540208,988,2,sem_split,198.0,8.886,234.26,13.33,19.888,68.0,0.0,0.0,84.0,46.0,0.0,0.0,0.0,6.8,0.0,0.0,1.764,0.322,0.0,0.0,0.0
dish_1572540238,988,2,sem_split,236.0,16.22,365.208,23.704,21.484,106.0,0.0,0.0,84.0,46.0,0.0,0.0,0.0,14.134,0.0,0.0,1.764,0.322,0.0,0.0,0.0
dish_1572540876,989,0,sem_split,65.0,1.365,91.65,4.42,11.7,0.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.365,0.0,0.0,0.0,0.0
dish_1572540910,989,0,sem_split,266.0,21.465,389.13,26.53,14.916,201.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,20.1,0.0,0.0,1.365,0.0,0.0,0.0,0.0
dish_1572540934,989,0,sem_split,357.0,39.028,702.716,51.373,18.738,292.0,0.0,0.0,65.0,0.0,0.0,0.0,0.0,37.663,0.0,0.0,1.365,0.0,0.0,0.0,0.0
dish_1572541487,990,4,sem_split,199.0,19.9,117.41,0.796,7.164,0.0,0.0,199.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.9,0.0,0.0,0.0,0.0,0.0
dish_1572550690,992,4,sem_split,169.0,10.14,267.02,1.521,52.39,0.0,0.0,0.0,169.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,10.14,0.0,0.0,0.0,0.0
dish_1572550743,992,4,sem_split,107.0,1.457,108.011,10.21,3.956,0.0,0.0,0.0,0.0,91.09,15.91,0.0,0.0,0.0,0.0,0.0,0.0,1.451,0.006,0.0,0.0
dish_1572552638,993,3,sem_split,165.0,9.9,260.7,1.485,51.15,0.0,0.0,0.0,165.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.9,0.0,0.0,0.0,0.0
dish_1572625763,994,1,sem_split,156.0,13.39,199.94,14.352,3.796,130.0,0.0,0.0,0.0,0.0,26.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,0.39,0.0,0.0
dish_1572625810,994,1,sem_split,199.0,29.3,432.57,32.412,4.398,173.0,0.0,0.0,0.0,0.0,26.0,0.0,0.0,28.91,0.0,0.0,0.0,0.0,0.39,0.0,0.0
dish_1572625859,994,1,sem_split,250.0,31.697,512.997,33.534,19.647,173.0,0.0,0.0,0.0,0.0,26.0,51.0,0.0,28.91,0.0,0.0,0.0,0.0,0.39,2.397,0.0
dish_1572625898,994,1,sem_split,271.0,31.928,516.147,33.618,20.214,173.0,0.0,0.0,0.0,21.0,26.0,51.0,0.0,28.91,0.0,0.0,0.0,0.231,0.39,2.397,0.0
dish_1572886987,996,3,sem_split,43.0,1.118,140.18,9.46,13.76,0.0,0.0,0.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.118,0.0,0.0,0.0,0.0
dish_1572887017,996,3,sem_split,274.0,20.945,251.445,9.922,17.648,192.5,0.0,0.0,43.0,0.0,38.5,0.0,0.0,19.25,0.0,0.0,1.118,0.0,0.578,0.0,0.0
dish_1572889151,997,2,sem_split,174.0,17.4,102.66,0.696,6.264,0.0,0.0,174.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,17.4,0.0,0.0,0.0,0.0,0.0
dish_1572889747,998,1,sem_split,39.0,14.43,210.99,16.38,0.546,39.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,14.43,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1572974428,1000,0,sem_split,74.0,3.199,144.599,10.264,11.957,0.0,15.57,0.0,13.73,38.93,3.94,0.0,1.83,0.0,1.713,0.0,0.563,0.576,0.0,0.0,0.347
dish_1572984706,1001,4,sem_split,118.0,2.075,47.964,2.492,5.813,0.0,0.0,0.0,0.0,115.85,2.15,0.0,0.0,0.0,0.0,0.0,0.0,2.075,0.0,0.0,0.0
dish_1572988741,1002,2,sem_split,54.0,8.257,99.544,5.105,5.026,25.21,0.0,3.98,8.19,8.0,8.62,0.0,0.0,6.554,0.0,0.995,0.491,0.088,0.129,0.0,0.0
dish_1572988785,1002,2,sem_split,107.0,12.921,219.324,11.995,15.679,25.21,0.0,3.98,8.19,8.0,8.62,53.0,0.0,6.554,0.0,0.995,0.491,0.088,0.129,4.664,0.0
dish_1573147034,1006,3,sem_split,101.0,5.673,228.96,18.499,12.148,23.03,0.0,0.0,13.61,49.47,7.46,0.0,7.42,2.879,0.0,0.0,0.558,0.574,0.0,0.0,1.662
dish_1573232835,1012,2,sem_split,27.0,9.99,146.07,11.34,0.378,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,9.99,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1573234760,1014,1,sem_split,21.0,7.77,113.61,8.82,0.294,21.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.77,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1573491305,1016,1,sem_split,70.0,7.0,36.4,0.14,0.49,70.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1573491330,1016,1,sem_split,123.0,7.371,53.89,0.299,4.73,70.0,0.0,0.0,0.0,53.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.371,0.0,0.0,0.0
dish_1573492999,1017,0,sem_split,113.0,10.653,257.474,12.241,29.677,0.0,0.0,55.52,0.0,8.81,20.9,0.0,27.76,0.0,0.0,5.552,0.0,0.097,0.062,0.0,4.942
dish_1573505208,1019,4,sem_split,192.0,13.824,259.2,9.6,29.568,0.0,0.0,0.0,0.0,0.0,0.0,192.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.824,0.0
dish_1573506378,1020,3,sem_split,78.0,19.408,123.097,3.534,2.234,61.72,0.0,0.0,6.17,8.56,1.54,0.0,0.0,19.134,0.0,0.0,0.16,0.114,0.0,0.0,0.0
dish_1573579355,1021,0,sem_split,36.0,13.32,194.76,15.12,0.504,36.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.32,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1573579406,1021,0,sem_split,110.0,15.244,436.0,31.4,24.184,36.0,0.0,0.0,74.0,0.0,0.0,0.0,0.0,13.32,0.0,0.0,1.924,0.0,0.0,0.0,0.0
dish_1573579439,1021,0,sem_split,216.0,24.342,571.857,41.152,26.763,124.33,0.0,0.0,74.0,0.0,17.67,0.0,0.0,22.153,0.0,0.0,1.924,0.0,0.265,0.0,0.0
dish_1573579470,1021,0,sem_split,301.0,25.837,606.407,42.947,30.95,124.33,0.0,0.0,74.0,83.45,19.22,0.0,0.0,22.153,0.0,0.0,1.924,1.495,0.265,0.0,0.0
dish_1573579635,1022,2,sem_split,54.0,3.043,171.811,10.355,16.76,0.0,0.0,27.08,9.55,0.23,17.15,0.0,0.0,0.0,0.0,1.679,1.289,0.002,0.072,0.0,0.0
dish_1573679065,1025,0,sem_split,90.0,18.374,184.081,10.211,3.332,68.97,0.0,0.0,9.2,11.26,0.57,0.0,0.0,17.931,0.0,0.0,0.248,0.195,0.0,0.0,0.0
dish_1573679099,1025,0,sem_split,127.0,18.929,188.891,10.285,4.146,68.97,0.0,0.0,9.2,48.26,0.57,0.0,0.0,17.931,0.0,0.0,0.248,0.75,0.0,0.0,0.0
dish_1573849852,1032,4,sem_split,182.0,11.284,418.6,9.828,70.98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,182.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.284
dish_1573849915,1032,4,sem_split,239.0,12.994,431.71,9.828,73.26,0.0,0.0,0.0,0.0,57.0,0.0,0.0,182.0,0.0,0.0,0.0,0.0,1.71,0.0,0.0,11.284
dish_1574097565,1034,3,sem_split,99.0,2.574,322.74,21.78,31.68,0.0,0.0,0.0,99.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.574,0.0,0.0,0.0,0.0
dish_1574097587,1034,3,sem_split,118.0,9.604,425.53,29.76,31.946,19.0,0.0,0.0,99.0,0.0,0.0,0.0,0.0,7.03,0.0,0.0,2.574,0.0,0.0,0.0,0.0
dish_1574184971,1038,1,sem_split,50.0,7.0,80.0,5.0,1.5,50.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1574269354,1041,0,sem_split,109.0,3.509,132.671,7.482,13.964,12.02,0.0,0.0,49.78,36.91,6.87,0.0,3.43,1.502,0.0,0.0,0.498,0.857,0.0,0.0,0.652
dish_1574269685,1042,4,sem_split,149.0,3.874,485.74,32.78,47.68,0.0,0.0,0.0,149.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.874,0.0,0.0,0.0,0.0
dish_1574280343,1043,2,sem_split,71.0,0.71,7.1,0.0,1.42,0.0,0.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.71,0.0,0.0,0.0
dish_1574280383,1043,2,sem_split,113.0,4.028,76.82,4.032,7.3,0.0,42.0,0.0,0.0,71.0,0.0,0.0,0.0,0.0,3.318,0.0,0.0,0.71,0.0,0.0,0.0
dish_1574357682,1045,3,sem_split,68.0,13.124,234.328,18.564,2.856,68.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.124,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1574369148,1047,2,sem_split,68.0,4.216,156.4,3.672,26.52,0.0,0.0,0.0,0.0,0.0,0.0,0.0,68.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.216
dish_1574444158,1050,1,sem_split,162.0,13.905,207.63,14.904,3.942,135.0,0.0,0.0,0.0,0.0,27.0,0.0,0.0,13.5,0.0,0.0,0.0,0.0,0.405,0.0,0.0
dish_1574702427,1052,4,sem_split,114.0,2.964,371.64,25.08,36.48,0.0,0.0,0.0,114.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2.964,0.0,0.0,0.0,0.0
dish_1574702473,1052,4,sem_split,161.0,20.354,625.91,44.82,37.138,47.0,0.0,0.0,114.0,0.0,0.0,0.0,0.0,17.39,0.0,0.0,2.964,0.0,0.0,0.0,0.0
dish_1574711418,1053,0,sem_split,55.0,1.365,30.196,1.237,4.243,0.0,0.0,0.0,0.73,53.28,0.99,0.0,0.0,0.0,0.0,0.0,0.096,1.269,0.0,0.0,0.0
dish_1574720429,1054,4,sem_split,96.0,13.44,153.6,9.6,2.88,96.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.44,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1575307466,1056,3,sem_split,82.0,7.731,186.84,8.883,21.536,0.0,0.0,40.29,0.0,6.4,15.17,0.0,20.15,0.0,0.0,4.029,0.0,0.07,0.045,0.0,3.586
dish_1575393132,1059,2,sem_split,162.0,7.003,316.555,22.47,26.176,0.0,34.09,0.0,30.06,85.23,8.62,0.0,4.0,0.0,3.75,0.0,1.233,1.261,0.0,0.0,0.759
dish_1575404926,1060,1,sem_split,89.0,5.518,204.7,4.806,34.71,0.0,0.0,0.0,0.0,0.0,0.0,0.0,89.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.518
dish_1575407477,1061,0,sem_split,77.0,4.774,177.1,4.158,30.03,0.0,0.0,0.0,0.0,0.0,0.0,0.0,77.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.774
dish_1575478635,1062,3,sem_split,132.0,13.2,195.36,14.52,2.112,132.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,13.2,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1575478663,1062,3,sem_split,220.0,15.488,482.24,33.88,30.272,132.0,0.0,0.0,88.0,0.0,0.0,0.0,0.0,13.2,0.0,0.0,2.288,0.0,0.0,0.0,0.0
dish_1575481406,1063,2,sem_split,27.0,5.211,93.042,7.371,1.134,27.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.211,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1575481443,1063,2,sem_split,85.0,6.719,282.122,20.131,19.694,27.0,0.0,0.0,58.0,0.0,0.0,0.0,0.0,5.211,0.0,0.0,1.508,0.0,0.0,0.0,0.0
dish_1575923584,1066,4,sem_split,224.0,11.424,294.784,2.464,55.552,0.0,0.0,0.0,224.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,11.424,0.0,0.0,0.0,0.0
dish_1575923893,1067,3,sem_split,115.0,5.865,151.34,1.265,28.52,0.0,0.0,0.0,115.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5.865,0.0,0.0,0.0,0.0
dish_1575999037,1071,2,sem_split,156.0,13.39,199.94,14.352,3.796,130.0,0.0,0.0,0.0,0.0,26.0,0.0,0.0,13.0,0.0,0.0,0.0,0.0,0.39,0.0,0.0
dish_1576171118,1073,1,sem_split,53.0,7.42,84.8,5.3,1.59,53.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,7.42,0.0,0.0,0.0,0.0,0.0,0.0,0.0
dish_1576171157,1073,1,sem_split,132.0,10.098,132.347,7.109,8.61,53.0,0.0,0.0,0.0,76.31,1.44,0.95,0.29,7.42,0.0,0.0,0.0,2.595,0.0,0.017,0.066
dish_1576171204,1073,1,sem_split,252.0,20.398,286.147,18.149,11.53,153.0,0.0,0.0,0.0,76.31,21.44,0.95,0.29,17.42,0.0,0.0,0.0,2.595,0.3,0.017,0.066
dish_1576171237,1073,1,sem_split,339.0,22.225,408.817,24.065,27.19,153.0,0.0,0.0,87.0,76.31,21.44,0.95,0.29,17.42,0.0,0.0,1.827,2.595,0.3,0.017,0.066


In [ ]:
%%writefile /tmp/nutrelle_codigo/listas/aval_ids.csv
prato_id,split,dish_id
d0001,calib,dish_1558457665
d0002,teste,dish_1563304301
d0003,teste,dish_1566329049
d0004,calib,dish_1568061202
d0005,teste,dish_1572627130
d0006,calib,dish_1565640772
d0007,teste,dish_1563824130
d0008,calib,dish_1558115641
d0009,calib,dish_1558375667
d0010,teste,dish_1563555510
d0011,teste,dish_1560368738
d0012,teste,dish_1561752035
d0013,calib,dish_1558637932
d0014,teste,dish_1564605328
d0015,teste,dish_1558642458
d0016,calib,dish_1563294575
d0017,teste,dish_1572029300
d0018,teste,dish_1573159868
d0019,teste,dish_1567022383
d0020,teste,dish_1576100046
d0021,calib,dish_1564516397
d0022,teste,dish_1573760187
d0023,calib,dish_1573156908
d0024,teste,dish_1567625343
d0025,calib,dish_1561481673
d0026,calib,dish_1557862783
d0027,teste,dish_1573232461
d0028,teste,dish_1565727791
d0029,teste,dish_1568666089
d0030,calib,dish_1561662216
d0031,teste,dish_1565810205
d0032,teste,dish_1565811000
d0033,teste,dish_1561577119
d0034,calib,dish_1561661500
d0035,calib,dish_1566920304
d0036,teste,dish_1558640604
d0037,calib,dish_1560370833
d0038,calib,dish_1572034012
d0039,teste,dish_1562774346
d0040,calib,dish_1567801716
d0041,calib,dish_1573232434
d0042,teste,dish_1558640705
d0043,teste,dish_1562774504
d0044,teste,dish_1573161475
d0045,calib,dish_1561049435
d0046,teste,dish_1562963289
d0047,calib,dish_1574443421
d0048,teste,dish_1564689108
d0049,teste,dish_1562689645
d0050,teste,dish_1563567319
d0051,teste,dish_1563305913
d0052,teste,dish_1563216412
d0053,teste,dish_1566923060
d0054,teste,dish_1560356523
d0055,teste,dish_1565972384
d0056,teste,dish_1571931782
d0057,teste,dish_1563468327
d0058,calib,dish_1566328724
d0059,teste,dish_1565383062
d0060,calib,dish_1562613918
d0061,teste,dish_1571934465
d0062,teste,dish_1566849862
d0063,teste,dish_1558636544
d0064,calib,dish_1571944238
d0065,calib,dish_1573503054
d0066,teste,dish_1563294465
d0067,calib,dish_1568401018
d0068,teste,dish_1573764136
d0069,teste,dish_1572291671
d0070,calib,dish_1562688482
d0071,calib,dish_1559667973
d0072,teste,dish_1573665056
d0073,calib,dish_1564775216
d0074,teste,dish_1558378634
d0075,teste,dish_1562010315
d0076,calib,dish_1566502537
d0077,teste,dish_1567023161
d0078,teste,dish_1563898722
d0079,calib,dish_1559845370
d0080,teste,dish_1562614980
d0081,teste,dish_1565813414
d0082,calib,dish_1562787848
d0083,calib,dish_1566316726
d0084,teste,dish_1566838286
d0085,calib,dish_1571932341
d0086,calib,dish_1567107049
d0087,teste,dish_1573848859
d0088,calib,dish_1565637123
d0089,teste,dish_1565896745
d0090,teste,dish_1559246053
d0091,teste,dish_1562959059
d0092,calib,dish_1562691409
d0093,calib,dish_1558458541
d0094,calib,dish_1558724723
d0095,teste,dish_1571944214
d0096,calib,dish_1564761488
d0097,teste,dish_1560526552
d0098,calib,dish_1558721757
d0099,teste,dish_1565711115
d0100,calib,dish_1562614121
d0101,teste,dish_1573763139
d0102,teste,dish_1565021054
d0103,teste,dish_1559844366
d0104,teste,dish_1559700851
d0105,teste,dish_1561737176
d0106,teste,dish_1563466280
d0107,teste,dish_1567526738
d0108,teste,dish_1575308389
d0109,teste,dish_1562871297
d0110,teste,dish_1561405836
d0111,teste,dish_1572365415
d0112,teste,dish_1558630207
d0113,teste,dish_1563295360
d0114,teste,dish_1559242221
d0115,teste,dish_1565117429
d0116,teste,dish_1563393366
d0117,teste,dish_1558639089
d0118,teste,dish_1562783691
d0119,teste,dish_1571932398
d0120,teste,dish_1563220171
d0121,teste,dish_1568318316
d0122,teste,dish_1568401167
d0123,teste,dish_1565207815
d0124,calib,dish_1565194781
d0125,teste,dish_1564168739
d0126,calib,dish_1565711183
d0127,teste,dish_1559581378
d0128,teste,dish_1562010085
d0129,teste,dish_1562958563
d0130,calib,dish_1558550218
d0131,calib,dish_1575997268
d0132,teste,dish_1565207631
d0133,teste,dish_1564432094
d0134,calib,dish_1565031338
d0135,teste,dish_1572548878
d0136,teste,dish_1561492703
d0137,teste,dish_1564502063
d0138,teste,dish_1560973984
d0139,teste,dish_1562603569
d0140,teste,dish_1573665025
d0141,teste,dish_1566316525
d0142,teste,dish_1566414611
d0143,teste,dish_1563565354
d0144,teste,dish_1564514997
d0145,teste,dish_1574281837
d0146,teste,dish_1563566725
d0147,teste,dish_1558114441
d0148,teste,dish_1561393550
d0149,teste,dish_1575653017
d0150,calib,dish_1563475874
d0151,teste,dish_1560543505
d0152,calib,dish_1571944549
d0153,teste,dish_1572031211
d0154,teste,dish_1562875320
d0155,teste,dish_1563206409
d0156,teste,dish_1562168696
d0157,teste,dish_1565021599
d0158,teste,dish_1573243053
d0159,calib,dish_1563305513
d0160,teste,dish_1562874049
d0161,teste,dish_1563913709
d0162,calib,dish_1557853229
d0163,teste,dish_1560357297
d0164,calib,dish_1576258613
d0165,teste,dish_1559668314
d0166,calib,dish_1560453132
d0167,teste,dish_1564688969
d0168,teste,dish_1573848236
d0169,calib,dish_1563218031
d0170,calib,dish_1575307845
d0171,teste,dish_1559238993
d0172,teste,dish_1563824406
d0173,teste,dish_1558724658
d0174,teste,dish_1563823423
d0175,teste,dish_1560356627
d0176,calib,dish_1563984729
d0177,teste,dish_1565117990
d0178,teste,dish_1562010266
d0179,calib,dish_1572278894
d0180,teste,dish_1559844063
d0181,teste,dish_1566417398
d0182,teste,dish_1563984199
d0183,teste,dish_1562958445
d0184,teste,dish_1559578551
d0185,teste,dish_1559589942
d0186,teste,dish_1562611853
d0187,teste,dish_1565021675
d0188,teste,dish_1561579867
d0189,teste,dish_1563208439
d0190,teste,dish_1574455483
d0191,calib,dish_1565109606
d0192,teste,dish_1574443234
d0193,teste,dish_1559242066
d0194,calib,dish_1568048018
d0195,teste,dish_1571944531
d0196,calib,dish_1563306039
d0197,teste,dish_1562086740
d0198,teste,dish_1562876732
d0199,teste,dish_1560787817
d0200,teste,dish_1565021965
d0201,calib,dish_1560887408
d0202,calib,dish_1566416299
d0203,teste,dish_1559240510
d0204,calib,dish_1558641897
d0205,teste,dish_1571946303
d0206,teste,dish_1559158354
d0207,teste,dish_1576258685
d0208,teste,dish_1565117962
d0209,calib,dish_1571934154
d0210,teste,dish_1571931648
d0211,teste,dish_1563207364
d0212,teste,dish_1566588066
d0213,calib,dish_1563221054
d0214,calib,dish_1565118729
d0215,teste,dish_1564159520
d0216,teste,dish_1558379712
d0217,teste,dish_1558373008
d0218,teste,dish_1566402041
d0219,calib,dish_1565987081
d0220,teste,dish_1562094877
d0221,teste,dish_1573159511
d0222,teste,dish_1561392459
d0223,teste,dish_1563205982
d0224,teste,dish_1561739954
d0225,calib,dish_1558376306
d0226,teste,dish_1561481634
d0227,calib,dish_1559240987
d0228,teste,dish_1560455224
d0229,teste,dish_1571944507
d0230,calib,dish_1558724959
d0231,teste,dish_1561576483
d0232,teste,dish_1562698100
d0233,calib,dish_1562688552
d0234,teste,dish_1562788794
d0235,teste,dish_1558636388
d0236,teste,dish_1568060722
d0237,teste,dish_1568315988
d0238,calib,dish_1566413469
d0239,calib,dish_1560800836
d0240,teste,dish_1561575951
d0241,teste,dish_1562173111
d0242,calib,dish_1563813308
d0243,teste,dish_1560357267
d0244,teste,dish_1558380557
d0245,teste,dish_1574111587
d0246,teste,dish_1565725188
d0247,calib,dish_1573156961
d0248,teste,dish_1558378553
d0249,calib,dish_1560357427
d0250,teste,dish_1559245697
d0251,teste,dish_1558639335
d0252,teste,dish_1572464329
d0253,teste,dish_1565884692
d0254,teste,dish_1573762622
d0255,teste,dish_1565023182
d0256,teste,dish_1563984140
d0257,teste,dish_1567526814
d0258,teste,dish_1559149688
d0259,teste,dish_1564590254
d0260,teste,dish_1562700718
d0261,calib,dish_1561752535
d0262,teste,dish_1568060461
d0263,calib,dish_1568318361
d0264,calib,dish_1572453701
d0265,calib,dish_1571944487
d0266,teste,dish_1566328745
d0267,teste,dish_1559235815
d0268,teste,dish_1565799565
d0269,calib,dish_1561060787
d0270,teste,dish_1567542813
d0271,teste,dish_1565194684
d0272,teste,dish_1558718974
d0273,teste,dish_1561753371
d0274,calib,dish_1563392128
d0275,teste,dish_1562613285
d0276,teste,dish_1565898230
d0277,calib,dish_1558031566
d0278,teste,dish_1572376754
d0279,calib,dish_1562777566
d0280,teste,dish_1565637162
d0281,teste,dish_1558719994
d0282,teste,dish_1574359199
d0283,calib,dish_1562788760
d0284,teste,dish_1561741174
d0285,calib,dish_1573232499
d0286,calib,dish_1560874751
d0287,teste,dish_1562875159
d0288,teste,dish_1564073860
d0289,teste,dish_1573233271
d0290,teste,dish_1558640008
d0291,teste,dish_1560961012
d0292,calib,dish_1558549413
d0293,teste,dish_1572464692
d0294,teste,dish_1566934487
d0295,teste,dish_1566328988
d0296,teste,dish_1567020394
d0297,teste,dish_1557861837
d0298,teste,dish_1562613010
d0299,calib,dish_1571946737
d0300,teste,dish_1559590157
d0301,calib,dish_1566316496
d0302,teste,dish_1572291722
d0303,calib,dish_1573763118
d0304,calib,dish_1562963648
d0305,teste,dish_1565711570
d0306,teste,dish_1558031675
d0307,teste,dish_1564429184
d0308,teste,dish_1573762582
d0309,teste,dish_1558724932
d0310,teste,dish_1561060613
d0311,teste,dish_1560455179
d0312,teste,dish_1573060410
d0313,teste,dish_1576259732
d0314,calib,dish_1566413388
d0315,calib,dish_1561739973
d0316,calib,dish_1565370004
d0317,teste,dish_1566332154
d0318,calib,dish_1558630072
d0319,teste,dish_1562691383
d0320,teste,dish_1562690074
d0321,teste,dish_1568315132
d0322,teste,dish_1561752680
d0323,teste,dish_1572462136
d0324,teste,dish_1560441487
d0325,teste,dish_1561753168
d0326,teste,dish_1558380527
d0327,teste,dish_1566931650
d0328,teste,dish_1572278972
d0329,calib,dish_1560801883
d0330,calib,dish_1558722931
d0331,calib,dish_1561737200
d0332,teste,dish_1574728229
d0333,teste,dish_1558725193
d0334,calib,dish_1560539340
d0335,teste,dish_1561747075
d0336,teste,dish_1561578188
d0337,teste,dish_1566488956
d0338,teste,dish_1565107119
d0339,calib,dish_1565195362
d0340,teste,dish_1561393715
d0341,teste,dish_1559849412
d0342,teste,dish_1562691674
d0343,calib,dish_1559751458
d0344,calib,dish_1558373492
d0345,calib,dish_1562700760
d0346,teste,dish_1573072479
d0347,calib,dish_1564169644
d0348,teste,dish_1563218201
d0349,teste,dish_1560360030
d0350,teste,dish_1558459395
d0351,calib,dish_1566502573
d0352,teste,dish_1573851651
d0353,calib,dish_1565637578
d0354,calib,dish_1562172643
d0355,teste,dish_1565896039
d0356,calib,dish_1566402197
d0357,calib,dish_1560888849
d0358,teste,dish_1565884738
d0359,teste,dish_1560455203
d0360,teste,dish_1562602589
d0361,teste,dish_1571944571
d0362,teste,dish_1563464429
d0363,calib,dish_1572379407
d0364,teste,dish_1560975601
d0365,teste,dish_1561749711
d0366,teste,dish_1559842274
d0367,calib,dish_1563305047
d0368,teste,dish_1575664962
d0369,teste,dish_1574098164
d0370,teste,dish_1562688524
d0371,calib,dish_1562085672
d0372,teste,dish_1566838378
d0373,teste,dish_1563826306
d0374,teste,dish_1563813833
d0375,teste,dish_1564518586
d0376,teste,dish_1562959190
d0377,teste,dish_1565036821
d0378,calib,dish_1564073716
d0379,teste,dish_1562873814
d0380,teste,dish_1563220143
d0381,calib,dish_1560802048
d0382,calib,dish_1565117892
d0383,teste,dish_1562168471
d0384,teste,dish_1558632080
d0385,calib,dish_1561576449
d0386,teste,dish_1567542778
d0387,calib,dish_1565209963
d0388,teste,dish_1561740321
d0389,teste,dish_1562959129
d0390,teste,dish_1562876531
d0391,calib,dish_1564675193
d0392,teste,dish_1565987116
d0393,teste,dish_1563389786
d0394,teste,dish_1559332418
d0395,teste,dish_1565972363
d0396,teste,dish_1563996596
d0397,calib,dish_1565974375
d0398,calib,dish_1558637393
d0399,teste,dish_1566931586
d0400,teste,dish_1564427406
n001,n5k_v1,dish_1561662842
n002,n5k_v1,dish_1563389548
n003,n5k_v1,dish_1565986702
n004,n5k_v1,dish_1565986733
n005,n5k_v1,dish_1561662814
n006,n5k_v1,dish_1566501594
n007,n5k_v1,dish_1565034681
n008,n5k_v1,dish_1565811139
n009,n5k_v1,dish_1560526408
n010,n5k_v1,dish_1562009934
n011,n5k_v1,dish_1564689078
n012,n5k_v1,dish_1565123881
n013,n5k_v1,dish_1565898193
n014,n5k_v1,dish_1558376984
n015,n5k_v1,dish_1560975572
n016,n5k_v1,dish_1566851677
n017,n5k_v1,dish_1558376874
n018,n5k_v1,dish_1562703345
n019,n5k_v1,dish_1564428102
n020,n5k_v1,dish_1566920339
n021,n5k_v1,dish_1558722207
n022,n5k_v1,dish_1564588889
n023,n5k_v1,dish_1567007751
n024,n5k_v1,dish_1564588859
n025,n5k_v1,dish_1562788568
n026,n5k_v1,dish_1560444869
n027,n5k_v1,dish_1566416701
n028,n5k_v1,dish_1562691032
n029,n5k_v1,dish_1566328776
n030,n5k_v1,dish_1557936645
n031,n5k_v1,dish_1558372948
n032,n5k_v1,dish_1566851704
n033,n5k_v1,dish_1560368464
n034,n5k_v1,dish_1563909580
n035,n5k_v1,dish_1565898158
n036,n5k_v1,dish_1558546558
n037,n5k_v1,dish_1559844272
n038,n5k_v1,dish_1563897981
n039,n5k_v1,dish_1561739238
n040,n5k_v1,dish_1561663580
n041,n5k_v1,dish_1558720236
n042,n5k_v1,dish_1563984167
n043,n5k_v1,dish_1557862345
n044,n5k_v1,dish_1561574394
n045,n5k_v1,dish_1563551164
n046,n5k_v1,dish_1566328805
n047,n5k_v1,dish_1561663606
n048,n5k_v1,dish_1562099076
n049,n5k_v1,dish_1561654374
n050,n5k_v1,dish_1557936555


In [ ]:
%%writefile /tmp/nutrelle_codigo/listas/aval2_ids.csv
prato_id,split,dish_id
e0001,teste2,dish_1558640671
e0002,teste2,dish_1565383026
e0003,teste2,dish_1558640941
e0004,teste2,dish_1562871408
e0005,teste2,dish_1568060542
e0006,teste2,dish_1564590190
e0007,teste2,dish_1558639195
e0008,teste2,dish_1572291695
e0009,teste2,dish_1566414642
e0010,teste2,dish_1573665080
e0011,teste2,dish_1566417442
e0012,teste2,dish_1564590226
e0013,teste2,dish_1566923030
e0014,teste2,dish_1560543605
e0015,teste2,dish_1559844506
e0016,teste2,dish_1562615103
e0017,teste2,dish_1558639673
e0018,teste2,dish_1561578461
e0019,teste2,dish_1561741139
e0020,teste2,dish_1561392570
e0021,teste2,dish_1565020822
e0022,teste2,dish_1565021141
e0023,teste2,dish_1562875241
e0024,teste2,dish_1558459462
e0025,teste2,dish_1573851578
e0026,teste2,dish_1561405887
e0027,teste2,dish_1558641287
e0028,teste2,dish_1563393341
e0029,teste2,dish_1558719066
e0030,teste2,dish_1562871363
e0031,teste2,dish_1563205877
e0032,teste2,dish_1565382833
e0033,teste2,dish_1573760161
e0034,teste2,dish_1565884833
e0035,teste2,dish_1560360112
e0036,teste2,dish_1562690047
e0037,teste2,dish_1563565454
e0038,teste2,dish_1565383097
e0039,teste2,dish_1560787781
e0040,teste2,dish_1562615070
e0041,teste2,dish_1563826223
e0042,teste2,dish_1565725156
e0043,teste2,dish_1561663068
e0044,teste2,dish_1572548822
e0045,teste2,dish_1565895879
e0046,teste2,dish_1562690917
e0047,teste2,dish_1566501258
e0048,teste2,dish_1562959344
e0049,teste2,dish_1562959766
e0050,teste2,dish_1563898880
e0051,teste2,dish_1558719570
e0052,teste2,dish_1562094931
e0053,teste2,dish_1560543755
e0054,teste2,dish_1561579824
e0055,teste2,dish_1562958775
e0056,teste2,dish_1567526888
e0057,teste2,dish_1560974009
e0058,teste2,dish_1562876494
e0059,teste2,dish_1563823390
e0060,teste2,dish_1566488905
e0061,teste2,dish_1562010293
e0062,teste2,dish_1565811025
e0063,teste2,dish_1559240441
e0064,teste2,dish_1563551006
e0065,teste2,dish_1566587958
e0066,teste2,dish_1562783605
e0067,teste2,dish_1559235690
e0068,teste2,dish_1562611824
e0069,teste2,dish_1565382762
e0070,teste2,dish_1565034605
e0071,teste2,dish_1564427455
e0072,teste2,dish_1559581356
e0073,teste2,dish_1560444701
e0074,teste2,dish_1561579617
e0075,teste2,dish_1558641416
e0076,teste2,dish_1559590031
e0077,teste2,dish_1559842326
e0078,teste2,dish_1568315001
e0079,teste2,dish_1561578344
e0080,teste2,dish_1562010394
e0081,teste2,dish_1566838351
e0082,teste2,dish_1564429270
e0083,teste2,dish_1558719377
e0084,teste2,dish_1562602860
e0085,teste2,dish_1560360148
e0086,teste2,dish_1559245979
e0087,teste2,dish_1563295334
e0088,teste2,dish_1566838407
e0089,teste2,dish_1558638954
e0090,teste2,dish_1564589072
e0091,teste2,dish_1564588820
e0092,teste2,dish_1560444724
e0093,teste2,dish_1568315088
e0094,teste2,dish_1574098218
e0095,teste2,dish_1571931457
e0096,teste2,dish_1558639440
e0097,teste2,dish_1561578839
e0098,teste2,dish_1561392710
e0099,teste2,dish_1562959485
e0100,teste2,dish_1566849987
e0101,teste2,dish_1566931519
e0102,teste2,dish_1567007783
e0103,teste2,dish_1558114484
e0104,teste2,dish_1558640418
e0105,teste2,dish_1562010130
e0106,teste2,dish_1562873790
e0107,teste2,dish_1559578579
e0108,teste2,dish_1558459528
e0109,teste2,dish_1559844683
e0110,teste2,dish_1558640311
e0111,teste2,dish_1562873838
e0112,teste2,dish_1562010440
e0113,teste2,dish_1564432128
e0114,teste2,dish_1568060501
e0115,teste2,dish_1565020895
e0116,teste2,dish_1568316085
e0117,teste2,dish_1560444616
e0118,teste2,dish_1563389719
e0119,teste2,dish_1563897794
e0120,teste2,dish_1566500988
e0121,teste2,dish_1562959700
e0122,teste2,dish_1566849955
e0123,teste2,dish_1562689519
e0124,teste2,dish_1563464670
e0125,teste2,dish_1565725095
e0126,teste2,dish_1559158263
e0127,teste2,dish_1558640979
e0128,teste2,dish_1562613376
e0129,teste2,dish_1561751958
e0130,teste2,dish_1560961261
e0131,teste2,dish_1561578603
e0132,teste2,dish_1558638924
e0133,teste2,dish_1564428181
e0134,teste2,dish_1561746876
e0135,teste2,dish_1561492519
e0136,teste2,dish_1561740353
e0137,teste2,dish_1562086595
e0138,teste2,dish_1567007719
e0139,teste2,dish_1559235771
e0140,teste2,dish_1563220109
e0141,teste2,dish_1561392632
e0142,teste2,dish_1564518454
e0143,teste2,dish_1561579902
e0144,teste2,dish_1574443257
e0145,teste2,dish_1563996464
e0146,teste2,dish_1558640748
e0147,teste2,dish_1568316041
e0148,teste2,dish_1561753021
e0149,teste2,dish_1561579933
e0150,teste2,dish_1563464651
e0151,teste2,dish_1560368570
e0152,teste2,dish_1557936599
e0153,teste2,dish_1558639849
e0154,teste2,dish_1562168450
e0155,teste2,dish_1558719040
e0156,teste2,dish_1560360102
e0157,teste2,dish_1563550976
e0158,teste2,dish_1558639362
e0159,teste2,dish_1566838317
e0160,teste2,dish_1561753649
e0161,teste2,dish_1564590115
e0162,teste2,dish_1558546663
e0163,teste2,dish_1558642490
e0164,teste2,dish_1561392410
e0165,teste2,dish_1559242155
e0166,teste2,dish_1558114609
e0167,teste2,dish_1576259776
e0168,teste2,dish_1567023296
e0169,teste2,dish_1565810146
e0170,teste2,dish_1567023375
e0171,teste2,dish_1565123783
e0172,teste2,dish_1562959732
e0173,teste2,dish_1561576074
e0174,teste2,dish_1563566626
e0175,teste2,dish_1572627152
e0176,teste2,dish_1559844458
e0177,teste2,dish_1558639752
e0178,teste2,dish_1564518542
e0179,teste2,dish_1562876645
e0180,teste2,dish_1566934366
e0181,teste2,dish_1563984296
e0182,teste2,dish_1562173175
e0183,teste2,dish_1568665970
e0184,teste2,dish_1561578441
e0185,teste2,dish_1558640917
e0186,teste2,dish_1566851549
e0187,teste2,dish_1566931548
e0188,teste2,dish_1558636483
e0189,teste2,dish_1565813446
e0190,teste2,dish_1564515049
e0191,teste2,dish_1562698276
e0192,teste2,dish_1568666000
e0193,teste2,dish_1558720364
e0194,teste2,dish_1563909469
e0195,teste2,dish_1562614952
e0196,teste2,dish_1567023132
e0197,teste2,dish_1558639818
e0198,teste2,dish_1568666061
e0199,teste2,dish_1561575977
e0200,teste2,dish_1574098245
e0201,teste2,dish_1561578417
e0202,teste2,dish_1558640450
e0203,teste2,dish_1567542843
e0204,teste2,dish_1561663541
e0205,teste2,dish_1568314857
e0206,teste2,dish_1562689548
e0207,teste2,dish_1563824250
e0208,teste2,dish_1559842374
e0209,teste2,dish_1560526490
e0210,teste2,dish_1565898758
e0211,teste2,dish_1559149719
e0212,teste2,dish_1561751897
e0213,teste2,dish_1558719336
e0214,teste2,dish_1564589025
e0215,teste2,dish_1560444815
e0216,teste2,dish_1563824155
e0217,teste2,dish_1563389446
e0218,teste2,dish_1562959379
e0219,teste2,dish_1558639876
e0220,teste2,dish_1562173411
e0221,teste2,dish_1563216381
e0222,teste2,dish_1571931594
e0223,teste2,dish_1561749908
e0224,teste2,dish_1564514953
e0225,teste2,dish_1562602832
e0226,teste2,dish_1562703447
e0227,teste2,dish_1562698209
e0228,teste2,dish_1567022296
e0229,teste2,dish_1558640145
e0230,teste2,dish_1559245848
e0231,teste2,dish_1563898786
e0232,teste2,dish_1563207332
e0233,teste2,dish_1564429235
e0234,teste2,dish_1568314823
e0235,teste2,dish_1560444637
e0236,teste2,dish_1565898128
e0237,teste2,dish_1561740385
e0238,teste2,dish_1566332085
e0239,teste2,dish_1565117463
e0240,teste2,dish_1563898107
e0241,teste2,dish_1566501012
e0242,teste2,dish_1563824377
e0243,teste2,dish_1567007848
e0244,teste2,dish_1562959024
e0245,teste2,dish_1562963109
e0246,teste2,dish_1562009894
e0247,teste2,dish_1562168427
e0248,teste2,dish_1566501067
e0249,teste2,dish_1561575996
e0250,teste2,dish_1562691064
e0251,teste2,dish_1568315177
e0252,teste2,dish_1567020417
e0253,teste2,dish_1561753589
e0254,teste2,dish_1562173381
e0255,teste2,dish_1559844479
e0256,teste2,dish_1565727875
e0257,teste2,dish_1561578394
e0258,teste2,dish_1562963084
e0259,teste2,dish_1566931620
e0260,teste2,dish_1565896662
e0261,teste2,dish_1559332487
e0262,teste2,dish_1561739160
e0263,teste2,dish_1563566939
e0264,teste2,dish_1558719401
e0265,teste2,dish_1561392543
e0266,teste2,dish_1559245654
e0267,teste2,dish_1565036890
e0268,teste2,dish_1560445069
e0269,teste2,dish_1562614895
e0270,teste2,dish_1566850099
e0271,teste2,dish_1560543729
e0272,teste2,dish_1563996656
e0273,teste2,dish_1560368691
e0274,teste2,dish_1566332193
e0275,teste2,dish_1559581333
e0276,teste2,dish_1563824339
e0277,teste2,dish_1558641253
e0278,teste2,dish_1562959599
e0279,teste2,dish_1558639588
e0280,teste2,dish_1560455421
e0281,teste2,dish_1565382886
e0282,teste2,dish_1559578599
e0283,teste2,dish_1563913627
e0284,teste2,dish_1566501041
e0285,teste2,dish_1560961202
e0286,teste2,dish_1562958603
e0287,teste2,dish_1560526514
e0288,teste2,dish_1558641391
e0289,teste2,dish_1567022212
e0290,teste2,dish_1560441511
e0291,teste2,dish_1563566810
e0292,teste2,dish_1558639266
e0293,teste2,dish_1560444756
e0294,teste2,dish_1566332116
e0295,teste2,dish_1558459276
e0296,teste2,dish_1558641007
e0297,teste2,dish_1558640105
e0298,teste2,dish_1563565323
e0299,teste2,dish_1563913675
e0300,teste2,dish_1565810237
e0301,teste2,dish_1562690950
e0302,teste2,dish_1559849537
e0303,teste2,dish_1561753477
e0304,teste2,dish_1563824279
e0305,teste2,dish_1565021104
e0306,teste2,dish_1559590202
e0307,teste2,dish_1565123845
e0308,teste2,dish_1560455059
e0309,teste2,dish_1560368395
e0310,teste2,dish_1562788601
e0311,teste2,dish_1565382969
e0312,teste2,dish_1562963059
e0313,teste2,dish_1561739188
e0314,teste2,dish_1558641108
e0315,teste2,dish_1565898432
e0316,teste2,dish_1566849923
e0317,teste2,dish_1563468199
e0318,teste2,dish_1566414723
e0319,teste2,dish_1567022251
e0320,teste2,dish_1561579673
e0321,teste2,dish_1565898361
e0322,teste2,dish_1562871488
e0323,teste2,dish_1559589873
e0324,teste2,dish_1561752970
e0325,teste2,dish_1558378516
e0326,teste2,dish_1565799617
e0327,teste2,dish_1560543465
e0328,teste2,dish_1558379639
e0329,teste2,dish_1566922978
e0330,teste2,dish_1563551134
e0331,teste2,dish_1563555605
e0332,teste2,dish_1561741063
e0333,teste2,dish_1559590005
e0334,teste2,dish_1563468269
e0335,teste2,dish_1568060616
e0336,teste2,dish_1562086702
e0337,teste2,dish_1558641075
e0338,teste2,dish_1558632130
e0339,teste2,dish_1558640039
e0340,teste2,dish_1566488981
e0341,teste2,dish_1561746970
e0342,teste2,dish_1567020340
e0343,teste2,dish_1565895966
e0344,teste2,dish_1563389693
e0345,teste2,dish_1559844030
e0346,teste2,dish_1563208459
e0347,teste2,dish_1560455442
e0348,teste2,dish_1564588919
e0349,teste2,dish_1565022027
e0350,teste2,dish_1558380414
e0351,teste2,dish_1563996441
e0352,teste2,dish_1560975491
e0353,teste2,dish_1567023110
e0354,teste2,dish_1564605293
e0355,teste2,dish_1564427430
e0356,teste2,dish_1562959547
e0357,teste2,dish_1559849497
e0358,teste2,dish_1562690886
e0359,teste2,dish_1562168399
e0360,teste2,dish_1564432238
e0361,teste2,dish_1565898511
e0362,teste2,dish_1563823359
e0363,teste2,dish_1565036788
e0364,teste2,dish_1557861605
e0365,teste2,dish_1561739212
e0366,teste2,dish_1563813806
e0367,teste2,dish_1558640294
e0368,teste2,dish_1568666032
e0369,teste2,dish_1563393148
e0370,teste2,dish_1562613045
e0371,teste2,dish_1574359130
e0372,teste2,dish_1558719514
e0373,teste2,dish_1563464543
e0374,teste2,dish_1558719298
e0375,teste2,dish_1562615153
e0376,teste2,dish_1562958809
e0377,teste2,dish_1564590352
e0378,teste2,dish_1560356566
e0379,teste2,dish_1565898402
e0380,teste2,dish_1559844090
e0381,teste2,dish_1565207694
e0382,teste2,dish_1562876618
e0383,teste2,dish_1568060697
e0384,teste2,dish_1563996485
e0385,teste2,dish_1558720097
e0386,teste2,dish_1564428144
e0387,teste2,dish_1565898544
e0388,teste2,dish_1559668238
e0389,teste2,dish_1558640238
e0390,teste2,dish_1563566965
e0391,teste2,dish_1560455343
e0392,teste2,dish_1558640513
e0393,teste2,dish_1558639221
e0394,teste2,dish_1560444570
e0395,teste2,dish_1563205953
e0396,teste2,dish_1564689219
e0397,teste2,dish_1565727851
e0398,teste2,dish_1558639526
e0399,teste2,dish_1562959103
e0400,teste2,dish_1565725123
e0401,teste2,dish_1559844113
e0402,teste2,dish_1560543583
e0403,teste2,dish_1558639138
e0404,teste2,dish_1558640614
e0405,teste2,dish_1560543557
e0406,teste2,dish_1561749664
e0407,teste2,dish_1559235639
e0408,teste2,dish_1563468299
e0409,teste2,dish_1563909507
e0410,teste2,dish_1564590146
e0411,teste2,dish_1562099157
e0412,teste2,dish_1563468240
e0413,teste2,dish_1561492628
e0414,teste2,dish_1565021185
e0415,teste2,dish_1559590130
e0416,teste2,dish_1561753432
e0417,teste2,dish_1562602888
e0418,teste2,dish_1563207389
e0419,teste2,dish_1562963006
e0420,teste2,dish_1571931678
e0421,teste2,dish_1565036922
e0422,teste2,dish_1568666230
e0423,teste2,dish_1566501126
e0424,teste2,dish_1558114569
e0425,teste2,dish_1560961107
e0426,teste2,dish_1563913741
e0427,teste2,dish_1563565415
e0428,teste2,dish_1566934401
e0429,teste2,dish_1562871448
e0430,teste2,dish_1562876709
e0431,teste2,dish_1567023184
e0432,teste2,dish_1565107171
e0433,teste2,dish_1563913588
e0434,teste2,dish_1560526472
e0435,teste2,dish_1558380464
e0436,teste2,dish_1560455030
e0437,teste2,dish_1561392734
e0438,teste2,dish_1560441464
e0439,teste2,dish_1565725244
e0440,teste2,dish_1561753400
e0441,teste2,dish_1563824304
e0442,teste2,dish_1558632198
e0443,teste2,dish_1558376938
e0444,teste2,dish_1562602657
e0445,teste2,dish_1561752996
e0446,teste2,dish_1564588719
e0447,teste2,dish_1562698132
e0448,teste2,dish_1563464457
e0449,teste2,dish_1565898489
e0450,teste2,dish_1562963033
e0451,teste2,dish_1562958998
e0452,teste2,dish_1558720133
e0453,teste2,dish_1558640267
e0454,teste2,dish_1567022180
e0455,teste2,dish_1565023157
e0456,teste2,dish_1566588016
e0457,teste2,dish_1568314951
e0458,teste2,dish_1564689168
e0459,teste2,dish_1562615128
e0460,teste2,dish_1565382856
e0461,teste2,dish_1563207406
e0462,teste2,dish_1558640849
e0463,teste2,dish_1565107066
e0464,teste2,dish_1563220087
e0465,teste2,dish_1558632233
e0466,teste2,dish_1565799663
e0467,teste2,dish_1562788536
e0468,teste2,dish_1564159588
e0469,teste2,dish_1563393182
e0470,teste2,dish_1560455158
e0471,teste2,dish_1561405864
e0472,teste2,dish_1568315040
e0473,teste2,dish_1563909550
e0474,teste2,dish_1565898723
e0475,teste2,dish_1563565289
e0476,teste2,dish_1566501293
e0477,teste2,dish_1565725009
e0478,teste2,dish_1560973958
e0479,teste2,dish_1565107092
e0480,teste2,dish_1563566909
e0481,teste2,dish_1563304333
e0482,teste2,dish_1562958721
e0483,teste2,dish_1561392593
e0484,teste2,dish_1568314907
e0485,teste2,dish_1561753203
e0486,teste2,dish_1559844657
e0487,teste2,dish_1566501153
e0488,teste2,dish_1566501352
e0489,teste2,dish_1562703408
e0490,teste2,dish_1558640373
e0491,teste2,dish_1567526845
e0492,teste2,dish_1561662869
e0493,teste2,dish_1558720396
e0494,teste2,dish_1567542876
e0495,teste2,dish_1566850142
e0496,teste2,dish_1565207662
e0497,teste2,dish_1565021220
e0498,teste2,dish_1559844490
e0499,teste2,dish_1562963246
e0500,teste2,dish_1565207865
e0501,teste2,dish_1566417516
e0502,teste2,dish_1561392386
e0503,teste2,dish_1565382995
e0504,teste2,dish_1565811091
e0505,teste2,dish_1565810969
e0506,teste2,dish_1563220058
e0507,teste2,dish_1557861795
e0508,teste2,dish_1572464400
e0509,teste2,dish_1568060646
e0510,teste2,dish_1561740411
e0511,teste2,dish_1568060746
e0512,teste2,dish_1562602918
e0513,teste2,dish_1561578373
e0514,teste2,dish_1567526785
e0515,teste2,dish_1558639907
e0516,teste2,dish_1561741090
e0517,teste2,dish_1562173247
e0518,teste2,dish_1562963153
e0519,teste2,dish_1558639979
e0520,teste2,dish_1563555543
e0521,teste2,dish_1559581236
e0522,teste2,dish_1573233324
e0523,teste2,dish_1566934455
e0524,teste2,dish_1562774475
e0525,teste2,dish_1562602803
e0526,teste2,dish_1565107211
e0527,teste2,dish_1561579505
e0528,teste2,dish_1560455000
e0529,teste2,dish_1558718906
e0530,teste2,dish_1567023335
e0531,teste2,dish_1562602627
e0532,teste2,dish_1563826272
e0533,teste2,dish_1563295292
e0534,teste2,dish_1563551220
e0535,teste2,dish_1572376778
e0536,teste2,dish_1564432191
e0537,teste2,dish_1558640781
e0538,teste2,dish_1558719145
e0539,teste2,dish_1565898590
e0540,teste2,dish_1561749818
e0541,teste2,dish_1561393649
e0542,teste2,dish_1565895808
e0543,teste2,dish_1565382803
e0544,teste2,dish_1561576026
e0545,teste2,dish_1563389600
e0546,teste2,dish_1567023262
e0547,teste2,dish_1563205903
e0548,teste2,dish_1563464480
e0549,teste2,dish_1560444985
e0550,teste2,dish_1565799597
e0551,teste2,dish_1561405912
e0552,teste2,dish_1562099053
e0553,teste2,dish_1567007814
e0554,teste2,dish_1561578226
e0555,teste2,dish_1560455263
e0556,teste2,dish_1562010369
e0557,teste2,dish_1563823292
e0558,teste2,dish_1558641320
e0559,teste2,dish_1560961059
e0560,teste2,dish_1562168365
e0561,teste2,dish_1558641442
e0562,teste2,dish_1558640074
e0563,teste2,dish_1563567372
e0564,teste2,dish_1562086564
e0565,teste2,dish_1566332040
e0566,teste2,dish_1565034661
e0567,teste2,dish_1563389413
e0568,teste2,dish_1558639683
e0569,teste2,dish_1565898331
e0570,teste2,dish_1568666329
e0571,teste2,dish_1562173200
e0572,teste2,dish_1562613340
e0573,teste2,dish_1560543525
e0574,teste2,dish_1558639787
e0575,teste2,dish_1565020796
e0576,teste2,dish_1558639482
e0577,teste2,dish_1562703382
e0578,teste2,dish_1566414686
e0579,teste2,dish_1561578800
e0580,teste2,dish_1565207896
e0581,teste2,dish_1558642420
e0582,teste2,dish_1561578570
e0583,teste2,dish_1560543682
e0584,teste2,dish_1565036953
e0585,teste2,dish_1567625294
e0586,teste2,dish_1560444967
e0587,teste2,dish_1561392430
e0588,teste2,dish_1563550950
e0589,teste2,dish_1561753623
e0590,teste2,dish_1559245801
e0591,teste2,dish_1560444892
e0592,teste2,dish_1559844611
e0593,teste2,dish_1568666184
e0594,teste2,dish_1564514906
e0595,teste2,dish_1561574368
e0596,teste2,dish_1560441529
e0597,teste2,dish_1566414924
e0598,teste2,dish_1564689138
e0599,teste2,dish_1558638896
e0600,teste2,dish_1568666263
e0601,teste2,dish_1562009810
e0602,teste2,dish_1558639555
e0603,teste2,dish_1561746912
e0604,teste2,dish_1563984242
e0605,teste2,dish_1566923004
e0606,teste2,dish_1565725071
e0607,teste2,dish_1561393619
e0608,teste2,dish_1558638844
e0609,teste2,dish_1561492665
e0610,teste2,dish_1561579706
e0611,teste2,dish_1566417475
e0612,teste2,dish_1562963325
e0613,teste2,dish_1561739265
e0614,teste2,dish_1562689601
e0615,teste2,dish_1557861697
e0616,teste2,dish_1563205927
e0617,teste2,dish_1560368764
e0618,teste2,dish_1566501447
e0619,teste2,dish_1560368506
e0620,teste2,dish_1562603598
e0621,teste2,dish_1563898688
e0622,teste2,dish_1565021640
e0623,teste2,dish_1562602945
e0624,teste2,dish_1568665935
e0625,teste2,dish_1562613403
e0626,teste2,dish_1563551105
e0627,teste2,dish_1559849466
e0628,teste2,dish_1561753275
e0629,teste2,dish_1561492687
e0630,teste2,dish_1564518488
e0631,teste2,dish_1560543704
e0632,teste2,dish_1566501382
e0633,teste2,dish_1567625434
e0634,teste2,dish_1563304261
e0635,teste2,dish_1564518512
e0636,teste2,dish_1562099134
e0637,teste2,dish_1558641164
e0638,teste2,dish_1560975517
e0639,teste2,dish_1565123816
e0640,teste2,dish_1561392512
e0641,teste2,dish_1558638983
e0642,teste2,dish_1563824092
e0643,teste2,dish_1565034635
e0644,teste2,dish_1558636582
e0645,teste2,dish_1561574456
e0646,teste2,dish_1561576051
e0647,teste2,dish_1561749857
e0648,teste2,dish_1562698239
e0649,teste2,dish_1558640593
e0650,teste2,dish_1563898084
e0651,teste2,dish_1566931482
e0652,teste2,dish_1564588688
e0653,teste2,dish_1560368712
e0654,teste2,dish_1561654346
e0655,teste2,dish_1558639012
e0656,teste2,dish_1565382733
e0657,teste2,dish_1563897837
e0658,teste2,dish_1562875273
e0659,teste2,dish_1560975628
e0660,teste2,dish_1566414754
e0661,teste2,dish_1558639662
e0662,teste2,dish_1563996509
e0663,teste2,dish_1562611878
e0664,teste2,dish_1558639412
e0665,teste2,dish_1562010048
e0666,teste2,dish_1560961179
e0667,teste2,dish_1561405934
e0668,teste2,dish_1565021714
e0669,teste2,dish_1563898813
e0670,teste2,dish_1561393764
e0671,teste2,dish_1567022338
e0672,teste2,dish_1562010418
e0673,teste2,dish_1571931482
e0674,teste2,dish_1560444914
e0675,teste2,dish_1562774531
e0676,teste2,dish_1565022062
e0677,teste2,dish_1564590288
e0678,teste2,dish_1563566674
e0679,teste2,dish_1562959799
e0680,teste2,dish_1558720335
e0681,teste2,dish_1567022147
e0682,teste2,dish_1561654326
e0683,teste2,dish_1561579845
e0684,teste2,dish_1564159636
e0685,teste2,dish_1562691000
e0686,teste2,dish_1560444948
e0687,teste2,dish_1566849895
e0688,teste2,dish_1563216440
e0689,teste2,dish_1565799507
e0690,teste2,dish_1558640563
e0691,teste2,dish_1561575908
e0692,teste2,dish_1563897870
e0693,teste2,dish_1560444592
e0694,teste2,dish_1563304369
e0695,teste2,dish_1560445004
e0696,teste2,dish_1558720289
e0697,teste2,dish_1561579576
e0698,teste2,dish_1565811061
e0699,teste2,dish_1559245589
e0700,teste2,dish_1558636514
e0701,teste2,dish_1563898019
e0702,teste2,dish_1566501473
e0703,teste2,dish_1562173267
e0704,teste2,dish_1558719097
e0705,teste2,dish_1565383128
e0706,teste2,dish_1568666303
e0707,teste2,dish_1558640190
e0708,teste2,dish_1562959570
e0709,teste2,dish_1568060414
e0710,teste2,dish_1561654392
e0711,teste2,dish_1561578156
e0712,teste2,dish_1564168695
e0713,teste2,dish_1562690112
e0714,teste2,dish_1571931763
e0715,teste2,dish_1561749778
e0716,teste2,dish_1565023128
e0717,teste2,dish_1563824065
e0718,teste2,dish_1565725047
e0719,teste2,dish_1560444785
e0720,teste2,dish_1562875211
e0721,teste2,dish_1566500963
e0722,teste2,dish_1574359160
e0723,teste2,dish_1565020926
e0724,teste2,dish_1560543625
e0725,teste2,dish_1565898659
e0726,teste2,dish_1558376734
e0727,teste2,dish_1574455444
e0728,teste2,dish_1572464354
e0729,teste2,dish_1566934430
e0730,teste2,dish_1562168800
e0731,teste2,dish_1561574428
e0732,teste2,dish_1558459664
e0733,teste2,dish_1560973936
e0734,teste2,dish_1566332268
e0735,teste2,dish_1561662926
e0736,teste2,dish_1565896701
e0737,teste2,dish_1562876756
e0738,teste2,dish_1563565508
e0739,teste2,dish_1562168648
e0740,teste2,dish_1563207301
e0741,teste2,dish_1560455380
e0742,teste2,dish_1563555563
e0743,teste2,dish_1560787754
e0744,teste2,dish_1565813386
e0745,teste2,dish_1563996625
e0746,teste2,dish_1562086501
e0747,teste2,dish_1558640477
e0748,teste2,dish_1566850031
e0749,teste2,dish_1567625392
e0750,teste2,dish_1565727921
e0751,teste2,dish_1562690144
e0752,teste2,dish_1562788506
e0753,teste2,dish_1558639167
e0754,teste2,dish_1561578623
e0755,teste2,dish_1559245920
e0756,teste2,dish_1559844323
e0757,teste2,dish_1562871537
e0758,teste2,dish_1563208412
e0759,teste2,dish_1561578775
e0760,teste2,dish_1565107143
e0761,teste2,dish_1559668272
e0762,teste2,dish_1558639306
e0763,teste2,dish_1561662948
e0764,teste2,dish_1565972429
e0765,teste2,dish_1562876674
e0766,teste2,dish_1566501324
e0767,teste2,dish_1560455090
e0768,teste2,dish_1563551194
e0769,teste2,dish_1563393244
e0770,teste2,dish_1560454970
e0771,teste2,dish_1562602687
e0772,teste2,dish_1562958629
e0773,teste2,dish_1567542986
e0774,teste2,dish_1560368815
e0775,teste2,dish_1560526530
e0776,teste2,dish_1566501575
e0777,teste2,dish_1566851584
e0778,teste2,dish_1566931674
e0779,teste2,dish_1559589902
e0780,teste2,dish_1566488928
e0781,teste2,dish_1563389626
e0782,teste2,dish_1566851616
e0783,teste2,dish_1563389742
e0784,teste2,dish_1563468362
e0785,teste2,dish_1567542953
e0786,teste2,dish_1561749750
e0787,teste2,dish_1572464284
e0788,teste2,dish_1561393741
e0789,teste2,dish_1565383159
e0790,teste2,dish_1562959517
e0791,teste2,dish_1571934494
e0792,teste2,dish_1559149615
e0793,teste2,dish_1558641358
e0794,teste2,dish_1561753238
e0795,teste2,dish_1565727821
e0796,teste2,dish_1566414780
e0797,teste2,dish_1563393217
e0798,teste2,dish_1558639949
e0799,teste2,dish_1558641037
e0800,teste2,dish_1567020366
e0801,teste2,dish_1560526447
e0802,teste2,dish_1565727765
e0803,teste2,dish_1558640882
e0804,teste2,dish_1560455399
e0805,teste2,dish_1564432165
e0806,teste2,dish_1561578865
e0807,teste2,dish_1565207727
e0808,teste2,dish_1566414891
e0809,teste2,dish_1559240382
e0810,teste2,dish_1562603626
e0811,teste2,dish_1561663035
e0812,teste2,dish_1559842409
e0813,teste2,dish_1561574479
e0814,teste2,dish_1563464627
e0815,teste2,dish_1563393403
e0816,teste2,dish_1558639630
e0817,teste2,dish_1558640808
e0818,teste2,dish_1567542914
e0819,teste2,dish_1560444931
e0820,teste2,dish_1559149650
e0821,teste2,dish_1565020867
e0822,teste2,dish_1565884766
e0823,teste2,dish_1573764183
e0824,teste2,dish_1561753075
e0825,teste2,dish_1563295400
e0826,teste2,dish_1560360055
e0827,teste2,dish_1562168673
e0828,teste2,dish_1561578676
e0829,teste2,dish_1563207277


In [ ]:
import subprocess, sys
cmd = [sys.executable, '-u', '/tmp/nutrelle_codigo/treino_gpu.py', '--saida', SAIDA, '--orcamento-h', str(ORCAMENTO_H)] + EXTRA
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for linha in p.stdout:
    print(linha, end='')
print('código de saída:', p.wait())   # não levanta erro: o que ficou pronto continua na pasta de saída

In [ ]:
import json, os
m = os.path.join(SAIDA, 'manifesto.json')
if os.path.exists(m):
    m = json.load(open(m))
    for nome, c in m['configs'].items():
        print(f"{c['estado']:14s} {nome:46s} {c['cv_treino'] or ''}")
        print(' ' * 15 + 'serviço:', c.get('servico', {}).get('estado'))
    print('melhor modelo:', m['melhor_modelo'])
    print('tamanho da saída:', m.get('tamanho_saida'))
print(sorted(os.listdir(SAIDA)))